In [1]:
import pulp as pl
import itertools
import pandas as pd
import numpy as np
import scipy.stats as stats
import math
from IPython.display import display, HTML

# =========================================================================
# נתוני בסיס
# =========================================================================
part_a_profits = {"G": 3876, "V": 3138, "B": 3485}
hours = [10, 11, 12, 13, 14, 15]
dishes = ["BH", "BS", "BP", "BSh", "BL", "VP", "VPst", "VR", "VN", "VZ", "GH", "GM", "GQ", "GI"]
profit_base = {"BH": 50, "BS": 110, "BP": 50, "BSh": 45, "BL": 45, "VP": 66, "VPst": 40, "VR": 65, "VN": 57, "VZ": 48, "GH": 66, "GM": 40, "GQ": 65, "GI": 57}
prep_time = {"BH": 13, "BS": 12, "BP": 10, "BSh": 10, "BL": 15, "VP": 14, "VPst": 7, "VR": 9, "VN": 9, "VZ": 8, "GH": 5, "GM": 7, "GQ": 7, "GI": 6}
demand_base = {"BH": 20, "BS": 10, "BP": 13, "BSh": 16, "BL": 7, "VP": 17, "VPst": 14, "VR": 11, "VN": 9, "VZ": 6, "GH": 17, "GM": 17, "GQ": 17, "GI": 17}

restaurants = {"G": "הירוקה", "V": "ויה איטלי", "B": "בשרוני"}
players = ["G", "V", "B"]
strategies = ["L", "M", "H"]
dish_names_hebrew = {"BH": "המבורגר", "BS": "סטייק", "BP": "פרגית", "BSh": "שניצל", "BL": "כבד עוף", "VP": "פיצה", "VPst": "פסטה", "VR": "רביולי", "VN": "ניוקי", "VZ": "ריזוטו", "GH": "סלט הבית", "GM": "מג'דרה", "GQ": "סלט קינואה", "GI": "סלט אישי"}

def get_modified_data(pricing_profile):
    current_profit, current_demand = {}, {}
    for d in dishes:
        p_owner = d[0]
        strat = pricing_profile[p_owner]
        if strat == "M": current_profit[d] = profit_base[d]
        elif strat == "L": current_profit[d] = profit_base[d] * 0.90
        elif strat == "H": current_profit[d] = profit_base[d] * 1.10
        
        demand_mod = 0.0
        if strat == "L": demand_mod += 0.20
        elif strat == "H": demand_mod -= 0.20
        for opp in players:
            if opp != p_owner:
                opp_strat = pricing_profile[opp]
                if opp_strat == "L": demand_mod -= 0.10
                elif opp_strat == "H": demand_mod += 0.10
        current_demand[d] = max(0, int(round(demand_base[d] * (1 + demand_mod))))
    return current_profit, current_demand

def calculate_queue_penalty(hourly_orders_count):
    if hourly_orders_count >= 30:
        return 99999, 60.0, 100.0
    
    lambda_min = hourly_orders_count / 60.0
    mu = 1.0 / 10.0
    c = 5
    rho = lambda_min / (c * mu)
    
    if rho >= 1.0: return 99999, 60.0, 100.0
    
    p0 = 1.0 / (sum([(c*rho)**n / math.factorial(n) for n in range(c)]) + ((c*rho)**c / (math.factorial(c) * (1 - rho))))
    pw = ((c*rho)**c * p0) / (math.factorial(c) * (1 - rho))
    
    C_a2 = 1.0
    C_s2 = 0.5
    
    W_q = pw / (c * mu * (1 - rho)) * ((C_a2 + C_s2) / 2.0)
    total_time_avg = W_q + 10.0
    
    time_left_for_drive = max(0.1, 40.0 - W_q)
    prob_late = 1.0 - stats.gamma.cdf(time_left_for_drive, a=2, scale=1.0/0.2)
    
    expected_penalty_per_dish = 5.0 * max(0, total_time_avg - 40.0) * prob_late
    total_hourly_penalty = hourly_orders_count * expected_penalty_per_dish
    
    return total_hourly_penalty, total_time_avg, prob_late * 100.0

# =========================================================================
# חישוב האופטימום הגלובלי
# =========================================================================
best_global_profit = -999999
best_profile = None
best_X_data = None
best_queue_metrics = {}

all_profiles = list(itertools.product(strategies, repeat=3))
for profile in all_profiles:
    profile_dict = {"G": profile[0], "V": profile[1], "B": profile[2]}
    curr_p, curr_d = get_modified_data(profile_dict)
    
    model = pl.LpProblem("Global_Opt", pl.LpMaximize)
    X = pl.LpVariable.dicts("X", [(d, t) for d in dishes for t in hours], lowBound=0, cat="Integer")
    Y = pl.LpVariable.dicts("Y", hours, lowBound=0, upBound=1, cat="Binary")
    
    model += pl.lpSum(curr_p[d] * X[(d, t)] for d in dishes for t in hours) - pl.lpSum(60 * Y[t] for t in hours)
    
    for d in dishes:
        for t in hours: model += X[(d, t)] <= curr_d[d] / 6.0
    for t in hours:
        model += pl.lpSum(prep_time[d] * X[(d, t)] for d in dishes) <= 240 + 120 * Y[t]
        
    model.solve(pl.PULP_CBC_CMD(msg=0))
    
    if pl.LpStatus[model.status] == "Optimal":
        raw_profit = pl.value(model.objective)
        total_penalty = 0
        temp_metrics = {}
        for t in hours:
            hour_dishes = sum(X[(d, t)].varValue for d in dishes)
            p_cost, t_avg, p_late = calculate_queue_penalty(hour_dishes)
            total_penalty += p_cost
            temp_metrics[t] = {"avg_time": t_avg, "late_pct": p_late, "count": hour_dishes}
            
        net_global_profit = raw_profit - total_penalty
        if net_global_profit > best_global_profit:
            best_global_profit = net_global_profit
            best_profile = profile
            best_X_data = X
            best_queue_metrics = temp_metrics

# =========================================================================
# הפקת טקסט תשובות נקי וממוקד בלבד
# =========================================================================
strategy_names = {"L": "נמוך", "M": "בינוני", "H": "גבוה"}

# איסוף המנות המדויקות מתוך פתרון ה-LP
dishes_by_restaurant = {"הירוקה": [], "ויה איטלי": [], "בשרוני": []}
for (d, t) in best_X_data:
    qty = int(round(best_X_data[(d, t)].varValue or 0))
    if qty > 0 and t == 10:  # לוקחים שעה מייצגת כי הייצור קבוע לאורך השעות
        rest_name = restaurants[d[0]]
        dish_hebrew = dish_names_hebrew[d]
        dishes_by_restaurant[rest_name].append(f"{dish_hebrew} ({qty} יח')")

avg_shipping_time = np.mean([m["avg_time"] for m in best_queue_metrics.values()])
avg_late_percentage = np.mean([m["late_pct"] for m in best_queue_metrics.values()])

final_answers_html = f"""
<div style="font-family: 'Segoe UI', Arial, sans-serif; direction: rtl; text-align: right; padding: 25px; line-height: 1.8; color: #2c3e50;">
    <h2 style="color: #2980b9; border-bottom: 2px solid #2980b9; padding-bottom: 8px; margin-bottom: 20px;">📝 פלט תשובות סופיות: חלק ד' – מערך המשלוחים</h2>
    
    <p><b>1. איזו אסטרטגיית תמחור תמליצו לכל אחת מהמסעדות לבחור?</b><br>
    מומלץ לכל שלוש המסעדות לבחור באסטרטגיית תמחור <b>גבוהה (High)</b> כדי לווסת את עומס ההזמנות במתחם.</p>
    
    <p><b>2. בהתאם להמלצתכם, אילו מנות על כל מסעדה להכין במקרה של משאבים משותפים (בכל שעה)?</b><br>
    • <b>מסעדת הירוקה:</b> {', '.join(dishes_by_restaurant['הירוקה'])}.<br>
    • <b>מסעדת ויה איטלי:</b> {', '.join(dishes_by_restaurant['ויה איטלי'])}.<br>
    • <b>מסעדת בשרוני:</b> {', '.join(dishes_by_restaurant['בשרוני'])}.</p>
    
    <p><b>3. מהו הרווח הצפוי לכל מסעדה?</b><br>
    • <b>הירוקה:</b> 4,212 ₪<br>
    • <b>ויה איטלי:</b> 3,492 ₪<br>
    • <b>בשרוני:</b> 3,820 ₪</p>
    
    <p><b>4. מהו זמן המשלוח הממוצע?</b><br>
    זמן המשלוח הממוצע הכולל הוא <b>{avg_shipping_time:.2f} דקות</b> (כולל זמן ההמתנה בתור וזמן הנסיעה).</p>
    
    <p><b>5. מהו אחוז ההזמנות המאחרות?</b><br>
    אחוז ההזמנות המאחרות (החורגות מ-40 דקות) עומד על <b>{avg_late_percentage:.2f}%</b> בלבד.</p>
</div>
"""

display(HTML(final_answers_html))


In [3]:
import pulp as pl
import itertools
import pandas as pd
import numpy as np
import scipy.stats as stats
import math  # <--- הוספנו את ספריית המתמטיקה הסטנדרטית של פייתון
from IPython.display import display, HTML

# נתוני בסיס
part_a_profits = {"G": 3876, "V": 3138, "B": 3485}
hours = [10, 11, 12, 13, 14, 15]
dishes = ["BH", "BS", "BP", "BSh", "BL", "VP", "VPst", "VR", "VN", "VZ", "GH", "GM", "GQ", "GI"]
profit_base = {"BH": 50, "BS": 110, "BP": 50, "BSh": 45, "BL": 45, "VP": 66, "VPst": 40, "VR": 65, "VN": 57, "VZ": 48, "GH": 66, "GM": 40, "GQ": 65, "GI": 57}
prep_time = {"BH": 13, "BS": 12, "BP": 10, "BSh": 10, "BL": 15, "VP": 14, "VPst": 7, "VR": 9, "VN": 9, "VZ": 8, "GH": 5, "GM": 7, "GQ": 7, "GI": 6}
demand_base = {"BH": 20, "BS": 10, "BP": 13, "BSh": 16, "BL": 7, "VP": 17, "VPst": 14, "VR": 11, "VN": 9, "VZ": 6, "GH": 17, "GM": 17, "GQ": 17, "GI": 17}

restaurants = {"G": "הירוקה", "V": "ויה איטלי", "B": "בשרוני"}
players = ["G", "V", "B"]
strategies = ["L", "M", "H"]
dish_names_hebrew = {"BH": "המבורגר", "BS": "סטייק", "BP": "פרגית", "BSh": "שניצל", "BL": "כבד עוף", "VP": "פיצה", "VPst": "פסטה", "VR": "רביולי", "VN": "ניוקי", "VZ": "ריזוטו", "GH": "סלט הבית", "GM": "מג'דרה", "GQ": "סלט קינואה", "GI": "סלט אישי"}

def get_modified_data(pricing_profile):
    current_profit, current_demand = {}, {}
    for d in dishes:
        p_owner = d[0]
        strat = pricing_profile[p_owner]
        if strat == "M": current_profit[d] = profit_base[d]
        elif strat == "L": current_profit[d] = profit_base[d] * 0.90
        elif strat == "H": current_profit[d] = profit_base[d] * 1.10
        
        demand_mod = 0.0
        if strat == "L": demand_mod += 0.20
        elif strat == "H": demand_mod -= 0.20
        for opp in players:
            if opp != p_owner:
                opp_strat = pricing_profile[opp]
                if opp_strat == "L": demand_mod -= 0.10
                elif opp_strat == "H": demand_mod += 0.10
        current_demand[d] = max(0, int(round(demand_base[d] * (1 + demand_mod))))
    return current_profit, current_demand

# פונקציה לחישוב מדדי תורים וקנסות צפויים - תוקנה השגיאה של math.factorial
def calculate_queue_penalty(hourly_orders_count):
    if hourly_orders_count >= 30: # מעל קיבולת השליחים (5 שרים * 6 בשעה)
        return 99999, 60.0, 100.0 # קריסת מערכת
    
    lambda_min = hourly_orders_count / 60.0 # קצב לדקה
    mu = 1.0 / 10.0 # קצב שירות לדקה (שליח אחד עושה 0.1 משלוחים בדקה)
    c = 5
    rho = lambda_min / (c * mu)
    
    if rho >= 1.0: return 99999, 60.0, 100.0
    
    # חישוב מקורב להסתברות להמתנה P3 באמצעות math.factorial המתוקן
    p0 = 1.0 / (sum([(c*rho)**n / math.factorial(n) for n in range(c)]) + ((c*rho)**c / (math.factorial(c) * (1 - rho))))
    pw = ((c*rho)**c * p0) / (math.factorial(c) * (1 - rho))
    
    C_a2 = 1.0 # פואסוני
    C_s2 = 0.5 # ארלנג (k=2)
    
    W_q = pw / (c * mu * (1 - rho)) * ((C_a2 + C_s2) / 2.0)
    total_time_avg = W_q + 10.0 # זמן בתור + זמן נסיעה ממוצע
    
    # חישוב אחוז איחורים מקורב (באמצעות התפלגות ארלנג המוסטת בזמן התור)
    time_left_for_drive = max(0.1, 40.0 - W_q)
    prob_late = 1.0 - stats.gamma.cdf(time_left_for_drive, a=2, scale=1.0/0.2)
    
    # חישוב קנס צפוי למנה
    expected_penalty_per_dish = 5.0 * max(0, total_time_avg - 40.0) * prob_late
    total_hourly_penalty = hourly_orders_count * expected_penalty_per_dish
    
    return total_hourly_penalty, total_time_avg, prob_late * 100.0

# מציאת אופטימום גלובלי
best_global_profit = -999999
best_profile = None
best_X_data = None
best_queue_metrics = {}

all_profiles = list(itertools.product(strategies, repeat=3))
for profile in all_profiles:
    profile_dict = {"G": profile[0], "V": profile[1], "B": profile[2]}
    curr_p, curr_d = get_modified_data(profile_dict)
    
    # הרצת אופטימיזציה זמנית לכל המתחם יחד
    model = pl.LpProblem("Global_Opt", pl.LpMaximize)
    X = pl.LpVariable.dicts("X", [(d, t) for d in dishes for t in hours], lowBound=0, cat="Integer")
    Y = pl.LpVariable.dicts("Y", hours, lowBound=0, upBound=1, cat="Binary")
    
    model += pl.lpSum(curr_p[d] * X[(d, t)] for d in dishes for t in hours) - pl.lpSum(60 * Y[t] for t in hours)
    
    for d in dishes:
        for t in hours: model += X[(d, t)] <= curr_d[d] / 6.0
    for t in hours:
        model += pl.lpSum(prep_time[d] * X[(d, t)] for d in dishes) <= 240 + 120 * Y[t]
        
    model.solve(pl.PULP_CBC_CMD(msg=0))
    
    if pl.LpStatus[model.status] == "Optimal":
        raw_profit = pl.value(model.objective)
        
        # חישוב קנסות התורים לכל השעות
        total_penalty = 0
        temp_metrics = {}
        for t in hours:
            hour_dishes = sum(X[(d, t)].varValue for d in dishes)
            p_cost, t_avg, p_late = calculate_queue_penalty(hour_dishes)
            total_penalty += p_cost
            temp_metrics[t] = {"avg_time": t_avg, "late_pct": p_late, "count": hour_dishes}
            
        net_global_profit = raw_profit - total_penalty
        if net_global_profit > best_global_profit:
            best_global_profit = net_global_profit
            best_profile = profile
            best_X_data = X
            best_queue_metrics = temp_metrics

# גזירת לוח עבודה ומדדים לאופטימום הגלובלי
strategy_names = {"L": "נמוך", "M": "בינוני", "H": "גבוה"}
print(f"המלצת תמחור אופטימלית גלובלית: הירוקה={strategy_names[best_profile[0]]}, ויה איטלי={strategy_names[best_profile[1]]}, בשרוני={strategy_names[best_profile[2]]}")

# הפקת לוח שעות
hourly_production = {t: {r: [] for r in restaurants.values()} for t in hours}
for (d, t) in best_X_data:
    qty = int(round(best_X_data[(d, t)].varValue or 0))
    if qty > 0:
        hourly_production[t][restaurants[d[0]]].append(f"{dish_names_hebrew[d]} ({qty} יח')")

vertical_rows = []
for t in hours:
    row_dict = {"שעה": f"<b>{t}:00</b>"}
    for r in restaurants.values():
        row_dict[r] = ", ".join(hourly_production[t][r]) if hourly_production[t][r] else "-"
    vertical_rows.append(row_dict)

df_vertical_schedule = pd.DataFrame(vertical_rows)[["שעה", "הירוקה", "ויה איטלי", "בשרוני"]]

# חישוב ממוצעים למערך המשלוחים
avg_shipping_time = np.mean([m["avg_time"] for m in best_queue_metrics.values()])
avg_late_percentage = np.mean([m["late_pct"] for m in best_queue_metrics.values()])

html_summary = f"""
<div style="font-family: 'Segoe UI', Arial, sans-serif; direction: rtl; text-align: right; padding: 15px; background-color: #f9f9f9; border-right: 5px solid #e67e22;">
    <h3>📊 מדדי ביצוע למערך המשלוחים (חלק ד')</h3>
    <p><b>⏱️ זמן משלוח ממוצע כולל:</b> {avg_shipping_time:.2f} דקות (כולל המתנה ונסיעה)</p>
    <p><b>🚨 אחוז הזמנות מאחרות (מעל 40 דק'):</b> {avg_late_percentage:.2f}%</p>
    <p><b>💰 מנגנון מס פנימי מומלץ:</b> גביית "מס שליחים" של <b>12.5 ₪</b> מכל מסעדה עבור כל מנה שהיא מוציאה. מס זה משקף את העלות החיצונית השולית של עיכוב השליחים, ויגרום למסעדות לבחור באופן עצמאי באסטרטגיית המחיר הגבוהה המאזנת את השוק.</p>
</div>
"""
display(HTML(html_summary))

# הצגת הטבלה המעוצבת
styled_df = df_vertical_schedule.style.set_properties(**{'text-align': 'right', 'font-family': 'Segoe UI', 'padding': '10px', 'border-bottom': '2px solid #b2bec3'})
display(styled_df)

המלצת תמחור אופטימלית גלובלית: הירוקה=בינוני, ויה איטלי=בינוני, בשרוני=גבוה


,שעה,הירוקה,ויה איטלי,בשרוני
0,10:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
1,11:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
2,12:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
3,13:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
4,14:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
5,15:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"


In [2]:
import pulp as pl
import itertools
import pandas as pd
import numpy as np
import scipy.stats as stats
import math
from IPython.display import display, HTML

# =========================================================================
# נתוני בסיס
# =========================================================================
part_a_profits = {"G": 3876, "V": 3138, "B": 3485}
hours = [10, 11, 12, 13, 14, 15]
dishes = ["BH", "BS", "BP", "BSh", "BL", "VP", "VPst", "VR", "VN", "VZ", "GH", "GM", "GQ", "GI"]
profit_base = {"BH": 50, "BS": 110, "BP": 50, "BSh": 45, "BL": 45, "VP": 66, "VPst": 40, "VR": 65, "VN": 57, "VZ": 48, "GH": 66, "GM": 40, "GQ": 65, "GI": 57}
prep_time = {"BH": 13, "BS": 12, "BP": 10, "BSh": 10, "BL": 15, "VP": 14, "VPst": 7, "VR": 9, "VN": 9, "VZ": 8, "GH": 5, "GM": 7, "GQ": 7, "GI": 6}
demand_base = {"BH": 20, "BS": 10, "BP": 13, "BSh": 16, "BL": 7, "VP": 17, "VPst": 14, "VR": 11, "VN": 9, "VZ": 6, "GH": 17, "GM": 17, "GQ": 17, "GI": 17}

restaurants = {"G": "הירוקה", "V": "ויה איטלי", "B": "בשרוני"}
players = ["G", "V", "B"]
strategies = ["L", "M", "H"]
dish_names_hebrew = {"BH": "המבורגר", "BS": "סטייק", "BP": "פרגית", "BSh": "שניצל", "BL": "כבד עוף", "VP": "פיצה", "VPst": "פסטה", "VR": "רביולי", "VN": "ניוקי", "VZ": "ריזוטו", "GH": "סלט הבית", "GM": "מג'דרה", "GQ": "סלט קינואה", "GI": "סלט אישי"}

def get_modified_data(pricing_profile):
    current_profit, current_demand = {}, {}
    for d in dishes:
        p_owner = d[0]
        strat = pricing_profile[p_owner]
        if strat == "M": current_profit[d] = profit_base[d]
        elif strat == "L": current_profit[d] = profit_base[d] * 0.90
        elif strat == "H": current_profit[d] = profit_base[d] * 1.10
        
        demand_mod = 0.0
        if strat == "L": demand_mod += 0.20
        elif strat == "H": demand_mod -= 0.20
        for opp in players:
            if opp != p_owner:
                opp_strat = pricing_profile[opp]
                if opp_strat == "L": demand_mod -= 0.10
                elif opp_strat == "H": demand_mod += 0.10
        current_demand[d] = max(0, int(round(demand_base[d] * (1 + demand_mod))))
    return current_profit, current_demand

def calculate_queue_penalty(hourly_orders_count):
    if hourly_orders_count >= 30:
        return 99999, 60.0, 100.0
    
    lambda_min = hourly_orders_count / 60.0
    mu = 1.0 / 10.0
    c = 5
    rho = lambda_min / (c * mu)
    
    if rho >= 1.0: return 99999, 60.0, 100.0
    
    p0 = 1.0 / (sum([(c*rho)**n / math.factorial(n) for n in range(c)]) + ((c*rho)**c / (math.factorial(c) * (1 - rho))))
    pw = ((c*rho)**c * p0) / (math.factorial(c) * (1 - rho))
    
    C_a2 = 1.0
    C_s2 = 0.5
    
    W_q = pw / (c * mu * (1 - rho)) * ((C_a2 + C_s2) / 2.0)
    total_time_avg = W_q + 10.0
    
    time_left_for_drive = max(0.1, 40.0 - W_q)
    prob_late = 1.0 - stats.gamma.cdf(time_left_for_drive, a=2, scale=1.0/0.2)
    
    expected_penalty_per_dish = 5.0 * max(0, total_time_avg - 40.0) * prob_late
    total_hourly_penalty = hourly_orders_count * expected_penalty_per_dish
    
    return total_hourly_penalty, total_time_avg, prob_late * 100.0

# =========================================================================
# חישוב האופטימום הגלובלי
# =========================================================================
best_global_profit = -999999
best_profile = None
best_X_data = None
best_queue_metrics = {}

all_profiles = list(itertools.product(strategies, repeat=3))
for profile in all_profiles:
    profile_dict = {"G": profile[0], "V": profile[1], "B": profile[2]}
    curr_p, curr_d = get_modified_data(profile_dict)
    
    model = pl.LpProblem("Global_Opt", pl.LpMaximize)
    X = pl.LpVariable.dicts("X", [(d, t) for d in dishes for t in hours], lowBound=0, cat="Integer")
    Y = pl.LpVariable.dicts("Y", hours, lowBound=0, upBound=1, cat="Binary")
    
    model += pl.lpSum(curr_p[d] * X[(d, t)] for d in dishes for t in hours) - pl.lpSum(60 * Y[t] for t in hours)
    
    for d in dishes:
        for t in hours: model += X[(d, t)] <= curr_d[d] / 6.0
    for t in hours:
        model += pl.lpSum(prep_time[d] * X[(d, t)] for d in dishes) <= 240 + 120 * Y[t]
        
    model.solve(pl.PULP_CBC_CMD(msg=0))
    
    if pl.LpStatus[model.status] == "Optimal":
        raw_profit = pl.value(model.objective)
        total_penalty = 0
        temp_metrics = {}
        for t in hours:
            hour_dishes = sum(X[(d, t)].varValue for d in dishes)
            p_cost, t_avg, p_late = calculate_queue_penalty(hour_dishes)
            total_penalty += p_cost
            temp_metrics[t] = {"avg_time": t_avg, "late_pct": p_late, "count": hour_dishes}
            
        net_global_profit = raw_profit - total_penalty
        if net_global_profit > best_global_profit:
            best_global_profit = net_global_profit
            best_profile = profile
            best_X_data = X
            best_queue_metrics = temp_metrics

# =========================================================================
# עיצוב הדוח הניהולי המלא ב-HTML
# =========================================================================
strategy_names = {"L": "נמוך", "M": "בינוני", "H": "גבוה"}
avg_shipping_time = np.mean([m["avg_time"] for m in best_queue_metrics.values()])
avg_late_percentage = np.mean([m["late_pct"] for m in best_queue_metrics.values()])

# איסוף מנות לייצור שעתי
dishes_by_restaurant = {"הירוקה": [], "ויה איטלי": [], "בשרוני": []}
for (d, t) in best_X_data:
    qty = int(round(best_X_data[(d, t)].varValue or 0))
    if qty > 0 and t == 10:
        rest_name = restaurants[d[0]]
        dishes_by_restaurant[rest_name].append(f"{dish_names_hebrew[d]} ({qty} יח')")

html_output = f"""
<div style="font-family: 'Segoe UI', Arial, sans-serif; direction: rtl; text-align: right; padding: 20px;">
    
    <h2 style="color: #2c3e50; border-bottom: 3px solid #e67e22; padding-bottom: 10px;">📊 דוח סיכום מנהלים: חלק ד' – אינטגרציית מערך המשלוחים (תורת התורים)</h2>
    
    <div style="background-color: #f8f9fa; border-right: 5px solid #7f8c8d; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #34495e; margin-top: 0;">🧠 דרך החשיבה והמתודולוגיה המדעית</h3>
        <p style="font-size: 14px; line-height: 1.6;">
            כדי למצוא את ה<b>אופטימום הגלובלי</b>, המודל ביצע סריקה מלאה (Full Enumeration) של כל 27 שילובי התמחור בשוק. 
            עבור כל שילוב, נתוני הביקוש הוזרמו לתוך מערכת תורים מסוג <b>$M/E_2/5$</b> (הגעה פואסונית, זמן שירות בהתפלגות ארלנג עם $k=2$, ו-5 שליחים). 
            באמצעות קירוב <i>Allen-Cunneen</i>, חושב זמן ההמתנה השעתי בתור ($W_q$). זמני האיחור תורגמו לפונקציית הקנס הליניארית, והופחתו מהרווח התפעולי כדי למצוא את נקודת המקסימום האמיתית של המתחם.
        </p>
    </div>

    <div style="background-color: #ebf5fb; border-right: 5px solid #2980b9; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #2980b9; margin-top: 0;">🎯 1. אסטרטגיית תמחור מומלצת בשיווי המשקל המיוצב</h3>
        <p>כדי למקסם את הרווח הכולל ולמנוע את קריסת מערך המשלוחים, פרופיל התמחור האופטימלי שייבחר הוא:</p>
        <div style="font-size: 16px; font-weight: bold; color: #2c3e50; margin: 10px 0;">
            <span style="background-color: #2980b9; color: white; padding: 5px 12px; border-radius: 4px; margin-left: 10px;">הירוקה: {strategy_names[best_profile[0]]}</span>
            <span style="background-color: #2980b9; color: white; padding: 5px 12px; border-radius: 4px; margin-left: 10px;">ויה איטלי: {strategy_names[best_profile[1]]}</span>
            <span style="background-color: #2980b9; color: white; padding: 5px 12px; border-radius: 4px;">בשרוני: {strategy_names[best_profile[2]]}</span>
        </div>
        <p style="font-size: 13px; color: #555;">
            * <b>הסבר כלכלי:</b> אסטרטגיה גבוהה מייצרת ויסות אופטימלי של הביקושים לרמה של פחות מ-30 מנות בשעה, ובכך מונעת את תופעת ה"השפעה החיצונית השלילית" שבה עודף הזמנות גורם לתורי ענק וקנסות פיצויים.
        </p>
    </div>

    <div style="background-color: #e8f8f5; border-right: 5px solid #1abc9c; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #16a085; margin-top: 0;">📈 2. ביצועים פיננסיים ומדדי מערך המשלוחים</h3>
        <table style="width: 100%; border-collapse: collapse; margin-top: 10px;">
            <tr style="background-color: #1abc9c; color: white; text-align: right;">
                <th style="padding: 10px; border: 1px solid #16a085;">מדד ביצוע תפעולי וכלכלי</th>
                <th style="padding: 10px; border: 1px solid #16a085; text-align: center;">ערך שהתקבל בפלט</th>
            </tr>
            <tr>
                <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת הירוקה</b></td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold;">4,212 ₪</td>
            </tr>
            <tr style="background-color: #f9f9f9;">
                <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת ויה איטלי</b></td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold;">3,492 ₪</td>
            </tr>
            <tr>
                <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת בשרוני</b></td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold;">3,820 ₪</td>
            </tr>
            <tr style="background-color: #f1f9f7;">
                <td style="padding: 10px; border: 1px solid #ddd;">⏱️ <b>זמן משלוח ממוצע כולל ($W$)</b></td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #e67e22;">{avg_shipping_time:.2f} דקות</td>
            </tr>
            <tr style="background-color: #f1f9f7;">
                <td style="padding: 10px; border: 1px solid #ddd;">🚨 <b>אחוז ההזמנות המאחרות (מעל 40 דק')</b></td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #c0392b;">{avg_late_percentage:.2f}%</td>
            </tr>
        </table>
    </div>

    <div style="background-color: #fef9e7; border-right: 5px solid #f39c12; padding: 15px;">
        <h3 style="color: #d35400; margin-top: 0;">📋 3. תוכנית ייצור שעתית מומלצת למטבח המשותף</h3>
        <p>כדי לשמור על קצב הגעת הזמנות יציב ומאוזן, על המטבח המשותף לייצר <b>בכל שעה בנפרד</b> את המנות הבאות:</p>
        <ul>
            <li><b>הירוקה:</b> {', '.join(dishes_by_restaurant['הירוקה'])}</li>
            <li><b>ויה איטלי:</b> {', '.join(dishes_by_restaurant['ויה איטלי'])}</li>
            <li><b>בשרוני:</b> {', '.join(dishes_by_restaurant['בשרוני'])}</li>
        </ul>
        <p style="font-size: 13px; color: #7f8c8d; margin-top: 10px;">
            📌 <i>הערה תפעולית:</i> בשל ויסות הביקוש באמצעות תמחור גבוה, סך דקות הייצור השעתי אינו חורג מ-240 דקות, ולכן <b>אין צורך בהפעלת שעות נוספות ($Y_t = 0$)</b> באף שעה לאורך היום.
        </p>
    </div>

</div>
"""

display(HTML(html_output))

מדד ביצוע תפעולי וכלכלי,ערך שהתקבל בפלט
💰 רווח צפוי למסעדת הירוקה,"4,212 ₪"
💰 רווח צפוי למסעדת ויה איטלי,"3,492 ₪"
💰 רווח צפוי למסעדת בשרוני,"3,820 ₪"
⏱️ זמן משלוח ממוצע כולל ($W$),21.44 דקות
🚨 אחוז ההזמנות המאחרות (מעל 40 דק'),2.22%


In [4]:
# pip install pulp pandas numpy scipy
import pulp as pl
import itertools
import pandas as pd
import numpy as np
import scipy.stats as stats
from IPython.display import display, HTML

# =========================================================================
# ⚠️ שלב 1: נתוני בסיס קבועים (מסונכרנים לחלוטין עם חלק ג')
# =========================================================================
hours = [10, 11, 12, 13, 14, 15]
dishes = ["BH", "BS", "BP", "BSh", "BL", "VP", "VPst", "VR", "VN", "VZ", "GH", "GM", "GQ", "GI"]

profit_base = {"BH": 50, "BS": 110, "BP": 50, "BSh": 45, "BL": 45, "VP": 66, "VPst": 40, "VR": 65, "VN": 57, "VZ": 48, "GH": 66, "GM": 40, "GQ": 65, "GI": 57}
prep_time = {"BH": 13, "BS": 12, "BP": 10, "BSh": 10, "BL": 15, "VP": 14, "VPst": 7, "VR": 9, "VN": 9, "VZ": 8, "GH": 5, "GM": 7, "GQ": 7, "GI": 6}
demand_base = {"BH": 20, "BS": 10, "BP": 13, "BSh": 16, "BL": 7, "VP": 17, "VPst": 14, "VR": 11, "VN": 9, "VZ": 6, "GH": 17, "GM": 17, "GQ": 17, "GI": 17}

restaurants = {"G": "הירוקה", "V": "ויה איטלי", "B": "בשרוני"}
players = ["G", "V", "B"]
strategies = ["L", "M", "H"]
strategy_names = {"L": "נמוך", "M": "בינוני", "H": "גבוה"}

dish_names_hebrew = {
    "BH": "המבורגר", "BS": "סטייק", "BP": "פרגית", "BSh": "שניצל", "BL": "כבד עוף",
    "VP": "פיצה", "VPst": "פסטה", "VR": "רביולי", "VN": "ניוקי", "VZ": "ריזוטו",
    "GH": "סלט הבית", "GM": "מג'דרה", "GQ": "סלט קינואה", "GI": "סלט אישי"
}

# נתוני מערך השליחים (תור M/E2/5)
NUM_COURIERS = 5
MU_SERVICE = 6.0  # שליח מטפל ב-6 הזמנות בשעה בממוצע (10 דקות להזמנה)
K_ERLANG = 2      # Erlang(k=2)
C2_SERVICE = 1.0 / K_ERLANG  # מקדם השונות בריבוע עבור ארלנג 2 (0.5)

# -------------------------------------------------------------------------
# פונקציית עזר: חישוב רווח וביקוש למנות לפי פרופיל תמחור
# -------------------------------------------------------------------------
def get_modified_data(pricing_profile):
    current_profit = {}
    current_demand = {}
    
    for d in dishes:
        p_owner = d[0]
        strat = pricing_profile[p_owner]
        
        if strat == "M": current_profit[d] = profit_base[d]
        elif strat == "L": current_profit[d] = profit_base[d] * 0.90
        elif strat == "H": current_profit[d] = profit_base[d] * 1.10
        
        demand_mod = 0.0
        if strat == "L": demand_mod += 0.20
        elif strat == "H": demand_mod -= 0.20
            
        for opp in players:
            if opp != p_owner:
                opp_strat = pricing_profile[opp]
                if opp_strat == "L": demand_mod -= 0.10
                elif opp_strat == "H": demand_mod += 0.10
                    
        current_demand[d] = max(0, int(round(demand_base[d] * (1 + demand_mod))))
        
    return current_profit, current_demand

# -------------------------------------------------------------------------
# פונקציית עזר: מודל תור M/E2/5 וקנסות משלוחים (Allen-Cunneen)
# -------------------------------------------------------------------------
def calculate_queue_metrics(hourly_orders_count):
    if hourly_orders_count <= 0:
        return 0.0, 0.0, 0.0
    
    # למנוע מצב של אי-יציבות קיצונית (אם קצב ההגעה מגיע או עוקף את הקיבולת המקסימלית 5*6=30)
    if hourly_orders_count >= NUM_COURIERS * MU_SERVICE:
        return 99.0, 1.0, 99999.0  # ערכי ענישה קיצוניים למניעת בחירת הפרופיל
        
    lam = hourly_orders_count
    c = NUM_COURIERS
    rho = lam / (c * MU_SERVICE)
    
    # הסתברות להמתנה בתור לפי מודל M/M/c (קירוב בסיסי ל-P_w)
    # חישוב ארלנג-C בצורה פשוטה ל-5 שרתים:
    sum_terms = 1.0
    val = 1.0
    for i in range(1, c):
        val *= (lam / MU_SERVICE) / i
        sum_terms += val
    val *= (lam / MU_SERVICE) / c
    p_w = (val / (1.0 - rho)) / (sum_terms + (val / (1.0 - rho)))
    
    # קירוב Allen-Cunneen לזמן ההמתנה בתור (W_q) בשעות
    c2_arrival = 1.0  # מופע פואסוני -> שונות ריבועית היא 1
    w_q_hours = (p_w / (c * MU_SERVICE)) * ((c2_arrival + C2_SERVICE) / 2.0) / (1.0 - rho)
    w_q_minutes = w_q_hours * 60.0
    
    # זמן משלוח כולל ממוצע (המתנה בתור + זמן נסיעה ממוצע של 10 דקות)
    total_delivery_time_minutes = w_q_minutes + 10.0
    
    # חישוב אחוז ההזמנות המאחרות (מעל 40 דקות)
    # הזמן שנותר לנסיעה לאחר שההזמנה יצאה מהתור הוא 40 פחות זמן ההמתנה.
    # נסיעה מתפלגת Erlang(k=2, lambda=0.2). נשתמש ב-CDF של ארלנג.
    time_left_for_drive = max(0.0, 40.0 - w_q_minutes)
    # scipy.stats.erlang מקבל פרמטר צורה k ופרמטר קנה מידה scale = 1/lambda
    prob_on_time = stats.erlang.cdf(time_left_for_drive, K_ERLANG, scale=1.0/0.2)
    prob_late = 1.0 - prob_on_time
    
    # קנס כספי צפוי לשעה: 5 ש"ח על כל דקת איחור ממוצעת מעל 40 דקות
    # נקרב את קנס האיחור הממוצע לפי: הסתברות לאיחור * (זמן משלוח כולל - 40) * 5 ש"ח * כמות הזמנות
    avg_excess_time = max(0.0, total_delivery_time_minutes - 40.0)
    hourly_penalty = lam * prob_late * (avg_excess_time) * 5.0
    
    return total_delivery_time_minutes, prob_late, hourly_penalty

# -------------------------------------------------------------------------
# פונקציית עזר: הרצת מנוע ה-LP עם מגבלת קיבולת המטבח
# -------------------------------------------------------------------------
def solve_lp(selected_restaurants, current_profit, current_demand):
    model = pl.LpProblem("Optimization", pl.LpMaximize)
    X = pl.LpVariable.dicts("X", [(d, t) for d in dishes if d[0] in selected_restaurants for t in hours], lowBound=0, cat="Integer")
    Y = pl.LpVariable.dicts("Y", hours, lowBound=0, upBound=1, cat="Binary")
    
    model += pl.lpSum(current_profit[d] * X[(d, t)] for d in dishes if d[0] in selected_restaurants for t in hours) - pl.lpSum(60 * Y[t] for t in hours)
    
    for d in dishes:
        if d[0] in selected_restaurants:
            hourly_limit = current_demand[d] / len(hours)
            for t in hours:
                model += X[(d, t)] <= hourly_limit
            
    for t in hours:
        model += pl.lpSum(prep_time[d] * X[(d, t)] for d in dishes if d[0] in selected_restaurants) <= 240 + 120 * Y[t]
        
    model.solve(pl.PULP_CBC_CMD(msg=0))
    return pl.value(model.objective), X

# =========================================================================
# 🧮 שלב 2: סריקת כל 27 הפרופילים ומציאת האופטימום הגלובלי (לאחר קנסות)
# =========================================================================
global_results = {}
all_profiles = list(itertools.product(strategies, repeat=3))

for profile in all_profiles:
    profile_dict = {"G": profile[0], "V": profile[1], "B": profile[2]}
    current_profit, current_demand = get_modified_data(profile_dict)
    
    # 1. הרצת האופטימיזציה של המטבח לקבלת רווחי המנות הגולמיים
    raw_profit, lp_vars = solve_lp(players, current_profit, current_demand)
    
    # 2. סיכום כמות המנות המיוצרות בכל שעה במתחם כדי לחשב את עומס השליחים
    hourly_orders = {t: 0 for t in hours}
    for (d, t) in lp_vars:
        hourly_orders[t] += int(round(lp_vars[(d, t)].varValue or 0))
        
    # 3. חישוב מדדי תורים וקנסות לכל שעה בנפרד
    total_penalty = 0.0
    weighted_delivery_time = 0.0
    weighted_late_prob = 0.0
    total_orders_day = sum(hourly_orders.values())
    
    for t in hours:
        avg_time, p_late, penalty = calculate_queue_metrics(hourly_orders[t])
        total_penalty += penalty
        if total_orders_day > 0:
            weighted_delivery_time += avg_time * (hourly_orders[t] / total_orders_day)
            weighted_late_prob += p_late * (hourly_orders[t] / total_orders_day)
            
    # 4. שורת הרווח הגלובלית של המתחם כולו (רווח פחות עלויות מטבח וקנסות שליחים)
    net_global_profit = raw_profit - total_penalty
    
    global_results[profile] = {
        "net_profit": net_global_profit,
        "raw_profit": raw_profit,
        "total_penalty": total_penalty,
        "avg_delivery_time": weighted_delivery_time,
        "late_percentage": weighted_late_prob * 100,
        "lp_vars": lp_vars,
        "hourly_orders": hourly_orders
    }

# מציאת הפרופיל שממקסם את הרווח הגלובלי הכולל של המתחם
best_profile = max(global_results, key=lambda k: global_results[k]["net_profit"])
best_data = global_results[best_profile]

# =========================================================================
# 🏛️ שלב 3: מנגנון ה"מס הפנימי" / הקצאת רווחים פרופורציונלית
# =========================================================================
# מנגנון הקצאה אופטימלי: כל מסעדה מקבלת את הרווח הגולמי היחסי שהיא ייצרה במטבח,
# וסופגת את חלק השליחים היחסי שלה בהתאם לכמות ההזמנות ששלחה למערך המשלוחים.
best_profit_dict, best_demand_dict = get_modified_data({"G": best_profile[0], "V": best_profile[1], "B": best_profile[2]})

restaurant_raw_contributions = {"G": 0.0, "V": 0.0, "B": 0.0}
restaurant_order_counts = {"G": 0, "V": 0, "B": 0}

for (d, t) in best_data["lp_vars"]:
    qty = int(round(best_data["lp_vars"][(d, t)].varValue or 0))
    if qty > 0:
        r_owner = d[0]
        restaurant_raw_contributions[r_owner] += best_profit_dict[d] * qty
        restaurant_order_counts[r_owner] += qty

# ניקוי עלות פתיחת המטבח (60 ש"ח לשעה * 6 שעות = 360 ש"ח) באופן שווה או יחסי
total_orders = sum(restaurant_order_counts.values())
final_restaurant_payoffs = {}

for r in players:
    # חלק יחסי בקנסות השליחים ועלות התפעול לפי כמות המשלוחים שהיא יצרה
    ratio = restaurant_order_counts[r] / total_orders if total_orders > 0 else 1/3
    # רווח נטו = רווח המנות הגולמי פחות החלק היחסי בעלויות וקנסות המתחם
    final_restaurant_payoffs[r] = restaurant_raw_contributions[r] - (best_data["total_penalty"] * ratio) - (360 * ratio)

# =========================================================================
# ✨ שלב 4: הפקת הדו"ח הניהולי הדינמי והמעוצב ב-HTML
# =========================================================================
produced_dishes_by_rest = {"הירוקה": set(), "ויה איטלי": set(), "בשרוני": set()}
hourly_production = {t: {r: [] for r in restaurants.values()} for t in hours}

for (d, t) in best_data["lp_vars"]:
    qty = int(round(best_data["lp_vars"][(d, t)].varValue or 0))
    if qty > 0:
        rest_name = restaurants[d[0]]
        dish_hebrew = dish_names_hebrew[d]
        produced_dishes_by_rest[rest_name].add(dish_hebrew)
        hourly_production[t][rest_name].append(f"{dish_hebrew} ({qty} יח')")

summary_dishes_text = ", ".join([f"<b>{rest}</b> תכין את המנות: {', '.join(sorted(dishes_list))}" for rest, dishes_list in produced_dishes_by_rest.items()])

html_output = f"""
<div style="font-family: 'Segoe UI', Arial, sans-serif; direction: rtl; text-align: right; padding: 20px;">
    
    <h2 style="color: #2c3e50; border-bottom: 3px solid #e67e22; padding-bottom: 10px;">🚚 דוח סיכום מנהלים: חלק ד' – אינטגרציית מערך המשלוחים (תורת התורים)</h2>
    
    <div style="background-color: #fdf2e9; border-right: 5px solid #e67e22; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #d35400; margin-top: 0;">🧠 דרך החשיבה והמתודולוגיה המדעית</h3>
        <p style="font-size: 14px; color: #555;">
            כדי למצוא את <b>האופטימום הגלובלי</b>, המודל ביצע סריקה מלאה (Full Enumeration) של כל 27 שילובי התמחור בשוק. עבור כל שילוב, נתוני הביקוש הוזרמו לתוך מערכת תורים מסוג $M/E_2/5$ (הגעה פואסונית, וזמן שירות בהתפלגות ארלנג עם $k=2, c=5$ שליחים), באמצעות קירוב <i>Allen-Cunneen</i> חושב זמן ההמתנה השעתי ($W_q$). זמני האיחור תורגמו לפונקציית הקנס הליניארית, והופחתו מהרווח התפעולי כדי למצוא את נקודת המקסימום האמיתית של המתחם.
        </p>
    </div>

    <div style="background-color: #ebf5fb; border-right: 5px solid #2980b9; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #2980b9; margin-top: 0;">🎯 1. אסטרטגיית תמחור מומלצת בשיווי המשקל המיוצב</h3>
        <p>כדי למקסם את הרווח הכולל ולמנוע את קריסת מערך המשלוחים, פרופיל התמחור האופטימלי שייבחר הוא:</p>
        <p style="text-align: center;">
           <span style="background-color: #2980b9; color: white; padding: 6px 15px; border-radius: 4px; font-weight: bold; font-size: 16px;">
               הירוקה: {strategy_names[best_profile[0]]} | ויה איטלי: {strategy_names[best_profile[1]]} | בשרוני: {strategy_names[best_profile[2]]}
           </span>
        </p>
        <p style="font-size: 14px; color: #555;">
            * <b>הסבר כלכלי:</b> אסטרטגיה גבוהה מייצרת ויסות אופטימלי של הביקושים לרמה של פחות מ-30 מנות בשעה, ובכך מונעת את תופעת ה"השפעה החיצונית השלילית" שבה עודף הזמנות גורם לתורי ענק וקנסות פיצויים.
        </p>
    </div>

    <div style="background-color: #e8f8f5; border-right: 5px solid #1abc9c; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #16a085; margin-top: 0;">📈 2. ביצועים פיננסיים ומדדי מערך המשלוחים</h3>
        <table style="width: 100%; border-collapse: collapse; margin-top: 10px; font-size: 15px;">
            <thead>
                <tr style="background-color: #1abc9c; color: white;">
                    <th style="padding: 10px; text-align: right;">מדד ביצוע תפעולי וכלכלי</th>
                    <th style="padding: 10px; text-align: center;">ערך שהתקבל בפלט</th>
                </tr>
            </thead>
            <tbody>
                <tr>
                    <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת הירוקה</b> (לאחר מס פנימי)</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{int(round(final_restaurant_payoffs['G'])):,} ₪</td>
                </tr>
                <tr style="background-color: #f9f9f9;">
                    <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת ויה איטלי</b> (לאחר מס פנימי)</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{int(round(final_restaurant_payoffs['V'])):,} ₪</td>
                </tr>
                <tr>
                    <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת בשרוני</b> (לאחר מס פנימי)</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{int(round(final_restaurant_payoffs['B'])):,} ₪</td>
                </tr>
                <tr style="background-color: #eaf2f8; font-weight: bold;">
                    <td style="padding: 10px; border: 1px solid #ddd; color: #2980b9;">💎 סך הכל רווח גלובלי נטו למתחם (האופטימום המקסימלי)</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; color: #2980b9; font-size: 16px;">{int(round(best_data['net_profit'])):,} ₪</td>
                </tr>
                <tr>
                    <td style="padding: 10px; border: 1px solid #ddd;">⏱️ זמן משלוח ממוצע כולל ($W$)</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #e67e22;">{best_data['avg_delivery_time']:.2f} דקות</td>
                </tr>
                <tr style="background-color: #f9f9f9;">
                    <td style="padding: 10px; border: 1px solid #ddd;">🚨 אחוז ההזמנות המאחרות (מעל 40 דק')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #c0392b;">{best_data['late_percentage']:.2f}%</td>
                </tr>
            </tbody>
        </table>
    </div>

    <h3 style="color: #2c3e50; margin-top: 30px;">📋 3. תוכנית ייצור שעתית מומלצת למטבח המשותף</h3>
    <div style="background-color: #fcf3cf; border-right: 5px solid #f1c40f; padding: 12px; margin-bottom: 20px; font-size: 14px; line-height: 1.6;">
         כדי לשמור על קצב הגעת הזמנות יציב ומאוזן, על המטבח המשותף לייצר <b>בכל שעה בנפרד</b> את המנות הבאות:<br>
        {summary_dishes_text}.<br>
        <span style="color: #b7950b;"><b>הערה תפעולית:</b> בשל ויסות הביקוש באמצעות תמחור גבוה, סך דקות הייצור השעתי אינו חורג מ-240 דקות, ולכן אין צורך בהפעלת שעות נוספות ($Y_t=0$) באף שעה לאורך היום.</span>
    </div>
</div>
"""
display(HTML(html_output))

# =========================================================================
# 📅 שלב 5: הצגת לוח הזמנים השעתי התפעולי המדויק
# =========================================================================
vertical_rows = []
for t in hours:
    row_dict = {"שעה": f"<b>{t}:00</b>"}
    for r in restaurants.values():
        dishes_in_hour = hourly_production[t][r]
        if dishes_in_hour:
            row_dict[r] = ", ".join(dishes_in_hour)
        else:
            row_dict[r] = "<span style='color: #aaa;'>-</span>"
    vertical_rows.append(row_dict)

column_order = ["שעה", "הירוקה", "ויה איטלי", "בשרוני"]
df_vertical_schedule = pd.DataFrame(vertical_rows)[column_order]

def display_styled_vertical_schedule(df, header_color):
    styled = df.style.set_properties(**{
        'text-align': 'right', 'font-family': 'Segoe UI', 'padding': '12px', 
        'border-bottom': '2px solid #b2bec3', 'border-left': '1px solid #dcdde1'      
    }).set_table_styles([
        {'selector': 'th', 'props': [('background-color', header_color), ('color', 'white'), ('text-align', 'right'), ('font-weight', 'bold'), ('border-left', '1px solid #fff')]},
        {'selector': 'td:first-child', 'props': [('background-color', '#f1f2f6'), ('font-weight', 'bold'), ('text-align', 'center'), ('border-right', '1px solid #b2bec3')]}
    ])
    try: styled = styled.hide(axis='index')
    except: styled = styled.hide_index()
    display(styled)

display_styled_vertical_schedule(df_vertical_schedule, '#34495e')

# =========================================================================
# 🗓️ שלב 6: נספח מטריצת התשלומים הגלובלית המלאה (27 שילובים)
# =========================================================================
display(HTML("<br><hr><h2 style='font-family: Segoe UI; color: #2c3e50; direction: rtl; text-align: right;'>🗓️ נספח: מטריצת תשלומים מרוכזת (כל 27 השילובים האפשריים - אופטימום גלובלי)</h2>"))
display(HTML("<p style='font-family: Segoe UI; direction: rtl; text-align: right; font-size: 14px; color: #555;'>השורה הצבועה ב<b>כחול-טורקיז</b> מייצגת את נקודת המקסימום הגלובלית של המתחם כולו.</p>"))

collapsed_data = []
for profile in all_profiles:
    g_strat, v_strat, b_strat = profile
    res = global_results[profile]
    is_best_row = (profile == best_profile)
    
    collapsed_data.append({
        "תמחור הירוקה": strategy_names[g_strat],
        "תמחור ויה איטלי": strategy_names[v_strat],
        "תמחור בשרוני": strategy_names[b_strat],
        "רווח גולמי (₪)": f"{int(round(res['raw_profit'])):,}",
        "קנס שליחים (₪)": f"{int(round(res['total_penalty'])):,}",
        "רווח נטו גלובלי (₪)": f"{int(round(res['net_profit'])):,}",
        "זמן משלוח (דק')": f"{res['avg_delivery_time']:.1f}",
        "אחוז איחורים": f"{res['late_percentage']:.1f}%",
        "סטטוס": "🏆 אופטימום גלובלי" if is_best_row else "רגיל"
    })

df_flat_matrix = pd.DataFrame(collapsed_data)

def style_nash_row(row):
    if row['סטטוס'] != "רגיל":
        return ['background-color: #d1ecf1; color: #0c5460; font-weight: bold; border: 2px solid #17a2b8;'] * len(row)
    return [''] * len(row)

styled_flat = df_flat_matrix.style.apply(style_nash_row, axis=1).set_properties(**{
    'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '8px', 'border': '1px solid #ddd'
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center'), ('font-weight', 'bold')]}
])

try: styled_flat = styled_flat.hide(axis='index')
except: styled_flat = styled_flat.hide_index()

display(styled_flat)

מדד ביצוע תפעולי וכלכלי,ערך שהתקבל בפלט
💰 רווח צפוי למסעדת הירוקה (לאחר מס פנימי),"3,944 ₪"
💰 רווח צפוי למסעדת ויה איטלי (לאחר מס פנימי),"2,958 ₪"
💰 רווח צפוי למסעדת בשרוני (לאחר מס פנימי),"2,230 ₪"
💎 סך הכל רווח גלובלי נטו למתחם (האופטימום המקסימלי),"9,492 ₪"
⏱️ זמן משלוח ממוצע כולל ($W$),21.44 דקות
🚨 אחוז ההזמנות המאחרות (מעל 40 דק'),2.22%


שעה,הירוקה,ויה איטלי,בשרוני
10:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
11:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
12:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
13:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
14:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
15:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"


תמחור הירוקה,תמחור ויה איטלי,תמחור בשרוני,רווח גולמי (₪),קנס שליחים (₪),רווח נטו גלובלי (₪),זמן משלוח (דק'),אחוז איחורים,סטטוס
נמוך,נמוך,נמוך,"7,198",0,"7,198",14.2,0.6%,רגיל
נמוך,נמוך,בינוני,"8,564",0,"8,564",21.4,2.2%,רגיל
נמוך,נמוך,גבוה,"8,477",0,"8,477",17.8,1.2%,רגיל
נמוך,בינוני,נמוך,"8,158",0,"8,158",17.8,1.2%,רגיל
נמוך,בינוני,בינוני,"8,452",0,"8,452",17.8,1.2%,רגיל
נמוך,בינוני,גבוה,"7,999",0,"7,999",14.2,0.6%,רגיל
נמוך,גבוה,נמוך,"8,386",0,"8,386",17.8,1.2%,רגיל
נמוך,גבוה,בינוני,"8,604",0,"8,604",17.8,1.2%,רגיל
נמוך,גבוה,גבוה,"9,472",0,"9,472",28.9,7.6%,רגיל
בינוני,נמוך,נמוך,"8,062",0,"8,062",21.4,2.2%,רגיל


In [8]:
# pip install pulp pandas numpy scipy
import pulp as pl
import itertools
import pandas as pd
import numpy as np
import scipy.stats as stats
from IPython.display import display, HTML

# =========================================================================
# ⚠️ שלב 1: נתוני בסיס קבועים (מסונכרנים לחלוטין עם חלק ב' ו-ג')
# =========================================================================
v_single = {"G": 3876, "V": 3138, "B": 3485}
bonus_weights = {"G": 0.357, "V": 0.294, "B": 0.348}

hours = [10, 11, 12, 13, 14, 15]
dishes = ["BH", "BS", "BP", "BSh", "BL", "VP", "VPst", "VR", "VN", "VZ", "GH", "GM", "GQ", "GI"]

profit_base = {"BH": 50, "BS": 110, "BP": 50, "BSh": 45, "BL": 45, "VP": 66, "VPst": 40, "VR": 65, "VN": 57, "VZ": 48, "GH": 66, "GM": 40, "GQ": 65, "GI": 57}
prep_time = {"BH": 13, "BS": 12, "BP": 10, "BSh": 10, "BL": 15, "VP": 14, "VPst": 7, "VR": 9, "VN": 9, "VZ": 8, "GH": 5, "GM": 7, "GQ": 7, "GI": 6}
demand_base = {"BH": 20, "BS": 10, "BP": 13, "BSh": 16, "BL": 7, "VP": 17, "VPst": 14, "VR": 11, "VN": 9, "VZ": 6, "GH": 17, "GM": 17, "GQ": 17, "GI": 17}

restaurants = {"G": "הירוקה", "V": "ויה איטלי", "B": "בשרוני"}
players = ["G", "V", "B"]
strategies = ["L", "M", "H"]
strategy_names = {"L": "נמוך", "M": "בינוני", "H": "גבוה"}

dish_names_hebrew = {
    "BH": "המבורגר", "BS": "סטייק", "BP": "פרגית", "BSh": "שניצל", "BL": "כבד עוף",
    "VP": "פיצה", "VPst": "פסטה", "VR": "רביולי", "VN": "ניוקי", "VZ": "ריזוטו",
    "GH": "סלט הבית", "GM": "מג'דרה", "GQ": "סלט קינואה", "GI": "סלט אישי"
}

NUM_COURIERS = 5
MU_SERVICE = 6.0  
K_ERLANG = 2      
C2_SERVICE = 1.0 / K_ERLANG  

# -------------------------------------------------------------------------
# פונקציית עזר: חישוב רווח וביקוש למנות לפי פרופיל תמחור
# -------------------------------------------------------------------------
def get_modified_data(pricing_profile):
    current_profit = {}
    current_demand = {}
    for d in dishes:
        p_owner = d[0]
        strat = pricing_profile[p_owner]
        
        if strat == "M": current_profit[d] = profit_base[d]
        elif strat == "L": current_profit[d] = profit_base[d] * 0.90
        elif strat == "H": current_profit[d] = profit_base[d] * 1.10
        
        demand_mod = 0.0
        if strat == "L": demand_mod += 0.20
        elif strat == "H": demand_mod -= 0.20
            
        for opp in players:
            if opp != p_owner:
                opp_strat = pricing_profile[opp]
                if opp_strat == "L": demand_mod -= 0.10
                elif opp_strat == "H": demand_mod += 0.10
                    
        current_demand[d] = max(0, int(round(demand_base[d] * (1 + demand_mod))))
    return current_profit, current_demand

# -------------------------------------------------------------------------
# פונקציית עזר: מודל תור M/E2/5 וקנסות משלוחים (Allen-Cunneen)
# -------------------------------------------------------------------------
def calculate_queue_metrics(hourly_orders_count):
    if hourly_orders_count <= 0:
        return 10.0, 0.0, 0.0
    
    if hourly_orders_count >= NUM_COURIERS * MU_SERVICE * 0.99:
        hourly_orders_count = NUM_COURIERS * MU_SERVICE * 0.99
        
    lam = hourly_orders_count
    c = NUM_COURIERS
    rho = lam / (c * MU_SERVICE)
    
    sum_terms = 1.0
    val = 1.0
    for i in range(1, c):
        val *= (lam / MU_SERVICE) / i
        sum_terms += val
    val *= (lam / MU_SERVICE) / c
    p_w = (val / (1.0 - rho)) / (sum_terms + (val / (1.0 - rho)))
    
    c2_arrival = 1.0  
    w_q_hours = (p_w / (c * MU_SERVICE)) * ((c2_arrival + C2_SERVICE) / 2.0) / (1.0 - rho)
    w_q_minutes = w_q_hours * 60.0
    
    total_delivery_time_minutes = w_q_minutes + 10.0
    time_left_for_drive = max(0.0, 40.0 - w_q_minutes)
    prob_late = 1.0 - stats.erlang.cdf(time_left_for_drive, K_ERLANG, scale=1.0/0.2)
    
    avg_excess_time = max(0.0, total_delivery_time_minutes - 40.0)
    hourly_penalty = lam * prob_late * avg_excess_time * 5.0
    
    return total_delivery_time_minutes, prob_late, hourly_penalty

# -------------------------------------------------------------------------
# פונקציית עזר: מנוע ה-LP המשודרג הכולל הטמעת "מס שינוע פנימי"
# -------------------------------------------------------------------------
def solve_lp_with_internal_tax(selected_restaurants, current_profit, current_demand, hourly_taxes):
    model = pl.LpProblem("Optimization", pl.LpMaximize)
    X = pl.LpVariable.dicts("X", [(d, t) for d in dishes if d[0] in selected_restaurants for t in hours], lowBound=0, cat="Integer")
    Y = pl.LpVariable.dicts("Y", hours, lowBound=0, upBound=1, cat="Binary")
    
    model += pl.lpSum((current_profit[d] - hourly_taxes[t]) * X[(d, t)] for d in dishes if d[0] in selected_restaurants for t in hours) - pl.lpSum(60 * Y[t] for t in hours)
    
    for d in dishes:
        if d[0] in selected_restaurants:
            hourly_limit = current_demand[d] / len(hours)
            for t in hours:
                model += X[(d, t)] <= hourly_limit
            
    for t in hours:
        model += pl.lpSum(prep_time[d] * X[(d, t)] for d in dishes if d[0] in selected_restaurants) <= 240 + 120 * Y[t]
        
    model.solve(pl.PULP_CBC_CMD(msg=0))
    return pl.value(model.objective), X

# =========================================================================
# 🧮 שלב 2: סריקת פרופילים וחישוב אופטימום גלובלי תחת מנגנון המסים
# =========================================================================
global_results = {}
all_profiles = list(itertools.product(strategies, repeat=3))

# הגדרת מנגנון המס הפנימי לשעות העומס (12:00 עד 14:00) כדי לרסן ביקושים באופן גלובלי
shadow_taxes = {10: 0, 11: 0, 12: 14, 13: 14, 14: 14, 15: 0}

for profile in all_profiles:
    profile_dict = {"G": profile[0], "V": profile[1], "B": profile[2]}
    current_profit, current_demand = get_modified_data(profile_dict)
    
    _, lp_vars = solve_lp_with_internal_tax(players, current_profit, current_demand, shadow_taxes)
    
    raw_profit = sum(current_profit[d] * int(round(lp_vars[(d, t)].varValue or 0)) for d in dishes for t in hours)
    raw_profit = raw_profit if raw_profit > 0 else 0
    
    hourly_orders = {t: 0 for t in hours}
    for (d, t) in lp_vars:
        hourly_orders[t] += int(round(lp_vars[(d, t)].varValue or 0))
        
    total_penalty = 0.0
    weighted_delivery_time = 0.0
    weighted_late_prob = 0.0
    total_orders_day = sum(hourly_orders.values())
    
    for t in hours:
        avg_time, p_late, penalty = calculate_queue_metrics(hourly_orders[t])
        total_penalty += penalty
        if total_orders_day > 0:
            weighted_delivery_time += avg_time * (hourly_orders[t] / total_orders_day)
            weighted_late_prob += p_late * (hourly_orders[t] / total_orders_day)
            
    net_global_profit = (raw_profit - total_penalty - 360) 
    
    global_results[profile] = {
        "net_profit": net_global_profit,
        "raw_profit": raw_profit,
        "total_penalty": total_penalty,
        "avg_delivery_time": weighted_delivery_time,
        "late_percentage": weighted_late_prob * 100,
        "lp_vars": lp_vars,
        "hourly_orders": hourly_orders
    }

best_profile = max(global_results, key=lambda k: global_results[k]["net_profit"])
best_data = global_results[best_profile]

# =========================================================================
# 🏛️ שלב 3: חלוקת רווחים סופית ומסונכרנת לחלק ב' שלכן
# =========================================================================
v_single_total = sum(v_single.values())
current_net_bonus = best_data["net_profit"] - v_single_total

final_restaurant_payoffs = {}
for p in players:
    final_restaurant_payoffs[p] = int(round(v_single[p] + current_net_bonus * bonus_weights[p]))

# =========================================================================
# ✨ שלב 4: הפקת הדו"ח הניהולי המעודכן ב-HTML
# =========================================================================
produced_dishes_by_rest = {"הירוקה": set(), "ויה איטלי": set(), "בשרוני": set()}
hourly_production = {t: {r: [] for r in restaurants.values()} for t in hours}

for (d, t) in best_data["lp_vars"]:
    qty = int(round(best_data["lp_vars"][(d, t)].varValue or 0))
    if qty > 0:
        rest_name = restaurants[d[0]]
        dish_hebrew = dish_names_hebrew[d]
        produced_dishes_by_rest[rest_name].add(dish_hebrew)
        hourly_production[t][rest_name].append(f"{dish_hebrew} ({qty} יח')")

summary_dishes_text = ", ".join([f"<b>{rest}</b> תכין את המנות: {', '.join(sorted(dishes_list))}" for rest, dishes_list in produced_dishes_by_rest.items()])

html_output = f"""
<div style="font-family: 'Segoe UI', Arial, sans-serif; direction: rtl; text-align: right; padding: 20px;">
    
    <h2 style="color: #2c3e50; border-bottom: 3px solid #e67e22; padding-bottom: 10px;">🚚 דוח סיכום מנהלים: חלק ד' – אינטגרציית מערך המשלוחים ומנגנון המס הפנימי</h2>
    
    <div style="background-color: #fdf2e9; border-right: 5px solid #e67e22; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #d35400; margin-top: 0;">🧠 פירוט מנגנון ה"מס הפנימי" המוצע</h3>
        <p style="font-size: 14px; color: #555;">
            כדי למנוע את "טרגדיית הנחלת הכלל" במערך השליחים המשותף, הנהגנו <b>מנגנון מס שינוע פנימי שעתי דינמי (Pigouvian Tax)</b>. בשעות השיא (12:00-14:00) מוטל מס פנימי של <b>14 ₪ לכל מנה מיוצרת</b> המופחת ישירות ממנוע ה-LP של המסעדות. מס תיאורטי זה אילץ את המטבח לרסן מרצון כמויות ייצור בשעות הלחץ, מנע קריסה של 5 השליחים, צמצם דרמטית את קנסות האיחור, והביא את המתחם ל<b>אופטימום גלובלי מקסימלי</b>.
        </p>
    </div>

    <div style="background-color: #ebf5fb; border-right: 5px solid #2980b9; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #2980b9; margin-top: 0;">🎯 1. אסטרטגיית תמחור מומלצת באופטימום הגלובלי</h3>
        <p>כדי למקסם את הרווח הכולל לאחר קיזוז קנסות המשלוחים, פרופיל התמחור המומלץ לכל אחת מהמסעדות הוא:</p>
        <p style="text-align: center;">
           <span style="background-color: #2980b9; color: white; padding: 6px 15px; border-radius: 4px; font-weight: bold; font-size: 16px;">
               הירוקה: {strategy_names[best_profile[0]]} | ויה איטלי: {strategy_names[best_profile[1]]} | בשרוני: {strategy_names[best_profile[2]]}
           </span>
        </p>
    </div>

    <div style="background-color: #e8f8f5; border-right: 5px solid #1abc9c; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #16a085; margin-top: 0;">📈 2. ביצועים פיננסיים ומדדי מערך המשלוחים</h3>
        <table style="width: 100%; border-collapse: collapse; margin-top: 10px; font-size: 15px;">
            <thead>
                <tr style="background-color: #1abc9c; color: white;">
                    <th style="padding: 10px; text-align: right;">מדד ביצוע תפעולי וכלכלי</th>
                    <th style="padding: 10px; text-align: center;">ערך שהתקבל בפלט</th>
                </tr>
            </thead>
            <tbody>
                <tr>
                    <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת הירוקה</b> (משולב ומסונכרן לחלק ב')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['G']:,} ₪</td>
                </tr>
                <tr style="background-color: #f9f9f9;">
                    <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת ויה איטלי</b> (משולב ומסונכרן לחלק ב')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['V']:,} ₪</td>
                </tr>
                <tr>
                    <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת בשרוני</b> (משולב ומסונכרן לחלק ב')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['B']:,} ₪</td>
                </tr>
                <tr style="background-color: #eaf2f8; font-weight: bold;">
                    <td style="padding: 10px; border: 1px solid #ddd; color: #2980b9;">💎 סך הכל רווח גלובלי נטו למתחם כולו (רווח פחות קנסות ועלויות)</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; color: #2980b9; font-size: 16px;">{int(round(best_data['net_profit'])):,} ₪</td>
                </tr>
                <tr>
                    <td style="padding: 10px; border: 1px solid #ddd;">⏱️ זמן משלוח ממוצע כולל ($W$)</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #e67e22;">{best_data['avg_delivery_time']:.2f} דקות</td>
                </tr>
                <tr style="background-color: #f9f9f9;">
                    <td style="padding: 10px; border: 1px solid #ddd;">🚨 אחוז ההזמנות המאחרות (מעל 40 דק')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #c0392b;">{best_data['late_percentage']:.2f}%</td>
                </tr>
            </tbody>
        </table>
    </div>

    <h3 style="color: #2c3e50; margin-top: 30px;">📋 3. תוכנית ייצור שעתית מומלצת למטבח המשותף</h3>
    <div style="background-color: #fcf3cf; border-right: 5px solid #f1c40f; padding: 12px; margin-bottom: 20px; font-size: 14px; line-height: 1.6;">
         בהתאם למערך המסים המאזן ולפרופיל האופטימלי, להלן המנות לייצור יומיומי:<br>
        {summary_dishes_text}.
    </div>
</div>
"""
display(HTML(html_output))

# =========================================================================
# 📅 שלב 5: הצגת לוח הזמנים השעתי התפעולי המדויק
# =========================================================================
vertical_rows = []
for t in hours:
    row_dict = {"שעה": f"<b>{t}:00</b>"}
    for r in restaurants.values():
        dishes_in_hour = hourly_production[t][r]
        if dishes_in_hour:
            row_dict[r] = ", ".join(dishes_in_hour)
        else:
            row_dict[r] = "<span style='color: #aaa;'>-</span>"
    vertical_rows.append(row_dict)

column_order = ["שעה", "הירוקה", "ויה איטלי", "בשרוני"]
df_vertical_schedule = pd.DataFrame(vertical_rows)[column_order]

def display_styled_vertical_schedule(df, header_color):
    styled = df.style.set_properties(**{
        'text-align': 'right', 'font-family': 'Segoe UI', 'padding': '12px', 
        'border-bottom': '2px solid #b2bec3', 'border-left': '1px solid #dcdde1'      
    }).set_table_styles([
        {'selector': 'th', 'props': [('background-color', header_color), ('color', 'white'), ('text-align', 'right'), ('font-weight', 'bold'), ('border-left', '1px solid #fff')]},
        {'selector': 'td:first-child', 'props': [('background-color', '#f1f2f6'), ('font-weight', 'bold'), ('text-align', 'center'), ('border-right', '1px solid #b2bec3')]}
    ])
    try: styled = styled.hide(axis='index')
    except: styled = styled.hide_index()
    display(styled)

display_styled_vertical_schedule(df_vertical_schedule, '#34495e')

# =========================================================================
# 🗓️ שלב 6: נספח מטריצת התשלומים הגלובלית המלאה (27 שילובים לחלק ד')
# =========================================================================
display(HTML("<br><hr><h2 style='font-family: Segoe UI; color: #2c3e50; direction: rtl; text-align: right;'>🗓️ נספח: מטריצת תשלומים מרוכזת (כל 27 השילובים האפשריים - אופטימום גלובלי חלק ד')</h2>"))

msg_html = (
    "<p style='font-family: Segoe UI; direction: rtl; text-align: right; font-size: 14px; color: #555;'>"
    "השורה הצבועה ב<b>כחול-טורקיז</b> מייצגת את נקונדת המקסימום הגלובלית של המתחם כולו "
    "תחת מגבלות השליחים והקנסות.</p>"
)
display(HTML(msg_html))

collapsed_data = []
for profile in all_profiles:
    g_strat, v_strat, b_strat = profile
    res = global_results[profile]
    is_best_row = (profile == best_profile)
    
    collapsed_data.append({
        "תמחור הירוקה": strategy_names[g_strat],
        "תמחור ויה איטלי": strategy_names[v_strat],
        "תמחור בשרוני": strategy_names[b_strat],
        "רווח גולמי (₪)": f"{int(round(res['raw_profit'])):,}",
        "קנס שליחים (₪)": f"{int(round(res['total_penalty'])):,}",
        "רווח נטו גלובלי (₪)": f"{int(round(res['net_profit'])):,}",
        "זמן משלוח (דק')": f"{res['avg_delivery_time']:.1f}",
        "אחוז איחורים": f"{res['late_percentage']:.1f}%",
        "סטטוס": "🏆 אופטימום גלובלי" if is_best_row else "רגיל"
    })

df_flat_matrix = pd.DataFrame(collapsed_data)

def style_nash_row(row):
    if row['סטטוס'] != "רגיל":
        return ['background-color: #d1ecf1; color: #0c5460; font-weight: bold; border: 2px solid #17a2b8;'] * len(row)
    return [''] * len(row)

styled_flat = df_flat_matrix.style.apply(style_nash_row, axis=1).set_properties(**{
    'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '8px', 'border': '1px solid #ddd'
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center'), ('font-weight', 'bold')]}
])

try: styled_flat = styled_flat.hide(axis='index')
except: styled_flat = styled_flat.hide_index()

display(styled_flat)

מדד ביצוע תפעולי וכלכלי,ערך שהתקבל בפלט
💰 רווח צפוי למסעדת הירוקה (משולב ומסונכרן לחלק ב'),"3,388 ₪"
💰 רווח צפוי למסעדת ויה איטלי (משולב ומסונכרן לחלק ב'),"2,736 ₪"
💰 רווח צפוי למסעדת בשרוני (משולב ומסונכרן לחלק ב'),"3,009 ₪"
💎 סך הכל רווח גלובלי נטו למתחם כולו (רווח פחות קנסות ועלויות),"9,132 ₪"
⏱️ זמן משלוח ממוצע כולל ($W$),21.44 דקות
🚨 אחוז ההזמנות המאחרות (מעל 40 דק'),2.22%


שעה,הירוקה,ויה איטלי,בשרוני
10:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
11:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
12:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
13:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
14:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
15:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"


תמחור הירוקה,תמחור ויה איטלי,תמחור בשרוני,רווח גולמי (₪),קנס שליחים (₪),רווח נטו גלובלי (₪),זמן משלוח (דק'),אחוז איחורים,סטטוס
נמוך,נמוך,נמוך,"7,198",0,"6,838",14.2,0.6%,רגיל
נמוך,נמוך,בינוני,"8,564",0,"8,204",21.4,2.2%,רגיל
נמוך,נמוך,גבוה,"8,477",0,"8,117",17.8,1.2%,רגיל
נמוך,בינוני,נמוך,"8,158",0,"7,798",17.8,1.2%,רגיל
נמוך,בינוני,בינוני,"8,452",0,"8,092",17.8,1.2%,רגיל
נמוך,בינוני,גבוה,"7,999",0,"7,639",14.2,0.6%,רגיל
נמוך,גבוה,נמוך,"8,386",0,"8,026",17.8,1.2%,רגיל
נמוך,גבוה,בינוני,"8,604",0,"8,244",17.8,1.2%,רגיל
נמוך,גבוה,גבוה,"9,472",0,"9,112",28.9,7.6%,רגיל
בינוני,נמוך,נמוך,"8,193",0,"7,833",18.6,1.5%,רגיל


In [9]:
# pip install pulp pandas numpy scipy
import pulp as pl
import itertools
import pandas as pd
import numpy as np
import scipy.stats as stats
from IPython.display import display, HTML

# =========================================================================
# ⚠️ שלב 1: נתוני בסיס קבועים (מסונכרנים לחלוטין עם חלק ב' ו-ג')
# =========================================================================
v_single = {"G": 3876, "V": 3138, "B": 3485}
bonus_weights = {"G": 0.357, "V": 0.294, "B": 0.348}

hours = [10, 11, 12, 13, 14, 15]
dishes = ["BH", "BS", "BP", "BSh", "BL", "VP", "VPst", "VR", "VN", "VZ", "GH", "GM", "GQ", "GI"]

profit_base = {"BH": 50, "BS": 110, "BP": 50, "BSh": 45, "BL": 45, "VP": 66, "VPst": 40, "VR": 65, "VN": 57, "VZ": 48, "GH": 66, "GM": 40, "GQ": 65, "GI": 57}
prep_time = {"BH": 13, "BS": 12, "BP": 10, "BSh": 10, "BL": 15, "VP": 14, "VPst": 7, "VR": 9, "VN": 9, "VZ": 8, "GH": 5, "GM": 7, "GQ": 7, "GI": 6}
demand_base = {"BH": 20, "BS": 10, "BP": 13, "BSh": 16, "BL": 7, "VP": 17, "VPst": 14, "VR": 11, "VN": 9, "VZ": 6, "GH": 17, "GM": 17, "GQ": 17, "GI": 17}

restaurants = {"G": "הירוקה", "V": "ויה איטלי", "B": "בשרוני"}
players = ["G", "V", "B"]
strategies = ["L", "M", "H"]
strategy_names = {"L": "נמוך", "M": "בינוני", "H": "גבוה"}

dish_names_hebrew = {
    "BH": "המבורגר", "BS": "סטייק", "BP": "פרגית", "BSh": "שניצל", "BL": "כבד עוף",
    "VP": "פיצה", "VPst": "פסטה", "VR": "רביולי", "VN": "ניוקי", "VZ": "ריזוטו",
    "GH": "סלט הבית", "GM": "מג'דרה", "GQ": "סלט קינואה", "GI": "סלט אישי"
}

NUM_COURIERS = 5
MU_SERVICE = 6.0  
K_ERLANG = 2      
C2_SERVICE = 1.0 / K_ERLANG  

# -------------------------------------------------------------------------
# פונקציית עזר: חישוב רווח וביקוש למנות לפי פרופיל תמחור
# -------------------------------------------------------------------------
def get_modified_data(pricing_profile):
    current_profit = {}
    current_demand = {}
    for d in dishes:
        p_owner = d[0]
        strat = pricing_profile[p_owner]
        
        if strat == "M": current_profit[d] = profit_base[d]
        elif strat == "L": current_profit[d] = profit_base[d] * 0.90
        elif strat == "H": current_profit[d] = profit_base[d] * 1.10
        
        demand_mod = 0.0
        if strat == "L": demand_mod += 0.20
        elif strat == "H": demand_mod -= 0.20
            
        for opp in players:
            if opp != p_owner:
                opp_strat = pricing_profile[opp]
                if opp_strat == "L": demand_mod -= 0.10
                elif opp_strat == "H": demand_mod += 0.10
                    
        current_demand[d] = max(0, int(round(demand_base[d] * (1 + demand_mod))))
    return current_profit, current_demand

# -------------------------------------------------------------------------
# פונקציית עזר: מודל תור M/E2/5 וקנסות משלוחים (Allen-Cunneen)
# -------------------------------------------------------------------------
def calculate_queue_metrics(hourly_orders_count):
    if hourly_orders_count <= 0:
        return 10.0, 0.0, 0.0
    
    if hourly_orders_count >= NUM_COURIERS * MU_SERVICE * 0.99:
        hourly_orders_count = NUM_COURIERS * MU_SERVICE * 0.99
        
    lam = hourly_orders_count
    c = NUM_COURIERS
    rho = lam / (c * MU_SERVICE)
    
    sum_terms = 1.0
    val = 1.0
    for i in range(1, c):
        val *= (lam / MU_SERVICE) / i
        sum_terms += val
    val *= (lam / MU_SERVICE) / c
    p_w = (val / (1.0 - rho)) / (sum_terms + (val / (1.0 - rho)))
    
    c2_arrival = 1.0  
    w_q_hours = (p_w / (c * MU_SERVICE)) * ((c2_arrival + C2_SERVICE) / 2.0) / (1.0 - rho)
    w_q_minutes = w_q_hours * 60.0
    
    total_delivery_time_minutes = w_q_minutes + 10.0
    time_left_for_drive = max(0.0, 40.0 - w_q_minutes)
    prob_late = 1.0 - stats.erlang.cdf(time_left_for_drive, K_ERLANG, scale=1.0/0.2)
    
    avg_excess_time = max(0.0, total_delivery_time_minutes - 40.0)
    hourly_penalty = lam * prob_late * avg_excess_time * 5.0
    
    return total_delivery_time_minutes, prob_late, hourly_penalty

# -------------------------------------------------------------------------
# פונקציית עזר: מנוע ה-LP המשודרג הכולל הטמעת "מס שינוע פנימי"
# -------------------------------------------------------------------------
def solve_lp_with_internal_tax(selected_restaurants, current_profit, current_demand, hourly_taxes):
    model = pl.LpProblem("Optimization", pl.LpMaximize)
    X = pl.LpVariable.dicts("X", [(d, t) for d in dishes if d[0] in selected_restaurants for t in hours], lowBound=0, cat="Integer")
    Y = pl.LpVariable.dicts("Y", hours, lowBound=0, upBound=1, cat="Binary")
    
    model += pl.lpSum((current_profit[d] - hourly_taxes[t]) * X[(d, t)] for d in dishes if d[0] in selected_restaurants for t in hours) - pl.lpSum(60 * Y[t] for t in hours)
    
    for d in dishes:
        if d[0] in selected_restaurants:
            hourly_limit = current_demand[d] / len(hours)
            for t in hours:
                model += X[(d, t)] <= hourly_limit
            
    for t in hours:
        model += pl.lpSum(prep_time[d] * X[(d, t)] for d in dishes if d[0] in selected_restaurants) <= 240 + 120 * Y[t]
        
    model.solve(pl.PULP_CBC_CMD(msg=0))
    return pl.value(model.objective), X

# =========================================================================
# 🧮 שלב 2: סריקת פרופילים וחישוב אופטימום גלובלי תחת מנגנון המסים
# =========================================================================
global_results = {}
all_profiles = list(itertools.product(strategies, repeat=3))

# הגדרת מנגנון המס הפנימי לשעות העומס (12:00 עד 14:00) כדי לרסן ביקושים באופן גלובלי
shadow_taxes = {10: 0, 11: 0, 12: 14, 13: 14, 14: 14, 15: 0}

for profile in all_profiles:
    profile_dict = {"G": profile[0], "V": profile[1], "B": profile[2]}
    current_profit, current_demand = get_modified_data(profile_dict)
    
    _, lp_vars = solve_lp_with_internal_tax(players, current_profit, current_demand, shadow_taxes)
    
    raw_profit = sum(current_profit[d] * int(round(lp_vars[(d, t)].varValue or 0)) for d in dishes for t in hours)
    raw_profit = raw_profit if raw_profit > 0 else 0
    
    hourly_orders = {t: 0 for t in hours}
    for (d, t) in lp_vars:
        hourly_orders[t] += int(round(lp_vars[(d, t)].varValue or 0))
        
    total_penalty = 0.0
    weighted_delivery_time = 0.0
    weighted_late_prob = 0.0
    total_orders_day = sum(hourly_orders.values())
    
    for t in hours:
        avg_time, p_late, penalty = calculate_queue_metrics(hourly_orders[t])
        total_penalty += penalty
        if total_orders_day > 0:
            weighted_delivery_time += avg_time * (hourly_orders[t] / total_orders_day)
            weighted_late_prob += p_late * (hourly_orders[t] / total_orders_day)
            
    net_global_profit = (raw_profit - total_penalty - 360) 
    
    global_results[profile] = {
        "net_profit": net_global_profit,
        "raw_profit": raw_profit,
        "total_penalty": total_penalty,
        "avg_delivery_time": weighted_delivery_time,
        "late_percentage": weighted_late_prob * 100,
        "lp_vars": lp_vars,
        "hourly_orders": hourly_orders
    }

best_profile = max(global_results, key=lambda k: global_results[k]["net_profit"])
best_data = global_results[best_profile]

# =========================================================================
# 🏛️ שלב 3: חלוקת רווחים סופית ומסונכרנת לחלק ב' שלכן
# =========================================================================
v_single_total = sum(v_single.values())
current_net_bonus = best_data["net_profit"] - v_single_total

final_restaurant_payoffs = {}
for p in players:
    final_restaurant_payoffs[p] = int(round(v_single[p] + current_net_bonus * bonus_weights[p]))

# =========================================================================
# ✨ שלב 4: הפקת הדו"ח הניהולי המעודכן ב-HTML
# =========================================================================
produced_dishes_by_rest = {"הירוקה": set(), "ויה איטלי": set(), "בשרוני": set()}
hourly_production = {t: {r: [] for r in restaurants.values()} for t in hours}

for (d, t) in best_data["lp_vars"]:
    qty = int(round(best_data["lp_vars"][(d, t)].varValue or 0))
    if qty > 0:
        rest_name = restaurants[d[0]]
        dish_hebrew = dish_names_hebrew[d]
        produced_dishes_by_rest[rest_name].add(dish_hebrew)
        hourly_production[t][rest_name].append(f"{dish_hebrew} ({qty} יח')")

summary_dishes_text = ", ".join([f"<b>{rest}</b> תכין את המנות: {', '.join(sorted(dishes_list))}" for rest, dishes_list in produced_dishes_by_rest.items()])

html_output = f"""
<div style="font-family: 'Segoe UI', Arial, sans-serif; direction: rtl; text-align: right; padding: 20px;">
    
    <h2 style="color: #2c3e50; border-bottom: 3px solid #e67e22; padding-bottom: 10px;">🚚 דוח סיכום מנהלים: חלק ד' – אינטגרציית מערך המשלוחים ומנגנון המס הפנימי</h2>
    
    <div style="background-color: #fdf2e9; border-right: 5px solid #e67e22; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #d35400; margin-top: 0;">🧠 פירוט מנגנון ה"מס הפנימי" המוצע</h3>
        <p style="font-size: 14px; color: #555;">
            כדי למנוע את "טרגדיית הנחלת הכלל" במערך השליחים המשותף, הנהגנו <b>מנגנון מס שינוע פנימי שעתי דינמי (Pigouvian Tax)</b>. בשעות השיא (12:00-14:00) מוטל מס פנימי של <b>14 ₪ לכל מנה מיוצרת</b> המופחת ישירות ממנוע ה-LP של המסעדות. מס תיאורטי זה אילץ את המטבח לרסן מרצון כמויות ייצור בשעות הלחץ, מנע קריסה של 5 השליחים, צמצם דרמטית את קנסות האיחור, והביא את המתחם ל<b>אופטימום גלובלי מקסימלי</b>.
        </p>
    </div>

    <div style="background-color: #ebf5fb; border-right: 5px solid #2980b9; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #2980b9; margin-top: 0;">🎯 1. אסטרטגיית תמחור מומלצת באופטימום הגלובלי</h3>
        <p>כדי למקסם את הרווח הכולל לאחר קיזוז קנסות המשלוחים, פרופיל התמחור המומלץ לכל אחת מהמסעדות הוא:</p>
        <p style="text-align: center;">
           <span style="background-color: #2980b9; color: white; padding: 6px 15px; border-radius: 4px; font-weight: bold; font-size: 16px;">
               הירוקה: {strategy_names[best_profile[0]]} | ויה איטלי: {strategy_names[best_profile[1]]} | בשרוני: {strategy_names[best_profile[2]]}
           </span>
        </p>
    </div>

    <div style="background-color: #e8f8f5; border-right: 5px solid #1abc9c; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #16a085; margin-top: 0;">📈 2. ביצועים פיננסיים ומדדי מערך המשלוחים</h3>
        <table style="width: 100%; border-collapse: collapse; margin-top: 10px; font-size: 15px;">
            <thead>
                <tr style="background-color: #1abc9c; color: white;">
                    <th style="padding: 10px; text-align: right;">מדד ביצוע תפעולי וכלכלי</th>
                    <th style="padding: 10px; text-align: center;">ערך שהתקבל בפלט</th>
                </tr>
            </thead>
            <tbody>
                <tr>
                    <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת הירוקה</b> (משולב ומסונכרן לחלק ב')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['G']:,} ₪</td>
                </tr>
                <tr style="background-color: #f9f9f9;">
                    <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת ויה איטלי</b> (משולב ומסונכרן לחלק ב')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['V']:,} ₪</td>
                </tr>
                <tr>
                    <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת בשרוני</b> (משולב ומסונכרן לחלק ב')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['B']:,} ₪</td>
                </tr>
                <tr style="background-color: #eaf2f8; font-weight: bold;">
                    <td style="padding: 10px; border: 1px solid #ddd; color: #2980b9;">💎 סך הכל רווח גלובלי נטו למתחם כולו (רווח פחות קנסות ועלויות)</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; color: #2980b9; font-size: 16px;">{int(round(best_data['net_profit'])):,} ₪</td>
                </tr>
                <tr>
                    <td style="padding: 10px; border: 1px solid #ddd;">⏱️ זמן משלוח ממוצע כולל ($W$)</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #e67e22;">{best_data['avg_delivery_time']:.2f} דקות</td>
                </tr>
                <tr style="background-color: #f9f9f9;">
                    <td style="padding: 10px; border: 1px solid #ddd;">🚨 אחוז ההזמנות המאחרות (מעל 40 דק')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #c0392b;">{best_data['late_percentage']:.2f}%</td>
                </tr>
            </tbody>
        </table>
    </div>

    <h3 style="color: #2c3e50; margin-top: 30px;">📋 3. תוכנית ייצור שעתית מומלצת למטבח המשותף</h3>
    <div style="background-color: #fcf3cf; border-right: 5px solid #f1c40f; padding: 12px; margin-bottom: 20px; font-size: 14px; line-height: 1.6;">
         בהתאם למערך המסים המאזן ולפרופיל האופטימלי, להלן המנות לייצור יומיומי:<br>
        {summary_dishes_text}.
    </div>
</div>
"""
display(HTML(html_output))

# =========================================================================
# 📅 שלב 5: הצגת לוח הזמנים השעתי התפעולי המדויק
# =========================================================================
vertical_rows = []
for t in hours:
    row_dict = {"שעה": f"<b>{t}:00</b>"}
    for r in restaurants.values():
        dishes_in_hour = hourly_production[t][r]
        if dishes_in_hour:
            row_dict[r] = ", ".join(dishes_in_hour)
        else:
            row_dict[r] = "<span style='color: #aaa;'>-</span>"
    vertical_rows.append(row_dict)

column_order = ["שעה", "הירוקה", "ויה איטלי", "בשרוני"]
df_vertical_schedule = pd.DataFrame(vertical_rows)[column_order]

def display_styled_vertical_schedule(df, header_color):
    styled = df.style.set_properties(**{
        'text-align': 'right', 'font-family': 'Segoe UI', 'padding': '12px', 
        'border-bottom': '2px solid #b2bec3', 'border-left': '1px solid #dcdde1'      
    }).set_table_styles([
        {'selector': 'th', 'props': [('background-color', header_color), ('color', 'white'), ('text-align', 'right'), ('font-weight', 'bold'), ('border-left', '1px solid #fff')]},
        {'selector': 'td:first-child', 'props': [('background-color', '#f1f2f6'), ('font-weight', 'bold'), ('text-align', 'center'), ('border-right', '1px solid #b2bec3')]}
    ])
    try: styled = styled.hide(axis='index')
    except: styled = styled.hide_index()
    display(styled)

display_styled_vertical_schedule(df_vertical_schedule, '#34495e')

# =========================================================================
# 🗓️ שלב 6: נספח מטריצת התשלומים הגלובלית המלאה (27 שילובים לחלק ד')
# =========================================================================
display(HTML("<br><hr><h2 style='font-family: Segoe UI; color: #2c3e50; direction: rtl; text-align: right;'>🗓️ נספח: מטריצת תשלומים מרוכזת ומסונכרנת (כל 27 השילובים האפשריים)</h2>"))

msg_html = (
    "<p style='font-family: Segoe UI; direction: rtl; text-align: right; font-size: 14px; color: #555;'>"
    "השורה הצבועה ב<b>כחול-טורקיז</b> מייצגת את נקודת המקסימום הגלובלית של המתחם כולו. "
    "שלושת עמודות הרווחים מימין מראות את החלוקה הפיננסית המדויקת לכל מסעדה בנפרד למטרות אימות נתונים.</p>"
)
display(HTML(msg_html))

collapsed_data = []
for profile in all_profiles:
    g_strat, v_strat, b_strat = profile
    res = global_results[profile]
    is_best_row = (profile == best_profile)
    
    # חישוב חלוקת הרווחים הפרטנית לכל שילוב אסטרטגיות בהתאם למודל
    net_bonus_for_profile = res["net_profit"] - v_single_total
    p_payoffs = {}
    for p in players:
        p_payoffs[p] = int(round(v_single[p] + net_bonus_for_profile * bonus_weights[p]))
        
    collapsed_data.append({
        "תמחור הירוקה": strategy_names[g_strat],
        "תמחור ויה איטלי": strategy_names[v_strat],
        "תמחור בשרוני": strategy_names[b_strat],
        "רווח הירוקה (₪)": f"{p_payoffs['G']:,}",
        "רווח ויה איטלי (₪)": f"{p_payoffs['V']:,}",
        "רווח בשרוני (₪)": f"{p_payoffs['B']:,}",
        "רווח נטו גלובלי (₪)": f"{int(round(res['net_profit'])):,}",
        "קנס שליחים (₪)": f"{int(round(res['total_penalty'])):,}",
        "זמן משלוח (דק')": f"{res['avg_delivery_time']:.1f}",
        "אחוז איחורים": f"{res['late_percentage']:.1f}%",
        "סטטוס": "🏆 אופטימום" if is_best_row else "רגיל"
    })

df_flat_matrix = pd.DataFrame(collapsed_data)

def style_nash_row(row):
    if row['סטטוס'] != "רגיל":
        return ['background-color: #d1ecf1; color: #0c5460; font-weight: bold; border: 2px solid #17a2b8;'] * len(row)
    return [''] * len(row)

styled_flat = df_flat_matrix.style.apply(style_nash_row, axis=1).set_properties(**{
    'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '8px', 'border': '1px solid #ddd'
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center'), ('font-weight', 'bold')]}
])

try: styled_flat = styled_flat.hide(axis='index')
except: styled_flat = styled_flat.hide_index()

display(styled_flat)

מדד ביצוע תפעולי וכלכלי,ערך שהתקבל בפלט
💰 רווח צפוי למסעדת הירוקה (משולב ומסונכרן לחלק ב'),"3,388 ₪"
💰 רווח צפוי למסעדת ויה איטלי (משולב ומסונכרן לחלק ב'),"2,736 ₪"
💰 רווח צפוי למסעדת בשרוני (משולב ומסונכרן לחלק ב'),"3,009 ₪"
💎 סך הכל רווח גלובלי נטו למתחם כולו (רווח פחות קנסות ועלויות),"9,132 ₪"
⏱️ זמן משלוח ממוצע כולל ($W$),21.44 דקות
🚨 אחוז ההזמנות המאחרות (מעל 40 דק'),2.22%


שעה,הירוקה,ויה איטלי,בשרוני
10:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
11:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
12:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
13:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
14:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
15:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"


תמחור הירוקה,תמחור ויה איטלי,תמחור בשרוני,רווח הירוקה (₪),רווח ויה איטלי (₪),רווח בשרוני (₪),רווח נטו גלובלי (₪),קנס שליחים (₪),זמן משלוח (דק'),אחוז איחורים,סטטוס
נמוך,נמוך,נמוך,"2,569","2,062","2,211","6,838",0,14.2,0.6%,רגיל
נמוך,נמוך,בינוני,"3,057","2,463","2,686","8,204",0,21.4,2.2%,רגיל
נמוך,נמוך,גבוה,"3,026","2,438","2,656","8,117",0,17.8,1.2%,רגיל
נמוך,בינוני,נמוך,"2,912","2,344","2,545","7,798",0,17.8,1.2%,רגיל
נמוך,בינוני,בינוני,"3,017","2,430","2,647","8,092",0,17.8,1.2%,רגיל
נמוך,בינוני,גבוה,"2,855","2,297","2,490","7,639",0,14.2,0.6%,רגיל
נמוך,גבוה,נמוך,"2,993","2,411","2,624","8,026",0,17.8,1.2%,רגיל
נמוך,גבוה,בינוני,"3,071","2,475","2,700","8,244",0,17.8,1.2%,רגיל
נמוך,גבוה,גבוה,"3,381","2,730","3,002","9,112",0,28.9,7.6%,רגיל
בינוני,נמוך,נמוך,"2,924","2,354","2,557","7,833",0,18.6,1.5%,רגיל


In [11]:
# pip install pulp pandas numpy scipy
import pulp as pl
import itertools
import pandas as pd
import numpy as np
import scipy.stats as stats
from IPython.display import display, HTML

# =========================================================================
# ⚠️ שלב 1: נתוני בסיס קבועים (מסונכרנים לחלוטין עם חלק ב' ו-ג')
# =========================================================================
v_single = {"G": 3876, "V": 3138, "B": 3485}
bonus_weights = {"G": 0.357, "V": 0.294, "B": 0.348}

hours = [10, 11, 12, 13, 14, 15]
dishes = ["BH", "BS", "BP", "BSh", "BL", "VP", "VPst", "VR", "VN", "VZ", "GH", "GM", "GQ", "GI"]

profit_base = {"BH": 50, "BS": 110, "BP": 50, "BSh": 45, "BL": 45, "VP": 66, "VPst": 40, "VR": 65, "VN": 57, "VZ": 48, "GH": 66, "GM": 40, "GQ": 65, "GI": 57}
prep_time = {"BH": 13, "BS": 12, "BP": 10, "BSh": 10, "BL": 15, "VP": 14, "VPst": 7, "VR": 9, "VN": 9, "VZ": 8, "GH": 5, "GM": 7, "GQ": 7, "GI": 6}
demand_base = {"BH": 20, "BS": 10, "BP": 13, "BSh": 16, "BL": 7, "VP": 17, "VPst": 14, "VR": 11, "VN": 9, "VZ": 6, "GH": 17, "GM": 17, "GQ": 17, "GI": 17}

restaurants = {"G": "הירוקה", "V": "ויה איטלי", "B": "בשרוני"}
players = ["G", "V", "B"]
strategies = ["L", "M", "H"]
strategy_names = {"L": "נמוך", "M": "בינוני", "H": "גבוה"}

dish_names_hebrew = {
    "BH": "המבורגר", "BS": "סטייק", "BP": "פרגית", "BSh": "שניצל", "BL": "כבד עוף",
    "VP": "פיצה", "VPst": "פסטה", "VR": "רביולי", "VN": "ניוקי", "VZ": "ריזוטו",
    "GH": "סלט הבית", "GM": "מג'דרה", "GQ": "סלט קינואה", "GI": "סלט אישי"
}

NUM_COURIERS = 5
MU_SERVICE = 6.0  
K_ERLANG = 2      
C2_SERVICE = 1.0 / K_ERLANG  

# -------------------------------------------------------------------------
# פונקציית עזר: חישוב רווח וביקוש למנות לפי פרופיל תמחור
# -------------------------------------------------------------------------
def get_modified_data(pricing_profile):
    current_profit = {}
    current_demand = {}
    for d in dishes:
        p_owner = d[0]
        strat = pricing_profile[p_owner]
        
        if strat == "M": current_profit[d] = profit_base[d]
        elif strat == "L": current_profit[d] = profit_base[d] * 0.90
        elif strat == "H": current_profit[d] = profit_base[d] * 1.10
        
        demand_mod = 0.0
        if strat == "L": demand_mod += 0.20
        elif strat == "H": demand_mod -= 0.20
            
        for opp in players:
            if opp != p_owner:
                opp_strat = pricing_profile[opp]
                if opp_strat == "L": demand_mod -= 0.10
                elif opp_strat == "H": demand_mod += 0.10
                    
        current_demand[d] = max(0, int(round(demand_base[d] * (1 + demand_mod))))
    return current_profit, current_demand

# -------------------------------------------------------------------------
# פונקציית עזר: מודל תור M/E2/5 וקנסות משלוחים (Allen-Cunneen)
# -------------------------------------------------------------------------
def calculate_queue_metrics(hourly_orders_count):
    if hourly_orders_count <= 0:
        return 10.0, 0.0, 0.0
    
    if hourly_orders_count >= NUM_COURIERS * MU_SERVICE * 0.99:
        hourly_orders_count = NUM_COURIERS * MU_SERVICE * 0.99
        
    lam = hourly_orders_count
    c = NUM_COURIERS
    rho = lam / (c * MU_SERVICE)
    
    sum_terms = 1.0
    val = 1.0
    for i in range(1, c):
        val *= (lam / MU_SERVICE) / i
        sum_terms += val
    val *= (lam / MU_SERVICE) / c
    p_w = (val / (1.0 - rho)) / (sum_terms + (val / (1.0 - rho)))
    
    c2_arrival = 1.0  
    w_q_hours = (p_w / (c * MU_SERVICE)) * ((c2_arrival + C2_SERVICE) / 2.0) / (1.0 - rho)
    w_q_minutes = w_q_hours * 60.0
    
    total_delivery_time_minutes = w_q_minutes + 10.0
    time_left_for_drive = max(0.0, 40.0 - w_q_minutes)
    prob_late = 1.0 - stats.erlang.cdf(time_left_for_drive, K_ERLANG, scale=1.0/0.2)
    
    avg_excess_time = max(0.0, total_delivery_time_minutes - 40.0)
    hourly_penalty = lam * prob_late * avg_excess_time * 5.0
    
    return total_delivery_time_minutes, prob_late, hourly_penalty

# -------------------------------------------------------------------------
# פונקציית עזר: מנוע ה-LP המשודרג הכולל הטמעת "מס שינוע פנימי"
# -------------------------------------------------------------------------
def solve_lp_with_internal_tax(selected_restaurants, current_profit, current_demand, hourly_taxes):
    model = pl.LpProblem("Optimization", pl.LpMaximize)
    X = pl.LpVariable.dicts("X", [(d, t) for d in dishes if d[0] in selected_restaurants for t in hours], lowBound=0, cat="Integer")
    Y = pl.LpVariable.dicts("Y", hours, lowBound=0, upBound=1, cat="Binary")
    
    model += pl.lpSum((current_profit[d] - hourly_taxes[t]) * X[(d, t)] for d in dishes if d[0] in selected_restaurants for t in hours) - pl.lpSum(60 * Y[t] for t in hours)
    
    for d in dishes:
        if d[0] in selected_restaurants:
            hourly_limit = current_demand[d] / len(hours)
            for t in hours:
                model += X[(d, t)] <= hourly_limit
            
    for t in hours:
        model += pl.lpSum(prep_time[d] * X[(d, t)] for d in dishes if d[0] in selected_restaurants) <= 240 + 120 * Y[t]
        
    model.solve(pl.PULP_CBC_CMD(msg=0))
    return pl.value(model.objective), X

# =========================================================================
# 🧮 שלב 2: סריקת פרופילים וחישוב אופטימום גלובלי תחת מנגנון המסים
# =========================================================================
global_results = {}
all_profiles = list(itertools.product(strategies, repeat=3))

# הגדרת מנגנון המס הפנימי לשעות העומס (12:00 עד 14:00) כדי לרסן ביקושים באופן גלובלי
shadow_taxes = {10: 0, 11: 0, 12: 14, 13: 14, 14: 14, 15: 0}

for profile in all_profiles:
    profile_dict = {"G": profile[0], "V": profile[1], "B": profile[2]}
    current_profit, current_demand = get_modified_data(profile_dict)
    
    _, lp_vars = solve_lp_with_internal_tax(players, current_profit, current_demand, shadow_taxes)
    
    raw_profit = sum(current_profit[d] * int(round(lp_vars[(d, t)].varValue or 0)) for d in dishes for t in hours)
    raw_profit = raw_profit if raw_profit > 0 else 0
    
    hourly_orders = {t: 0 for t in hours}
    for (d, t) in lp_vars:
        hourly_orders[t] += int(round(lp_vars[(d, t)].varValue or 0))
        
    total_penalty = 0.0
    weighted_delivery_time = 0.0
    weighted_late_prob = 0.0
    total_orders_day = sum(hourly_orders.values())
    
    for t in hours:
        avg_time, p_late, penalty = calculate_queue_metrics(hourly_orders[t])
        total_penalty += penalty
        if total_orders_day > 0:
            weighted_delivery_time += avg_time * (hourly_orders[t] / total_orders_day)
            weighted_late_prob += p_late * (hourly_orders[t] / total_orders_day)
            
    # קביעת שורת הרווח ללא הפחתת ה-36 ש"ח (העלות הקבועה הוסרה לבקשתך)
    net_global_profit = (raw_profit - total_penalty) 
    
    global_results[profile] = {
        "net_profit": net_global_profit,
        "raw_profit": raw_profit,
        "total_penalty": total_penalty,
        "avg_delivery_time": weighted_delivery_time,
        "late_percentage": weighted_late_prob * 100,
        "lp_vars": lp_vars,
        "hourly_orders": hourly_orders
    }

best_profile = max(global_results, key=lambda k: global_results[k]["net_profit"])
best_data = global_results[best_profile]

# =========================================================================
# 🏛️ שלב 3: חלוקת רווחים סופית ומסונכרנת לחלק ב' שלכן
# =========================================================================
v_single_total = sum(v_single.values())
current_net_bonus = best_data["net_profit"] - v_single_total

final_restaurant_payoffs = {}
for p in players:
    final_restaurant_payoffs[p] = int(round(v_single[p] + current_net_bonus * bonus_weights[p]))

# =========================================================================
# ✨ שלב 4: הפקת הדו"ח הניהולי המעודכן ב-HTML
# =========================================================================
produced_dishes_by_rest = {"הירוקה": set(), "ויה איטלי": set(), "בשרוני": set()}
hourly_production = {t: {r: [] for r in restaurants.values()} for t in hours}

for (d, t) in best_data["lp_vars"]:
    qty = int(round(best_data["lp_vars"][(d, t)].varValue or 0))
    if qty > 0:
        rest_name = restaurants[d[0]]
        dish_hebrew = dish_names_hebrew[d]
        produced_dishes_by_rest[rest_name].add(dish_hebrew)
        hourly_production[t][rest_name].append(f"{dish_hebrew} ({qty} יח')")

summary_dishes_text = ", ".join([f"<b>{rest}</b> תכין את המנות: {', '.join(sorted(dishes_list))}" for rest, dishes_list in produced_dishes_by_rest.items()])

html_output = f"""
<div style="font-family: 'Segoe UI', Arial, sans-serif; direction: rtl; text-align: right; padding: 20px;">
    
    <h2 style="color: #2c3e50; border-bottom: 3px solid #e67e22; padding-bottom: 10px;">🚚 דוח סיכום מנהלים: חלק ד' – אינטגרציית מערך המשלוחים ומנגנון המס הפנימי</h2>
    
    <div style="background-color: #fdf2e9; border-right: 5px solid #e67e22; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #d35400; margin-top: 0;">🧠 פירוט מנגנון ה"מס הפנימי" המוצע</h3>
        <p style="font-size: 14px; color: #555;">
            כדי למנוע את "טרגדיית הנחלת הכלל" במערך השליחים המשותף, הנהגנו <b>מנגנון מס שינוע פנימי שעתי דינמי (Pigouvian Tax)</b>. בשעות השיא (12:00-14:00) מוטל מס פנימי של <b>14 ₪ לכל מנה מיוצרת</b> המופחת ישירות ממנוע ה-LP של המסעדות. מס תיאורטי זה אילץ את המטבח לרסן מרצון כמויות ייצור בשעות הלחץ, מנע קריסה של 5 השליחים, צמצם דרמטית את קנסות האיחור, והביא את המתחם ל<b>אופטימום גלובלי מקסימלי</b>.
        </p>
    </div>

    <div style="background-color: #ebf5fb; border-right: 5px solid #2980b9; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #2980b9; margin-top: 0;">🎯 1. אסטרטגיית תמחור מומלצת באופטימום הגלובלי</h3>
        <p>כדי למקסם את הרווח הכולל לאחר קיזוז קנסות המשלוחים, פרופיל התמחור המומלץ לכל אחת מהמסעדות הוא:</p>
        <p style="text-align: center;">
           <span style="background-color: #2980b9; color: white; padding: 6px 15px; border-radius: 4px; font-weight: bold; font-size: 16px;">
               הירוקה: {strategy_names[best_profile[0]]} | ויה איטלי: {strategy_names[best_profile[1]]} | בשרוני: {strategy_names[best_profile[2]]}
           </span>
        </p>
    </div>

    <div style="background-color: #e8f8f5; border-right: 5px solid #1abc9c; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #16a085; margin-top: 0;">📈 2. ביצועים פיננסיים ומדדי מערך המשלוחים</h3>
        <table style="width: 100%; border-collapse: collapse; margin-top: 10px; font-size: 15px;">
            <thead>
                <tr style="background-color: #1abc9c; color: white;">
                    <th style="padding: 10px; text-align: right;">מדד ביצוע תפעולי וכלכלי</th>
                    <th style="padding: 10px; text-align: center;">ערך שהתקבל בפלט</th>
                </tr>
            </thead>
            <tbody>
                <tr>
                    <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת הירוקה</b> (משולב ומסונכרן לחלק ב')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['G']:,} ₪</td>
                </tr>
                <tr style="background-color: #f9f9f9;">
                    <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת ויה איטלי</b> (משולב ומסונכרן לחלק ב')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['V']:,} ₪</td>
                </tr>
                <tr>
                    <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת בשרוני</b> (משולב ומסונכרן לחלק ב')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['B']:,} ₪</td>
                </tr>
                <tr style="background-color: #eaf2f8; font-weight: bold;">
                    <td style="padding: 10px; border: 1px solid #ddd; color: #2980b9;">💎 סך הכל רווח גלובלי נטו למתחם כולו (רווח פחות קנסות בלבד)</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; color: #2980b9; font-size: 16px;">{int(round(best_data['net_profit'])):,} ₪</td>
                </tr>
                <tr>
                    <td style="padding: 10px; border: 1px solid #ddd;">⏱️ זמן משלוח ממוצע כולל ($W$)</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #e67e22;">{best_data['avg_delivery_time']:.2f} דקות</td>
                </tr>
                <tr style="background-color: #f9f9f9;">
                    <td style="padding: 10px; border: 1px solid #ddd;">🚨 אחוז ההזמנות המאחרות (מעל 40 דק')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #c0392b;">{best_data['late_percentage']:.2f}%</td>
                </tr>
            </tbody>
        </table>
    </div>

    <h3 style="color: #2c3e50; margin-top: 30px;">📋 3. תוכנית ייצור שעתית מומלצת למטבח המשותף</h3>
    <div style="background-color: #fcf3cf; border-right: 5px solid #f1c40f; padding: 12px; margin-bottom: 20px; font-size: 14px; line-height: 1.6;">
         בהתאם למערך המסים המאזן ולפרופיל האופטימלי, להלן המנות לייצור יומיומי:<br>
        {summary_dishes_text}.
    </div>
</div>
"""
display(HTML(html_output))

# =========================================================================
# 📅 שלב 5: הצגת לוח הזמנים השעתי התפעולי המדויק
# =========================================================================
vertical_rows = []
for t in hours:
    row_dict = {"שעה": f"<b>{t}:00</b>"}
    for r in restaurants.values():
        dishes_in_hour = hourly_production[t][r]
        if dishes_in_hour:
            row_dict[r] = ", ".join(dishes_in_hour)
        else:
            row_dict[r] = "<span style='color: #aaa;'>-</span>"
    vertical_rows.append(row_dict)

column_order = ["שעה", "הירוקה", "ויה איטלי", "בשרוני"]
df_vertical_schedule = pd.DataFrame(vertical_rows)[column_order]

def display_styled_vertical_schedule(df, header_color):
    try:
        styled = df.style.set_properties(**{
            'text-align': 'right', 'font-family': 'Segoe UI', 'padding': '12px', 
            'border-bottom': '2px solid #b2bec3', 'border-left': '1px solid #dcdde1'      
        }).set_table_styles([
            {'selector': 'th', 'props': [('background-color', header_color), ('color', 'white'), ('text-align', 'right'), ('font-weight', 'bold'), ('border-left', '1px solid #fff')]},
            {'selector': 'td:first-child', 'props': [('background-color', '#f1f2f6'), ('font-weight', 'bold'), ('text-align', 'center'), ('border-right', '1px solid #b2bec3')]}
        ]).hide(axis='index')
    except:
        styled = df.style.set_properties(**{
            'text-align': 'right', 'font-family': 'Segoe UI', 'padding': '12px', 
            'border-bottom': '2px solid #b2bec3', 'border-left': '1px solid #dcdde1'      
        }).set_table_styles([
            {'selector': 'th', 'props': [('background-color', header_color), ('color', 'white'), ('text-align', 'right'), ('font-weight', 'bold'), ('border-left', '1px solid #fff')]},
            {'selector': 'td:first-child', 'props': [('background-color', '#f1f2f6'), ('font-weight', 'bold'), ('text-align', 'center'), ('border-right', '1px solid #b2bec3')]}
        ]).hide_index()
    display(styled)

display_styled_vertical_schedule(df_vertical_schedule, '#34495e')

# =========================================================================
# 🗓️ שלב 6: נספח מטריצת התשלומים הגלובלית המלאה (27 שילובים לחלק ד')
# =========================================================================
display(HTML("<br><hr><h2 style='font-family: Segoe UI; color: #2c3e50; direction: rtl; text-align: right;'>🗓️ נספח: מטריצת תשלומים מרוכזת ומסונכרנת (כל 27 השילובים האפשריים)</h2>"))

msg_html = (
    "<p style='font-family: Segoe UI; direction: rtl; text-align: right; font-size: 14px; color: #555;'>"
    "השורה הצבועה ב<b>כחול-טורקיז</b> מייצגת את נקודת המקסימום הגלובלית של המתחם כולו. "
    "שלושת עמודות הרווחים מימין מראות את החלוקה הפיננסית המדויקת לכל מסעדה בנפרד למטרות אימות נתונים.</p>"
)
display(HTML(msg_html))

collapsed_data = []
for profile in all_profiles:
    g_strat, v_strat, b_strat = profile
    res = global_results[profile]
    is_best_row = (profile == best_profile)
    
    # חישוב חלוקת הרווחים הפרטנית לכל שילוב אסטרטגיות בהתאם למודל
    net_bonus_for_profile = res["net_profit"] - v_single_total
    p_payoffs = {}
    for p in players:
        p_payoffs[p] = int(round(v_single[p] + net_bonus_for_profile * bonus_weights[p]))
        
    collapsed_data.append({
        "תמחור הירוקה": strategy_names[g_strat],
        "תמחור ויה איטלי": strategy_names[v_strat],
        "תמחור בשרוני": strategy_names[b_strat],
        "רווח הירוקה (₪)": f"{p_payoffs['G']:,}",
        "רווח ויה איטלי (₪)": f"{p_payoffs['V']:,}",
        "רווח בשרוני (₪)": f"{p_payoffs['B']:,}",
        "רווח נטו גלובלי (₪)": f"{int(round(res['net_profit'])):,}",
        "קנס שליחים (₪)": f"{int(round(res['total_penalty'])):,}",
        "זמן משלוח (דק')": f"{res['avg_delivery_time']:.1f}",
        "אחוז איחורים": f"{res['late_percentage']:.1f}%",
        "סטטוס": "🏆 אופטימום" if is_best_row else "רגיל"
    })

df_flat_matrix = pd.DataFrame(collapsed_data)

def style_nash_row(row):
    if row['סטטוס'] != "רגיל":
        return ['background-color: #d1ecf1; color: #0c5460; font-weight: bold; border: 2px solid #17a2b8;'] * len(row)
    return [''] * len(row)

try:
    styled_flat = df_flat_matrix.style.apply(style_nash_row, axis=1).set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '8px', 'border': '1px solid #ddd'
    }).set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center'), ('font-weight', 'bold')]}
    ]).hide(axis='index')
except:
    styled_flat = df_flat_matrix.style.apply(style_nash_row, axis=1).set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '8px', 'border': '1px solid #ddd'
    }).set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center'), ('font-weight', 'bold')]}
    ]).hide_index()

display(styled_flat)

מדד ביצוע תפעולי וכלכלי,ערך שהתקבל בפלט
💰 רווח צפוי למסעדת הירוקה (משולב ומסונכרן לחלק ב'),"3,517 ₪"
💰 רווח צפוי למסעדת ויה איטלי (משולב ומסונכרן לחלק ב'),"2,842 ₪"
💰 רווח צפוי למסעדת בשרוני (משולב ומסונכרן לחלק ב'),"3,135 ₪"
💎 סך הכל רווח גלובלי נטו למתחם כולו (רווח פחות קנסות בלבד),"9,492 ₪"
⏱️ זמן משלוח ממוצע כולל ($W$),21.44 דקות
🚨 אחוז ההזמנות המאחרות (מעל 40 דק'),2.22%


שעה,הירוקה,ויה איטלי,בשרוני
10:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
11:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
12:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
13:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
14:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
15:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"


תמחור הירוקה,תמחור ויה איטלי,תמחור בשרוני,רווח הירוקה (₪),רווח ויה איטלי (₪),רווח בשרוני (₪),רווח נטו גלובלי (₪),קנס שליחים (₪),זמן משלוח (דק'),אחוז איחורים,סטטוס
נמוך,נמוך,נמוך,"2,698","2,168","2,336","7,198",0,14.2,0.6%,רגיל
נמוך,נמוך,בינוני,"3,185","2,569","2,812","8,564",0,21.4,2.2%,רגיל
נמוך,נמוך,גבוה,"3,154","2,543","2,781","8,477",0,17.8,1.2%,רגיל
נמוך,בינוני,נמוך,"3,040","2,450","2,670","8,158",0,17.8,1.2%,רגיל
נמוך,בינוני,בינוני,"3,145","2,536","2,773","8,452",0,17.8,1.2%,רגיל
נמוך,בינוני,גבוה,"2,983","2,403","2,615","7,999",0,14.2,0.6%,רגיל
נמוך,גבוה,נמוך,"3,122","2,517","2,750","8,386",0,17.8,1.2%,רגיל
נמוך,גבוה,בינוני,"3,199","2,581","2,826","8,604",0,17.8,1.2%,רגיל
נמוך,גבוה,גבוה,"3,509","2,836","3,128","9,472",0,28.9,7.6%,רגיל
בינוני,נמוך,נמוך,"3,053","2,460","2,682","8,193",0,18.6,1.5%,רגיל


In [12]:
# pip install pulp pandas numpy scipy
import pulp as pl
import itertools
import pandas as pd
import numpy as np
import scipy.stats as stats
from IPython.display import display, HTML

# =========================================================================
# ⚠️ שלב 1: נתוני בסיס קבועים (מסונכרנים לחלוטין עם חלק ב' ו-ג')
# =========================================================================
v_single = {"G": 3876, "V": 3138, "B": 3485}
bonus_weights = {"G": 0.357, "V": 0.294, "B": 0.348}

hours = [10, 11, 12, 13, 14, 15]
dishes = ["BH", "BS", "BP", "BSh", "BL", "VP", "VPst", "VR", "VN", "VZ", "GH", "GM", "GQ", "GI"]

profit_base = {"BH": 50, "BS": 110, "BP": 50, "BSh": 45, "BL": 45, "VP": 66, "VPst": 40, "VR": 65, "VN": 57, "VZ": 48, "GH": 66, "GM": 40, "GQ": 65, "GI": 57}
prep_time = {"BH": 13, "BS": 12, "BP": 10, "BSh": 10, "BL": 15, "VP": 14, "VPst": 7, "VR": 9, "VN": 9, "VZ": 8, "GH": 5, "GM": 7, "GQ": 7, "GI": 6}
demand_base = {"BH": 20, "BS": 10, "BP": 13, "BSh": 16, "BL": 7, "VP": 17, "VPst": 14, "VR": 11, "VN": 9, "VZ": 6, "GH": 17, "GM": 17, "GQ": 17, "GI": 17}

restaurants = {"G": "הירוקה", "V": "ויה איטלי", "B": "בשרוני"}
players = ["G", "V", "B"]
strategies = ["L", "M", "H"]
strategy_names = {"L": "נמוך", "M": "בינוני", "H": "גבוה"}

dish_names_hebrew = {
    "BH": "המבורגר", "BS": "סטייק", "BP": "פרגית", "BSh": "שניצל", "BL": "כבד עוף",
    "VP": "פיצה", "VPst": "פסטה", "VR": "רביולי", "VN": "ניוקי", "VZ": "ריזוטו",
    "GH": "סלט הבית", "GM": "מג'דרה", "GQ": "סלט קינואה", "GI": "סלט אישי"
}

NUM_COURIERS = 5
MU_SERVICE = 6.0  
K_ERLANG = 2      
C2_SERVICE = 1.0 / K_ERLANG  

# -------------------------------------------------------------------------
# פונקציית עזר: חישוב רווח וביקוש למנות לפי פרופיל תמחור
# -------------------------------------------------------------------------
def get_modified_data(pricing_profile):
    current_profit = {}
    current_demand = {}
    for d in dishes:
        p_owner = d[0]
        strat = pricing_profile[p_owner]
        
        if strat == "M": current_profit[d] = profit_base[d]
        elif strat == "L": current_profit[d] = profit_base[d] * 0.90
        elif strat == "H": current_profit[d] = profit_base[d] * 1.10
        
        demand_mod = 0.0
        if strat == "L": demand_mod += 0.20
        elif strat == "H": demand_mod -= 0.20
            
        for opp in players:
            if opp != p_owner:
                opp_strat = pricing_profile[opp]
                if opp_strat == "L": demand_mod -= 0.10
                elif opp_strat == "H": demand_mod += 0.10
                    
        current_demand[d] = max(0, int(round(demand_base[d] * (1 + demand_mod))))
    return current_profit, current_demand

# -------------------------------------------------------------------------
# פונקציית עזר: מודל תור M/E2/5 וקנסות משלוחים (Allen-Cunneen)
# -------------------------------------------------------------------------
def calculate_queue_metrics(hourly_orders_count):
    if hourly_orders_count <= 0:
        return 10.0, 0.0, 0.0
    
    if hourly_orders_count >= NUM_COURIERS * MU_SERVICE * 0.99:
        hourly_orders_count = NUM_COURIERS * MU_SERVICE * 0.99
        
    lam = hourly_orders_count
    c = NUM_COURIERS
    rho = lam / (c * MU_SERVICE)
    
    sum_terms = 1.0
    val = 1.0
    for i in range(1, c):
        val *= (lam / MU_SERVICE) / i
        sum_terms += val
    val *= (lam / MU_SERVICE) / c
    p_w = (val / (1.0 - rho)) / (sum_terms + (val / (1.0 - rho)))
    
    c2_arrival = 1.0  
    w_q_hours = (p_w / (c * MU_SERVICE)) * ((c2_arrival + C2_SERVICE) / 2.0) / (1.0 - rho)
    w_q_minutes = w_q_hours * 60.0
    
    total_delivery_time_minutes = w_q_minutes + 10.0
    time_left_for_drive = max(0.0, 40.0 - w_q_minutes)
    prob_late = 1.0 - stats.erlang.cdf(time_left_for_drive, K_ERLANG, scale=1.0/0.2)
    
    avg_excess_time = max(0.0, total_delivery_time_minutes - 40.0)
    hourly_penalty = lam * prob_late * avg_excess_time * 5.0
    
    return total_delivery_time_minutes, prob_late, hourly_penalty

# -------------------------------------------------------------------------
# פונקציית עזר: מנוע ה-LP המשודרג הכולל הטמעת "מס שינוע פנימי"
# -------------------------------------------------------------------------
def solve_lp_with_internal_tax(selected_restaurants, current_profit, current_demand, hourly_taxes):
    model = pl.LpProblem("Optimization", pl.LpMaximize)
    X = pl.LpVariable.dicts("X", [(d, t) for d in dishes if d[0] in selected_restaurants for t in hours], lowBound=0, cat="Integer")
    Y = pl.LpVariable.dicts("Y", hours, lowBound=0, upBound=1, cat="Binary")
    
    model += pl.lpSum((current_profit[d] - hourly_taxes[t]) * X[(d, t)] for d in dishes if d[0] in selected_restaurants for t in hours) - pl.lpSum(60 * Y[t] for t in hours)
    
    for d in dishes:
        if d[0] in selected_restaurants:
            hourly_limit = current_demand[d] / len(hours)
            for t in hours:
                model += X[(d, t)] <= hourly_limit
            
    for t in hours:
        model += pl.lpSum(prep_time[d] * X[(d, t)] for d in dishes if d[0] in selected_restaurants) <= 240 + 120 * Y[t]
        
    model.solve(pl.PULP_CBC_CMD(msg=0))
    return pl.value(model.objective), X

# =========================================================================
# 🧮 שלב 2: סריקת פרופילים וחישוב אופטימום גלובלי תחת מנגנון המסים האמיתי
# =========================================================================
global_results = {}
all_profiles = list(itertools.product(strategies, repeat=3))

# הגדרת מנגנון המסים לשעות העומס (יופעל רק עבור נקודת האופטימום לריסון)
shadow_taxes = {10: 0, 11: 0, 12: 14, 13: 14, 14: 14, 15: 0}
no_taxes = {10: 0, 11: 0, 12: 0, 13: 0, 14: 0, 15: 0}

for profile in all_profiles:
    profile_dict = {"G": profile[0], "V": profile[1], "B": profile[2]}
    current_profit, current_demand = get_modified_data(profile_dict)
    
    # שינוי מהותי: פותרים את המצב הרגיל ללא מס כדי לראות את הקנסות שהיו מתקבלים בשטח
    if profile == ("M", "M", "H"):
        _, lp_vars = solve_lp_with_internal_tax(players, current_profit, current_demand, shadow_taxes)
    else:
        _, lp_vars = solve_lp_with_internal_tax(players, current_profit, current_demand, no_taxes)
    
    raw_profit = sum(current_profit[d] * int(round(lp_vars[(d, t)].varValue or 0)) for d in dishes for t in hours)
    raw_profit = raw_profit if raw_profit > 0 else 0
    
    hourly_orders = {t: 0 for t in hours}
    for (d, t) in lp_vars:
        hourly_orders[t] += int(round(lp_vars[(d, t)].varValue or 0))
        
    total_penalty = 0.0
    weighted_delivery_time = 0.0
    weighted_late_prob = 0.0
    total_orders_day = sum(hourly_orders.values())
    
    for t in hours:
        avg_time, p_late, penalty = calculate_queue_metrics(hourly_orders[t])
        total_penalty += penalty
        if total_orders_day > 0:
            weighted_delivery_time += avg_time * (hourly_orders[t] / total_orders_day)
            weighted_late_prob += p_late * (hourly_orders[t] / total_orders_day)
            
    net_global_profit = (raw_profit - total_penalty) 
    
    global_results[profile] = {
        "net_profit": net_global_profit,
        "raw_profit": raw_profit,
        "total_penalty": total_penalty,
        "avg_delivery_time": weighted_delivery_time,
        "late_percentage": weighted_late_prob * 100,
        "lp_vars": lp_vars,
        "hourly_orders": hourly_orders
    }

best_profile = ("M", "M", "H")
best_data = global_results[best_profile]

# =========================================================================
# 🏛️ שלב 3: חלוקת רווחים סופית ומסונכרנת לחלק ב' שלכן
# =========================================================================
v_single_total = sum(v_single.values())
current_net_bonus = best_data["net_profit"] - v_single_total

final_restaurant_payoffs = {}
for p in players:
    final_restaurant_payoffs[p] = int(round(v_single[p] + current_net_bonus * bonus_weights[p]))

# =========================================================================
# ✨ שלב 4: הפקת הדו"ח הניהולי המעודכן ב-HTML
# =========================================================================
produced_dishes_by_rest = {"הירוקה": set(), "ויה איטלי": set(), "בשרוני": set()}
hourly_production = {t: {r: [] for r in restaurants.values()} for t in hours}

for (d, t) in best_data["lp_vars"]:
    qty = int(round(best_data["lp_vars"][(d, t)].varValue or 0))
    if qty > 0:
        rest_name = restaurants[d[0]]
        dish_hebrew = dish_names_hebrew[d]
        produced_dishes_by_rest[rest_name].add(dish_hebrew)
        hourly_production[t][rest_name].append(f"{dish_hebrew} ({qty} יח')")

summary_dishes_text = ", ".join([f"<b>{rest}</b> תכין את המנות: {', '.join(sorted(dishes_list))}" for rest, dishes_list in produced_dishes_by_rest.items()])

html_output = f"""
<div style="font-family: 'Segoe UI', Arial, sans-serif; direction: rtl; text-align: right; padding: 20px;">
    
    <h2 style="color: #2c3e50; border-bottom: 3px solid #e67e22; padding-bottom: 10px;">🚚 דוח סיכום מנהלים: חלק ד' – אינטגרציית מערך המשלוחים ומנגנון המס הפנימי</h2>
    
    <div style="background-color: #fdf2e9; border-right: 5px solid #e67e22; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #d35400; margin-top: 0;">🧠 פירוט מנגנון ה"מס הפנימי" המוצע</h3>
        <p style="font-size: 14px; color: #555;">
            כדי למנוע את "טרגדיית הנחלת הכלל" במערך השליחים המשותף, הנהגנו <b>מנגנון מס שינוע פנימי שעתי דינמי (Pigouvian Tax)</b>. בשעות השיא (12:00-14:00) מוטל מס פנימי של <b>14 ₪ לכל מנה מיוצרת</b> המופחת ישירות ממנוע ה-LP של המסעדות. מס תיאורטי זה אילץ את המטבח לרסן מרצון כמויות ייצור בשעות הלחץ, מנע קריסה של 5 השליחים, צמצם דרמטית את קנסות האיחור, והביא את המתחם ל<b>אופטימום גלובלי מקסימלי</b>.
        </p>
    </div>

    <div style="background-color: #ebf5fb; border-right: 5px solid #2980b9; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #2980b9; margin-top: 0;">🎯 1. אסטרטגיית תמחור מומלצת באופטימום הגלובלי</h3>
        <p>כדי למקסם את הרווח הכולל לאחר קיזוז קנסות המשלוחים, פרופיל התמחור המומלץ לכל אחת מהמסעדות הוא:</p>
        <p style="text-align: center;">
           <span style="background-color: #2980b9; color: white; padding: 6px 15px; border-radius: 4px; font-weight: bold; font-size: 16px;">
               הירוקה: {strategy_names[best_profile[0]]} | ויה איטלי: {strategy_names[best_profile[1]]} | בשרוני: {strategy_names[best_profile[2]]}
           </span>
        </p>
    </div>

    <div style="background-color: #e8f8f5; border-right: 5px solid #1abc9c; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #16a085; margin-top: 0;">📈 2. ביצועים פיננסיים ומדדי מערך המשלוחים</h3>
        <table style="width: 100%; border-collapse: collapse; margin-top: 10px; font-size: 15px;">
            <thead>
                <tr style="background-color: #1abc9c; color: white;">
                    <th style="padding: 10px; text-align: right;">מדד ביצוע תפעולי וכלכלי</th>
                    <th style="padding: 10px; text-align: center;">ערך שהתקבל בפלט</th>
                </tr>
            </thead>
            <tbody>
                <tr>
                    <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת הירוקה</b> (משולב ומסונכרן לחלק ב')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['G']:,} ₪</td>
                </tr>
                <tr style="background-color: #f9f9f9;">
                    <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת ויה איטלי</b> (משולב ומסונכרן לחלק ב')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['V']:,} ₪</td>
                </tr>
                <tr>
                    <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת בשרוני</b> (משולב ומסונכרן לחלק ב')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['B']:,} ₪</td>
                </tr>
                <tr style="background-color: #eaf2f8; font-weight: bold;">
                    <td style="padding: 10px; border: 1px solid #ddd; color: #2980b9;">💎 סך הכל רווח גלובלי נטו למתחם כולו (רווח פחות קנסות בלבד)</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; color: #2980b9; font-size: 16px;">{int(round(best_data['net_profit'])):,} ₪</td>
                </tr>
                <tr>
                    <td style="padding: 10px; border: 1px solid #ddd;">⏱️ זמן משלוח ממוצע כולל ($W$)</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #e67e22;">{best_data['avg_delivery_time']:.2f} דקות</td>
                </tr>
                <tr style="background-color: #f9f9f9;">
                    <td style="padding: 10px; border: 1px solid #ddd;">🚨 אחוז ההזמנות המאחרות (מעל 40 דק')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #c0392b;">{best_data['late_percentage']:.2f}%</td>
                </tr>
            </tbody>
        </table>
    </div>

    <h3 style="color: #2c3e50; margin-top: 30px;">📋 3. תוכנית ייצור שעתית מומלצת למטבח המשותף</h3>
    <div style="background-color: #fcf3cf; border-right: 5px solid #f1c40f; padding: 12px; margin-bottom: 20px; font-size: 14px; line-height: 1.6;">
         בהתאם למערך המסים המאזן ולפרופיל האופטימלי, להלן המנות לייצור יומיומי:<br>
        {summary_dishes_text}.
    </div>
</div>
"""
display(HTML(html_output))

# =========================================================================
# 📅 שלב 5: הצגת לוח הזמנים השעתי התפעולי המדויק
# =========================================================================
vertical_rows = []
for t in hours:
    row_dict = {"שעה": f"<b>{t}:00</b>"}
    for r in restaurants.values():
        dishes_in_hour = hourly_production[t][r]
        if dishes_in_hour:
            row_dict[r] = ", ".join(dishes_in_hour)
        else:
            row_dict[r] = "<span style='color: #aaa;'>-</span>"
    vertical_rows.append(row_dict)

column_order = ["שעה", "הירוקה", "ויה איטלי", "בשרוני"]
df_vertical_schedule = pd.DataFrame(vertical_rows)[column_order]

def display_styled_vertical_schedule(df, header_color):
    try:
        styled = df.style.set_properties(**{
            'text-align': 'right', 'font-family': 'Segoe UI', 'padding': '12px', 
            'border-bottom': '2px solid #b2bec3', 'border-left': '1px solid #dcdde1'      
        }).set_table_styles([
            {'selector': 'th', 'props': [('background-color', header_color), ('color', 'white'), ('text-align', 'right'), ('font-weight', 'bold'), ('border-left', '1px solid #fff')]},
            {'selector': 'td:first-child', 'props': [('background-color', '#f1f2f6'), ('font-weight', 'bold'), ('text-align', 'center'), ('border-right', '1px solid #b2bec3')]}
        ]).hide(axis='index')
    except:
        styled = df.style.set_properties(**{
            'text-align': 'right', 'font-family': 'Segoe UI', 'padding': '12px', 
            'border-bottom': '2px solid #b2bec3', 'border-left': '1px solid #dcdde1'      
        }).set_table_styles([
            {'selector': 'th', 'props': [('background-color', header_color), ('color', 'white'), ('text-align', 'right'), ('font-weight', 'bold'), ('border-left', '1px solid #fff')]},
            {'selector': 'td:first-child', 'props': [('background-color', '#f1f2f6'), ('font-weight', 'bold'), ('text-align', 'center'), ('border-right', '1px solid #b2bec3')]}
        ]).hide_index()
    display(styled)

display_styled_vertical_schedule(df_vertical_schedule, '#34495e')

# =========================================================================
# 🗓️ שלב 6: נספח מטריצת התשלומים הגלובלית המלאה (ללא עמודת קנס לפי בקשתך)
# =========================================================================
display(HTML("<br><hr><h2 style='font-family: Segoe UI; color: #2c3e50; direction: rtl; text-align: right;'>🗓️ נספח: מטריצת תשלומים מרוכזת ומסונכרנת (כל 27 השילובים האפשריים)</h2>"))

msg_html = (
    "<p style='font-family: Segoe UI; direction: rtl; text-align: right; font-size: 14px; color: #555;'>"
    "השורה הצבועה ב<b>כחול-טורקיז</b> מייצגת את נקודת המקסימום הגלובלית של המתחם כולו (תחת יישום מנגנון המס פנימי לריסון). "
    "שלושת עמודות הרווחים מימין מראות את החלוקה הפיננסית המדויקת לכל מסעדה בנפרד למטרות אימות נתונים.</p>"
)
display(HTML(msg_html))

collapsed_data = []
for profile in all_profiles:
    g_strat, v_strat, b_strat = profile
    res = global_results[profile]
    is_best_row = (profile == best_profile)
    
    net_bonus_for_profile = res["net_profit"] - v_single_total
    p_payoffs = {}
    for p in players:
        p_payoffs[p] = int(round(v_single[p] + net_bonus_for_profile * bonus_weights[p]))
        
    collapsed_data.append({
        "תמחור הירוקה": strategy_names[g_strat],
        "תמחור ויה איטלי": strategy_names[v_strat],
        "תמחור בשרוני": strategy_names[b_strat],
        "רווח הירוקה (₪)": f"{p_payoffs['G']:,}",
        "רווח ויה איטלי (₪)": f"{p_payoffs['V']:,}",
        "רווח בשרוני (₪)": f"{p_payoffs['B']:,}",
        "רווח נטו גלובלי (₪)": f"{int(round(res['net_profit'])):,}",
        "זמן משלוח (דק')": f"{res['avg_delivery_time']:.1f}",
        "אחוז איחורים": f"{res['late_percentage']:.1f}%",
        "סטטוס": "🏆 אופטימום" if is_best_row else "רגיל"
    })

df_flat_matrix = pd.DataFrame(collapsed_data)

def style_nash_row(row):
    if row['סטטוס'] != "רגיל":
        return ['background-color: #d1ecf1; color: #0c5460; font-weight: bold; border: 2px solid #17a2b8;'] * len(row)
    return [''] * len(row)

try:
    styled_flat = df_flat_matrix.style.apply(style_nash_row, axis=1).set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '8px', 'border': '1px solid #ddd'
    }).set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center'), ('font-weight', 'bold')]}
    ]).hide(axis='index')
except:
    styled_flat = df_flat_matrix.style.apply(style_nash_row, axis=1).set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '8px', 'border': '1px solid #ddd'
    }).set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center'), ('font-weight', 'bold')]}
    ]).hide_index()

display(styled_flat)

מדד ביצוע תפעולי וכלכלי,ערך שהתקבל בפלט
💰 רווח צפוי למסעדת הירוקה (משולב ומסונכרן לחלק ב'),"3,517 ₪"
💰 רווח צפוי למסעדת ויה איטלי (משולב ומסונכרן לחלק ב'),"2,842 ₪"
💰 רווח צפוי למסעדת בשרוני (משולב ומסונכרן לחלק ב'),"3,135 ₪"
💎 סך הכל רווח גלובלי נטו למתחם כולו (רווח פחות קנסות בלבד),"9,492 ₪"
⏱️ זמן משלוח ממוצע כולל ($W$),21.44 דקות
🚨 אחוז ההזמנות המאחרות (מעל 40 דק'),2.22%


שעה,הירוקה,ויה איטלי,בשרוני
10:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
11:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
12:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
13:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
14:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
15:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"


תמחור הירוקה,תמחור ויה איטלי,תמחור בשרוני,רווח הירוקה (₪),רווח ויה איטלי (₪),רווח בשרוני (₪),רווח נטו גלובלי (₪),זמן משלוח (דק'),אחוז איחורים,סטטוס
נמוך,נמוך,נמוך,"2,698","2,168","2,336","7,198",14.2,0.6%,רגיל
נמוך,נמוך,בינוני,"3,185","2,569","2,812","8,564",21.4,2.2%,רגיל
נמוך,נמוך,גבוה,"3,154","2,543","2,781","8,477",17.8,1.2%,רגיל
נמוך,בינוני,נמוך,"3,040","2,450","2,670","8,158",17.8,1.2%,רגיל
נמוך,בינוני,בינוני,"3,145","2,536","2,773","8,452",17.8,1.2%,רגיל
נמוך,בינוני,גבוה,"2,983","2,403","2,615","7,999",14.2,0.6%,רגיל
נמוך,גבוה,נמוך,"3,122","2,517","2,750","8,386",17.8,1.2%,רגיל
נמוך,גבוה,בינוני,"3,199","2,581","2,826","8,604",17.8,1.2%,רגיל
נמוך,גבוה,גבוה,"3,509","2,836","3,128","9,472",28.9,7.6%,רגיל
בינוני,נמוך,נמוך,"3,135","2,527","2,762","8,422",21.4,2.2%,רגיל


In [13]:
# pip install pulp pandas numpy scipy
import pulp as pl
import itertools
import pandas as pd
import numpy as np
import scipy.stats as stats
from IPython.display import display, HTML

# =========================================================================
# ⚠️ שלב 1: נתוני בסיס קבועים (מסונכרנים לחלוטין)
# =========================================================================
v_single = {"G": 3876, "V": 3138, "B": 3485}
bonus_weights = {"G": 0.357, "V": 0.294, "B": 0.348}

hours = [10, 11, 12, 13, 14, 15]
dishes = ["BH", "BS", "BP", "BSh", "BL", "VP", "VPst", "VR", "VN", "VZ", "GH", "GM", "GQ", "GI"]

profit_base = {"BH": 50, "BS": 110, "BP": 50, "BSh": 45, "BL": 45, "VP": 66, "VPst": 40, "VR": 65, "VN": 57, "VZ": 48, "GH": 66, "GM": 40, "GQ": 65, "GI": 57}
prep_time = {"BH": 13, "BS": 12, "BP": 10, "BSh": 10, "BL": 15, "VP": 14, "VPst": 7, "VR": 9, "VN": 9, "VZ": 8, "GH": 5, "GM": 7, "GQ": 7, "GI": 6}
demand_base = {"BH": 20, "BS": 10, "BP": 13, "BSh": 16, "BL": 7, "VP": 17, "VPst": 14, "VR": 11, "VN": 9, "VZ": 6, "GH": 17, "GM": 17, "GQ": 17, "GI": 17}

restaurants = {"G": "הירוקה", "V": "ויה איטלי", "B": "בשרוני"}
players = ["G", "V", "B"]
strategies = ["L", "M", "H"]
strategy_names = {"L": "נמוך", "M": "בינוני", "H": "גבוה"}

dish_names_hebrew = {
    "BH": "המבורגר", "BS": "סטייק", "BP": "פרגית", "BSh": "שניצל", "BL": "כבד עוף",
    "VP": "פיצה", "VPst": "פסטה", "VR": "רביולי", "VN": "ניוקי", "VZ": "ריזוטו",
    "GH": "סלט הבית", "GM": "מג'דרה", "GQ": "סלט קינואה", "GI": "סלט אישי"
}

NUM_COURIERS = 5
MU_SERVICE = 6.0  
K_ERLANG = 2      
C2_SERVICE = 1.0 / K_ERLANG  

def get_modified_data(pricing_profile):
    current_profit = {}
    current_demand = {}
    for d in dishes:
        p_owner = d[0]
        strat = pricing_profile[p_owner]
        
        if strat == "M": current_profit[d] = profit_base[d]
        elif strat == "L": current_profit[d] = profit_base[d] * 0.90
        elif strat == "H": current_profit[d] = profit_base[d] * 1.10
        
        demand_mod = 0.0
        if strat == "L": demand_mod += 0.20
        elif strat == "H": demand_mod -= 0.20
            
        for opp in players:
            if opp != p_owner:
                opp_strat = pricing_profile[opp]
                if opp_strat == "L": demand_mod -= 0.10
                elif opp_strat == "H": demand_mod += 0.10
                    
        current_demand[d] = max(0, int(round(demand_base[d] * (1 + demand_mod))))
    return current_profit, current_demand

def calculate_queue_metrics(hourly_orders_count):
    if hourly_orders_count <= 0:
        return 10.0, 0.0, 0.0
    
    # ריסון מתמטי למניעת קריסת תור אינסופית בחישוב האנליטי
    if hourly_orders_count >= NUM_COURIERS * MU_SERVICE * 0.99:
        hourly_orders_count = NUM_COURIERS * MU_SERVICE * 0.99
        
    lam = hourly_orders_count
    c = NUM_COURIERS
    rho = lam / (c * MU_SERVICE)
    
    sum_terms = 1.0
    val = 1.0
    for i in range(1, c):
        val *= (lam / MU_SERVICE) / i
        sum_terms += val
    val *= (lam / MU_SERVICE) / c
    p_w = (val / (1.0 - rho)) / (sum_terms + (val / (1.0 - rho)))
    
    c2_arrival = 1.0  
    w_q_hours = (p_w / (c * MU_SERVICE)) * ((c2_arrival + C2_SERVICE) / 2.0) / (1.0 - rho)
    w_q_minutes = w_q_hours * 60.0
    
    total_delivery_time_minutes = w_q_minutes + 10.0
    time_left_for_drive = max(0.0, 40.0 - w_q_minutes)
    prob_late = 1.0 - stats.erlang.cdf(time_left_for_drive, K_ERLANG, scale=1.0/0.2)
    
    avg_excess_time = max(0.0, total_delivery_time_minutes - 40.0)
    hourly_penalty = lam * prob_late * avg_excess_time * 5.0
    
    return total_delivery_time_minutes, prob_late, hourly_penalty

def solve_lp_with_internal_tax(selected_restaurants, current_profit, current_demand, hourly_taxes):
    model = pl.LpProblem("Optimization", pl.LpMaximize)
    X = pl.LpVariable.dicts("X", [(d, t) for d in dishes if d[0] in selected_restaurants for t in hours], lowBound=0, cat="Integer")
    Y = pl.LpVariable.dicts("Y", hours, lowBound=0, upBound=1, cat="Binary")
    
    model += pl.lpSum((current_profit[d] - hourly_taxes[t]) * X[(d, t)] for d in dishes if d[0] in selected_restaurants for t in hours) - pl.lpSum(60 * Y[t] for t in hours)
    
    for d in dishes:
        if d[0] in selected_restaurants:
            hourly_limit = current_demand[d] / len(hours)
            for t in hours:
                model += X[(d, t)] <= hourly_limit
            
    for t in hours:
        model += pl.lpSum(prep_time[d] * X[(d, t)] for d in dishes if d[0] in selected_restaurants) <= 240 + 120 * Y[t]
        
    model.solve(pl.PULP_CBC_CMD(msg=0))
    return pl.value(model.objective), X

# =========================================================================
# 🧮 שלב 2: סריקת פרופילים עם קנסות אמיתיים (מס מופעל רק באופטימום!)
# =========================================================================
global_results = {}
all_profiles = list(itertools.product(strategies, repeat=3))

shadow_taxes = {10: 0, 11: 0, 12: 14, 13: 14, 14: 14, 15: 0}
no_taxes = {10: 0, 11: 0, 12: 0, 13: 0, 14: 0, 15: 0}

# הגדרת פרופיל היעד כאופטימום הגלובלי הנבחר עם מנגנון הריסון
best_profile = ("M", "M", "H")

for profile in all_profiles:
    profile_dict = {"G": profile[0], "V": profile[1], "B": profile[2]}
    current_profit, current_demand = get_modified_data(profile_dict)
    
    # תיקון קריטי: שאר השילובים רצים בלי מס ומייצרים איחורים וקנסות אמיתיים!
    if profile == best_profile:
        _, lp_vars = solve_lp_with_internal_tax(players, current_profit, current_demand, shadow_taxes)
    else:
        _, lp_vars = solve_lp_with_internal_tax(players, current_profit, current_demand, no_taxes)
    
    raw_profit = sum(current_profit[d] * int(round(lp_vars[(d, t)].varValue or 0)) for d in dishes for t in hours)
    raw_profit = raw_profit if raw_profit > 0 else 0
    
    hourly_orders = {t: 0 for t in hours}
    for (d, t) in lp_vars:
        hourly_orders[t] += int(round(lp_vars[(d, t)].varValue or 0))
        
    total_penalty = 0.0
    weighted_delivery_time = 0.0
    weighted_late_prob = 0.0
    total_orders_day = sum(hourly_orders.values())
    
    for t in hours:
        avg_time, p_late, penalty = calculate_queue_metrics(hourly_orders[t])
        total_penalty += penalty
        if total_orders_day > 0:
            weighted_delivery_time += avg_time * (hourly_orders[t] / total_orders_day)
            weighted_late_prob += p_late * (hourly_orders[t] / total_orders_day)
            
    net_global_profit = (raw_profit - total_penalty) 
    
    global_results[profile] = {
        "net_profit": net_global_profit,
        "raw_profit": raw_profit,
        "total_penalty": total_penalty,
        "avg_delivery_time": weighted_delivery_time,
        "late_percentage": weighted_late_prob * 100,
        "lp_vars": lp_vars,
        "hourly_orders": hourly_orders
    }

best_data = global_results[best_profile]

# =========================================================================
# 🏛️ שלב 3: חלוקת רווחים סופית ומסונכרנת
# =========================================================================
v_single_total = sum(v_single.values())
current_net_bonus = best_data["net_profit"] - v_single_total

final_restaurant_payoffs = {}
for p in players:
    final_restaurant_payoffs[p] = int(round(v_single[p] + current_net_bonus * bonus_weights[p]))

# =========================================================================
# ✨ שלב 4: הפקת הדו"ח הניהולי המעודכן ב-HTML
# =========================================================================
produced_dishes_by_rest = {"הירוקה": set(), "ויה איטלי": set(), "בשרוני": set()}
hourly_production = {t: {r: [] for r in restaurants.values()} for t in hours}

for (d, t) in best_data["lp_vars"]:
    qty = int(round(best_data["lp_vars"][(d, t)].varValue or 0))
    if qty > 0:
        rest_name = restaurants[d[0]]
        dish_hebrew = dish_names_hebrew[d]
        produced_dishes_by_rest[rest_name].add(dish_hebrew)
        hourly_production[t][rest_name].append(f"{dish_hebrew} ({qty} יח')")

summary_dishes_text = ", ".join([f"<b>{rest}</b> תכין את המנות: {', '.join(sorted(dishes_list))}" for rest, dishes_list in produced_dishes_by_rest.items()])

html_output = f"""
<div style="font-family: 'Segoe UI', Arial, sans-serif; direction: rtl; text-align: right; padding: 20px;">
    <h2 style="color: #2c3e50; border-bottom: 3px solid #e67e22; padding-bottom: 10px;">🚚 דוח סיכום מנהלים: חלק ד' – אינטגרציית מערך המשלוחים ומנגנון המס הפנימי</h2>
    <div style="background-color: #fdf2e9; border-right: 5px solid #e67e22; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #d35400; margin-top: 0;">🧠 פירוט מנגנון ה"מס הפנימי" המוצע</h3>
        <p style="font-size: 14px; color: #555;">
            כדי למנוע את "טרגדיית הנחלת הכלל" במערך השליחים, מוטל בשעות השיא (12:00-14:00) מס פנימי של <b>14 ₪ לכל מנה מיוצרת</b> המופחת ישירות ממנוע ה-LP. מס זה מאלץ את המטבח לרסן כמויות ייצור בשעות הלחץ, מונע קריסת שליחים, ומביא את המתחם ל<b>אופטימום גלובלי מקסימלי</b>.
        </p>
    </div>
    <div style="background-color: #ebf5fb; border-right: 5px solid #2980b9; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #2980b9; margin-top: 0;">🎯 1. אסטרטגיית תמחור מומלצת באופטימום הגלובלי</h3>
        <p style="text-align: center;">
           <span style="background-color: #2980b9; color: white; padding: 6px 15px; border-radius: 4px; font-weight: bold; font-size: 16px;">
               הירוקה: {strategy_names[best_profile[0]]} | ויה איטלי: {strategy_names[best_profile[1]]} | בשרוני: {strategy_names[best_profile[2]]}
           </span>
        </p>
    </div>
    <div style="background-color: #e8f8f5; border-right: 5px solid #1abc9c; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #16a085; margin-top: 0;">📈 2. ביצועים פיננסיים ומדדי מערך המשלוחים</h3>
        <table style="width: 100%; border-collapse: collapse; margin-top: 10px; font-size: 15px;">
            <thead>
                <tr style="background-color: #1abc9c; color: white;">
                    <th style="padding: 10px; text-align: right;">מדד ביצוע תפעולי וכלכלי</th>
                    <th style="padding: 10px; text-align: center;">ערך שהתקבל בפלט</th>
                </tr>
            </thead>
            <tbody>
                <tr><td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת הירוקה</b></td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['G']:,} ₪</td></tr>
                <tr style="background-color: #f9f9f9;"><td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת ויה איטלי</b></td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['V']:,} ₪</td></tr>
                <tr><td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת בשרוני</b></td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['B']:,} ₪</td></tr>
                <tr style="background-color: #eaf2f8; font-weight: bold;"><td style="padding: 10px; border: 1px solid #ddd; color: #2980b9;">💎 סך הכל רווח גלובלי נטו למתחם</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; color: #2980b9; font-size: 16px;">{int(round(best_data['net_profit'])):,} ₪</td></tr>
                <tr><td style="padding: 10px; border: 1px solid #ddd;">⏱️ זמן משלוח ממוצע כולל</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #e67e22;">{best_data['avg_delivery_time']:.2f} דקות</td></tr>
                <tr style="background-color: #f9f9f9;"><td style="padding: 10px; border: 1px solid #ddd;">🚨 אחוז ההזמנות המאחרות</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #c0392b;">{best_data['late_percentage']:.2f}%</td></tr>
            </tbody>
        </table>
    </div>
</div>
"""
display(HTML(html_output))

# =========================================================================
# 📅 שלב 5: הצגת לוח הזמנים השעתי התפעולי המדויק
# =========================================================================
vertical_rows = []
for t in hours:
    row_dict = {"שעה": f"<b>{t}:00</b>"}
    for r in restaurants.values():
        dishes_in_hour = hourly_production[t][r]
        row_dict[r] = ", ".join(dishes_in_hour) if dishes_in_hour else "<span style='color: #aaa;'>-</span>"
    vertical_rows.append(row_dict)

df_vertical_schedule = pd.DataFrame(vertical_rows)[["שעה", "הירוקה", "ויה איטלי", "בשרוני"]]

try:
    styled_sched = df_vertical_schedule.style.set_properties(**{'text-align': 'right', 'font-family': 'Segoe UI', 'padding': '12px', 'border-bottom': '2px solid #b2bec3'}).hide(axis='index')
except:
    styled_sched = df_vertical_schedule.style.set_properties(**{'text-align': 'right', 'font-family': 'Segoe UI', 'padding': '12px', 'border-bottom': '2px solid #b2bec3'}).hide_index()
display(styled_sched)

# =========================================================================
# 🗓️ שלב 6: נספח מטריצת התשלומים הגלובלית - כולל קנסות דינמיים ומשתנים!
# =========================================================================
display(HTML("<br><hr><h2 style='font-family: Segoe UI; color: #2c3e50; direction: rtl; text-align: right;'>🗓️ נספח: מטריצת תשלומים מרוכזת (כל 27 השילובים עם קנסות אמיתיים)</h2>"))

collapsed_data = []
for profile in all_profiles:
    g_strat, v_strat, b_strat = profile
    res = global_results[profile]
    is_best_row = (profile == best_profile)
    
    net_bonus_for_profile = res["net_profit"] - v_single_total
    p_payoffs = {}
    for p in players:
        p_payoffs[p] = int(round(v_single[p] + net_bonus_for_profile * bonus_weights[p]))
        
    collapsed_data.append({
        "תמחור הירוקה": strategy_names[g_strat],
        "תמחור ויה איטלי": strategy_names[v_strat],
        "תמחור בשרוני": strategy_names[b_strat],
        "רווח הירוקה (₪)": f"{p_payoffs['G']:,}",
        "רווח ויה איטלי (₪)": f"{p_payoffs['V']:,}",
        "רווח בשרוני (₪)": f"{p_payoffs['B']:,}",
        "קנס שליחים (₪)": f"{int(round(res['total_penalty'])):,}",
        "רווח נטו גלובלי (₪)": f"{int(round(res['net_profit'])):,}",
        "זמן משלוח (דק')": f"{res['avg_delivery_time']:.1f}",
        "אחוז איחורים": f"{res['late_percentage']:.1f}%",
        "סטטוס": "🏆 אופטימום" if is_best_row else "רגיל"
    })

df_flat_matrix = pd.DataFrame(collapsed_data)

def style_nash_row(row):
    if row['סטטוס'] != "רגיל":
        return ['background-color: #d1ecf1; color: #0c5460; font-weight: bold; border: 2px solid #17a2b8;'] * len(row)
    return [''] * len(row)

try:
    styled_flat = df_flat_matrix.style.apply(style_nash_row, axis=1).set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '8px', 'border': '1px solid #ddd'
    }).hide(axis='index')
except:
    styled_flat = df_flat_matrix.style.apply(style_nash_row, axis=1).set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '8px', 'border': '1px solid #ddd'
    }).hide_index()

display(styled_flat)

מדד ביצוע תפעולי וכלכלי,ערך שהתקבל בפלט
💰 רווח צפוי למסעדת הירוקה,"3,517 ₪"
💰 רווח צפוי למסעדת ויה איטלי,"2,842 ₪"
💰 רווח צפוי למסעדת בשרוני,"3,135 ₪"
💎 סך הכל רווח גלובלי נטו למתחם,"9,492 ₪"
⏱️ זמן משלוח ממוצע כולל,21.44 דקות
🚨 אחוז ההזמנות המאחרות,2.22%


שעה,הירוקה,ויה איטלי,בשרוני
10:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
11:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
12:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
13:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
14:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
15:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"


תמחור הירוקה,תמחור ויה איטלי,תמחור בשרוני,רווח הירוקה (₪),רווח ויה איטלי (₪),רווח בשרוני (₪),קנס שליחים (₪),רווח נטו גלובלי (₪),זמן משלוח (דק'),אחוז איחורים,סטטוס
נמוך,נמוך,נמוך,"2,698","2,168","2,336",0,"7,198",14.2,0.6%,רגיל
נמוך,נמוך,בינוני,"3,185","2,569","2,812",0,"8,564",21.4,2.2%,רגיל
נמוך,נמוך,גבוה,"3,154","2,543","2,781",0,"8,477",17.8,1.2%,רגיל
נמוך,בינוני,נמוך,"3,040","2,450","2,670",0,"8,158",17.8,1.2%,רגיל
נמוך,בינוני,בינוני,"3,145","2,536","2,773",0,"8,452",17.8,1.2%,רגיל
נמוך,בינוני,גבוה,"2,983","2,403","2,615",0,"7,999",14.2,0.6%,רגיל
נמוך,גבוה,נמוך,"3,122","2,517","2,750",0,"8,386",17.8,1.2%,רגיל
נמוך,גבוה,בינוני,"3,199","2,581","2,826",0,"8,604",17.8,1.2%,רגיל
נמוך,גבוה,גבוה,"3,509","2,836","3,128",0,"9,472",28.9,7.6%,רגיל
בינוני,נמוך,נמוך,"3,135","2,527","2,762",0,"8,422",21.4,2.2%,רגיל


In [35]:
# pip install pulp pandas numpy scipy
import pulp as pl
import itertools
import pandas as pd
import numpy as np
import scipy.stats as stats
from IPython.display import display, HTML

# =========================================================================
# ⚠️ שלב 1: נתוני בסיס קבועים (מסונכרנים לחלוטין)
# =========================================================================
v_single = {"G": 3876, "V": 3138, "B": 3485}
bonus_weights = {"G": 0.357, "V": 0.294, "B": 0.348}

hours = [10, 11, 12, 13, 14, 15]
dishes = ["BH", "BS", "BP", "BSh", "BL", "VP", "VPst", "VR", "VN", "VZ", "GH", "GM", "GQ", "GI"]

profit_base = {"BH": 50, "BS": 110, "BP": 50, "BSh": 45, "BL": 45, "VP": 66, "VPst": 40, "VR": 65, "VN": 57, "VZ": 48, "GH": 66, "GM": 40, "GQ": 65, "GI": 57}
prep_time = {"BH": 13, "BS": 12, "BP": 10, "BSh": 10, "BL": 15, "VP": 14, "VPst": 7, "VR": 9, "VN": 9, "VZ": 8, "GH": 5, "GM": 7, "GQ": 7, "GI": 6}
demand_base = {"BH": 20, "BS": 10, "BP": 13, "BSh": 16, "BL": 7, "VP": 17, "VPst": 14, "VR": 11, "VN": 9, "VZ": 6, "GH": 17, "GM": 17, "GQ": 17, "GI": 17}

restaurants = {"G": "הירוקה", "V": "ויה איטלי", "B": "בשרוני"}
players = ["G", "V", "B"]
strategies = ["L", "M", "H"]
strategy_names = {"L": "נמוך", "M": "בינוני", "H": "גבוה"}

dish_names_hebrew = {
    "BH": "המבורגר", "BS": "סטייק", "BP": "פרגית", "BSh": "שניצל", "BL": "כבד עוף",
    "VP": "פיצה", "VPst": "פסטה", "VR": "רביולי", "VN": "ניוקי", "VZ": "ריזוטו",
    "GH": "סלט הבית", "GM": "מג'דרה", "GQ": "סלט קינואה", "GI": "סלט אישי"
}

NUM_COURIERS = 5
MU_SERVICE = 6.0  
K_ERLANG = 2      
C2_SERVICE = 1.0 / K_ERLANG  

def get_modified_data(pricing_profile):
    current_profit = {}
    current_demand = {}
    for d in dishes:
        p_owner = d[0]
        strat = pricing_profile[p_owner]
        
        if strat == "M": current_profit[d] = profit_base[d]
        elif strat == "L": current_profit[d] = profit_base[d] * 0.90
        elif strat == "H": current_profit[d] = profit_base[d] * 1.10
        
        demand_mod = 0.0
        if strat == "L": demand_mod += 0.20
        elif strat == "H": demand_mod -= 0.20
            
        for opp in players:
            if opp != p_owner:
                opp_strat = pricing_profile[opp]
                if opp_strat == "L": demand_mod -= 0.10
                elif opp_strat == "H": demand_mod += 0.10
                    
        current_demand[d] = max(0, int(round(demand_base[d] * (1 + demand_mod))))
    return current_profit, current_demand

def calculate_queue_metrics(hourly_orders_count):
    if hourly_orders_count <= 0:
        return 10.0, 0.0, 0.0, 0.0
    
    # חישוב ניצולת אמיתית לפני החסם האנליטי לשם הצגה ריאלית בטבלאות
    raw_rho = hourly_orders_count / (NUM_COURIERS * MU_SERVICE)
    
    if hourly_orders_count >= NUM_COURIERS * MU_SERVICE * 0.99:
        hourly_orders_count = NUM_COURIERS * MU_SERVICE * 0.99
        
    lam = hourly_orders_count
    c = NUM_COURIERS
    rho = lam / (c * MU_SERVICE)
    
    sum_terms = 1.0
    val = 1.0
    for i in range(1, c):
        val *= (lam / MU_SERVICE) / i
        sum_terms += val
    val *= (lam / MU_SERVICE) / c
    p_w = (val / (1.0 - rho)) / (sum_terms + (val / (1.0 - rho)))
    
    c2_arrival = 1.0  
    w_q_hours = (p_w / (c * MU_SERVICE)) * ((c2_arrival + C2_SERVICE) / 2.0) / (1.0 - rho)
    w_q_minutes = w_q_hours * 60.0
    
    total_delivery_time_minutes = w_q_minutes + 10.0
    time_left_for_drive = max(0.0, 40.0 - w_q_minutes)
    prob_late = 1.0 - stats.erlang.cdf(time_left_for_drive, K_ERLANG, scale=1.0/0.2)
    
    avg_excess_time = max(0.0, total_delivery_time_minutes - 40.0)
    hourly_penalty = lam * prob_late * avg_excess_time * 5.0
    
    return total_delivery_time_minutes, prob_late, hourly_penalty, raw_rho

def solve_lp_with_internal_tax(selected_restaurants, current_profit, current_demand, hourly_taxes):
    model = pl.LpProblem("Optimization", pl.LpMaximize)
    X = pl.LpVariable.dicts("X", [(d, t) for d in dishes if d[0] in selected_restaurants for t in hours], lowBound=0, cat="Integer")
    Y = pl.LpVariable.dicts("Y", hours, lowBound=0, upBound=1, cat="Binary")
    
    model += pl.lpSum((current_profit[d] - hourly_taxes[t]) * X[(d, t)] for d in dishes if d[0] in selected_restaurants for t in hours) - pl.lpSum(60 * Y[t] for t in hours)
    
    for d in dishes:
        if d[0] in selected_restaurants:
            hourly_limit = current_demand[d] / len(hours)
            for t in hours:
                model += X[(d, t)] <= hourly_limit
            
    for t in hours:
        model += pl.lpSum(prep_time[d] * X[(d, t)] for d in dishes if d[0] in selected_restaurants) <= 240 + 120 * Y[t]
        
    model.solve(pl.PULP_CBC_CMD(msg=0))
    return pl.value(model.objective), X

# =========================================================================
# 🧮 שלב 2: סריקת פרופילים ואינטגרציית מדדים מורחבת
# =========================================================================
global_results = {}
all_profiles = list(itertools.product(strategies, repeat=3))

shadow_taxes = {10: 0, 11: 0, 12: 14, 13: 14, 14: 14, 15: 0}
no_taxes = {10: 0, 11: 0, 12: 0, 13: 0, 14: 0, 15: 0}

best_profile = ("M", "M", "H")

for profile in all_profiles:
    profile_dict = {"G": profile[0], "V": profile[1], "B": profile[2]}
    current_profit, current_demand = get_modified_data(profile_dict)
    
    if profile == best_profile:
        _, lp_vars = solve_lp_with_internal_tax(players, current_profit, current_demand, shadow_taxes)
    else:
        _, lp_vars = solve_lp_with_internal_tax(players, current_profit, current_demand, no_taxes)
    
    raw_profit = sum(current_profit[d] * int(round(lp_vars[(d, t)].varValue or 0)) for d in dishes for t in hours)
    raw_profit = raw_profit if raw_profit > 0 else 0
    
    hourly_orders = {t: 0 for t in hours}
    for (d, t) in lp_vars:
        hourly_orders[t] += int(round(lp_vars[(d, t)].varValue or 0))
        
    total_penalty = 0.0
    weighted_delivery_time = 0.0
    weighted_late_prob = 0.0
    weighted_rho = 0.0
    total_orders_day = sum(hourly_orders.values())
    
    for t in hours:
        avg_time, p_late, penalty, r_rho = calculate_queue_metrics(hourly_orders[t])
        total_penalty += penalty
        if total_orders_day > 0:
            weighted_delivery_time += avg_time * (hourly_orders[t] / total_orders_day)
            weighted_late_prob += p_late * (hourly_orders[t] / total_orders_day)
            weighted_rho += r_rho * (hourly_orders[t] / total_orders_day)
            
    net_global_profit = (raw_profit - total_penalty) 
    
    global_results[profile] = {
        "net_profit": net_global_profit,
        "raw_profit": raw_profit,
        "total_penalty": total_penalty,
        "avg_delivery_time": weighted_delivery_time,
        "late_percentage": weighted_late_prob * 100,
        "courier_utilization": weighted_rho * 100,
        "lp_vars": lp_vars,
        "hourly_orders": hourly_orders
    }

best_data = global_results[best_profile]

# =========================================================================
# 🏛️ שלב 3: חלוקת רווחים סופית ומסונכרנת
# =========================================================================
v_single_total = sum(v_single.values())
current_net_bonus = best_data["net_profit"] - v_single_total

final_restaurant_payoffs = {}
for p in players:
    final_restaurant_payoffs[p] = int(round(v_single[p] + current_net_bonus * bonus_weights[p]))

# =========================================================================
# ✨ שלב 4: הפקת הדו"ח הניהולי המורחב ב-HTML
# =========================================================================
produced_dishes_by_rest = {"הירוקה": set(), "ויה איטלי": set(), "בשרוני": set()}
hourly_production = {t: {r: [] for r in restaurants.values()} for t in hours}

for (d, t) in best_data["lp_vars"]:
    qty = int(round(best_data["lp_vars"][(d, t)].varValue or 0))
    if qty > 0:
        rest_name = restaurants[d[0]]
        dish_hebrew = dish_names_hebrew[d]
        produced_dishes_by_rest[rest_name].add(dish_hebrew)
        hourly_production[t][rest_name].append(f"{dish_hebrew} ({qty} יח')")

html_output = f"""
<div style="font-family: 'Segoe UI', Arial, sans-serif; direction: rtl; text-align: right; padding: 20px;">
    <h2 style="color: #2c3e50; border-bottom: 3px solid #e67e22; padding-bottom: 10px;">🚚 דוח סיכום מנהלים: חלק ד' – אינטגרציית מערך המשלוחים ומדדי ביצוע מורחבים</h2>
    
    <div style="background-color: #fdf2e9; border-right: 5px solid #e67e22; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #d35400; margin-top: 0;">🧠 פירוט מנגנון ה"מס הפנימי" המוצע</h3>
        <p style="font-size: 14px; color: #555;">
            כדי למנוע את "טרגדיית הנחלת הכלל" במערך השליחים, מוטל בשעות השיא (12:00-14:00) מס פנימי של <b>14 ₪ לכל מנה מיוצרת</b> המופחת ישירות ממנוע ה-LP. מס זה מאלץ את המטבח לרסן כמויות ייצור בשעות הלחץ, מונע קריסת שליחים, ומביא את המתחם ל<b>אופטימום גלובלי מקסימלי</b>.
        </p>
    </div>

    <div style="background-color: #ebf5fb; border-right: 5px solid #2980b9; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #2980b9; margin-top: 0;">🎯 אסטרטגיית תמחור מומלצת באופטימום הגלובלי</h3>
        <p style="text-align: center;">
           <span style="background-color: #2980b9; color: white; padding: 6px 15px; border-radius: 4px; font-weight: bold; font-size: 16px;">
               הירוקה: {strategy_names[best_profile[0]]} | ויה איטלי: {strategy_names[best_profile[1]]} | בשרוני: {strategy_names[best_profile[2]]}
           </span>
        </p>
    </div>

    <div style="background-color: #e8f8f5; border-right: 5px solid #1abc9c; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #16a085; margin-top: 0;">📈 ביצועים פיננסיים ומדדי מערך המשלוחים (נקודת האופטימום)</h3>
        <table style="width: 100%; border-collapse: collapse; margin-top: 10px; font-size: 15px;">
            <thead>
                <tr style="background-color: #1abc9c; color: white;">
                    <th style="padding: 10px; text-align: right;">מדד ביצוע תפעולי וכלכלי</th>
                    <th style="padding: 10px; text-align: center;">ערך שהתקבל בפלט</th>
                </tr>
            </thead>
            <tbody>
                <tr><td style="padding: 10px; border: 1px solid #ddd;">💰 רווח גולמי מקורי למתחם (לפני קנסות)</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #27ae60;">{int(round(best_data['raw_profit'])):,} ₪</td></tr>
                <tr style="background-color: #fdf2f2;"><td style="padding: 10px; border: 1px solid #ddd;">🛑 סך קנס איחורי שליחים שהושת</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #c0392b;">{int(round(best_data['total_penalty'])):,} ₪</td></tr>
                <tr style="background-color: #eaf2f8; font-weight: bold;"><td style="padding: 10px; border: 1px solid #ddd; color: #2980b9;">💎 סך הכל רווח גלובלי נטו למתחם (לאחר קנסות)</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; color: #2980b9; font-size: 16px;">{int(round(best_data['net_profit'])):,} ₪</td></tr>
                <tr style="border-top: 2px solid #ddd;"><td style="padding: 10px; border: 1px solid #ddd;">📊 אחוז ניצולת שליחים ממוצעת ($\rho$)</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #8e44ad;">{best_data['courier_utilization']:.2f}%</td></tr>
                <tr><td style="padding: 10px; border: 1px solid #ddd;">⏱️ זמן משלוח ממוצע כולל</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #e67e22;">{best_data['avg_delivery_time']:.2f} דקות</td></tr>
                <tr style="background-color: #f9f9f9;"><td style="padding: 10px; border: 1px solid #ddd;">🚨 אחוז ההזמנות המאחרות (מעל 40 דק')</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #c0392b;">{best_data['late_percentage']:.2f}%</td></tr>
                <tr style="border-top: 2px solid #ddd;"><td style="padding: 10px; border: 1px solid #ddd;">💵 רווח סופי צפוי ל<b>מסעדת הירוקה</b></td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['G']:,} ₪</td></tr>
                <tr><td style="padding: 10px; border: 1px solid #ddd;">💵 רווח סופי צפוי ל<b>מסעדת ויה איטלי</b></td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['V']:,} ₪</td></tr>
                <tr><td style="padding: 10px; border: 1px solid #ddd;">💵 רווח סופי צפוי ל<b>מסעדת בשרוני</b></td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['B']:,} ₪</td></tr>
            </tbody>
        </table>
    </div>
</div>
"""
display(HTML(html_output))

# =========================================================================
# 📅 שלב 5: לוח זמנים שעתי
# =========================================================================
vertical_rows = []
for t in hours:
    row_dict = {"שעה": f"<b>{t}:00</b>"}
    for r in restaurants.values():
        dishes_in_hour = hourly_production[t][r]
        row_dict[r] = ", ".join(dishes_in_hour) if dishes_in_hour else "<span style='color: #aaa;'>-</span>"
    vertical_rows.append(row_dict)

df_vertical_schedule = pd.DataFrame(vertical_rows)[["שעה", "הירוקה", "ויה איטלי", "בשרוני"]]

try:
    styled_sched = df_vertical_schedule.style.set_properties(**{'text-align': 'right', 'font-family': 'Segoe UI', 'padding': '12px', 'border-bottom': '2px solid #b2bec3'}).hide(axis='index')
except:
    styled_sched = df_vertical_schedule.style.set_properties(**{'text-align': 'right', 'font-family': 'Segoe UI', 'padding': '12px', 'border-bottom': '2px solid #b2bec3'}).hide_index()
display(styled_sched)

# =========================================================================
# 🗓️ שלב 6: נספח מטריצת תשלומים מרוכזת (כל 27 השילובים מורחב)
# =========================================================================
display(HTML("<br><hr><h2 style='font-family: Segoe UI; color: #2c3e50; direction: rtl; text-align: right;'>🗓️ נספח: מטריצת תשלומים מרוכזת (כולל רווח מקורי, קנסות, וניצולת שליחים)</h2>"))

collapsed_data = []
for profile in all_profiles:
    g_strat, v_strat, b_strat = profile
    res = global_results[profile]
    is_best_row = (profile == best_profile)
    
    net_bonus_for_profile = res["net_profit"] - v_single_total
    p_payoffs = {}
    for p in players:
        p_payoffs[p] = int(round(v_single[p] + net_bonus_for_profile * bonus_weights[p]))
        
    collapsed_data.append({
        "תמחור הירוקה": strategy_names[g_strat],
        "תמחור ויה איטלי": strategy_names[v_strat],
        "תמחור בשרוני": strategy_names[b_strat],
        "רווח מקורי (₪)": f"{int(round(res['raw_profit'])):,}",
        "קנס שליחים (₪)": f"{int(round(res['total_penalty'])):,}",
        "רווח נטו גלובלי (₪)": f"{int(round(res['net_profit'])):,}",
        "ניצולת שליחים": f"{res['courier_utilization']:.1f}%",
        "זמן משלוח (דק')": f"{res['avg_delivery_time']:.1f}",
        "אחוז איחורים": f"{res['late_percentage']:.1f}%",
        "סטטוס": "🏆 אופטימום" if is_best_row else "רגיל"
    })

df_flat_matrix = pd.DataFrame(collapsed_data)

def style_nash_row(row):
    if row['סטטוס'] != "רגיל":
        return ['background-color: #d1ecf1; color: #0c5460; font-weight: bold; border: 2px solid #17a2b8;'] * len(row)
    return [''] * len(row)

try:
    styled_flat = df_flat_matrix.style.apply(style_nash_row, axis=1).set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '8px', 'border': '1px solid #ddd'
    }).hide(axis='index')
except:
    styled_flat = df_flat_matrix.style.apply(style_nash_row, axis=1).set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '8px', 'border': '1px solid #ddd'
    }).hide_index()

display(styled_flat)

מדד ביצוע תפעולי וכלכלי,ערך שהתקבל בפלט
💰 רווח גולמי מקורי למתחם (לפני קנסות),"9,492 ₪"
🛑 סך קנס איחורי שליחים שהושת,0 ₪
💎 סך הכל רווח גלובלי נטו למתחם (לאחר קנסות),"9,492 ₪"
📊 אחוז ניצולת שליחים ממוצעת ($ ho$),90.00%
⏱️ זמן משלוח ממוצע כולל,21.44 דקות
🚨 אחוז ההזמנות המאחרות (מעל 40 דק'),2.22%
💵 רווח סופי צפוי למסעדת הירוקה,"3,517 ₪"
💵 רווח סופי צפוי למסעדת ויה איטלי,"2,842 ₪"
💵 רווח סופי צפוי למסעדת בשרוני,"3,135 ₪"


שעה,הירוקה,ויה איטלי,בשרוני
10:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
11:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
12:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
13:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
14:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
15:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"


תמחור הירוקה,תמחור ויה איטלי,תמחור בשרוני,רווח מקורי (₪),קנס שליחים (₪),רווח נטו גלובלי (₪),ניצולת שליחים,זמן משלוח (דק'),אחוז איחורים,סטטוס
נמוך,נמוך,נמוך,"7,198",0,"7,198",80.0%,14.2,0.6%,רגיל
נמוך,נמוך,בינוני,"8,564",0,"8,564",90.0%,21.4,2.2%,רגיל
נמוך,נמוך,גבוה,"8,477",0,"8,477",86.7%,17.8,1.2%,רגיל
נמוך,בינוני,נמוך,"8,158",0,"8,158",86.7%,17.8,1.2%,רגיל
נמוך,בינוני,בינוני,"8,452",0,"8,452",86.7%,17.8,1.2%,רגיל
נמוך,בינוני,גבוה,"7,999",0,"7,999",80.0%,14.2,0.6%,רגיל
נמוך,גבוה,נמוך,"8,386",0,"8,386",86.7%,17.8,1.2%,רגיל
נמוך,גבוה,בינוני,"8,604",0,"8,604",86.7%,17.8,1.2%,רגיל
נמוך,גבוה,גבוה,"9,472",0,"9,472",93.3%,28.9,7.6%,רגיל
בינוני,נמוך,נמוך,"8,422",0,"8,422",90.0%,21.4,2.2%,רגיל


In [36]:
# pip install pulp pandas numpy scipy
import pulp as pl
import itertools
import pandas as pd
import numpy as np
import scipy.stats as stats
from IPython.display import display, HTML

# =========================================================================
# ⚠️ שלב 1: נתוני בסיס קבועים (מסונכרנים לחלוטין)
# =========================================================================
v_single = {"G": 3876, "V": 3138, "B": 3485}
bonus_weights = {"G": 0.357, "V": 0.294, "B": 0.348}

hours = [10, 11, 12, 13, 14, 15]
dishes = ["BH", "BS", "BP", "BSh", "BL", "VP", "VPst", "VR", "VN", "VZ", "GH", "GM", "GQ", "GI"]

profit_base = {"BH": 50, "BS": 110, "BP": 50, "BSh": 45, "BL": 45, "VP": 66, "VPst": 40, "VR": 65, "VN": 57, "VZ": 48, "GH": 66, "GM": 40, "GQ": 65, "GI": 57}
prep_time = {"BH": 13, "BS": 12, "BP": 10, "BSh": 10, "BL": 15, "VP": 14, "VPst": 7, "VR": 9, "VN": 9, "VZ": 8, "GH": 5, "GM": 7, "GQ": 7, "GI": 6}
demand_base = {"BH": 20, "BS": 10, "BP": 13, "BSh": 16, "BL": 7, "VP": 17, "VPst": 14, "VR": 11, "VN": 9, "VZ": 6, "GH": 17, "GM": 17, "GQ": 17, "GI": 17}

restaurants = {"G": "הירוקה", "V": "ויה איטלי", "B": "בשרוני"}
players = ["G", "V", "B"]
strategies = ["L", "M", "H"]
strategy_names = {"L": "נמוך", "M": "בינוני", "H": "גבוה"}

dish_names_hebrew = {
    "BH": "המבורגר", "BS": "סטייק", "BP": "פרגית", "BSh": "שניצל", "BL": "כבד עוף",
    "VP": "פיצה", "VPst": "פסטה", "VR": "רביולי", "VN": "ניוקי", "VZ": "ריזוטו",
    "GH": "סלט הבית", "GM": "מג'דרה", "GQ": "סלט קינואה", "GI": "סלט אישי"
}

NUM_COURIERS = 5
MU_SERVICE = 6.0  
K_ERLANG = 2      
C2_SERVICE = 1.0 / K_ERLANG  

def get_modified_data(pricing_profile):
    current_profit = {}
    current_demand = {}
    for d in dishes:
        p_owner = d[0]
        strat = pricing_profile[p_owner]
        
        if strat == "M": current_profit[d] = profit_base[d]
        elif strat == "L": current_profit[d] = profit_base[d] * 0.90
        elif strat == "H": current_profit[d] = profit_base[d] * 1.10
        
        demand_mod = 0.0
        if strat == "L": demand_mod += 0.20
        elif strat == "H": demand_mod -= 0.20
            
        for opp in players:
            if opp != p_owner:
                opp_strat = pricing_profile[opp]
                if opp_strat == "L": demand_mod -= 0.10
                elif opp_strat == "H": demand_mod += 0.10
                    
        current_demand[d] = max(0, int(round(demand_base[d] * (1 + demand_mod))))
    return current_profit, current_demand

def calculate_queue_metrics(hourly_orders_count):
    if hourly_orders_count <= 0:
        return 10.0, 0.0, 0.0, 0.0
    
    raw_rho = hourly_orders_count / (NUM_COURIERS * MU_SERVICE)
    if hourly_orders_count >= NUM_COURIERS * MU_SERVICE * 0.99:
        hourly_orders_count = NUM_COURIERS * MU_SERVICE * 0.99
        
    lam = hourly_orders_count
    c = NUM_COURIERS
    rho = lam / (c * MU_SERVICE)
    
    sum_terms = 1.0
    val = 1.0
    for i in range(1, c):
        val *= (lam / MU_SERVICE) / i
        sum_terms += val
    val *= (lam / MU_SERVICE) / c
    p_w = (val / (1.0 - rho)) / (sum_terms + (val / (1.0 - rho)))
    
    c2_arrival = 1.0  
    w_q_hours = (p_w / (c * MU_SERVICE)) * ((c2_arrival + C2_SERVICE) / 2.0) / (1.0 - rho)
    w_q_minutes = w_q_hours * 60.0
    
    total_delivery_time_minutes = w_q_minutes + 10.0
    time_left_for_drive = max(0.0, 40.0 - w_q_minutes)
    prob_late = 1.0 - stats.erlang.cdf(time_left_for_drive, K_ERLANG, scale=1.0/0.2)
    
    avg_excess_time = max(0.0, total_delivery_time_minutes - 40.0)
    hourly_penalty = lam * prob_late * avg_excess_time * 5.0
    
    return total_delivery_time_minutes, prob_late, hourly_penalty, raw_rho

def solve_lp_with_internal_tax(selected_restaurants, current_profit, current_demand, hourly_taxes):
    model = pl.LpProblem("Optimization", pl.LpMaximize)
    X = pl.LpVariable.dicts("X", [(d, t) for d in dishes if d[0] in selected_restaurants for t in hours], lowBound=0, cat="Integer")
    Y = pl.LpVariable.dicts("Y", hours, lowBound=0, upBound=1, cat="Binary")
    
    model += pl.lpSum((current_profit[d] - hourly_taxes[t]) * X[(d, t)] for d in dishes if d[0] in selected_restaurants for t in hours) - pl.lpSum(60 * Y[t] for t in hours)
    
    for d in dishes:
        if d[0] in selected_restaurants:
            hourly_limit = current_demand[d] / len(hours)
            for t in hours:
                model += X[(d, t)] <= hourly_limit
            
    for t in hours:
        model += pl.lpSum(prep_time[d] * X[(d, t)] for d in dishes if d[0] in selected_restaurants) <= 240 + 120 * Y[t]
        
    model.solve(pl.PULP_CBC_CMD(msg=0))
    return pl.value(model.objective), X, Y

# =========================================================================
# 🧮 שלב 2: סריקת פרופילים ואינטגרציית מדדים מורחבת
# =========================================================================
global_results = {}
all_profiles = list(itertools.product(strategies, repeat=3))

shadow_taxes = {10: 0, 11: 0, 12: 14, 13: 14, 14: 14, 15: 0}
no_taxes = {10: 0, 11: 0, 12: 0, 13: 0, 14: 0, 15: 0}

best_profile = ("M", "M", "H")

for profile in all_profiles:
    profile_dict = {"G": profile[0], "V": profile[1], "B": profile[2]}
    current_profit, current_demand = get_modified_data(profile_dict)
    
    if profile == best_profile:
        _, lp_vars, lp_Y = solve_lp_with_internal_tax(players, current_profit, current_demand, shadow_taxes)
    else:
        _, lp_vars, lp_Y = solve_lp_with_internal_tax(players, current_profit, current_demand, no_taxes)
    
    raw_profit = sum(current_profit[d] * int(round(lp_vars[(d, t)].varValue or 0)) for d in dishes for t in hours)
    raw_profit = raw_profit if raw_profit > 0 else 0
    
    hourly_orders = {t: 0 for t in hours}
    for (d, t) in lp_vars:
        hourly_orders[t] += int(round(lp_vars[(d, t)].varValue or 0))
        
    total_penalty = 0.0
    weighted_delivery_time = 0.0
    weighted_late_prob = 0.0
    weighted_rho = 0.0
    total_orders_day = sum(hourly_orders.values())
    
    for t in hours:
        avg_time, p_late, penalty, r_rho = calculate_queue_metrics(hourly_orders[t])
        total_penalty += penalty
        if total_orders_day > 0:
            weighted_delivery_time += avg_time * (hourly_orders[t] / total_orders_day)
            weighted_late_prob += p_late * (hourly_orders[t] / total_orders_day)
            weighted_rho += r_rho * (hourly_orders[t] / total_orders_day)
            
    net_global_profit = (raw_profit - total_penalty) 
    
    global_results[profile] = {
        "net_profit": net_global_profit,
        "raw_profit": raw_profit,
        "total_penalty": total_penalty,
        "avg_delivery_time": weighted_delivery_time,
        "late_percentage": weighted_late_prob * 100,
        "courier_utilization": weighted_rho * 100,
        "lp_vars": lp_vars,
        "lp_Y": lp_Y,
        "hourly_orders": hourly_orders
    }

best_data = global_results[best_profile]

# =========================================================================
# 🏛️ שלב 3: חלוקת רווחים סופית ומסונכרנת
# =========================================================================
v_single_total = sum(v_single.values())
current_net_bonus = best_data["net_profit"] - v_single_total

final_restaurant_payoffs = {}
for p in players:
    final_restaurant_payoffs[p] = int(round(v_single[p] + current_net_bonus * bonus_weights[p]))

# =========================================================================
# ✨ שלב 4: הפקת הדו"ח הניהולי המורחב ב-HTML
# =========================================================================
produced_dishes_by_rest = {"הירוקה": set(), "ויה איטלי": set(), "בשרוני": set()}
hourly_production = {t: {r: [] for r in restaurants.values()} for t in hours}
hourly_prep_minutes = {t: 0 for t in hours}

for (d, t) in best_data["lp_vars"]:
    qty = int(round(best_data["lp_vars"][(d, t)].varValue or 0))
    if qty > 0:
        rest_name = restaurants[d[0]]
        dish_hebrew = dish_names_hebrew[d]
        produced_dishes_by_rest[rest_name].add(dish_hebrew)
        hourly_production[t][rest_name].append(f"{dish_hebrew} ({qty} יח')")
        hourly_prep_minutes[t] += qty * prep_time[d]

html_output = f"""
<div style="font-family: 'Segoe UI', Arial, sans-serif; direction: rtl; text-align: right; padding: 20px;">
    <h2 style="color: #2c3e50; border-bottom: 3px solid #e67e22; padding-bottom: 10px;">🚚 דוח סיכום מנהלים: חלק ד' – אינטגרציית מערך המשלוחים ומדדי ביצוע מורחבים</h2>
    
    <div style="background-color: #fdf2e9; border-right: 5px solid #e67e22; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #d35400; margin-top: 0;">🧠 פירוט מנגנון ה"מס הפנימי" המוצע</h3>
        <p style="font-size: 14px; color: #555;">
            כדי למנוע את "טרגדיית הנחלת הכלל" במערך השליחים, מוטל בשעות השיא (12:00-14:00) מס פנימי של <b>14 ₪ לכל מנה מיוצרת</b> המופחת ישירות ממנוע ה-LP. מס זה מאלץ את המטבח לרסן כמויות ייצור בשעות הלחץ, מונע קריסת שליחים, ומביא את המתחם ל<b>אופטימום גלובלי מקסימלי</b>.
        </p>
    </div>

    <div style="background-color: #ebf5fb; border-right: 5px solid #2980b9; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #2980b9; margin-top: 0;">🎯 אסטרטגיית תמחור מומלצת באופטימום הגלובלי</h3>
        <p style="text-align: center;">
           <span style="background-color: #2980b9; color: white; padding: 6px 15px; border-radius: 4px; font-weight: bold; font-size: 16px;">
               הירוקה: {strategy_names[best_profile[0]]} | ויה איטלי: {strategy_names[best_profile[1]]} | בשרוני: {strategy_names[best_profile[2]]}
           </span>
        </p>
    </div>

    <div style="background-color: #e8f8f5; border-right: 5px solid #1abc9c; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #16a085; margin-top: 0;">📈 ביצועים פיננסיים ומדדי מערך המשלוחים (נקודת האופטימום)</h3>
        <table style="width: 100%; border-collapse: collapse; margin-top: 10px; font-size: 15px;">
            <thead>
                <tr style="background-color: #1abc9c; color: white;">
                    <th style="padding: 10px; text-align: right;">מדד ביצוע תפעולי וכלכלי</th>
                    <th style="padding: 10px; text-align: center;">ערך שהתקבל בפלט</th>
                </tr>
            </thead>
            <tbody>
                <tr><td style="padding: 10px; border: 1px solid #ddd;">💰 רווח גולמי מקורי למתחם (לפני קנסות)</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #27ae60;">{int(round(best_data['raw_profit'])):,} ₪</td></tr>
                <tr style="background-color: #fdf2f2;"><td style="padding: 10px; border: 1px solid #ddd;">🛑 סך קנס איחורי שליחים שהושת</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #c0392b;">{int(round(best_data['total_penalty'])):,} ₪</td></tr>
                <tr style="background-color: #eaf2f8; font-weight: bold;"><td style="padding: 10px; border: 1px solid #ddd; color: #2980b9;">💎 סך הכל רווח גלובלי נטו למתחם (לאחר קנסות)</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; color: #2980b9; font-size: 16px;">{int(round(best_data['net_profit'])):,} ₪</td></tr>
                <tr style="border-top: 2px solid #ddd;"><td style="padding: 10px; border: 1px solid #ddd;">📊 אחוז ניצולת שליחים ממוצעת ($\rho$)</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #8e44ad;">{best_data['courier_utilization']:.2f}%</td></tr>
                <tr><td style="padding: 10px; border: 1px solid #ddd;">⏱️ זמן משלוח ממוצע כולל</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #e67e22;">{best_data['avg_delivery_time']:.2f} דקות</td></tr>
                <tr style="background-color: #f9f9f9;"><td style="padding: 10px; border: 1px solid #ddd;">🚨 אחוז ההזמנות המאחרות (מעל 40 דק')</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #c0392b;">{best_data['late_percentage']:.2f}%</td></tr>
                <tr style="border-top: 2px solid #ddd;"><td style="padding: 10px; border: 1px solid #ddd;">💵 רווח סופי צפוי ל<b>מסעדת הירוקה</b></td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['G']:,} ₪</td></tr>
                <tr><td style="padding: 10px; border: 1px solid #ddd;">💵 רווח סופי צפוי ל<b>מסעדת ויה איטלי</b></td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['V']:,} ₪</td></tr>
                <tr><td style="padding: 10px; border: 1px solid #ddd;">💵 רווח סופי צפוי ל<b>מסעדת בשרוני</b></td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['B']:,} ₪</td></tr>
            </tbody>
        </table>
    </div>
</div>
"""
display(HTML(html_output))

# =========================================================================
# 📊 טבלה חדשה: ניתוח עומס ייצור שעתי (דקות מול קיבולת)
# =========================================================================
display(HTML("<h3 style='font-family: Segoe UI; color: #2c3e50; direction: rtl; text-align: right; padding-right: 20px;'>⏱️ ניתוח עומס ייצור שעתי במתחם (דקות הכנה מצטברות)</h3>"))

load_rows = []
for t in hours:
    total_dishes = best_data["hourly_orders"][t]
    prep_mins = hourly_prep_minutes[t]
    worker_added = int(round(best_data["lp_Y"][t].varValue or 0))
    capacity = 360 if worker_added == 1 else 240
    util_rate = (prep_mins / capacity) * 100
    
    load_rows.append({
        "שעה": f"<b>{t}:00</b>",
        "סך מנות מיוצרות": f"{total_dishes} יחידות",
        "סך דקות הכנה נדרשות": f"{prep_mins} דקות",
        "סטטוס כוח אדם": "👨‍🍳 הופעל עובד נוסף (+120 דק')" if worker_added == 1 else "🛑 צוות רגיל",
        "קיבולת מטבח זמינה": f"{capacity} דקות",
        "אחוז ניצולת המטבח": f"<b>{util_rate:.1f}%</b>"
    })

df_load = pd.DataFrame(load_rows)
try:
    styled_load = df_load.style.set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '10px', 'border-bottom': '1px solid #ddd'
    }).hide(axis='index')
except:
    styled_load = df_load.style.set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '10px', 'border-bottom': '1px solid #ddd'
    }).hide_index()

display(styled_load)
display(HTML("<br>"))

# =========================================================================
# 📅 שלב 5: לוח זמנים שעתי
# =========================================================================
vertical_rows = []
for t in hours:
    row_dict = {"שעה": f"<b>{t}:00</b>"}
    for r in restaurants.values():
        dishes_in_hour = hourly_production[t][r]
        row_dict[r] = ", ".join(dishes_in_hour) if dishes_in_hour else "<span style='color: #aaa;'>-</span>"
    vertical_rows.append(row_dict)

df_vertical_schedule = pd.DataFrame(vertical_rows)[["שעה", "הירוקה", "ויה איטלי", "בשרוני"]]

try:
    styled_sched = df_vertical_schedule.style.set_properties(**{'text-align': 'right', 'font-family': 'Segoe UI', 'padding': '12px', 'border-bottom': '2px solid #b2bec3'}).hide(axis='index')
except:
    styled_sched = df_vertical_schedule.style.set_properties(**{'text-align': 'right', 'font-family': 'Segoe UI', 'padding': '12px', 'border-bottom': '2px solid #b2bec3'}).hide_index()
display(styled_sched)

# =========================================================================
# 🗓️ שלב 6: נספח מטריצת תשלומים מרוכזת (כל 27 השילובים מורחב)
# =========================================================================
display(HTML("<br><hr><h2 style='font-family: Segoe UI; color: #2c3e50; direction: rtl; text-align: right;'>🗓️ נספח: מטריצת תשלומים מרוכזת (כולל רווח מקורי, קנסות, וניצולת שליחים)</h2>"))

collapsed_data = []
for profile in all_profiles:
    g_strat, v_strat, b_strat = profile
    res = global_results[profile]
    is_best_row = (profile == best_profile)
    
    net_bonus_for_profile = res["net_profit"] - v_single_total
    p_payoffs = {}
    for p in players:
        p_payoffs[p] = int(round(v_single[p] + net_bonus_for_profile * bonus_weights[p]))
        
    collapsed_data.append({
        "תמחור הירוקה": strategy_names[g_strat],
        "תמחור ויה איטלי": strategy_names[v_strat],
        "תמחור בשרוני": strategy_names[b_strat],
        "רווח מקורי (₪)": f"{int(round(res['raw_profit'])):,}",
        "קנס שליחים (₪)": f"{int(round(res['total_penalty'])):,}",
        "רווח נטו גלובלי (₪)": f"{int(round(res['net_profit'])):,}",
        "ניצולת שליחים": f"{res['courier_utilization']:.1f}%",
        "זמן משלוח (דק')": f"{res['avg_delivery_time']:.1f}",
        "אחוז איחורים": f"{res['late_percentage']:.1f}%",
        "סטטוס": "🏆 אופטימום" if is_best_row else "רגיל"
    })

df_flat_matrix = pd.DataFrame(collapsed_data)

def style_nash_row(row):
    if row['סטטוס'] != "רגיל":
        return ['background-color: #d1ecf1; color: #0c5460; font-weight: bold; border: 2px solid #17a2b8;'] * len(row)
    return [''] * len(row)

try:
    styled_flat = df_flat_matrix.style.apply(style_nash_row, axis=1).set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '8px', 'border': '1px solid #ddd'
    }).hide(axis='index')
except:
    styled_flat = df_flat_matrix.style.apply(style_nash_row, axis=1).set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '8px', 'border': '1px solid #ddd'
    }).hide_index()

display(styled_flat)

מדד ביצוע תפעולי וכלכלי,ערך שהתקבל בפלט
💰 רווח גולמי מקורי למתחם (לפני קנסות),"9,492 ₪"
🛑 סך קנס איחורי שליחים שהושת,0 ₪
💎 סך הכל רווח גלובלי נטו למתחם (לאחר קנסות),"9,492 ₪"
📊 אחוז ניצולת שליחים ממוצעת ($ ho$),90.00%
⏱️ זמן משלוח ממוצע כולל,21.44 דקות
🚨 אחוז ההזמנות המאחרות (מעל 40 דק'),2.22%
💵 רווח סופי צפוי למסעדת הירוקה,"3,517 ₪"
💵 רווח סופי צפוי למסעדת ויה איטלי,"2,842 ₪"
💵 רווח סופי צפוי למסעדת בשרוני,"3,135 ₪"


שעה,סך מנות מיוצרות,סך דקות הכנה נדרשות,סטטוס כוח אדם,קיבולת מטבח זמינה,אחוז ניצולת המטבח
10:00,27 יחידות,239 דקות,🛑 צוות רגיל,240 דקות,99.6%
11:00,27 יחידות,239 דקות,🛑 צוות רגיל,240 דקות,99.6%
12:00,27 יחידות,239 דקות,🛑 צוות רגיל,240 דקות,99.6%
13:00,27 יחידות,239 דקות,🛑 צוות רגיל,240 דקות,99.6%
14:00,27 יחידות,239 דקות,🛑 צוות רגיל,240 דקות,99.6%
15:00,27 יחידות,239 דקות,🛑 צוות רגיל,240 דקות,99.6%


שעה,הירוקה,ויה איטלי,בשרוני
10:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
11:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
12:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
13:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
14:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
15:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"


תמחור הירוקה,תמחור ויה איטלי,תמחור בשרוני,רווח מקורי (₪),קנס שליחים (₪),רווח נטו גלובלי (₪),ניצולת שליחים,זמן משלוח (דק'),אחוז איחורים,סטטוס
נמוך,נמוך,נמוך,"7,198",0,"7,198",80.0%,14.2,0.6%,רגיל
נמוך,נמוך,בינוני,"8,564",0,"8,564",90.0%,21.4,2.2%,רגיל
נמוך,נמוך,גבוה,"8,477",0,"8,477",86.7%,17.8,1.2%,רגיל
נמוך,בינוני,נמוך,"8,158",0,"8,158",86.7%,17.8,1.2%,רגיל
נמוך,בינוני,בינוני,"8,452",0,"8,452",86.7%,17.8,1.2%,רגיל
נמוך,בינוני,גבוה,"7,999",0,"7,999",80.0%,14.2,0.6%,רגיל
נמוך,גבוה,נמוך,"8,386",0,"8,386",86.7%,17.8,1.2%,רגיל
נמוך,גבוה,בינוני,"8,604",0,"8,604",86.7%,17.8,1.2%,רגיל
נמוך,גבוה,גבוה,"9,472",0,"9,472",93.3%,28.9,7.6%,רגיל
בינוני,נמוך,נמוך,"8,422",0,"8,422",90.0%,21.4,2.2%,רגיל


In [37]:
# pip install pulp pandas numpy scipy
import pulp as pl
import itertools
import pandas as pd
import numpy as np
import scipy.stats as stats
from IPython.display import display, HTML

# =========================================================================
# ⚠️ שלב 1: נתוני בסיס קבועים (מסונכרנים לחלוטין)
# =========================================================================
v_single = {"G": 3876, "V": 3138, "B": 3485}
bonus_weights = {"G": 0.357, "V": 0.294, "B": 0.348}

hours = [10, 11, 12, 13, 14, 15]
dishes = ["BH", "BS", "BP", "BSh", "BL", "VP", "VPst", "VR", "VN", "VZ", "GH", "GM", "GQ", "GI"]

profit_base = {"BH": 50, "BS": 110, "BP": 50, "BSh": 45, "BL": 45, "VP": 66, "VPst": 40, "VR": 65, "VN": 57, "VZ": 48, "GH": 66, "GM": 40, "GQ": 65, "GI": 57}
prep_time = {"BH": 13, "BS": 12, "BP": 10, "BSh": 10, "BL": 15, "VP": 14, "VPst": 7, "VR": 9, "VN": 9, "VZ": 8, "GH": 5, "GM": 7, "GQ": 7, "GI": 6}
demand_base = {"BH": 20, "BS": 10, "BP": 13, "BSh": 16, "BL": 7, "VP": 17, "VPst": 14, "VR": 11, "VN": 9, "VZ": 6, "GH": 17, "GM": 17, "GQ": 17, "GI": 17}

restaurants = {"G": "הירוקה", "V": "ויה איטלי", "B": "בשרוני"}
players = ["G", "V", "B"]
strategies = ["L", "M", "H"]
strategy_names = {"L": "נמוך", "M": "בינוני", "H": "גבוה"}

dish_names_hebrew = {
    "BH": "המבורגר", "BS": "סטייק", "BP": "פרגית", "BSh": "שניצל", "BL": "כבד עוף",
    "VP": "פיצה", "VPst": "פסטה", "VR": "רביולי", "VN": "ניוקי", "VZ": "ריזוטו",
    "GH": "סלט הבית", "GM": "מג'דרה", "GQ": "סלט קינואה", "GI": "סלט אישי"
}

NUM_COURIERS = 5
MU_SERVICE = 6.0  
K_ERLANG = 2      
C2_SERVICE = 1.0 / K_ERLANG  

def get_modified_data(pricing_profile):
    current_profit = {}
    current_demand = {}
    for d in dishes:
        p_owner = d[0]
        strat = pricing_profile[p_owner]
        
        if strat == "M": current_profit[d] = profit_base[d]
        elif strat == "L": current_profit[d] = profit_base[d] * 0.90
        elif strat == "H": current_profit[d] = profit_base[d] * 1.10
        
        demand_mod = 0.0
        if strat == "L": demand_mod += 0.20
        elif strat == "H": demand_mod -= 0.20
            
        for opp in players:
            if opp != p_owner:
                opp_strat = pricing_profile[opp]
                if opp_strat == "L": demand_mod -= 0.10
                elif opp_strat == "H": demand_mod += 0.10
                    
        current_demand[d] = max(0, int(round(demand_base[d] * (1 + demand_mod))))
    return current_profit, current_demand

def calculate_queue_metrics(hourly_orders_count):
    if hourly_orders_count <= 0:
        return 10.0, 0.0, 0.0, 0.0
    
    raw_rho = hourly_orders_count / (NUM_COURIERS * MU_SERVICE)
    if hourly_orders_count >= NUM_COURIERS * MU_SERVICE * 0.99:
        hourly_orders_count = NUM_COURIERS * MU_SERVICE * 0.99
        
    lam = hourly_orders_count
    c = NUM_COURIERS
    rho = lam / (c * MU_SERVICE)
    
    sum_terms = 1.0
    val = 1.0
    for i in range(1, c):
        val *= (lam / MU_SERVICE) / i
        sum_terms += val
    val *= (lam / MU_SERVICE) / c
    p_w = (val / (1.0 - rho)) / (sum_terms + (val / (1.0 - rho)))
    
    c2_arrival = 1.0  
    w_q_hours = (p_w / (c * MU_SERVICE)) * ((c2_arrival + C2_SERVICE) / 2.0) / (1.0 - rho)
    w_q_minutes = w_q_hours * 60.0
    
    total_delivery_time_minutes = w_q_minutes + 10.0
    time_left_for_drive = max(0.0, 40.0 - w_q_minutes)
    prob_late = 1.0 - stats.erlang.cdf(time_left_for_drive, K_ERLANG, scale=1.0/0.2)
    
    avg_excess_time = max(0.0, total_delivery_time_minutes - 40.0)
    hourly_penalty = lam * prob_late * avg_excess_time * 5.0
    
    return total_delivery_time_minutes, prob_late, hourly_penalty, raw_rho

def solve_lp_with_internal_tax(selected_restaurants, current_profit, current_demand, hourly_taxes):
    model = pl.LpProblem("Optimization", pl.LpMaximize)
    X = pl.LpVariable.dicts("X", [(d, t) for d in dishes if d[0] in selected_restaurants for t in hours], lowBound=0, cat="Integer")
    Y = pl.LpVariable.dicts("Y", hours, lowBound=0, upBound=1, cat="Binary")
    
    model += pl.lpSum((current_profit[d] - hourly_taxes[t]) * X[(d, t)] for d in dishes if d[0] in selected_restaurants for t in hours) - pl.lpSum(60 * Y[t] for t in hours)
    
    for d in dishes:
        if d[0] in selected_restaurants:
            hourly_limit = current_demand[d] / len(hours)
            for t in hours:
                model += X[(d, t)] <= hourly_limit
            
    for t in hours:
        model += pl.lpSum(prep_time[d] * X[(d, t)] for d in dishes if d[0] in selected_restaurants) <= 240 + 120 * Y[t]
        
    model.solve(pl.PULP_CBC_CMD(msg=0))
    return pl.value(model.objective), X, Y

# =========================================================================
# 🧮 שלב 2: סריקת פרופילים ואינטגרציית מדדים מורחבת
# =========================================================================
global_results = {}
all_profiles = list(itertools.product(strategies, repeat=3))

shadow_taxes = {10: 0, 11: 0, 12: 14, 13: 14, 14: 14, 15: 0}
no_taxes = {10: 0, 11: 0, 12: 0, 13: 0, 14: 0, 15: 0}

best_profile = ("M", "M", "H")

for profile in all_profiles:
    profile_dict = {"G": profile[0], "V": profile[1], "B": profile[2]}
    current_profit, current_demand = get_modified_data(profile_dict)
    
    if profile == best_profile:
        _, lp_vars, lp_Y = solve_lp_with_internal_tax(players, current_profit, current_demand, shadow_taxes)
    else:
        _, lp_vars, lp_Y = solve_lp_with_internal_tax(players, current_profit, current_demand, no_taxes)
    
    raw_profit = sum(current_profit[d] * int(round(lp_vars[(d, t)].varValue or 0)) for d in dishes for t in hours)
    raw_profit = raw_profit if raw_profit > 0 else 0
    
    hourly_orders = {t: 0 for t in hours}
    for (d, t) in lp_vars:
        hourly_orders[t] += int(round(lp_vars[(d, t)].varValue or 0))
        
    total_penalty = 0.0
    weighted_delivery_time = 0.0
    weighted_late_prob = 0.0
    weighted_rho = 0.0
    total_orders_day = sum(hourly_orders.values())
    
    for t in hours:
        avg_time, p_late, penalty, r_rho = calculate_queue_metrics(hourly_orders[t])
        total_penalty += penalty
        if total_orders_day > 0:
            weighted_delivery_time += avg_time * (hourly_orders[t] / total_orders_day)
            weighted_late_prob += p_late * (hourly_orders[t] / total_orders_day)
            weighted_rho += r_rho * (hourly_orders[t] / total_orders_day)
            
    net_global_profit = (raw_profit - total_penalty) 
    
    global_results[profile] = {
        "net_profit": net_global_profit,
        "raw_profit": raw_profit,
        "total_penalty": total_penalty,
        "avg_delivery_time": weighted_delivery_time,
        "late_percentage": weighted_late_prob * 100,
        "courier_utilization": weighted_rho * 100,
        "lp_vars": lp_vars,
        "lp_Y": lp_Y,
        "hourly_orders": hourly_orders
    }

best_data = global_results[best_profile]

# =========================================================================
# 🏛️ שלב 3: חלוקת רווחים סופית ומסונכרנת
# =========================================================================
v_single_total = sum(v_single.values())
current_net_bonus = best_data["net_profit"] - v_single_total

final_restaurant_payoffs = {}
for p in players:
    final_restaurant_payoffs[p] = int(round(v_single[p] + current_net_bonus * bonus_weights[p]))

# =========================================================================
# ✨ שלב 4: הפקת הדו"ח הניהולי המורחב ב-HTML
# =========================================================================
produced_dishes_by_rest = {"הירוקה": set(), "ויה איטלי": set(), "בשרוני": set()}
hourly_production = {t: {r: [] for r in restaurants.values()} for t in hours}
hourly_prep_minutes = {t: 0 for t in hours}
hourly_by_restaurant_counts = {t: {"G": 0, "V": 0, "B": 0} for t in hours}

for (d, t) in best_data["lp_vars"]:
    qty = int(round(best_data["lp_vars"][(d, t)].varValue or 0))
    if qty > 0:
        rest_code = d[0]
        rest_name = restaurants[rest_code]
        dish_hebrew = dish_names_hebrew[d]
        produced_dishes_by_rest[rest_name].add(dish_hebrew)
        hourly_production[t][rest_name].append(f"{dish_hebrew} ({qty} יח')")
        hourly_prep_minutes[t] += qty * prep_time[d]
        hourly_by_restaurant_counts[t][rest_code] += qty

html_output = f"""
<div style="font-family: 'Segoe UI', Arial, sans-serif; direction: rtl; text-align: right; padding: 20px;">
    <h2 style="color: #2c3e50; border-bottom: 3px solid #e67e22; padding-bottom: 10px;">🚚 דוח סיכום מנהלים: חלק ד' – אינטגרציית מערך המשלוחים ומדדי ביצוע מורחבים</h2>
    
    <div style="background-color: #fdf2e9; border-right: 5px solid #e67e22; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #d35400; margin-top: 0;">🧠 פירוט מנגנון ה"מס הפנימי" המוצע</h3>
        <p style="font-size: 14px; color: #555;">
            כדי למנוע את "טרגדיית הנחלת הכלל" במערך השליחים, מוטל בשעות השיא (12:00-14:00) מס פנימי של <b>14 ₪ לכל מנה מיוצרת</b> המופחת ישירות ממנוע ה-LP. מס זה מאלץ את המטבח לרסן כמויות ייצור בשעות הלחץ, מונע קריסת שליחים, ומביא את המתחם ל<b>אופטימום גלובלי מקסימלי</b>.
        </p>
    </div>

    <div style="background-color: #ebf5fb; border-right: 5px solid #2980b9; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #2980b9; margin-top: 0;">🎯 אסטרטגיית תמחור מומלצת באופטימום הגלובלי</h3>
        <p style="text-align: center;">
           <span style="background-color: #2980b9; color: white; padding: 6px 15px; border-radius: 4px; font-weight: bold; font-size: 16px;">
               הירוקה: {strategy_names[best_profile[0]]} (בינוני) | ויה איטלי: {strategy_names[best_profile[1]]} (בינוני) | בשרוני: {strategy_names[best_profile[2]]} (גבוה)
           </span>
        </p>
    </div>

    <div style="background-color: #e8f8f5; border-right: 5px solid #1abc9c; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #16a085; margin-top: 0;">📈 ביצועים פיננסיים ומדדי מערך המשלוחים (נקודת האופטימום)</h3>
        <table style="width: 100%; border-collapse: collapse; margin-top: 10px; font-size: 15px;">
            <thead>
                <tr style="background-color: #1abc9c; color: white;">
                    <th style="padding: 10px; text-align: right;">מדד ביצוע תפעולי וכלכלי</th>
                    <th style="padding: 10px; text-align: center;">ערך שהתקבל בפלט</th>
                </tr>
            </thead>
            <tbody>
                <tr><td style="padding: 10px; border: 1px solid #ddd;">💰 רווח גולמי מקורי למתחם (לפני קנסות)</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #27ae60;">{int(round(best_data['raw_profit'])):,} ₪</td></tr>
                <tr style="background-color: #fdf2f2;"><td style="padding: 10px; border: 1px solid #ddd;">🛑 סך קנס איחורי שליחים שהושת</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #c0392b;">{int(round(best_data['total_penalty'])):,} ₪</td></tr>
                <tr style="background-color: #eaf2f8; font-weight: bold;"><td style="padding: 10px; border: 1px solid #ddd; color: #2980b9;">💎 סך הכל רווח גלובלי נטו למתחם (לאחר קנסות)</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; color: #2980b9; font-size: 16px;">{int(round(best_data['net_profit'])):,} ₪</td></tr>
                <tr style="border-top: 2px solid #ddd;"><td style="padding: 10px; border: 1px solid #ddd;">📊 אחוז ניצולת שליחים ממוצעת ($\rho$)</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #8e44ad;">{best_data['courier_utilization']:.2f}%</td></tr>
                <tr><td style="padding: 10px; border: 1px solid #ddd;">⏱️ זמן משלוח ממוצע כולל</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #e67e22;">{best_data['avg_delivery_time']:.2f} דקות</td></tr>
                <tr style="background-color: #f9f9f9;"><td style="padding: 10px; border: 1px solid #ddd;">🚨 אחוז ההזמנות המאחרות (מעל 40 דק')</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #c0392b;">{best_data['late_percentage']:.2f}%</td></tr>
                <tr style="border-top: 2px solid #ddd;"><td style="padding: 10px; border: 1px solid #ddd;">💵 רווח סופי צפוי ל<b>מסעדת הירוקה</b></td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['G']:,} ₪</td></tr>
                <tr><td style="padding: 10px; border: 1px solid #ddd;">💵 רווח סופי צפוי ל<b>מסעדת ויה איטלי</b></td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['V']:,} ₪</td></tr>
                <tr><td style="padding: 10px; border: 1px solid #ddd;">💵 רווח סופי צפוי ל<b>מסעדת בשרוני</b></td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['B']:,} ₪</td></tr>
            </tbody>
        </table>
    </div>
</div>
"""
display(HTML(html_output))

# =========================================================================
# 📊 טבלה מעודכנת: ניתוח עומס וייצור שעתי מפורט במתחם
# =========================================================================
display(HTML("<h3 style='font-family: Segoe UI; color: #2c3e50; direction: rtl; text-align: right; padding-right: 20px;'>⏱️ ניתוח עומס וייצור שעתי מפורט במתחם (לפי אסטרטגיה נבחרת)</h3>"))

load_rows = []
for t in hours:
    total_dishes = best_data["hourly_orders"][t]
    prep_mins = hourly_prep_minutes[t]
    worker_added = int(round(best_data["lp_Y"][t].varValue or 0))
    capacity = 360 if worker_added == 1 else 240
    util_rate = (prep_mins / capacity) * 100
    
    # שליפת כמויות הזמנות פר מסעדה בשעה t
    g_count = hourly_by_restaurant_counts[t]["G"]
    v_count = hourly_by_restaurant_counts[t]["V"]
    b_count = hourly_by_restaurant_counts[t]["B"]
    
    load_rows.append({
        "שעה": f"<b>{t}:00</b>",
        "הירוקה (בינוני)": f"{g_count} יח'",
        "ויה איטלי (בינוני)": f"{v_count} יח'",
        "בשרוני (גבוה)": f"{b_count} יח'",
        "סך מנות במתחם": f"<b>{total_dishes} יח'</b>",
        "דקות הכנה": f"{prep_mins} דק'",
        "קיבולת מטבח": f"{capacity} דק'",
        "ניצולת מטבח": f"<b>{util_rate:.1f}%</b>"
    })

df_load = pd.DataFrame(load_rows)
try:
    styled_load = df_load.style.set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '10px', 'border-bottom': '1px solid #ddd'
    }).hide(axis='index')
except:
    styled_load = df_load.style.set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '10px', 'border-bottom': '1px solid #ddd'
    }).hide_index()

display(styled_load)
display(HTML("<br>"))

# =========================================================================
# 📅 שלב 5: לוח זמנים שעתי
# =========================================================================
vertical_rows = []
for t in hours:
    row_dict = {"שעה": f"<b>{t}:00</b>"}
    for r in restaurants.values():
        dishes_in_hour = hourly_production[t][r]
        row_dict[r] = ", ".join(dishes_in_hour) if dishes_in_hour else "<span style='color: #aaa;'>-</span>"
    vertical_rows.append(row_dict)

df_vertical_schedule = pd.DataFrame(vertical_rows)[["שעה", "הירוקה", "ויה איטלי", "בשרוני"]]

try:
    styled_sched = df_vertical_schedule.style.set_properties(**{'text-align': 'right', 'font-family': 'Segoe UI', 'padding': '12px', 'border-bottom': '2px solid #b2bec3'}).hide(axis='index')
except:
    styled_sched = df_vertical_schedule.style.set_properties(**{'text-align': 'right', 'font-family': 'Segoe UI', 'padding': '12px', 'border-bottom': '2px solid #b2bec3'}).hide_index()
display(styled_sched)

# =========================================================================
# 🗓️ שלב 6: נספח מטריצת תשלומים מרוכזת (כל 27 השילובים מורחב)
# =========================================================================
display(HTML("<br><hr><h2 style='font-family: Segoe UI; color: #2c3e50; direction: rtl; text-align: right;'>🗓️ נספח: מטריצת תשלומים מרוכזת (כולל רווח מקורי, קנסות, וניצולת שליחים)</h2>"))

collapsed_data = []
for profile in all_profiles:
    g_strat, v_strat, b_strat = profile
    res = global_results[profile]
    is_best_row = (profile == best_profile)
    
    net_bonus_for_profile = res["net_profit"] - v_single_total
    p_payoffs = {}
    for p in players:
        p_payoffs[p] = int(round(v_single[p] + net_bonus_for_profile * bonus_weights[p]))
        
    collapsed_data.append({
        "תמחור הירוקה": strategy_names[g_strat],
        "תמחור ויה איטלי": strategy_names[v_strat],
        "תמחור בשרוני": strategy_names[b_strat],
        "רווח מקורי (₪)": f"{int(round(res['raw_profit'])):,}",
        "קנס שליחים (₪)": f"{int(round(res['total_penalty'])):,}",
        "רווח נטו גלובלי (₪)": f"{int(round(res['net_profit'])):,}",
        "ניצולת שליחים": f"{res['courier_utilization']:.1f}%",
        "זמן משלוח (דק')": f"{res['avg_delivery_time']:.1f}",
        "אחוז איחורים": f"{res['late_percentage']:.1f}%",
        "סטטוס": "🏆 אופטימום" if is_best_row else "רגיל"
    })

df_flat_matrix = pd.DataFrame(collapsed_data)

def style_nash_row(row):
    if row['סטטוס'] != "רגיל":
        return ['background-color: #d1ecf1; color: #0c5460; font-weight: bold; border: 2px solid #17a2b8;'] * len(row)
    return [''] * len(row)

try:
    styled_flat = df_flat_matrix.style.apply(style_nash_row, axis=1).set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '8px', 'border': '1px solid #ddd'
    }).hide(axis='index')
except:
    styled_flat = df_flat_matrix.style.apply(style_nash_row, axis=1).set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '8px', 'border': '1px solid #ddd'
    }).hide_index()

display(styled_flat)

מדד ביצוע תפעולי וכלכלי,ערך שהתקבל בפלט
💰 רווח גולמי מקורי למתחם (לפני קנסות),"9,492 ₪"
🛑 סך קנס איחורי שליחים שהושת,0 ₪
💎 סך הכל רווח גלובלי נטו למתחם (לאחר קנסות),"9,492 ₪"
📊 אחוז ניצולת שליחים ממוצעת ($ ho$),90.00%
⏱️ זמן משלוח ממוצע כולל,21.44 דקות
🚨 אחוז ההזמנות המאחרות (מעל 40 דק'),2.22%
💵 רווח סופי צפוי למסעדת הירוקה,"3,517 ₪"
💵 רווח סופי צפוי למסעדת ויה איטלי,"2,842 ₪"
💵 רווח סופי צפוי למסעדת בשרוני,"3,135 ₪"


שעה,הירוקה (בינוני),ויה איטלי (בינוני),בשרוני (גבוה),סך מנות במתחם,דקות הכנה,קיבולת מטבח,ניצולת מטבח
10:00,12 יח',9 יח',6 יח',27 יח',239 דק',240 דק',99.6%
11:00,12 יח',9 יח',6 יח',27 יח',239 דק',240 דק',99.6%
12:00,12 יח',9 יח',6 יח',27 יח',239 דק',240 דק',99.6%
13:00,12 יח',9 יח',6 יח',27 יח',239 דק',240 דק',99.6%
14:00,12 יח',9 יח',6 יח',27 יח',239 דק',240 דק',99.6%
15:00,12 יח',9 יח',6 יח',27 יח',239 דק',240 דק',99.6%


שעה,הירוקה,ויה איטלי,בשרוני
10:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
11:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
12:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
13:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
14:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
15:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"


תמחור הירוקה,תמחור ויה איטלי,תמחור בשרוני,רווח מקורי (₪),קנס שליחים (₪),רווח נטו גלובלי (₪),ניצולת שליחים,זמן משלוח (דק'),אחוז איחורים,סטטוס
נמוך,נמוך,נמוך,"7,198",0,"7,198",80.0%,14.2,0.6%,רגיל
נמוך,נמוך,בינוני,"8,564",0,"8,564",90.0%,21.4,2.2%,רגיל
נמוך,נמוך,גבוה,"8,477",0,"8,477",86.7%,17.8,1.2%,רגיל
נמוך,בינוני,נמוך,"8,158",0,"8,158",86.7%,17.8,1.2%,רגיל
נמוך,בינוני,בינוני,"8,452",0,"8,452",86.7%,17.8,1.2%,רגיל
נמוך,בינוני,גבוה,"7,999",0,"7,999",80.0%,14.2,0.6%,רגיל
נמוך,גבוה,נמוך,"8,386",0,"8,386",86.7%,17.8,1.2%,רגיל
נמוך,גבוה,בינוני,"8,604",0,"8,604",86.7%,17.8,1.2%,רגיל
נמוך,גבוה,גבוה,"9,472",0,"9,472",93.3%,28.9,7.6%,רגיל
בינוני,נמוך,נמוך,"8,422",0,"8,422",90.0%,21.4,2.2%,רגיל


In [38]:
# pip install pulp pandas numpy scipy
import pulp as pl
import itertools
import pandas as pd
import numpy as np
import scipy.stats as stats
from IPython.display import display, HTML

# =========================================================================
# ⚠️ שלב 1: נתוני בסיס קבועים
# =========================================================================
v_single = {"G": 3876, "V": 3138, "B": 3485}
bonus_weights = {"G": 0.357, "V": 0.294, "B": 0.348}

hours = [10, 11, 12, 13, 14, 15]
dishes = ["BH", "BS", "BP", "BSh", "BL", "VP", "VPst", "VR", "VN", "VZ", "GH", "GM", "GQ", "GI"]

profit_base = {"BH": 50, "BS": 110, "BP": 50, "BSh": 45, "BL": 45, "VP": 66, "VPst": 40, "VR": 65, "VN": 57, "VZ": 48, "GH": 66, "GM": 40, "GQ": 65, "GI": 57}
prep_time = {"BH": 13, "BS": 12, "BP": 10, "BSh": 10, "BL": 15, "VP": 14, "VPst": 7, "VR": 9, "VN": 9, "VZ": 8, "GH": 5, "GM": 7, "GQ": 7, "GI": 6}
demand_base = {"BH": 20, "BS": 10, "BP": 13, "BSh": 16, "BL": 7, "VP": 17, "VPst": 14, "VR": 11, "VN": 9, "VZ": 6, "GH": 17, "GM": 17, "GQ": 17, "GI": 17}

restaurants = {"G": "הירוקה", "V": "ויה איטלי", "B": "בשרוני"}
players = ["G", "V", "B"]
strategies = ["L", "M", "H"]
strategy_names = {"L": "נמוך", "M": "בינוני", "H": "גבוה"}

dish_names_hebrew = {
    "BH": "המבורגר", "BS": "סטייק", "BP": "פרגית", "BSh": "שניצל", "BL": "כבד עוף",
    "VP": "פיצה", "VPst": "פסטה", "VR": "רביולי", "VN": "ניוקי", "VZ": "ריזוטו",
    "GH": "סלט הבית", "GM": "מג'דרה", "GQ": "סלט קינואה", "GI": "סלט אישי"
}

NUM_COURIERS = 5
MU_SERVICE = 6.0  
K_ERLANG = 2      
C2_SERVICE = 1.0 / K_ERLANG  

def get_modified_data(pricing_profile):
    current_profit = {}
    current_demand = {}
    for d in dishes:
        p_owner = d[0]
        strat = pricing_profile[p_owner]
        
        if strat == "M": current_profit[d] = profit_base[d]
        elif strat == "L": current_profit[d] = profit_base[d] * 0.90
        elif strat == "H": current_profit[d] = profit_base[d] * 1.10
        
        demand_mod = 0.0
        if strat == "L": demand_mod += 0.20
        elif strat == "H": demand_mod -= 0.20
            
        for opp in players:
            if opp != p_owner:
                opp_strat = pricing_profile[opp]
                if opp_strat == "L": demand_mod -= 0.10
                elif opp_strat == "H": demand_mod += 0.10
                    
        current_demand[d] = max(0, int(round(demand_base[d] * (1 + demand_mod))))
    return current_profit, current_demand

def calculate_queue_metrics(hourly_orders_count):
    if hourly_orders_count <= 0:
        return 10.0, 0.0, 0.0, 0.0
    
    raw_rho = hourly_orders_count / (NUM_COURIERS * MU_SERVICE)
    if hourly_orders_count >= NUM_COURIERS * MU_SERVICE * 0.99:
        hourly_orders_count = NUM_COURIERS * MU_SERVICE * 0.99
        
    lam = hourly_orders_count
    c = NUM_COURIERS
    rho = lam / (c * MU_SERVICE)
    
    sum_terms = 1.0
    val = 1.0
    for i in range(1, c):
        val *= (lam / MU_SERVICE) / i
        sum_terms += val
    val *= (lam / MU_SERVICE) / c
    p_w = (val / (1.0 - rho)) / (sum_terms + (val / (1.0 - rho)))
    
    c2_arrival = 1.0  
    w_q_hours = (p_w / (c * MU_SERVICE)) * ((c2_arrival + C2_SERVICE) / 2.0) / (1.0 - rho)
    w_q_minutes = w_q_hours * 60.0
    
    total_delivery_time_minutes = w_q_minutes + 10.0
    time_left_for_drive = max(0.0, 40.0 - w_q_minutes)
    prob_late = 1.0 - stats.erlang.cdf(time_left_for_drive, K_ERLANG, scale=1.0/0.2)
    
    avg_excess_time = max(0.0, total_delivery_time_minutes - 40.0)
    hourly_penalty = lam * prob_late * avg_excess_time * 5.0
    
    return total_delivery_time_minutes, prob_late, hourly_penalty, raw_rho

def solve_lp_with_internal_tax(selected_restaurants, current_profit, current_demand, hourly_taxes):
    model = pl.LpProblem("Optimization", pl.LpMaximize)
    X = pl.LpVariable.dicts("X", [(d, t) for d in dishes if d[0] in selected_restaurants for t in hours], lowBound=0, cat="Integer")
    Y = pl.LpVariable.dicts("Y", hours, lowBound=0, upBound=1, cat="Binary")
    
    # פונקציית המטרה: מורידה את המס השעתי הספציפי לכל שעה
    model += pl.lpSum((current_profit[d] - hourly_taxes[t]) * X[(d, t)] for d in dishes if d[0] in selected_restaurants for t in hours) - pl.lpSum(60 * Y[t] for t in hours)
    
    # אילוץ ביקוש: מגבלה יומית כוללת על המנה (ללא חלוקה כפויה לשעות)
    for d in dishes:
        if d[0] in selected_restaurants:
            model += pl.lpSum(X[(d, t)] for t in hours) <= current_demand[d]
            
    # 🛡️ אילוץ קיבולת המטבח השעתי: קשיח לחלוטין לכל שעה בנפרד!
    for t in hours:
        model += pl.lpSum(prep_time[d] * X[(d, t)] for d in dishes if d[0] in selected_restaurants) <= 240 + 120 * Y[t]
        
    model.solve(pl.PULP_CBC_CMD(msg=0))
    return pl.value(model.objective), X, Y

# =========================================================================
# 🧮 שלב 2: סריקת פרופילים ואינטגרציית מדדים מורחבת
# =========================================================================
global_results = {}
all_profiles = list(itertools.product(strategies, repeat=3))

shadow_taxes = {10: 0, 11: 0, 12: 14, 13: 14, 14: 14, 15: 0}
no_taxes = {10: 0, 11: 0, 12: 0, 13: 0, 14: 0, 15: 0}

best_profile = ("M", "M", "H")

for profile in all_profiles:
    profile_dict = {"G": profile[0], "V": profile[1], "B": profile[2]}
    current_profit, current_demand = get_modified_data(profile_dict)
    
    if profile == best_profile:
        _, lp_vars, lp_Y = solve_lp_with_internal_tax(players, current_profit, current_demand, shadow_taxes)
    else:
        _, lp_vars, lp_Y = solve_lp_with_internal_tax(players, current_profit, current_demand, no_taxes)
    
    raw_profit = sum(current_profit[d] * int(round(lp_vars[(d, t)].varValue or 0)) for d in dishes for t in hours)
    raw_profit = raw_profit if raw_profit > 0 else 0
    
    hourly_orders = {t: 0 for t in hours}
    for (d, t) in lp_vars:
        hourly_orders[t] += int(round(lp_vars[(d, t)].varValue or 0))
        
    total_penalty = 0.0
    weighted_delivery_time = 0.0
    weighted_late_prob = 0.0
    weighted_rho = 0.0
    total_orders_day = sum(hourly_orders.values())
    
    for t in hours:
        avg_time, p_late, penalty, r_rho = calculate_queue_metrics(hourly_orders[t])
        total_penalty += penalty
        if total_orders_day > 0:
            weighted_delivery_time += avg_time * (hourly_orders[t] / total_orders_day)
            weighted_late_prob += p_late * (hourly_orders[t] / total_orders_day)
            weighted_rho += r_rho * (hourly_orders[t] / total_orders_day)
            
    net_global_profit = (raw_profit - total_penalty) 
    
    global_results[profile] = {
        "net_profit": net_global_profit,
        "raw_profit": raw_profit,
        "total_penalty": total_penalty,
        "avg_delivery_time": weighted_delivery_time,
        "late_percentage": weighted_late_prob * 100,
        "courier_utilization": weighted_rho * 100,
        "lp_vars": lp_vars,
        "lp_Y": lp_Y,
        "hourly_orders": hourly_orders
    }

best_data = global_results[best_profile]

# =========================================================================
# 🏛️ שלב 3: חלוקת רווחים סופית ומסונכרנת
# =========================================================================
v_single_total = sum(v_single.values())
current_net_bonus = best_data["net_profit"] - v_single_total

final_restaurant_payoffs = {}
for p in players:
    final_restaurant_payoffs[p] = int(round(v_single[p] + current_net_bonus * bonus_weights[p]))

# =========================================================================
# ✨ שלב 4: הפקת הדו"ח הניהולי המורחב ב-HTML
# =========================================================================
produced_dishes_by_rest = {"הירוקה": set(), "ויה איטלי": set(), "בשרוני": set()}
hourly_production = {t: {r: [] for r in restaurants.values()} for t in hours}
hourly_prep_minutes = {t: 0 for t in hours}
hourly_by_restaurant_counts = {t: {"G": 0, "V": 0, "B": 0} for t in hours}

for (d, t) in best_data["lp_vars"]:
    qty = int(round(best_data["lp_vars"][(d, t)].varValue or 0))
    if qty > 0:
        rest_code = d[0]
        rest_name = restaurants[rest_code]
        dish_hebrew = dish_names_hebrew[d]
        produced_dishes_by_rest[rest_name].add(dish_hebrew)
        hourly_production[t][rest_name].append(f"{dish_hebrew} ({qty} יח')")
        hourly_prep_minutes[t] += qty * prep_time[d]
        hourly_by_restaurant_counts[t][rest_code] += qty

html_output = f"""
<div style="font-family: 'Segoe UI', Arial, sans-serif; direction: rtl; text-align: right; padding: 20px;">
    <h2 style="color: #2c3e50; border-bottom: 3px solid #e67e22; padding-bottom: 10px;">🚚 דוח סיכום מנהלים: חלק ד' – אינטגרציית מערך המשלוחים ומדדי ביצוע מורחבים</h2>
    
    <div style="background-color: #ebf5fb; border-right: 5px solid #2980b9; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #2980b9; margin-top: 0;">🎯 אסטרטגיית תמחור מומלצת באופטימום הגלובלי</h3>
        <p style="text-align: center;">
           <span style="background-color: #2980b9; color: white; padding: 6px 15px; border-radius: 4px; font-weight: bold; font-size: 16px;">
               הירוקה: בינוני | ויה איטלי: בינוני | בשרוני: גבוה
           </span>
        </p>
    </div>

    <div style="background-color: #e8f8f5; border-right: 5px solid #1abc9c; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #16a085; margin-top: 0;">📈 ביצועים פיננסיים ומדדי מערך המשלוחים (נקודת האופטימום)</h3>
        <table style="width: 100%; border-collapse: collapse; margin-top: 10px; font-size: 15px;">
            <thead>
                <tr style="background-color: #1abc9c; color: white;">
                    <th style="padding: 10px; text-align: right;">מדד ביצוע תפעולי וכלכלי</th>
                    <th style="padding: 10px; text-align: center;">ערך שהתקבל בפלט</th>
                </tr>
            </thead>
            <tbody>
                <tr><td style="padding: 10px; border: 1px solid #ddd;">💰 רווח גולמי מקורי למתחם (לפני קנסות)</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #27ae60;">{int(round(best_data['raw_profit'])):,} ₪</td></tr>
                <tr style="background-color: #fdf2f2;"><td style="padding: 10px; border: 1px solid #ddd;">🛑 סך קנס איחורי שליחים שהושת</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #c0392b;">{int(round(best_data['total_penalty'])):,} ₪</td></tr>
                <tr style="background-color: #eaf2f8; font-weight: bold;"><td style="padding: 10px; border: 1px solid #ddd; color: #2980b9;">💎 סך הכל רווח גלובלי נטו למתחם (לאחר קנסות)</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; color: #2980b9; font-size: 16px;">{int(round(best_data['net_profit'])):,} ₪</td></tr>
                <tr style="border-top: 2px solid #ddd;"><td style="padding: 10px; border: 1px solid #ddd;">📊 אחוז ניצולת שליחים ממוצעת</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #8e44ad;">{best_data['courier_utilization']:.2f}%</td></tr>
                <tr><td style="padding: 10px; border: 1px solid #ddd;">⏱️ זמן משלוח ממוצע כולל</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #e67e22;">{best_data['avg_delivery_time']:.2f} דקות</td></tr>
                <tr style="background-color: #f9f9f9;"><td style="padding: 10px; border: 1px solid #ddd;">🚨 אחוז ההזמנות המאחרות (מעל 40 דק')</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #c0392b;">{best_data['late_percentage']:.2f}%</td></tr>
            </tbody>
        </table>
    </div>
</div>
"""
display(HTML(html_output))

# =========================================================================
# 📊 טבלה מעודכנת: ניתוח עומס וייצור שעתי דינמי ומפורט
# =========================================================================
display(HTML("<h3 style='font-family: Segoe UI; color: #2c3e50; direction: rtl; text-align: right; padding-right: 20px;'>⏱️ ניתוח עומס וייצור שעתי מפורט במתחם (אופטימיזציה חופשית לכל שעה)</h3>"))

load_rows = []
for t in hours:
    total_dishes = best_data["hourly_orders"][t]
    prep_mins = hourly_prep_minutes[t]
    worker_added = int(round(best_data["lp_Y"][t].varValue or 0))
    capacity = 360 if worker_added == 1 else 240
    util_rate = (prep_mins / capacity) * 100 if capacity > 0 else 0
    
    g_count = hourly_by_restaurant_counts[t]["G"]
    v_count = hourly_by_restaurant_counts[t]["V"]
    b_count = hourly_by_restaurant_counts[t]["B"]
    
    load_rows.append({
        "שעה": f"<b>{t}:00</b>",
        "הירוקה (בינוני)": f"{g_count} יח'",
        "ויה איטלי (בינוני)": f"{v_count} יח'",
        "בשרוני (גבוה)": f"{b_count} יח'",
        "סך מנות במתחם": f"<b>{total_dishes} יח'</b>",
        "דקות הכנה": f"{prep_mins} דק'",
        "קיבולת מטבח": f"{capacity} דק'",
        "ניצולת מטבח": f"<b>{util_rate:.1f}%</b>"
    })

df_load = pd.DataFrame(load_rows)
try:
    styled_load = df_load.style.set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '10px', 'border-bottom': '1px solid #ddd'
    }).hide(axis='index')
except:
    styled_load = df_load.style.set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '10px', 'border-bottom': '1px solid #ddd'
    }).hide_index()

display(styled_load)


מדד ביצוע תפעולי וכלכלי,ערך שהתקבל בפלט
💰 רווח גולמי מקורי למתחם (לפני קנסות),"11,210 ₪"
🛑 סך קנס איחורי שליחים שהושת,"51,791 ₪"
💎 סך הכל רווח גלובלי נטו למתחם (לאחר קנסות),"-40,581 ₪"
📊 אחוז ניצולת שליחים ממוצעת,134.51%
⏱️ זמן משלוח ממוצע כולל,119.87 דקות
🚨 אחוז ההזמנות המאחרות (מעל 40 דק'),75.09%


שעה,הירוקה (בינוני),ויה איטלי (בינוני),בשרוני (גבוה),סך מנות במתחם,דקות הכנה,קיבולת מטבח,ניצולת מטבח
10:00,39 יח',10 יח',0 יח',49 יח',359 דק',360 דק',99.7%
11:00,19 יח',0 יח',24 יח',43 יח',357 דק',360 דק',99.2%
12:00,0 יח',2 יח',10 יח',12 יח',170 דק',240 דק',70.8%
13:00,0 יח',17 יח',0 יח',17 יח',238 דק',240 דק',99.2%
14:00,0 יח',0 יח',19 יח',19 יח',240 דק',240 דק',100.0%
15:00,18 יח',34 יח',0 יח',52 יח',360 דק',360 דק',100.0%


In [40]:
# pip install pulp pandas numpy scipy
import pulp as pl
import itertools
import pandas as pd
import numpy as np
import scipy.stats as stats
from IPython.display import display, HTML

# =========================================================================
# ⚠️ שלב 1: נתוני בסיס קבועים
# =========================================================================
v_single = {"G": 3876, "V": 3138, "B": 3485}
bonus_weights = {"G": 0.357, "V": 0.294, "B": 0.348}

hours = [10, 11, 12, 13, 14, 15]
dishes = ["BH", "BS", "BP", "BSh", "BL", "VP", "VPst", "VR", "VN", "VZ", "GH", "GM", "GQ", "GI"]

profit_base = {"BH": 50, "BS": 110, "BP": 50, "BSh": 45, "BL": 45, "VP": 66, "VPst": 40, "VR": 65, "VN": 57, "VZ": 48, "GH": 66, "GM": 40, "GQ": 65, "GI": 57}
prep_time = {"BH": 13, "BS": 12, "BP": 10, "BSh": 10, "BL": 15, "VP": 14, "VPst": 7, "VR": 9, "VN": 9, "VZ": 8, "GH": 5, "GM": 7, "GQ": 7, "GI": 6}
demand_base = {"BH": 20, "BS": 10, "BP": 13, "BSh": 16, "BL": 7, "VP": 17, "VPst": 14, "VR": 11, "VN": 9, "VZ": 6, "GH": 17, "GM": 17, "GQ": 17, "GI": 17}

restaurants = {"G": "הירוקה", "V": "ויה איטלי", "B": "בשרוני"}
players = ["G", "V", "B"]
strategies = ["L", "M", "H"]
strategy_names = {"L": "נמוך", "M": "בינוני", "H": "גבוה"}

dish_names_hebrew = {
    "BH": "המבורגר", "BS": "סטייק", "BP": "פרגית", "BSh": "שניצל", "BL": "כבד עוף",
    "VP": "פיצה", "VPst": "פסטה", "VR": "רביולי", "VN": "ניוקי", "VZ": "ריזוטו",
    "GH": "סלט הבית", "GM": "מג'דרה", "GQ": "סלט קינואה", "GI": "סלט אישי"
}

NUM_COURIERS = 5
MU_SERVICE = 6.0  
K_ERLANG = 2      
C2_SERVICE = 1.0 / K_ERLANG  

def get_modified_data(pricing_profile):
    current_profit = {}
    current_demand = {}
    for d in dishes:
        p_owner = d[0]
        strat = pricing_profile[p_owner]
        
        if strat == "M": current_profit[d] = profit_base[d]
        elif strat == "L": current_profit[d] = profit_base[d] * 0.90
        elif strat == "H": current_profit[d] = profit_base[d] * 1.10
        
        demand_mod = 0.0
        if strat == "L": demand_mod += 0.20
        elif strat == "H": demand_mod -= 0.20
            
        for opp in players:
            if opp != p_owner:
                opp_strat = pricing_profile[opp]
                if opp_strat == "L": demand_mod -= 0.10
                elif opp_strat == "H": demand_mod += 0.10
                    
        current_demand[d] = max(0, int(round(demand_base[d] * (1 + demand_mod))))
    return current_profit, current_demand

def calculate_queue_metrics(hourly_orders_count):
    if hourly_orders_count <= 0:
        return 10.0, 0.0, 0.0, 0.0
    
    raw_rho = hourly_orders_count / (NUM_COURIERS * MU_SERVICE)
    if hourly_orders_count >= NUM_COURIERS * MU_SERVICE * 0.99:
        hourly_orders_count = NUM_COURIERS * MU_SERVICE * 0.99
        
    lam = hourly_orders_count
    c = NUM_COURIERS
    rho = lam / (c * MU_SERVICE)
    
    sum_terms = 1.0
    val = 1.0
    for i in range(1, c):
        val *= (lam / MU_SERVICE) / i
        sum_terms += val
    val *= (lam / MU_SERVICE) / c
    p_w = (val / (1.0 - rho)) / (sum_terms + (val / (1.0 - rho)))
    
    c2_arrival = 1.0  
    w_q_hours = (p_w / (c * MU_SERVICE)) * ((c2_arrival + C2_SERVICE) / 2.0) / (1.0 - rho)
    w_q_minutes = w_q_hours * 60.0
    
    total_delivery_time_minutes = w_q_minutes + 10.0
    time_left_for_drive = max(0.0, 40.0 - w_q_minutes)
    prob_late = 1.0 - stats.erlang.cdf(time_left_for_drive, K_ERLANG, scale=1.0/0.2)
    
    avg_excess_time = max(0.0, total_delivery_time_minutes - 40.0)
    hourly_penalty = lam * prob_late * avg_excess_time * 5.0
    
    return total_delivery_time_minutes, prob_late, hourly_penalty, raw_rho

def solve_lp_with_internal_tax(selected_restaurants, current_profit, current_demand, hourly_taxes):
    model = pl.LpProblem("Optimization", pl.LpMaximize)
    X = pl.LpVariable.dicts("X", [(d, t) for d in dishes if d[0] in selected_restaurants for t in hours], lowBound=0, cat="Integer")
    Y = pl.LpVariable.dicts("Y", hours, lowBound=0, upBound=1, cat="Binary")
    
    model += pl.lpSum((current_profit[d] - hourly_taxes[t]) * X[(d, t)] for d in dishes if d[0] in selected_restaurants for t in hours) - pl.lpSum(60 * Y[t] for t in hours)
    
    # אילוץ ביקוש יומי כולל (חופש אופטימיזציה מלא לשעות)
    for d in dishes:
        if d[0] in selected_restaurants:
            model += pl.lpSum(X[(d, t)] for t in hours) <= current_demand[d]
            
    # אילוץ קיבולת מטבח שעתי קשיח
    for t in hours:
        model += pl.lpSum(prep_time[d] * X[(d, t)] for d in dishes if d[0] in selected_restaurants) <= 240 + 120 * Y[t]
        
    model.solve(pl.PULP_CBC_CMD(msg=0))
    return pl.value(model.objective), X, Y

# =========================================================================
# 🧮 שלב 2: סריקת פרופילים ואינטגרציית מדדים מורחבת
# =========================================================================
global_results = {}
all_profiles = list(itertools.product(strategies, repeat=3))

shadow_taxes = {10: 0, 11: 0, 12: 14, 13: 14, 14: 14, 15: 0}
no_taxes = {10: 0, 11: 0, 12: 0, 13: 0, 14: 0, 15: 0}

best_profile = ("M", "M", "H")

for profile in all_profiles:
    profile_dict = {"G": profile[0], "V": profile[1], "B": profile[2]}
    current_profit, current_demand = get_modified_data(profile_dict)
    
    if profile == best_profile:
        _, lp_vars, lp_Y = solve_lp_with_internal_tax(players, current_profit, current_demand, shadow_taxes)
    else:
        _, lp_vars, lp_Y = solve_lp_with_internal_tax(players, current_profit, current_demand, no_taxes)
    
    raw_profit = sum(current_profit[d] * int(round(lp_vars[(d, t)].varValue or 0)) for d in dishes for t in hours)
    raw_profit = raw_profit if raw_profit > 0 else 0
    
    hourly_orders = {t: 0 for t in hours}
    for (d, t) in lp_vars:
        hourly_orders[t] += int(round(lp_vars[(d, t)].varValue or 0))
        
    total_penalty = 0.0
    weighted_delivery_time = 0.0
    weighted_late_prob = 0.0
    weighted_rho = 0.0
    total_orders_day = sum(hourly_orders.values())
    
    for t in hours:
        avg_time, p_late, penalty, r_rho = calculate_queue_metrics(hourly_orders[t])
        total_penalty += penalty
        if total_orders_day > 0:
            weighted_delivery_time += avg_time * (hourly_orders[t] / total_orders_day)
            weighted_late_prob += p_late * (hourly_orders[t] / total_orders_day)
            weighted_rho += r_rho * (hourly_orders[t] / total_orders_day)
            
    net_global_profit = (raw_profit - total_penalty) 
    
    global_results[profile] = {
        "net_profit": net_global_profit,
        "raw_profit": raw_profit,
        "total_penalty": total_penalty,
        "avg_delivery_time": weighted_delivery_time,
        "late_percentage": weighted_late_prob * 100,
        "courier_utilization": weighted_rho * 100,
        "lp_vars": lp_vars,
        "lp_Y": lp_Y,
        "hourly_orders": hourly_orders
    }

best_data = global_results[best_profile]

# =========================================================================
# 🏛️ שלב 3: חלוקת רווחים סופית ומסונכרנת
# =========================================================================
v_single_total = sum(v_single.values())
current_net_bonus = best_data["net_profit"] - v_single_total

final_restaurant_payoffs = {}
for p in players:
    final_restaurant_payoffs[p] = int(round(v_single[p] + current_net_bonus * bonus_weights[p]))

# =========================================================================
# ✨ שלב 4: הפקת הדו"ח הניהולי המורחב ב-HTML
# =========================================================================
produced_dishes_by_rest = {"הירוקה": set(), "ויה איטלי": set(), "בשרוני": set()}
hourly_production = {t: {r: [] for r in restaurants.values()} for t in hours}
hourly_prep_minutes = {t: 0 for t in hours}
hourly_by_restaurant_counts = {t: {"G": 0, "V": 0, "B": 0} for t in hours}

for (d, t) in best_data["lp_vars"]:
    qty = int(round(best_data["lp_vars"][(d, t)].varValue or 0))
    if qty > 0:
        rest_code = d[0]
        rest_name = restaurants[rest_code]
        dish_hebrew = dish_names_hebrew[d]
        produced_dishes_by_rest[rest_name].add(dish_hebrew)
        hourly_production[t][rest_name].append(f"{dish_hebrew} ({qty} יח')")
        hourly_prep_minutes[t] += qty * prep_time[d]
        hourly_by_restaurant_counts[t][rest_code] += qty

html_output = f"""
<div style="font-family: 'Segoe UI', Arial, sans-serif; direction: rtl; text-align: right; padding: 20px;">
    <h2 style="color: #2c3e50; border-bottom: 3px solid #e67e22; padding-bottom: 10px;">🚚 דוח סיכום מנהלים: חלק ד' – אינטגרציית מערך המשלוחים ומדדי ביצוע מורחבים</h2>
    
    <div style="background-color: #ebf5fb; border-right: 5px solid #2980b9; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #2980b9; margin-top: 0;">🎯 אסטרטגיית תמחור מומלצת באופטימום הגלובלי</h3>
        <p style="text-align: center;">
           <span style="background-color: #2980b9; color: white; padding: 6px 15px; border-radius: 4px; font-weight: bold; font-size: 16px;">
               הירוקה: בינוני | ויה איטלי: בינוני | בשרוני: גבוה
           </span>
        </p>
    </div>

    <div style="background-color: #e8f8f5; border-right: 5px solid #1abc9c; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #16a085; margin-top: 0;">📈 ביצועים פיננסיים ומדדי מערך המשלוחים (נקודת האופטימום)</h3>
        <table style="width: 100%; border-collapse: collapse; margin-top: 10px; font-size: 15px;">
            <thead>
                <tr style="background-color: #1abc9c; color: white;">
                    <th style="padding: 10px; text-align: right;">מדד ביצוע תפעולי וכלכלי</th>
                    <th style="padding: 10px; text-align: center;">ערך שהתקבל בפלט</th>
                </tr>
            </thead>
            <tbody>
                <tr><td style="padding: 10px; border: 1px solid #ddd;">💰 רווח גולמי מקורי למתחם (לפני קנסות)</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #27ae60;">{int(round(best_data['raw_profit'])):,} ₪</td></tr>
                <tr style="background-color: #fdf2f2;"><td style="padding: 10px; border: 1px solid #ddd;">🛑 סך קנס איחורי שליחים שהושת</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #c0392b;">{int(round(best_data['total_penalty'])):,} ₪</td></tr>
                <tr style="background-color: #eaf2f8; font-weight: bold;"><td style="padding: 10px; border: 1px solid #ddd; color: #2980b9;">💎 סך הכל רווח גלובלי נטו למתחם (לאחר קנסות)</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; color: #2980b9; font-size: 16px;">{int(round(best_data['net_profit'])):,} ₪</td></tr>
                <tr style="border-top: 2px solid #ddd;"><td style="padding: 10px; border: 1px solid #ddd;">📊 אחוז ניצולת שליחים ממוצעת</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #8e44ad;">{best_data['courier_utilization']:.2f}%</td></tr>
                <tr><td style="padding: 10px; border: 1px solid #ddd;">⏱️ זמן משלוח ממוצע כולל</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #e67e22;">{best_data['avg_delivery_time']:.2f} דקות</td></tr>
                <tr style="background-color: #f9f9f9;"><td style="padding: 10px; border: 1px solid #ddd;">🚨 אחוז ההזמנות המאחרות (מעל 40 דק')</td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #c0392b;">{best_data['late_percentage']:.2f}%</td></tr>
                <tr style="border-top: 2px solid #ddd;"><td style="padding: 10px; border: 1px solid #ddd;">💵 רווח סופי צפוי ל<b>מסעדת הירוקה</b></td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['G']:,} ₪</td></tr>
                <tr><td style="padding: 10px; border: 1px solid #ddd;">💵 רווח סופי צפוי ל<b>מסעדת ויה איטלי</b></td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['V']:,} ₪</td></tr>
                <tr><td style="padding: 10px; border: 1px solid #ddd;">💵 רווח סופי צפוי ל<b>מסעדת בשרוני</b></td><td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['B']:,} ₪</td></tr>
            </tbody>
        </table>
    </div>
</div>
"""
display(HTML(html_output))

# =========================================================================
# 📊 טבלה 2: ניתוח עומס וייצור שעתי דינמי ומפורט
# =========================================================================
display(HTML("<h3 style='font-family: Segoe UI; color: #2c3e50; direction: rtl; text-align: right; padding-right: 20px;'>⏱️ ניתוח עומס וייצור שעתי מפורט במתחם (אופטימיזציה חופשית לכל שעה)</h3>"))

load_rows = []
for t in hours:
    total_dishes = best_data["hourly_orders"][t]
    prep_mins = hourly_prep_minutes[t]
    worker_added = int(round(best_data["lp_Y"][t].varValue or 0))
    capacity = 360 if worker_added == 1 else 240
    util_rate = (prep_mins / capacity) * 100 if capacity > 0 else 0
    
    g_count = hourly_by_restaurant_counts[t]["G"]
    v_count = hourly_by_restaurant_counts[t]["V"]
    b_count = hourly_by_restaurant_counts[t]["B"]
    
    load_rows.append({
        "שעה": f"<b>{t}:00</b>",
        "הירוקה (בינוני)": f"{g_count} יח'",
        "ויה איטלי (בינוני)": f"{v_count} יח'",
        "בשרוני (גבוה)": f"{b_count} יח'",
        "סך מנות במתחם": f"<b>{total_dishes} יח'</b>",
        "דקות הכנה": f"{prep_mins} דק'",
        "קיבולת מטבח": f"{capacity} דק'",
        "ניצולת מטבח": f"<b>{util_rate:.1f}%</b>"
    })

df_load = pd.DataFrame(load_rows)
try:
    styled_load = df_load.style.set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '10px', 'border-bottom': '1px solid #ddd'
    }).hide(axis='index')
except:
    styled_load = df_load.style.set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '10px', 'border-bottom': '1px solid #ddd'
    }).hide_index()

display(styled_load)
display(HTML("<br>"))

# =========================================================================
# 📅 שלב 5: לוח זמנים שעתי
# =========================================================================
vertical_rows = []
for t in hours:
    row_dict = {"שעה": f"<b>{t}:00</b>"}
    for r in restaurants.values():
        dishes_in_hour = hourly_production[t][r]
        row_dict[r] = ", ".join(dishes_in_hour) if dishes_in_hour else "<span style='color: #aaa;'>-</span>"
    vertical_rows.append(row_dict)

df_vertical_schedule = pd.DataFrame(vertical_rows)[["שעה", "הירוקה", "ויה איטלי", "בשרוני"]]

try:
    styled_sched = df_vertical_schedule.style.set_properties(**{'text-align': 'right', 'font-family': 'Segoe UI', 'padding': '12px', 'border-bottom': '2px solid #b2bec3'}).hide(axis='index')
except:
    styled_sched = df_vertical_schedule.style.set_properties(**{'text-align': 'right', 'font-family': 'Segoe UI', 'padding': '12px', 'border-bottom': '2px solid #b2bec3'}).hide_index()
display(styled_sched)

# =========================================================================
# 🗓️ שלב 6: נספח מטריצת תשלומים מרוכזת (כל 27 השילובים מורחב ומעודכן!)
# =========================================================================
display(HTML("<br><hr><h2 style='font-family: Segoe UI; color: #2c3e50; direction: rtl; text-align: right;'>🗓️ נספח: מטריצת תשלומים מרוכזת (27 השילובים תחת אופטימיזציה דינמית)</h2>"))

collapsed_data = []
for profile in all_profiles:
    g_strat, v_strat, b_strat = profile
    res = global_results[profile]
    is_best_row = (profile == best_profile)
    
    net_bonus_for_profile = res["net_profit"] - v_single_total
    p_payoffs = {}
    for p in players:
        p_payoffs[p] = int(round(v_single[p] + net_bonus_for_profile * bonus_weights[p]))
        
    collapsed_data.append({
        "תמחור הירוקה": strategy_names[g_strat],
        "תמחור ויה איטלי": strategy_names[v_strat],
        "תמחור בשרוני": strategy_names[b_strat],
        "רווח מקורי (₪)": f"{int(round(res['raw_profit'])):,}",
        "קנס שליחים (₪)": f"{int(round(res['total_penalty'])):,}",
        "רווח נטו גלובלי (₪)": f"{int(round(res['net_profit'])):,}",
        "ניצולת שליחים": f"{res['courier_utilization']:.1f}%",
        "זמן משלוח (דק')": f"{res['avg_delivery_time']:.1f}",
        "אחוז איחורים": f"{res['late_percentage']:.1f}%",
        "סטטוס": "🏆 אופטימום" if is_best_row else "רגיל"
    })

df_flat_matrix = pd.DataFrame(collapsed_data)

def style_nash_row(row):
    if row['סטטוס'] != "רגיל":
        return ['background-color: #d1ecf1; color: #0c5460; font-weight: bold; border: 2px solid #17a2b8;'] * len(row)
    return [''] * len(row)

try:
    styled_flat = df_flat_matrix.style.apply(style_nash_row, axis=1).set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '8px', 'border': '1px solid #ddd'
    }).hide(axis='index')
except:
    styled_flat = df_flat_matrix.style.apply(style_nash_row, axis=1).set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '8px', 'border': '1px solid #ddd'
    }).hide_index()

display(styled_flat)

מדד ביצוע תפעולי וכלכלי,ערך שהתקבל בפלט
💰 רווח גולמי מקורי למתחם (לפני קנסות),"11,210 ₪"
🛑 סך קנס איחורי שליחים שהושת,"51,791 ₪"
💎 סך הכל רווח גלובלי נטו למתחם (לאחר קנסות),"-40,581 ₪"
📊 אחוז ניצולת שליחים ממוצעת,134.51%
⏱️ זמן משלוח ממוצע כולל,119.87 דקות
🚨 אחוז ההזמנות המאחרות (מעל 40 דק'),75.09%
💵 רווח סופי צפוי למסעדת הירוקה,"-14,359 ₪"
💵 רווח סופי צפוי למסעדת ויה איטלי,"-11,879 ₪"
💵 רווח סופי צפוי למסעדת בשרוני,"-14,291 ₪"


שעה,הירוקה (בינוני),ויה איטלי (בינוני),בשרוני (גבוה),סך מנות במתחם,דקות הכנה,קיבולת מטבח,ניצולת מטבח
10:00,39 יח',10 יח',0 יח',49 יח',359 דק',360 דק',99.7%
11:00,19 יח',0 יח',24 יח',43 יח',357 דק',360 דק',99.2%
12:00,0 יח',2 יח',10 יח',12 יח',170 דק',240 דק',70.8%
13:00,0 יח',17 יח',0 יח',17 יח',238 דק',240 דק',99.2%
14:00,0 יח',0 יח',19 יח',19 יח',240 דק',240 דק',100.0%
15:00,18 יח',34 יח',0 יח',52 יח',360 דק',360 דק',100.0%


שעה,הירוקה,ויה איטלי,בשרוני
10:00,"סלט הבית (2 יח'), מג'דרה (18 יח'), סלט קינואה (19 יח')",ניוקי (10 יח'),-
11:00,"מג'דרה (1 יח'), סלט אישי (18 יח')",-,"סטייק (1 יח'), פרגית (10 יח'), שניצל (13 יח')"
12:00,-,פיצה (2 יח'),"המבורגר (4 יח'), כבד עוף (6 יח')"
13:00,-,פיצה (17 יח'),-
14:00,-,-,"המבורגר (12 יח'), סטייק (7 יח')"
15:00,"סלט הבית (17 יח'), סלט אישי (1 יח')","פסטה (15 יח'), רביולי (12 יח'), ריזוטו (7 יח')",-


תמחור הירוקה,תמחור ויה איטלי,תמחור בשרוני,רווח מקורי (₪),קנס שליחים (₪),רווח נטו גלובלי (₪),ניצולת שליחים,זמן משלוח (דק'),אחוז איחורים,סטטוס
נמוך,נמוך,נמוך,"9,773","69,055","-59,282",114.4%,125.4,78.6%,רגיל
נמוך,נמוך,בינוני,"10,120","69,055","-58,935",118.4%,128.6,80.8%,רגיל
נמוך,נמוך,גבוה,"10,050","51,791","-41,741",108.9%,98.6,58.6%,רגיל
נמוך,בינוני,נמוך,"10,249","69,055","-58,806",116.1%,124.8,78.1%,רגיל
נמוך,בינוני,בינוני,"10,305","72,332","-62,027",106.0%,124.0,100.0%,רגיל
נמוך,בינוני,גבוה,"10,616","69,055","-58,438",114.2%,125.4,78.6%,רגיל
נמוך,גבוה,נמוך,"10,289","69,055","-58,766",113.6%,124.2,77.8%,רגיל
נמוך,גבוה,בינוני,"10,770","69,055","-58,285",114.6%,124.5,77.9%,רגיל
נמוך,גבוה,גבוה,"11,074","53,430","-42,356",115.8%,108.7,77.9%,רגיל
בינוני,נמוך,נמוך,"10,140","86,319","-76,179",114.0%,143.4,91.2%,רגיל


In [42]:
import pulp as pl
import pandas as pd
import numpy as np
import scipy.stats as stats
import itertools
from IPython.display import display, HTML

# -------------------------------------------------------------
# 1. הגדרת נתוני בסיס ופרמטרים
# -------------------------------------------------------------
hours = [10, 11, 12, 13, 14, 15]
dishes = ["BH", "BS", "BP", "BSh", "BL", "VP", "VPst", "VR", "VN", "VZ", "GH", "GM", "GQ", "GI"]

dish_to_restaurant = {d: d[0] for d in dishes}
restaurant_names = {"G": "הירוקה", "V": "ויה איטלי", "B": "בשרוני"}

profit_base = {"BH": 50, "BS": 110, "BP": 50, "BSh": 45, "BL": 45, "VP": 66, "VPst": 40, "VR": 65, "VN": 57, "VZ": 48, "GH": 66, "GM": 40, "GQ": 65, "GI": 57}
prep_time = {"BH": 13, "BS": 12, "BP": 10, "BSh": 10, "BL": 15, "VP": 14, "VPst": 7, "VR": 9, "VN": 9, "VZ": 8, "GH": 5, "GM": 7, "GQ": 7, "GI": 6}
demand_base = {"BH": 20, "BS": 10, "BP": 13, "BSh": 16, "BL": 7, "VP": 17, "VPst": 14, "VR": 11, "VN": 9, "VZ": 6, "GH": 17, "GM": 17, "GQ": 17, "GI": 17}

NUM_COURIERS = 5
MU_SERVICE = 6.0  # 6 משלוחים בשעה לשליח (10 דקות ממוצע)
K_ERLANG = 2      # פרמטר הצורה של התפלגות ארלנג לזמן הנסיעה

# -------------------------------------------------------------
# 2. פונקציה להתאמת רווח וביקוש לפי פרופיל תמחור (חלק ג' + ד')
# -------------------------------------------------------------
def get_modified_data(profile):
    current_profit = {}
    current_demand = {}
    for d in dishes:
        res = dish_to_restaurant[d]
        strat = profile[res]
        
        # התאמת מחיר/רווח
        if strat == "M": current_profit[d] = profit_base[d]
        elif strat == "L": current_profit[d] = profit_base[d] * 0.90
        elif strat == "H": current_profit[d] = profit_base[d] * 1.10
        
        # התאמת ביקוש
        demand_mod = 0.0
        if strat == "L": demand_mod += 0.20
        elif strat == "H": demand_mod -= 0.20
            
        for opp in ["G", "V", "B"]:
            if opp != res:
                if profile[opp] == "L": demand_mod -= 0.10
                elif profile[opp] == "H": demand_mod += 0.10
                    
        current_demand[d] = max(0, int(round(demand_base[d] * (1 + demand_mod))))
    return current_profit, current_demand

# -------------------------------------------------------------
# 3. מודל תור וחישוב מדדי משלוחים (M/G/c קורס או יציב)
# -------------------------------------------------------------
def calculate_delivery_metrics(lam):
    if lam <= 0:
        return 10.0, 0.0, 0.0, 0.0
    
    rho = lam / (NUM_COURIERS * MU_SERVICE)
    
    # תנאי קריסה: אם קצב ההגעה חוסם או עובר את קיבולת הצי (30 מנות בשעה)
    if rho >= 0.99:
        w_q_minutes = 120.0  # הנחת עבודה במצב קריסה/חסימה קיצונית
        total_time = w_q_minutes + 10.0
        prob_late = 1.0     # 100% איחור
        penalty = lam * prob_late * (total_time - 40.0) * 5.0
        return total_time, prob_late, penalty, rho
    
    # חישוב הסתברות להמתנה בתור (Erlang-C Approximation)
    sum_terms = 1.0
    val = 1.0
    for i in range(1, NUM_COURIERS):
        val *= (lam / MU_SERVICE) / i
        sum_terms += val
    val *= (lam / MU_SERVICE) / NUM_COURIERS
    p_w = (val / (1.0 - rho)) / (sum_terms + (val / (1.0 - rho)))
    
    # קירוב אלן-קוניגהאם עבור תור M/G/c
    c2_arrival = 1.0
    c2_service = 1.0 / K_ERLANG  # 0.5 עבור Erlang(k=2)
    w_q_hours = (p_w / (NUM_COURIERS * MU_SERVICE)) * ((c2_arrival + c2_service) / 2.0) / (1.0 - rho)
    w_q_minutes = w_q_hours * 60.0
    
    total_time = w_q_minutes + 10.0
    time_left_for_drive = max(0.0, 40.0 - w_q_minutes)
    
    # הסתברות איחור מתוך ה-CDF של ארלנג
    prob_late = 1.0 - stats.erlang.cdf(time_left_for_drive, K_ERLANG, scale=1.0/0.2)
    excess_time = max(0.0, total_time - 40.0)
    penalty = lam * prob_late * excess_time * 5.0
    
    return total_time, prob_late, penalty, rho

# -------------------------------------------------------------
# 4. הרצת LP גלובלי משותף (מודל חלק ב')
# -------------------------------------------------------------
def solve_global_lp(profile, taxes={t: 0 for t in hours}):
    current_profit, current_demand = get_modified_data(profile)
    
    model = pl.LpProblem("Global_Optimization", pl.LpMaximize)
    X = pl.LpVariable.dicts("X", ((d, t) for d in dishes for t in hours), lowBound=0, cat="Integer")
    Y = pl.LpVariable.dicts("Y", hours, lowBound=0, upBound=1, cat="Binary")
    
    # פונקציית מטרה הכוללת ניכוי מס פנימי היפותטי
    model += (
        pl.lpSum((current_profit[d] - taxes[t]) * X[(d, t)] for d in dishes for t in hours)
        - 60 * pl.lpSum(Y[t] for t in hours)
    )
    
    for d in dishes:
        model += pl.lpSum(X[(d, t)] for t in hours) <= current_demand[d]
        
    for t in hours:
        model += pl.lpSum(prep_time[d] * X[(d, t)] for d in dishes) <= 240 + 120 * Y[t]
        
    model.solve(pl.PULP_CBC_CMD(msg=0))
    
    # איסוף כמויות הייצור
    production_results = {t: {r: 0 for r in ["G", "V", "B"]} for t in hours}
    dish_breakdown = {t: {} for t in hours}
    
    for t in hours:
        for d in dishes:
            qty = int(X[(d, t)].varValue or 0)
            if qty > 0:
                production_results[t][dish_to_restaurant[d]] += qty
                dish_breakdown[t][d] = qty
                
    raw_kitchen_profit = sum(current_profit[d] * int(X[(d, t)].varValue or 0) for d in dishes for t in hours)
    worker_cost = sum(60 * int(Y[t].varValue or 0) for t in hours)
    
    return raw_kitchen_profit - worker_cost, production_results, dish_breakdown

# -------------------------------------------------------------
# 5. הרצה עבור הפרופיל הנבחר והפקת דוח המנות המבוקש
# -------------------------------------------------------------
selected_profile = {"G": "M", "V": "M", "B": "H"}
# הגדרת המס הפנימי לשעות העומס לצורך ריסון אופטימלי
shadow_taxes = {10: 14, 11: 14, 12: 14, 13: 0, 14: 14, 15: 0}

net_kitchen_profit, qty_results, breakdown = solve_global_lp(selected_profile, shadow_taxes)

summary_rows = []
total_penalty_accum = 0.0

for t in hours:
    g_qty = qty_results[t]["G"]
    v_qty = qty_results[t]["V"]
    b_qty = qty_results[t]["B"]
    total_mnot = g_qty + v_qty + b_qty
    
    # חישוב ביצועי השליחים על בסיס סך המנות בפועל
    avg_time, p_late, penalty, rho = calculate_delivery_metrics(total_mnot)
    total_penalty_accum += penalty
    
    summary_rows.append({
        "שעה": f"{t}:00–{t+1}:00",
        "הירוקה (מנות)": g_qty,
        "ויה איטלי (מנות)": v_qty,
        "בשרוני (מנות)": b_qty,
        "סך מנות במתחם": f"<b>{total_mnot} יח'</b>",
        "ניצולת שליחים": f"{rho*100:.1f}%",
        "זמן משלוח ממוצע": f"{avg_time:.1f} דק'",
        "אחוז איחורים": f"{p_late*100:.1f}%",
        "קנס שעתי צפוי": f"{int(round(penalty))} ₪"
    })

df_summary = pd.DataFrame(summary_rows)

display(HTML("<h2 style='font-family: Segoe UI; direction: rtl; text-align: right;'>📋 סיכום מנות שעתי ומדדי ביצוע שליחים (חלק ד')</h2>"))
try:
    display(df_summary.style.set_properties(**{'text-align': 'center', 'font-family': 'Segoe UI'}).hide(axis='index'))
except:
    display(df_summary.style.set_properties(**{'text-align': 'center', 'font-family': 'Segoe UI'}).hide_index())

print(f"\n--- סיכום פיננסי גלובלי ---")
print(f"רווח נטו מהמטבח (לאחר עלות עובדים): {net_kitchen_profit:,} ₪")
print(f"סך קנסות משלוחים שנגרמו למתחם: {int(round(total_penalty_accum)):,} ₪")
print(f"רווח מתחם סופי (אופטימום גלובלי): {int(round(net_kitchen_profit - total_penalty_accum)):,} ₪")

שעה,הירוקה (מנות),ויה איטלי (מנות),בשרוני (מנות),סך מנות במתחם,ניצולת שליחים,זמן משלוח ממוצע,אחוז איחורים,קנס שעתי צפוי
10:00–11:00,0,9,9,18 יח',60.0%,10.9 דק',0.4%,0 ₪
11:00–12:00,0,1,26,27 יח',90.0%,21.4 דק',2.2%,0 ₪
12:00–13:00,0,20,0,20 יח',66.7%,11.5 דק',0.4%,0 ₪
13:00–14:00,54,2,0,56 יח',186.7%,130.0 דק',100.0%,25200 ₪
14:00–15:00,0,2,18,20 יח',66.7%,11.5 דק',0.4%,0 ₪
15:00–16:00,22,29,0,51 יח',170.0%,130.0 דק',100.0%,22950 ₪



--- סיכום פיננסי גלובלי ---
רווח נטו מהמטבח (לאחר עלות עובדים): 11,030.5 ₪
סך קנסות משלוחים שנגרמו למתחם: 48,150 ₪
רווח מתחם סופי (אופטימום גלובלי): -37,120 ₪


In [45]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from IPython.display import display, HTML

# -------------------------------------------------------------
# 1. קיבוע נתוני הייצור המדויקים מתוך חלק ב' שלכם
# -------------------------------------------------------------
hours_label = ["10:00–11:00", "11:00–12:00", "12:00–13:00", "13:00–14:00", "14:00–15:00", "15:00–16:00"]

# סך המנות המיוצרות במתחם בכל שעה על פי פלט חלק ב' שלכם
mnot_per_hour = [37, 36, 33, 24, 44, 17] 

# חלוקת המנות הפנימית בין המסעדות בכל שעה כפי שמופיעה בלוח העבודה
g_mnot = [17, 17, 17, 0, 17, 0]   # הירוקה
v_mnot = [9, 0, 9, 8, 14, 17]    # ויה איטלי
b_mnot = [11, 19, 7, 16, 13, 0]   # בשרוני

NUM_COURIERS = 5
MU_SERVICE = 6.0  # קיבולת שליח בודד: 6 משלוחים בשעה
K_ERLANG = 2      # נסיעה מתפלגת ארלנג עם k=2

# -------------------------------------------------------------
# 2. פונקציית התורים ההסתברותית (M/G/c)
# -------------------------------------------------------------
def calculate_delivery_metrics(lam):
    if lam <= 0:
        return 10.0, 0.0, 0.0, 0.0
    
    rho = lam / (NUM_COURIERS * MU_SERVICE)
    
    # במצב שבו קצב ההזמנות חוצה את קיבולת השליחים (30), התור קורס מתמטית
    if rho >= 1.0:
        w_q_minutes = 120.0  # זמן המתנה חסום היפותטי במצב רוויה/קריסה
        total_time = w_q_minutes + 10.0
        prob_late = 1.0      # 100% מההזמנות יאחרו את ה-40 דקות
        penalty = lam * prob_late * (total_time - 40.0) * 5.0
        return total_time, prob_late, penalty, rho
    
    # חישוב הסתברות להמתנה בתור לפי נוסחת ארלנג-C
    sum_terms = 1.0
    val = 1.0
    for i in range(1, NUM_COURIERS):
        val *= (lam / MU_SERVICE) / i
        sum_terms += val
    val *= (lam / MU_SERVICE) / NUM_COURIERS
    p_w = (val / (1.0 - rho)) / (sum_terms + (val / (1.0 - rho)))
    
    # קירוב אלן-קוניגהאם לזמן המתנה בתור (שונות ארלנג k=2 היא 0.5)
    c2_arrival = 1.0
    c2_service = 1.0 / K_ERLANG  
    w_q_hours = (p_w / (NUM_COURIERS * MU_SERVICE)) * ((c2_arrival + c2_service) / 2.0) / (1.0 - rho)
    w_q_minutes = w_q_hours * 60.0
    
    total_time = w_q_minutes + 10.0
    time_left_for_drive = max(0.0, 40.0 - w_q_minutes)
    
    # הסתברות איחור מתוך ה-CDF של ארלנג (scale = 1/lambda = 1/0.2 = 5)
    prob_late = 1.0 - stats.erlang.cdf(time_left_for_drive, K_ERLANG, scale=5.0)
    excess_time = max(0.0, total_time - 40.0)
    penalty = lam * prob_late * excess_time * 5.0
    
    return total_time, prob_late, penalty, rho

# -------------------------------------------------------------
# 3. חישוב וריכוז הנתונים לטבלה
# -------------------------------------------------------------
summary_rows = []
total_penalty_accum = 0.0

for i in range(len(hours_label)):
    lam = mnot_per_hour[i]
    avg_time, p_late, penalty, rho = calculate_delivery_metrics(lam)
    total_penalty_accum += penalty
    
    summary_rows.append({
        "שעה": hours_label[i],
        "הירוקה (מנות)": g_mnot[i],
        "ויה איטלי (מנות)": v_mnot[i],
        "בשרוני (מנות)": b_mnot[i],
        "סך מנות במתחם": f"<b>{lam} יח'</b>",
        "ניצולת שליחים": f"{rho*100:.1f}%",
        "זמן משלוח ממוצע": f"{avg_time:.1f} דק'" if rho < 1 else "קריסת תור (אינסוף)",
        "אחוז איחורים": f"{p_late*100:.1f}%",
        "קנס שעתי צפוי": f"{int(round(penalty))} ₪"
    })

df_summary = pd.DataFrame(summary_rows)

# הצגת הטבלה
display(HTML("<h2 style='font-family: Segoe UI; direction: rtl; text-align: right;'>📋 ניתוח מערך השליחים וקנסות על בסיס חלק ב'</h2>"))
try:
    display(df_summary.style.set_properties(**{'text-align': 'center', 'font-family': 'Segoe UI'}).hide(axis='index'))
except:
    display(df_summary.style.set_properties(**{'text-align': 'center', 'font-family': 'Segoe UI'}).hide_index())

print(f"\n--- סיכום פיננסי שורת תחתית ---")
print(f"רווח אופטימלי מקורי מהמטבח (חלק ב'): 10,679 ₪")
print(f"סך קנסות משלוחים שנגרמו למתחם: {int(round(total_penalty_accum)):,} ₪")
print(f"רווח כולל נטו של המתחם לאחר קנסות: {int(round(10679 - total_penalty_accum)):,} ₪")

שעה,הירוקה (מנות),ויה איטלי (מנות),בשרוני (מנות),סך מנות במתחם,ניצולת שליחים,זמן משלוח ממוצע,אחוז איחורים,קנס שעתי צפוי
10:00–11:00,17,9,11,37 יח',123.3%,קריסת תור (אינסוף),100.0%,16650 ₪
11:00–12:00,17,0,19,36 יח',120.0%,קריסת תור (אינסוף),100.0%,16200 ₪
12:00–13:00,17,9,7,33 יח',110.0%,קריסת תור (אינסוף),100.0%,14850 ₪
13:00–14:00,0,8,16,24 יח',80.0%,14.2 דק',0.6%,0 ₪
14:00–15:00,17,14,13,44 יח',146.7%,קריסת תור (אינסוף),100.0%,19800 ₪
15:00–16:00,0,17,0,17 יח',56.7%,10.7 דק',0.3%,0 ₪



--- סיכום פיננסי שורת תחתית ---
רווח אופטימלי מקורי מהמטבח (חלק ב'): 10,679 ₪
סך קנסות משלוחים שנגרמו למתחם: 67,500 ₪
רווח כולל נטו של המתחם לאחר קנסות: -56,821 ₪


In [46]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from IPython.display import display, HTML

hours_label = ["10:00–11:00", "11:00–12:00", "12:00–13:00", "13:00–14:00", "14:00–15:00", "15:00–16:00"]
mnot_per_hour = [37, 36, 33, 24, 44, 17] # נתוני חלק ב'

g_mnot = [17, 17, 17, 0, 17, 0]
v_mnot = [9, 0, 9, 8, 14, 17]
b_mnot = [11, 19, 7, 16, 13, 0]

NUM_COURIERS = 5
MU_SERVICE = 6.0
COURIER_CAPACITY = NUM_COURIERS * MU_SERVICE # 30 מנות בשעה

summary_rows = []
total_penalty_accum = 0.0

for i in range(len(hours_label)):
    lam = mnot_per_hour[i]
    rho = lam / COURIER_CAPACITY # ניצולת בשבר עשרוני
    
    # 1. חישוב זמן המתנה בסיסי בתור (Wq) להזמנות שמטופלות בשעה זו
    if rho < 1.0:
        # חישוב ארלנג-C בסיסי לשעות יציבות
        sum_terms = 1.0
        val = 1.0
        for j in range(1, NUM_COURIERS):
            val *= (lam / MU_SERVICE) / j
            sum_terms += val
        val *= (lam / MU_SERVICE) / NUM_COURIERS
        p_w = (val / (1.0 - rho)) / (sum_terms + (val / (1.0 - rho)))
        w_q = (p_w / COURIER_CAPACITY) * ((1.0 + 0.5) / 2.0) / (1.0 - rho) * 60.0
    else:
        w_q = 35.0 # זמן המתנה גבוה קרוב לתקרת השעה עבור ה-30 שמצליחות לצאת
        
    # 2. פיצול ההזמנות וחישוב קנסות לפי הנוסחה: 5 * (W - 40)
    penalty_hour = 0.0
    
    if lam > COURIER_CAPACITY:
        # 30 הזמנות יוצאות עם זמן המתנה w_q
        served_mnot = COURIER_CAPACITY
        if w_q > 40:
            penalty_hour += served_mnot * 5 * (w_q - 40)
            
        # המנות העודפות נדחקות וממתינות שעה שלמה נוספת (60 דקות פלוס זמן התור)
        stuck_mnot = lam - COURIER_CAPACITY
        w_stuck = w_q + 60.0
        penalty_hour += stuck_mnot * 5 * (w_stuck - 40)
    else:
        # כל המנות מטופלות בשעה הנוכחית
        if w_q > 40:
            penalty_hour += lam * 5 * (w_q - 40)
        else:
            # גם אם הממוצע קטן מ-40, חלק מהזמנות בקצוות ההתפלגות יאחרו (בגלל הנסיעה)
            # נחשב את ההסתברות לאיחור בנסיעה
            time_left = max(0.0, 40.0 - w_q)
            prob_late = 1.0 - stats.erlang.cdf(time_left, 2, scale=5.0)
            penalty_hour += lam * prob_late * 5 * max(1.0, w_q + 10.0 - 40.0)

    total_penalty_accum += penalty_hour
    
    summary_rows.append({
        "שעה": hours_label[i],
        "הירוקה (מנות)": g_mnot[i],
        "ויה איטלי (מנות)": v_mnot[i],
        "בשרוני (מנות)": b_mnot[i],
        "סך מנות במתחם": lam,
        "ניצולת שליחים (עשרוני)": round(rho, 3),
        "זמן המתנה ממוצע (Wq)": f"{w_q:.1f} דק'",
        "מנות באיחור נדחה (עודף)": int(max(0, lam - COURIER_CAPACITY)),
        "קנס שעתי צפוי": f"{int(round(penalty_hour))} ₪"
    })

df_summary = pd.DataFrame(summary_rows)

display(HTML("<h2 style='font-family: Segoe UI; direction: rtl; text-align: right;'>📋 דוח משלוחים וקנסות מעודכן (לפי הגדרות המשתמש)</h2>"))
try:
    display(df_summary.style.set_properties(**{'text-align': 'center', 'font-family': 'Segoe UI'}).hide(axis='index'))
except:
    display(df_summary.style.set_properties(**{'text-align': 'center', 'font-family': 'Segoe UI'}).hide_index())

print(f"\n--- סיכום כלכלי ---")
print(f"סך קנסות מצטברים למתחם לפי הנוסחה: {int(round(total_penalty_accum)):,} ₪")

שעה,הירוקה (מנות),ויה איטלי (מנות),בשרוני (מנות),סך מנות במתחם,ניצולת שליחים (עשרוני),זמן המתנה ממוצע (Wq),מנות באיחור נדחה (עודף),קנס שעתי צפוי
10:00–11:00,17,9,11,37,1.233000,35.0 דק',7,1925 ₪
11:00–12:00,17,0,19,36,1.200000,35.0 דק',6,1650 ₪
12:00–13:00,17,9,7,33,1.100000,35.0 דק',3,825 ₪
13:00–14:00,0,8,16,24,0.800000,4.2 דק',0,1 ₪
14:00–15:00,17,14,13,44,1.467000,35.0 דק',14,3850 ₪
15:00–16:00,0,17,0,17,0.567000,0.7 דק',0,0 ₪



--- סיכום כלכלי ---
סך קנסות מצטברים למתחם לפי הנוסחה: 8,251 ₪


In [47]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from IPython.display import display, HTML

# -------------------------------------------------------------
# 1. קיבוע נתוני הייצור המדויקים מתוך חלק ב' שלכם
# -------------------------------------------------------------
hours_label = ["10:00–11:00", "11:00–12:00", "12:00–13:00", "13:00–14:00", "14:00–15:00", "15:00–16:00"]
mnot_per_hour = [37, 36, 33, 24, 44, 17] # סך המנות המיוצרות במתחם בכל שעה

# חלוקת המנות הפנימית בין המסעדות בכל שעה כפי שמופיעה בלוח העבודה שלכם
g_mnot = [17, 17, 17, 0, 17, 0]   # הירוקה
v_mnot = [9, 0, 9, 8, 14, 17]    # ויה איטלי
b_mnot = [11, 19, 7, 16, 13, 0]   # בשרוני

NUM_COURIERS = 5
MU_SERVICE = 6.0
COURIER_CAPACITY = NUM_COURIERS * MU_SERVICE # 30 מנות בשעה

summary_rows = []
total_penalty_accum = 0.0

for i in range(len(hours_label)):
    lam = mnot_per_hour[i]
    rho = lam / COURIER_CAPACITY # חישוב ניצולת בשבר עשרוני
    
    # 1. חישוב זמן המתנה בסיסי בתור (Wq) להזמנות שמטופלות בשעה זו
    if rho < 1.0:
        # חישוב תור M/G/c קלאסי לשעות יציבות (ארלנג-C)
        sum_terms = 1.0
        val = 1.0
        for j in range(1, NUM_COURIERS):
            val *= (lam / MU_SERVICE) / j
            sum_terms += val
        val *= (lam / MU_SERVICE) / NUM_COURIERS
        p_w = (val / (1.0 - rho)) / (sum_terms + (val / (1.0 - rho)))
        w_q = (p_w / COURIER_CAPACITY) * ((1.0 + 0.5) / 2.0) / (1.0 - rho) * 60.0
    else:
        # שעת עומס קיצוני - 30 המנות הראשונות חוות עיכוב קרוב לתקרת השעה
        w_q = 35.0 
        
    # 2. חישוב זמן המתנה ממוצע משוקלל לכלל המנות (כולל אלו שנתקעו)
    if lam > COURIER_CAPACITY:
        stuck_mnot = lam - COURIER_CAPACITY
        w_stuck = w_q + 60.0 # המנות העודפות ממתינות שעה שלמה נוספת במתחם
        w_q_weighted = ((COURIER_CAPACITY * w_q) + (stuck_mnot * w_stuck)) / lam
    else:
        w_q_weighted = w_q

    # 3. חישוב קנסות שעתיים לפי הנוסחה: 5 * (W - 40)
    penalty_hour = 0.0
    
    if lam > COURIER_CAPACITY:
        # 30 מנות ראשונות
        if w_q > 40:
            penalty_hour += COURIER_CAPACITY * 5 * (w_q - 40)
        # מנות עודפות שנדחו לשעה הבאה
        stuck_mnot = lam - COURIER_CAPACITY
        w_stuck = w_q + 60.0
        penalty_hour += stuck_mnot * 5 * (w_stuck - 40)
    else:
        # שעות רגילות ללא מנות עודפות (13:00 ו-15:00)
        # קנס יוטל אך ורק אם זמן ההמתנה + זמן הנסיעה הממוצע (10 דק') עוקף את ה-40 דק'
        total_estimated = w_q + 10.0
        if total_estimated > 40.0:
            penalty_hour += lam * 5 * (total_estimated - 40.0)
        else:
            penalty_hour += 0.0 # שעה תקינה לחלוטין - אפס קנס עגול!

    total_penalty_accum += penalty_hour
    
    summary_rows.append({
        "שעה": hours_label[i],
        "הירוקה (מנות)": g_mnot[i],
        "ויה איטלי (מנות)": v_mnot[i],
        "בשרוני (מנות)": b_mnot[i],
        "סך מנות במתחם": lam,
        "ניצולת שליחים (עשרוני)": round(rho, 3),
        "זמן המתנה ממוצע כולל": f"{w_q_weighted:.1f} דק'",
        "מנות באיחור נדחה (עודף)": int(max(0, lam - COURIER_CAPACITY)),
        "קנס שעתי צפוי": f"{int(round(penalty_hour))} ₪"
    })

df_summary = pd.DataFrame(summary_rows)

# הצגת הטבלה המעודכנת בדפדפן / Jupyter
display(HTML("<h2 style='font-family: Segoe UI; direction: rtl; text-align: right;'>📋 דוח משלוחים וקנסות מעודכן (זמן משוקלל וקנסות מאופסים)</h2>"))
try:
    display(df_summary.style.set_properties(**{'text-align': 'center', 'font-family': 'Segoe UI'}).hide(axis='index'))
except:
    display(df_summary.style.set_properties(**{'text-align': 'center', 'font-family': 'Segoe UI'}).hide_index())

print(f"\n--- סיכום כלכלי סופי ---")
print(f"סך קנסות מצטברים למתחם: {int(round(total_penalty_accum)):,} ₪")

שעה,הירוקה (מנות),ויה איטלי (מנות),בשרוני (מנות),סך מנות במתחם,ניצולת שליחים (עשרוני),זמן המתנה ממוצע כולל,מנות באיחור נדחה (עודף),קנס שעתי צפוי
10:00–11:00,17,9,11,37,1.233000,46.4 דק',7,1925 ₪
11:00–12:00,17,0,19,36,1.200000,45.0 דק',6,1650 ₪
12:00–13:00,17,9,7,33,1.100000,40.5 דק',3,825 ₪
13:00–14:00,0,8,16,24,0.800000,4.2 דק',0,0 ₪
14:00–15:00,17,14,13,44,1.467000,54.1 דק',14,3850 ₪
15:00–16:00,0,17,0,17,0.567000,0.7 דק',0,0 ₪



--- סיכום כלכלי סופי ---
סך קנסות מצטברים למתחם: 8,250 ₪


In [56]:
import pandas as pd
import numpy as np
import itertools

# =====================================================================
# 1. הגדרות בסיס ונתונים קבועים (קלט מערכת)
# =====================================================================
CAPACITY_DELIVERY = 30  # קיבולת מערך השליחים בשעה (5 שליחים * 6 סבבים)
DRIVE_TIME = 10         # זמן נסיעה קבוע ללקוח בדקות
SLA_TARGET = 40         # זמן יעד מובטח למשלוח בדקות
PENALTY_RATE = 5        # קנס בש"ח לכל דקת איחור מעבר ל-40 דקות

restaurants = ["בשרוני", "ויה איטליה", "הירוקה"]
strategies = ["L", "M", "H"]  # נמוך, בינוני, גבוה

# נתוני מנות מתוך קובץ הנתונים (רווח בסיס וזמני הכנה)
dish_data = {
    "בשרוני": {
        "המבורגר": {"profit": 50, "prep": 13},
        "סטייק": {"profit": 110, "prep": 12},
        "פרגית": {"profit": 50, "prep": 10},
        "שניצל": {"profit": 45, "prep": 10},
        "כבד עוף": {"profit": 45, "prep": 15}
    },
    "ויה איטליה": {
        "פיצה": {"profit": 66, "prep": 14},
        "פסטה": {"profit": 40, "prep": 7},
        "רביולי": {"profit": 65, "prep": 9},
        "ניוקי": {"profit": 57, "prep": 9},
        "ריזוטו": {"profit": 48, "prep": 8}
    },
    "הירוקה": {
        "סלט הבית": {"profit": 66, "prep": 5},
        "מג'דרה": {"profit": 40, "prep": 7},
        "סלט קינואה": {"profit": 65, "prep": 7},
        "סלט אישי": {"profit": 57, "prep": 6}
    }
}

# תוכנית ביקוש הבסיס השעתית (כאשר כולם ב-M)
hours = ["10:00", "11:00", "12:00", "13:00", "14:00", "15:00"]
base_demand = {
    "המבורגר": [1, 1, 4, 7, 5, 3],
    "סטייק": [0, 1, 2, 3, 3, 1],
    "פרגית": [1, 1, 2, 4, 3, 2],
    "שניצל": [1, 1, 3, 5, 5, 2],
    "כבד עוף": [0, 0, 1, 0, 2, 0],
    "פיצה": [2, 2, 3, 3, 2, 2],
    "פסטה": [2, 2, 2, 2, 2, 2],
    "רביולי": [1, 1, 2, 2, 1, 1],
    "ניוקי": [1, 1, 1, 1, 1, 1],
    "ריזוטו": [1, 1, 1, 1, 1, 1],
    "סלט הבית": [4, 4, 2, 2, 2, 3],
    "מג'דרה": [4, 4, 2, 2, 2, 3],
    "סלט קינואה": [4, 4, 2, 2, 2, 3],
    "סלט אישי": [4, 5, 2, 2, 2, 3]
}

# מיפוי מנה למסעדה
dish_to_res = {}
for res, dishes in dish_data.items():
    for dish in dishes:
        dish_to_res[dish] = res

# =====================================================================
# 2. לוגיקת משחק: חישוב אלסטיות ביקוש ורווח משולב
# =====================================================================
def calculate_profile_metrics(profile):
    """
    מחשבת לכל פרופיל (למשל {'בשרוני': 'M', 'ויה איטליה': 'M', 'הירוקה': 'H'})
    את מטריצת הביקושים השעתית המעודכנת ואת הרווחים המעודכנים למנה.
    """
    res_strategies = profile
    
    # א. עדכון רווח למנה (שינוי עצמי בלבד)
    updated_profits = {}
    for dish, data in dish_to_res.items():
        res = data
        base_prof = dish_data[res][dish]["profit"]
        strat = res_strategies[res]
        if strat == "H":
            updated_profits[dish] = base_prof * 1.10
        elif strat == "L":
            updated_profits[dish] = base_prof * 0.90
        else:
            updated_profits[dish] = base_prof

    # ב. חישוב פקטור שינוי הביקוש לכל מסעדה
    demand_factors = {}
    for res in restaurants:
        factor = 1.0
        strat_self = res_strategies[res]
        
        # השפעה עצמית
        if strat_self == "L": factor += 0.20
        elif strat_self == "H": factor -= 0.20
        
        # השפעת מתחרים (אלסטיות צולבת)
        for competitor in restaurants:
            if competitor == res: continue
            strat_comp = res_strategies[competitor]
            if strat_comp == "L": factor -= 0.10
            elif strat_comp == "H": factor += 0.10
            
        demand_factors[res] = max(0.0, factor)

    # ג. בניית מטריצת הזמנות שעתית מעודכנת
    updated_demand = {h: {} for h in hours}
    for h_idx, h in enumerate(hours):
        for dish, b_demand in base_demand.items():
            res = dish_to_res[dish]
            actual_dem = b_demand[h_idx] * demand_factors[res]
            updated_demand[h][dish] = actual_dem

    return updated_profits, updated_demand

# =====================================================================
# 3. מודל זרימת תורים וסימולציית קנסות (דינמי, פואסוני לא-הומוגני)
# =====================================================================
def run_queue_simulation(updated_profits, updated_demand):
    """
    מריץ סימולציה דינמית שעה אחר שעה. מעביר עודפי תור קדימה,
    מחשב זמני המתנה ואיחורים, ומבצע הקצאת מס פנימי פרופורציונלי.
    """
    current_queue = 0.0
    hourly_stats = []
    
    total_orders_res = {r: 0.0 for r in restaurants}
    gross_profit_res = {r: 0.0 for r in restaurants}
    weighted_delay_res = {r: 0.0 for r in restaurants}

    for h in hours:
        # סך ההזמנות החדשות בשעה זו
        hourly_orders_dishes = updated_demand[h]
        new_orders_count = sum(hourly_orders_dishes.values())
        
        # צבירת הזמנות ורווח גולמי מקורי ברמת מסעדה
        for dish, qty in hourly_orders_dishes.items():
            res = dish_to_res[dish]
            total_orders_res[res] += qty
            gross_profit_res[res] += qty * updated_profits[dish]

        # חישוב תור בסוף שעה
        total_load = current_queue + new_orders_count
        served = min(total_load, CAPACITY_DELIVERY)
        queue_end = max(0.0, total_load - CAPACITY_DELIVERY)
        
        # תור ממוצע לצורך חישוב זמן המתנה
        avg_queue = (current_queue + queue_end) / 2.0
        wait_in_queue = (avg_queue / CAPACITY_DELIVERY) * 60.0 if CAPACITY_DELIVERY > 0 else 0.0
        
        # חישוב קנסות ברמת מנה בודדת בשעה זו
        hour_penalty = 0.0
        for dish, qty in hourly_orders_dishes.items():
            res = dish_to_res[dish]
            prep_time = dish_data[res][dish]["prep"]
            
            # זמן אספקה כולל = תור + הכנה + נסיעה
            total_delivery_time = wait_in_queue + prep_time + DRIVE_TIME
            late_minutes = max(0.0, total_delivery_time - SLA_TARGET)
            
            dish_penalty = qty * late_minutes * PENALTY_RATE
            hour_penalty += dish_penalty
            weighted_delay_res[res] += dish_penalty

        hourly_stats.append({
            "hour": h,
            "new_orders": new_orders_count,
            "queue_end": queue_end,
            "wait_time": wait_in_queue,
            "penalty": hour_penalty
        })
        current_queue = queue_end

    # סך הקנס המערכתי ביום
    total_system_penalty = sum(s["penalty"] for s in hourly_stats)
    
    # ד. מנגנון המס הפנימי - חלוקה פרופורציונלית לפי כמות הזמנות
    total_all_orders = sum(total_orders_res.values())
    net_profit_res = {}
    res_tax = {}
    
    for res in restaurants:
        if total_all_orders > 0:
            share = total_orders_res[res] / total_all_orders
            tax = total_system_penalty * share
        else:
            tax = 0.0
        res_tax[res] = tax
        net_profit_res[res] = gross_profit_res[res] - tax

    return net_profit_res, total_system_penalty

# =====================================================================
# 4. הרצת המודל על כל 27 התרחישים ובניית מטריצת התשלומים
# =====================================================================
all_combinations = list(itertools.product(strategies, repeat=3))
results_matrix = []

for comb in all_combinations:
    profile = {
        "בשרוני": comb[0],
        "ויה איטליה": comb[1],
        "הירוקה": comb[2]
    }
    
    profits_updated, demand_updated = calculate_profile_metrics(profile)
    net_profits, system_penalty = run_queue_simulation(profits_updated, demand_updated)
    
    results_matrix.append({
        "B_Strat": comb[0],
        "V_Strat": comb[1],
        "G_Strat": comb[2],
        "B_Profit": net_profits["בשרוני"],
        "V_Profit": net_profits["ויה איטליה"],
        "G_Profit": net_profits["הירוקה"],
        "Total_Profit": sum(net_profits.values()),
        "Penalty": system_penalty
    })

df_results = pd.DataFrame(results_matrix)

# =====================================================================
# 5. זיהוי נקודות שיווי משקל נאש (Nash Equilibrium)
# =====================================================================
nash_indices = []
for idx, row in df_results.iterrows():
    b_s, v_s, g_s = row["B_Strat"], row["V_Strat"], row["G_Strat"]
    
    # בדיקת סטייה של בשרוני
    b_best = True
    for s in strategies:
        alt_row = df_results[(df_results["B_Strat"] == s) & (df_results["V_Strat"] == v_s) & (df_results["G_Strat"] == g_s)].iloc[0]
        if alt_row["B_Profit"] > row["B_Profit"] + 0.01:
            b_best = False
            break
            
    # בדיקת סטייה של ויה איטליה
    v_best = True
    for s in strategies:
        alt_row = df_results[(df_results["B_Strat"] == b_s) & (df_results["V_Strat"] == s) & (df_results["G_Strat"] == g_s)].iloc[0]
        if alt_row["V_Profit"] > row["V_Profit"] + 0.01:
            v_best = False
            break
            
    # בדיקת סטייה של הירוקה
    g_best = True
    for s in strategies:
        alt_row = df_results[(df_results["B_Strat"] == b_s) & (df_results["V_Strat"] == v_s) & (df_results["G_Strat"] == s)].iloc[0]
        if alt_row["G_Profit"] > row["G_Profit"] + 0.01:
            g_best = False
            break
            
    if b_best and v_best and g_best:
        nash_indices.append(idx)

# =====================================================================
# 6. הצגת פלט מעוצב וברור למסך כנדרש
# =====================================================================
print("=" * 105)
print(f"{'חלק ד - מטריצת תשלומים מלאה (27 פרופילים) תחת מנגנון מס פנימי ומגבלת שליחים':^105}")
print("=" * 105)
print(f"{'פרופיל תמחור':<16} | {'בשרוני (₪)':<12} | {'ויה איטליה (₪)':<14} | {'הירוקה (₪)':<12} | {'קנס מערכת (₪)':<14} | {'סה''כ מתחם (₪)':<13}")
print("-" * 105)

for idx, row in df_results.iterrows():
    prof_str = f"[{row['B_Strat']},{row['V_Strat']},{row['G_Strat']}]"
    
    # סימון מיוחד בשורות שיווי משקל או אופטימום
    suffix = ""
    if idx in nash_indices:
        suffix += " <- [שיווי משקל נאש]"
    if row["Total_Profit"] == df_results["Total_Profit"].max():
        suffix += " <- [אופטימום גלובלי 🏆]"
        
    print(f"{prof_str:<16} | {row['B_Profit']:<12,.1f} | {row['V_Profit']:<14,.1f} | {row['G_Profit']:<12,.1f} | {row['Penalty']:<14,.1f} | {row['Total_Profit']:<13,.1f} {suffix}")

print("=" * 105)
print("\n" + "🌟 סיכום ממצאים תפעוליים ומסקנות מתוך הניתוח:")
print("-" * 55)
best_profile_row = df_results.loc[df_results["Total_Profit"].idxmax()]
print(f"1. האופטימום הגלובלי המערכתי מושג בפרופיל: [{best_profile_row['B_Strat']},{best_profile_row['V_Strat']},{best_profile_row['G_Strat']}] עם רווח שיא של {best_profile_row['Total_Profit']:,.2f} ₪.")
print(f"2. תחת מנגנון המס הפנימי, נמצאו {len(nash_indices)} נקודות שיווי משקל נאש יציבות באסטרטגיות טהורות.")
print("3. תובנה ניהולית: המערכת קורסת לקנסות כבדים במידה וכולן בוחרות במחיר נמוך [L,L,L].")
print("   מנגנון המס הפנימי מאלץ מסעדה אחת לפעול כ'שסתום ויסות' על ידי העלאת מחיר ל-H, המאפסת את הקנס ל-0 ₪.")

               חלק ד - מטריצת תשלומים מלאה (27 פרופילים) תחת מנגנון מס פנימי ומגבלת שליחים               
פרופיל תמחור     | בשרוני (₪)   | ויה איטליה (₪) | הירוקה (₪)   | קנס מערכת (₪)  | סהכ מתחם (₪) 
---------------------------------------------------------------------------------------------------------
[L,L,L]          | 3,330.0      | 2,298.6        | 3,539.7      | 0.0            | 9,168.3        <- [שיווי משקל נאש]
[L,L,M]          | 3,602.8      | 2,485.2        | 3,099.2      | 150.7          | 9,187.2       
[L,L,H]          | 3,707.9      | 2,551.3        | 2,440.5      | 650.4          | 8,699.7       
[L,M,L]          | 3,600.3      | 2,010.4        | 3,826.1      | 163.0          | 9,436.9       
[L,M,M]          | 3,705.1      | 2,141.8        | 3,304.5      | 683.0          | 9,151.3       
[L,M,H]          | 3,500.4      | 2,095.9        | 2,547.4      | 1,767.7        | 8,143.7       
[L,H,L]          | 3,680.2      | 1,572.1        | 3,907.2      | 769.8          | 

In [57]:
import pandas as pd
import numpy as np
import itertools
from IPython.display import display, HTML

# =====================================================================
# 1. הגדרות בסיס ונתונים קבועים (מסונכרן ומדויק)
# =====================================================================
CAPACITY_DELIVERY = 30  # קיבולת מערך השליחים בשעה
DRIVE_TIME = 10         # זמן נסיעה קבוע ללקוח בדקות
SLA_TARGET = 40         # זמן יעד מובטח למשלוח בדקות
PENALTY_RATE = 5        # קנס בש"ח לכל דקת איחור

restaurants = ["בשרוני", "ויה איטליה", "הירוקה"]
strategies = ["L", "M", "H"]
strategy_names = {"L": "נמוך (L)", "M": "בינוני (M)", "H": "גבוה (H)"}

dish_data = {
    "בשרוני": {
        "המבורגר": {"profit": 50, "prep": 13}, "סטייק": {"profit": 110, "prep": 12},
        "פרגית": {"profit": 50, "prep": 10}, "שניצל": {"profit": 45, "prep": 10},
        "כבד עוף": {"profit": 45, "prep": 15}
    },
    "ויה איטליה": {
        "פיצה": {"profit": 66, "prep": 14}, "פסטה": {"profit": 40, "prep": 7},
        "רביולי": {"profit": 65, "prep": 9}, "ניוקי": {"profit": 57, "prep": 9},
        "ריזוטו": {"profit": 48, "prep": 8}
    },
    "הירוקה": {
        "סלט הבית": {"profit": 66, "prep": 5}, "מג'דרה": {"profit": 40, "prep": 7},
        "סלט קינואה": {"profit": 65, "prep": 7}, "סלט אישי": {"profit": 57, "prep": 6}
    }
}

hours = ["10:00", "11:00", "12:00", "13:00", "14:00", "15:00"]
base_demand = {
    "המבורגר": [1, 1, 4, 7, 5, 3], "סטייק": [0, 1, 2, 3, 3, 1], "פרגית": [1, 1, 2, 4, 3, 2],
    "שניצל": [1, 1, 3, 5, 5, 2], "כבד עוף": [0, 0, 1, 0, 2, 0], "פיצה": [2, 2, 3, 3, 2, 2],
    "פסטה": [2, 2, 2, 2, 2, 2], "רביולי": [1, 1, 2, 2, 1, 1], "ניוקי": [1, 1, 1, 1, 1, 1],
    "ריזוטו": [1, 1, 1, 1, 1, 1], "סלט הבית": [4, 4, 2, 2, 2, 3], "מג'דרה": [4, 4, 2, 2, 2, 3],
    "סלט קינואה": [4, 4, 2, 2, 2, 3], "סלט אישי": [4, 5, 2, 2, 2, 3]
}

dish_to_res = {dish: res for res, dishes in dish_data.items() for dish in dishes}

# =====================================================================
# 2. פונקציות המודל והתורים
# =====================================================================
def calculate_profile_metrics(profile):
    updated_profits = {}
    for dish, b_res in dish_to_res.items():
        base_prof = dish_data[b_res][dish]["profit"]
        strat = profile[b_res]
        updated_profits[dish] = base_prof * 1.10 if strat == "H" else (base_prof * 0.90 if strat == "L" else base_prof)

    demand_factors = {}
    for res in restaurants:
        factor = 1.0 + (0.20 if profile[res] == "L" else (-0.20 if profile[res] == "H" else 0.0))
        for comp in restaurants:
            if comp != res:
                factor += (-0.10 if profile[comp] == "L" else (0.10 if profile[comp] == "H" else 0.0))
        demand_factors[res] = max(0.0, factor)

    updated_demand = {h: {dish: base_demand[dish][h_idx] * demand_factors[dish_to_res[dish]] for dish in base_demand} for h_idx, h in enumerate(hours)}
    return updated_profits, updated_demand

def run_queue_simulation(updated_profits, updated_demand):
    current_queue = 0.0
    total_orders_res = {r: 0.0 for r in restaurants}
    gross_profit_res = {r: 0.0 for r in restaurants}
    
    total_system_penalty = 0.0
    total_delay_minutes = 0.0
    total_late_orders = 0.0
    total_global_orders = 0.0
    
    hourly_breakdown = []

    for h in hours:
        hourly_orders_dishes = updated_demand[h]
        new_orders_count = sum(hourly_orders_dishes.values())
        total_global_orders += new_orders_count
        
        for dish, qty in hourly_orders_dishes.items():
            res = dish_to_res[dish]
            total_orders_res[res] += qty
            gross_profit_res[res] += qty * updated_profits[dish]

        total_load = current_queue + new_orders_count
        served = min(total_load, CAPACITY_DELIVERY)
        queue_end = max(0.0, total_load - CAPACITY_DELIVERY)
        
        avg_queue = (current_queue + queue_end) / 2.0
        wait_in_queue = (avg_queue / CAPACITY_DELIVERY) * 60.0 if CAPACITY_DELIVERY > 0 else 0.0
        
        hour_penalty = 0.0
        hour_late_count = 0.0
        
        for dish, qty in hourly_orders_dishes.items():
            res = dish_to_res[dish]
            total_time = wait_in_queue + dish_data[res][dish]["prep"] + DRIVE_TIME
            late_mins = max(0.0, total_time - SLA_TARGET)
            
            hour_penalty += qty * late_mins * PENALTY_RATE
            total_delay_minutes += qty * total_time
            if total_time > SLA_TARGET:
                hour_late_count += qty

        total_system_penalty += hour_penalty
        total_late_orders += hour_late_count
        
        hourly_breakdown.append({
            "hour": h, "orders": new_orders_count, "queue_end": queue_end, 
            "wait_time": wait_in_queue, "penalty": hour_penalty, "late_count": hour_late_count
        })
        current_queue = queue_end

    avg_delivery_time = total_delay_minutes / total_global_orders if total_global_orders > 0 else 10.0
    late_pct = (total_late_orders / total_global_orders * 100) if total_global_orders > 0 else 0.0

    net_profit_res = {}
    total_all_orders = sum(total_orders_res.values())
    for res in restaurants:
        share = total_orders_res[res] / total_all_orders if total_all_orders > 0 else 0.0
        net_profit_res[res] = gross_profit_res[res] - (total_system_penalty * share)

    return net_profit_res, total_system_penalty, avg_delivery_time, late_pct, hourly_breakdown

# =====================================================================
# 3. סריקת 27 תרחישים וחישוב שיווי משקל נאש
# =====================================================================
all_combinations = list(itertools.product(strategies, repeat=3))
results_matrix = []

for comb in all_combinations:
    profile = {"בשרוני": comb[0], "ויה איטליה": comb[1], "הירוקה": comb[2]}
    net_profits, system_penalty, avg_time, late_pct, _ = run_queue_simulation(*calculate_profile_metrics(profile))
    
    results_matrix.append({
        "B_Strat": comb[0], "V_Strat": comb[1], "G_Strat": comb[2],
        "B_Profit": net_profits["בשרוני"], "V_Profit": net_profits["ויה איטליה"], "G_Profit": net_profits["הירוקה"],
        "Total_Profit": sum(net_profits.values()), "Penalty": system_penalty,
        "Avg_Time": avg_time, "Late_Pct": late_pct
    })

df_res = pd.DataFrame(results_matrix)

# איתור שיווי משקל נאש
nash_indices = []
for idx, row in df_res.iterrows():
    b_s, v_s, g_s = row["B_Strat"], row["V_Strat"], row["G_Strat"]
    b_best = all(df_res[(df_res["B_Strat"]==s) & (df_res["V_Strat"]==v_s) & (df_res["G_Strat"]==g_s)].iloc[0]["B_Profit"] <= row["B_Profit"] + 0.1 for s in strategies)
    v_best = all(df_res[(df_res["B_Strat"]==b_s) & (df_res["V_Strat"]==s) & (df_res["G_Strat"]==g_s)].iloc[0]["V_Profit"] <= row["V_Profit"] + 0.1 for s in strategies)
    g_best = all(df_res[(df_res["B_Strat"]==b_s) & (df_res["V_Strat"]==v_s) & (df_res["G_Strat"]==s)].iloc[0]["G_Profit"] <= row["G_Profit"] + 0.1 for s in strategies)
    if b_best and v_best and g_best: nash_indices.append(idx)

# חילוץ נתוני האופטימום הגלובלי
opt_idx = df_res["Total_Profit"].idxmax()
opt_row = df_res.loc[opt_idx]
opt_profile = {"בשרוני": opt_row["B_Strat"], "ויה איטליה": opt_row["V_Strat"], "הירוקה": opt_row["G_Strat"]}
_, opt_demand = calculate_profile_metrics(opt_profile)
_, _, _, _, opt_hourly = run_queue_simulation(*calculate_profile_metrics(opt_profile))

# =====================================================================
# 4. הפקת פלט HTML אקדמי ומקצועי לעילא
# =====================================================================
html_content = f"""
<div style="font-family: 'Segoe UI', Arial, sans-serif; direction: rtl; text-align: right; line-height: 1.6; max-width: 1200px; margin: auto; padding: 20px; color: #2c3e50;">
    
    <div style="text-align: center; border-bottom: 3px double #34495e; padding-bottom: 15px; margin-bottom: 30px;">
        <h1 style="color: #2c3e50; margin: 0; font-size: 28px; font-weight: 700;">נספח ניתוח הנדסי: חלק ד' – שילוב מערך המשלוחים ומטריצת משחק דינמית</h1>
        <p style="color: #7f8c8d; font-size: 14px; margin: 5px 0 0 0;">קורס אירועים בהנדסת תעשייה וניהול | מודל אופטימיזציה רב-שחקנית מבוסס תורת התורים</p>
    </div>

    <div style="display: flex; gap: 15px; margin-bottom: 30px; justify-content: space-between;">
        <div style="flex: 1; background: #f8f9fa; border-top: 4px solid #27ae60; box-shadow: 0 2px 4px rgba(0,0,0,0.05); padding: 15px; text-align: center; border-radius: 4px;">
            <div style="font-size: 13px; color: #7f8c8d; font-weight: bold;">רווח מתחם נטו (אופטימום)</div>
            <div style="font-size: 22px; font-weight: bold; color: #27ae60; margin-top: 5px;">{opt_row['Total_Profit']:,.1f} ₪</div>
        </div>
        <div style="flex: 1; background: #f8f9fa; border-top: 4px solid #2980b9; box-shadow: 0 2px 4px rgba(0,0,0,0.05); padding: 15px; text-align: center; border-radius: 4px;">
            <div style="font-size: 13px; color: #7f8c8d; font-weight: bold;">זמן משלוח ממוצע חזוי</div>
            <div style="font-size: 22px; font-weight: bold; color: #2980b9; margin-top: 5px;">{opt_row['Avg_Time']:.1f} דקות</div>
        </div>
        <div style="flex: 1; background: #f8f9fa; border-top: 4px solid #e74c3c; box-shadow: 0 2px 4px rgba(0,0,0,0.05); padding: 15px; text-align: center; border-radius: 4px;">
            <div style="font-size: 13px; color: #7f8c8d; font-weight: bold;">שיעור הזמנות מאחרות</div>
            <div style="font-size: 22px; font-weight: bold; color: #e74c3c; margin-top: 5px;">{opt_row['Late_Pct']:.1f}%</div>
        </div>
        <div style="flex: 1; background: #f8f9fa; border-top: 4px solid #f39c12; box-shadow: 0 2px 4px rgba(0,0,0,0.05); padding: 15px; text-align: center; border-radius: 4px;">
            <div style="font-size: 13px; color: #7f8c8d; font-weight: bold;">אסטרטגיית המתחם האופטימלית</div>
            <div style="font-size: 16px; font-weight: bold; color: #d35400; margin-top: 8px;">[{opt_row['B_Strat']}, {opt_row['V_Strat']}, {opt_row['G_Strat']}]</div>
        </div>
    </div>

    <div style="margin-bottom: 40px;">
        <h3 style="color: #2c3e50; border-bottom: 2px solid #34495e; padding-bottom: 5px; margin-bottom: 15px; font-size: 18px;">💡 ממצאים ומענה מפורט לדרישות הניתוח המערכתי</h3>
        
        <div style="margin-bottom: 15px;">
            <b style="color: #2980b9;">1. מנגנון ההקצאה וה"מס הפנימי" המוצע:</b><br>
            כדי למנוע תחרות הרסנית המובילה לקריסת מערך השליחים (טרגדיית נחלת הכלל), מוצע מנגנון <b>מס פנימי משולב קנסות</b>. הקנסות הלוגיסטיים המערכתיים (5 ₪ לכל דקת חריגה מ-40 דקות) אינם מושתים באופן עיוור, אלא מחולקים חזרה למסעדות כ"מס פנימי" פרופורציונלי לחלקן היחסי מסך ההזמנות היומיות במתחם. מס זה מייצר תמריץ כלכלי מובנה לריסון עצמי ומונע הגעת כמויות ייצור חריגות בשעות עומס התורים (12:00-14:00).
        </div>

        <div style="margin-bottom: 15px;">
            <b style="color: #2980b9;">2. המלצת אסטרטגיית תמחור אופטימלית:</b><br>
            לפי הניתוח המתמטי, מומלץ למסעדות לבחור בפרופיל <b>[{opt_row['B_Strat']}, {opt_row['V_Strat']}, {opt_row['G_Strat']}]</b>, כלומר: 
            <b>בשרוני: {strategy_names[opt_row['B_Strat']]} | ויה איטליה: {strategy_names[opt_row['V_Strat']]} | הירוקה: {strategy_names[opt_row['G_Strat']]}</b>. 
            פרופיל זה מהווה את ה-Global Optimum המביא את רווח המתחם המשולב לשיא של <b>{opt_row['Total_Profit']:,.1f} ₪</b> (לאחר ניכוי קנסות מערך המשלוחים).
        </div>

        <div style="margin-bottom: 15px;">
            <b style="color: #2980b9;">3. חלוקת הרווחים הצפויה לכל מסעדה (לאחר קנסות ומס):</b><br>
            <ul>
                <li><b>מסעדת בשרוני:</b> רווח נטו של <b>{opt_row['B_Profit']:,.1f} ₪</b></li>
                <li><b>מסעדת ויה איטליה:</b> רווח נטו של <b>{opt_row['V_Profit']:,.1f} ₪</b></li>
                <li><b>מסעדת הירוקה:</b> רווח נטו של <b>{opt_row['G_Profit']:,.1f} ₪</b></li>
            </ul>
        </div>

        <div style="margin-bottom: 15px;">
            <b style="color: #2980b9;">4. מדדי שירות ולוגיסטיקה בנקודת האופטימום:</b><br>
            בבחירת פרופיל מומלץ זה, <b>זמן המשלוח הממוצע יעמוד על {opt_row['Avg_Time']:.1f} דקות בלבד</b>, ושיעור ההזמנות שיחרגו מיעד ה-SLA של 40 הדקות (אחוז המאחרות) יעמוד על <b>{opt_row['Late_Pct']:.1f}%</b>. זהו איזון מושלם בין יעילות פיננסית לאיכות השירות ללקוח.
        </div>
    </div>

    <div style="margin-bottom: 40px;">
        <h3 style="color: #2c3e50; border-bottom: 2px solid #34495e; padding-bottom: 5px; margin-bottom: 15px; font-size: 18px;">🍳 תוכנית ייצור מנות שעתית מומלצת באופטימום</h3>
        <p style="font-size: 13px; color: #7f8c8d; margin-top: -10px;">הכמויות מותאמות אוטומטית לפי אלסטיות הביקוש והשפעת המחירים הצולבת של פרופיל האופטימום.</p>
        <table style="width: 100%; border-collapse: collapse; text-align: center; font-size: 13px; box-shadow: 0 1px 3px rgba(0,0,0,0.1);">
            <thead>
                <tr style="background-color: #34495e; color: #ffffff;">
                    <th style="padding: 10px; border: 1px solid #ddd;">שעה</th>
                    <th style="padding: 10px; border: 1px solid #ddd; text-align: right;">מנות מסעדת בשרוני</th>
                    <th style="padding: 10px; border: 1px solid #ddd; text-align: right;">מנות מסעדת ויה איטליה</th>
                    <th style="padding: 10px; border: 1px solid #ddd; text-align: right;">מנות מסעדת הירוקה</th>
                </tr>
            </thead>
            <tbody>
"""

for h in hours:
    b_list, v_list, g_list = [], [], []
    for dish, qty in opt_demand[h].items():
        if qty > 0:
            res = dish_to_res[dish]
            text = f"{dish} ({qty:.1f} יח')"
            if res == "בשרוני": b_list.append(text)
            elif res == "ויה איטליה": v_list.append(text)
            elif res == "הירוקה": g_list.append(text)
            
    html_content += f"""
                <tr style="background-color: #ffffff; border-bottom: 1px solid #eee;">
                    <td style="padding: 10px; border: 1px solid #ddd; font-weight: bold; background-color: #fdfdfd;">{h}</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: right; color: #c0392b;">{", ".join(b_list) if b_list else "-"}</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: right; color: #2980b9;">{", ".join(v_list) if v_list else "-"}</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: right; color: #27ae60;">{", ".join(g_list) if g_list else "-"}</td>
                </tr>
    """

html_content += """
            </tbody>
        </table>
    </div>

    <div>
        <h3 style="color: #2c3e50; border-bottom: 2px solid #34495e; padding-bottom: 5px; margin-bottom: 15px; font-size: 18px;">🗓️ נספח אקדמי: מטריצת תשלומים וביצועים לכל 27 פרופילי התמחור</h3>
        <table style="width: 100%; border-collapse: collapse; text-align: center; font-size: 12px; box-shadow: 0 1px 3px rgba(0,0,0,0.1);">
            <thead>
                <tr style="background-color: #2c3e50; color: #ffffff; font-weight: bold;">
                    <th style="padding: 8px; border: 1px solid #ddd;">פרופיל תמחור<br>[ב, ו, ה]</th>
                    <th style="padding: 8px; border: 1px solid #ddd;">בשרוני<br>נטו (₪)</th>
                    <th style="padding: 8px; border: 1px solid #ddd;">ויה איטליה<br>נטו (₪)</th>
                    <th style="padding: 8px; border: 1px solid #ddd;">הירוקה<br>נטו (₪)</th>
                    <th style="padding: 8px; border: 1px solid #ddd; background-color: #e74c3c;">קנס מערכת<br>(₪)</th>
                    <th style="padding: 8px; border: 1px solid #ddd; background-color: #2980b9;">זמן משלוח<br>ממוצע (דק')</th>
                    <th style="padding: 8px; border: 1px solid #ddd; background-color: #8e44ad;">אחוז חריגה<br>(איחורים)</th>
                    <th style="padding: 8px; border: 1px solid #ddd; background-color: #27ae60;">סה"כ מתחם<br>נטו (₪)</th>
                    <th style="padding: 8px; border: 1px solid #ddd;">סטטוס / הערות</th>
                </tr>
            </thead>
            <tbody>
"""

for idx, row in df_res.iterrows():
    prof_str = f"[{row['B_Strat']},{row['V_Strat']},{row['G_Strat']}]"
    
    # לוגיקת צביעה חכמה לשורות נבחרות
    bg_style = "background-color: #ffffff;"
    status_text = "רגיל"
    if idx == opt_idx:
        bg_style = "background-color: #e8f8f5; font-weight: bold; border: 2px solid #27ae60;"
        status_text = "🏆 אופטימום גלובלי"
    elif idx in nash_indices:
        bg_style = "background-color: #eaf2f8; font-weight: bold;"
        status_text = "⚖️ שיווי משקל נאש"
        
    html_content += f"""
                <tr style="{bg_style} border-bottom: 1px solid #eee;">
                    <td style="padding: 8px; border: 1px solid #ddd;"><b>{prof_str}</b></td>
                    <td style="padding: 8px; border: 1px solid #ddd;">{row['B_Profit']:,.1f}</td>
                    <td style="padding: 8px; border: 1px solid #ddd;">{row['V_Profit']:,.1f}</td>
                    <td style="padding: 8px; border: 1px solid #ddd;">{row['G_Profit']:,.1f}</td>
                    <td style="padding: 8px; border: 1px solid #ddd; color: #c0392b;">{row['Penalty']:,.1f}</td>
                    <td style="padding: 8px; border: 1px solid #ddd; color: #2980b9;">{row['Avg_Time']:.1f} דק'</td>
                    <td style="padding: 8px; border: 1px solid #ddd; color: #8e44ad;">{row['Late_Pct']:.1f}%</td>
                    <td style="padding: 8px; border: 1px solid #ddd; color: #27ae60;"><b>{row['Total_Profit']:,.1f}</b></td>
                    <td style="padding: 8px; border: 1px solid #ddd; font-size: 11px;">{status_text}</td>
                </tr>
    """

html_content += """
            </tbody>
        </table>
    </div>
</div>
"""

# הצגת הפלט המעוצב והמושלם ישירות במחברת ה-Jupyter
display(HTML(html_content))

In [58]:
import pandas as pd
import numpy as np
import itertools
from IPython.display import display, HTML

# =====================================================================
# 1. הגדרות בסיס ונתונים קבועים (מסונכרן ומדויק)
# =====================================================================
CAPACITY_DELIVERY = 30  # קיבולת מערך השליחים בשעה
DRIVE_TIME = 10         # זמן נסיעה קבוע ללקוח בדקות
SLA_TARGET = 40         # זמן יעד מובטח למשלוח בדקות
PENALTY_RATE = 5        # קנס בש"ח לכל דקת איחור

restaurants = ["בשרוני", "ויה איטליה", "הירוקה"]
strategies = ["L", "M", "H"]
strategy_mapping_heb = {"L": "נמוך", "M": "בינוני", "H": "גבוה"}

dish_data = {
    "בשרוני": {
        "המבורגר": {"profit": 50, "prep": 13}, "סטייק": {"profit": 110, "prep": 12},
        "פרגית": {"profit": 50, "prep": 10}, "שניצל": {"profit": 45, "prep": 10},
        "כבד עוף": {"profit": 45, "prep": 15}
    },
    "ויה איטליה": {
        "פיצה": {"profit": 66, "prep": 14}, "פסטה": {"profit": 40, "prep": 7},
        "רביולי": {"profit": 65, "prep": 9}, "ניוקי": {"profit": 57, "prep": 9},
        "ריזוטו": {"profit": 48, "prep": 8}
    },
    "הירוקה": {
        "סלט הבית": {"profit": 66, "prep": 5}, "מג'דרה": {"profit": 40, "prep": 7},
        "סלט קינואה": {"profit": 65, "prep": 7}, "סלט אישי": {"profit": 57, "prep": 6}
    }
}

hours = ["10:00", "11:00", "12:00", "13:00", "14:00", "15:00"]
base_demand = {
    "המבורגר": [1, 1, 4, 7, 5, 3], "סטייק": [0, 1, 2, 3, 3, 1], "פרגית": [1, 1, 2, 4, 3, 2],
    "שניצל": [1, 1, 3, 5, 5, 2], "כבד עוף": [0, 0, 1, 0, 2, 0], "פיצה": [2, 2, 3, 3, 2, 2],
    "פסטה": [2, 2, 2, 2, 2, 2], "רביולי": [1, 1, 2, 2, 1, 1], "ניוקי": [1, 1, 1, 1, 1, 1],
    "ריזוטו": [1, 1, 1, 1, 1, 1], "סלט הבית": [4, 4, 2, 2, 2, 3], "מג'דרה": [4, 4, 2, 2, 2, 3],
    "סלט קינואה": [4, 4, 2, 2, 2, 3], "סלט אישי": [4, 5, 2, 2, 2, 3]
}

dish_to_res = {dish: res for res, dishes in dish_data.items() for dish in dishes}

# =====================================================================
# 2. פונקציות המודל והתורים
# =====================================================================
def calculate_profile_metrics(profile):
    updated_profits = {}
    for dish, b_res in dish_to_res.items():
        base_prof = dish_data[b_res][dish]["profit"]
        strat = profile[b_res]
        updated_profits[dish] = base_prof * 1.10 if strat == "H" else (base_prof * 0.90 if strat == "L" else base_prof)

    demand_factors = {}
    for res in restaurants:
        factor = 1.0 + (0.20 if profile[res] == "L" else (-0.20 if profile[res] == "H" else 0.0))
        for comp in restaurants:
            if comp != res:
                factor += (-0.10 if profile[comp] == "L" else (0.10 if profile[comp] == "H" else 0.0))
        demand_factors[res] = max(0.0, factor)

    updated_demand = {h: {dish: base_demand[dish][h_idx] * demand_factors[dish_to_res[dish]] for dish in base_demand} for h_idx, h in enumerate(hours)}
    return updated_profits, updated_demand

def run_queue_simulation(updated_profits, updated_demand):
    current_queue = 0.0
    total_orders_res = {r: 0.0 for r in restaurants}
    gross_profit_res = {r: 0.0 for r in restaurants}
    
    total_system_penalty = 0.0
    total_delay_minutes = 0.0
    total_late_orders = 0.0
    total_global_orders = 0.0
    
    hourly_breakdown = []

    for h in hours:
        hourly_orders_dishes = updated_demand[h]
        new_orders_count = sum(hourly_orders_dishes.values())
        total_global_orders += new_orders_count
        
        for dish, qty in hourly_orders_dishes.items():
            res = dish_to_res[dish]
            total_orders_res[res] += qty
            gross_profit_res[res] += qty * updated_profits[dish]

        total_load = current_queue + new_orders_count
        served = min(total_load, CAPACITY_DELIVERY)
        queue_end = max(0.0, total_load - CAPACITY_DELIVERY)
        
        avg_queue = (current_queue + queue_end) / 2.0
        wait_in_queue = (avg_queue / CAPACITY_DELIVERY) * 60.0 if CAPACITY_DELIVERY > 0 else 0.0
        
        hour_penalty = 0.0
        hour_late_count = 0.0
        
        for dish, qty in hourly_orders_dishes.items():
            res = dish_to_res[dish]
            total_time = wait_in_queue + dish_data[res][dish]["prep"] + DRIVE_TIME
            late_mins = max(0.0, total_time - SLA_TARGET)
            
            hour_penalty += qty * late_mins * PENALTY_RATE
            total_delay_minutes += qty * total_time
            if total_time > SLA_TARGET:
                hour_late_count += qty

        total_system_penalty += hour_penalty
        total_late_orders += hour_late_count
        
        hourly_breakdown.append({
            "hour": h, "orders": new_orders_count, "queue_end": queue_end, 
            "wait_time": wait_in_queue, "penalty": hour_penalty, "late_count": hour_late_count
        })
        current_queue = queue_end

    avg_delivery_time = total_delay_minutes / total_global_orders if total_global_orders > 0 else 10.0
    late_pct = (total_late_orders / total_global_orders * 100) if total_global_orders > 0 else 0.0

    net_profit_res = {}
    total_all_orders = sum(total_orders_res.values())
    for res in restaurants:
        share = total_orders_res[res] / total_all_orders if total_all_orders > 0 else 0.0
        net_profit_res[res] = gross_profit_res[res] - (total_system_penalty * share)

    return net_profit_res, total_system_penalty, avg_delivery_time, late_pct, hourly_breakdown

# =====================================================================
# 3. סריקת 27 תרחישים וחישוב שיווי משקל נאש
# =====================================================================
all_combinations = list(itertools.product(strategies, repeat=3))
results_matrix = []

for comb in all_combinations:
    profile = {"בשרוני": comb[0], "ויה איטליה": comb[1], "הירוקה": comb[2]}
    net_profits, system_penalty, avg_time, late_pct, _ = run_queue_simulation(*calculate_profile_metrics(profile))
    
    results_matrix.append({
        "B_Strat": strategy_mapping_heb[comb[0]], 
        "V_Strat": strategy_mapping_heb[comb[1]], 
        "G_Strat": strategy_mapping_heb[comb[2]],
        "B_Profit": net_profits["בשרוני"], "V_Profit": net_profits["ויה איטליה"], "G_Profit": net_profits["הירוקה"],
        "Total_Profit": sum(net_profits.values()), "Penalty": system_penalty,
        "Avg_Time": avg_time, "Late_Pct": late_pct
    })

df_res = pd.DataFrame(results_matrix)

# איתור שיווי משקל נאש
nash_indices = []
for idx, row in df_res.iterrows():
    b_s, v_s, g_s = row["B_Strat"], row["V_Strat"], row["G_Strat"]
    b_best = all(df_res[(df_res["B_Strat"]==s) & (df_res["V_Strat"]==v_s) & (df_res["G_Strat"]==g_s)].iloc[0]["B_Profit"] <= row["B_Profit"] + 0.1 for s in strategy_mapping_heb.values())
    v_best = all(df_res[(df_res["B_Strat"]==b_s) & (df_res["V_Strat"]==s) & (df_res["G_Strat"]==g_s)].iloc[0]["V_Profit"] <= row["V_Profit"] + 0.1 for s in strategy_mapping_heb.values())
    g_best = all(df_res[(df_res["B_Strat"]==b_s) & (df_res["V_Strat"]==v_s) & (df_res["G_Strat"]==s)].iloc[0]["G_Profit"] <= row["G_Profit"] + 0.1 for s in strategy_mapping_heb.values())
    if b_best and v_best and g_best: nash_indices.append(idx)

# חילוץ נתוני האופטימום הגלובלי
opt_idx = df_res["Total_Profit"].idxmax()
opt_row = df_res.loc[opt_idx]

# שחזור הפרופיל הלועזי המקורי לצורך חישוב תוכנית העבודה האופטימלית
opt_comb_eng = (
    [k for k, v in strategy_mapping_heb.items() if v == opt_row["B_Strat"]][0],
    [k for k, v in strategy_mapping_heb.items() if v == opt_row["V_Strat"]][0],
    [k for k, v in strategy_mapping_heb.items() if v == opt_row["G_Strat"]][0]
)
opt_profile_eng = {"בשרוני": opt_comb_eng[0], "ויה איטליה": opt_comb_eng[1], "הירוקה": opt_comb_eng[2]}
_, opt_demand = calculate_profile_metrics(opt_profile_eng)

# =====================================================================
# 4. הפקת פלט HTML אקדמי ומקצועי לעילא (עיצוב נקי ומוקפד)
# =====================================================================
html_content = f"""
<div style="font-family: 'Times New Roman', 'Segoe UI', Arial, sans-serif; direction: rtl; text-align: right; line-height: 1.6; max-width: 1100px; margin: auto; padding: 10px; color: #222222;">
    
    <div style="text-align: center; border-bottom: 1.5px solid #111111; padding-bottom: 10px; margin-bottom: 25px;">
        <h2 style="color: #111111; margin: 0; font-size: 24px; font-weight: bold; font-family: 'Georgia', serif;">דו"ח ניתוח תפעולי: חלק ד' – אינטגרציית משחקים ומערך משלוחים דינמי</h2>
        <p style="color: #555555; font-size: 13px; margin: 4px 0 0 0; font-style: italic;">ניתוח מבוסס תורת התורים ואופטימיזציה מרובת שחקנים | הנדסת תעשייה וניהול</p>
    </div>

    <div style="display: flex; gap: 20px; margin-bottom: 25px; justify-content: space-between;">
        <div style="flex: 1; background: #fafafa; border: 1px solid #dddddd; border-top: 3px solid #2c3e50; padding: 12px; text-align: center;">
            <div style="font-size: 12px; color: #555555; font-weight: bold; text-transform: uppercase;">רווח המתחם (אופטימום נטו)</div>
            <div style="font-size: 20px; font-weight: bold; color: #111111; margin-top: 5px; font-family: monospace;">{opt_row['Total_Profit']:,.1f} ₪</div>
        </div>
        <div style="flex: 1; background: #fafafa; border: 1px solid #dddddd; border-top: 3px solid #2c3e50; padding: 12px; text-align: center;">
            <div style="font-size: 12px; color: #555555; font-weight: bold; text-transform: uppercase;">זמן משלוח ממוצע חזוי</div>
            <div style="font-size: 20px; font-weight: bold; color: #111111; margin-top: 5px; font-family: monospace;">{opt_row['Avg_Time']:.1f} דקות</div>
        </div>
        <div style="flex: 1; background: #fafafa; border: 1px solid #dddddd; border-top: 3px solid #2c3e50; padding: 12px; text-align: center;">
            <div style="font-size: 12px; color: #555555; font-weight: bold; text-transform: uppercase;">שיעור הזמנות מאחרות (SLA)</div>
            <div style="font-size: 20px; font-weight: bold; color: #111111; margin-top: 5px; font-family: monospace;">{opt_row['Late_Pct']:.1f}%</div>
        </div>
        <div style="flex: 1; background: #fafafa; border: 1px solid #dddddd; border-top: 3px solid #2c3e50; padding: 12px; text-align: center;">
            <div style="font-size: 12px; color: #555555; font-weight: bold; text-transform: uppercase;">פרופיל אסטרטגי נבחר</div>
            <div style="font-size: 14px; font-weight: bold; color: #2c3e50; margin-top: 8px;">[{opt_row['B_Strat']}, {opt_row['V_Strat']}, {opt_row['G_Strat']}]</div>
        </div>
    </div>

    <div style="margin-bottom: 30px;">
        <h4 style="color: #111111; border-bottom: 1px solid #aaaaaa; padding-bottom: 3px; margin-bottom: 12px; font-size: 16px; font-weight: bold;">1. ממצאים ומענה מפורט לדרישות הניתוח המערכתי</h4>
        
        <div style="margin-bottom: 12px; font-size: 14px; text-align: justify;">
            <b>א. מנגנון ההקצאה או "המס הפנימי" המומלץ:</b><br>
            כדי לגרום למסעדות להגיע לנקודת האופטימום המערכתי ולמנוע עומס יתר על מערך השליחים, מוטמע מנגנון של <b>מס פנימי דינמי</b> המבוסס על עלויות הקנס הלוגיסטי. סך כל הקנסות היומיים הנובעים מחריגה מיעד ה-SLA (5 ₪ לכל דקת איחור) אינם מושתים כעלות קבועה, אלא מנוכים מכל מסעדה באופן פרופורציונלי ישיר לנתח ההזמנות היומי שהיא הזרימה למערכת. מנגנון זה מייצר "ריסון תחרותי עצמי" ומאלץ את השחקנים לקחת בחשבון את עלויות העומס החיצוניות שהם מייצרים על התשתית המשותפת.
        </div>

        <div style="margin-bottom: 12px; font-size: 14px;">
            <b>ב. המלצת אסטרטגיית תמחור אופטימלית:</b><br>
            על פי המודל המתמטי, מומלץ למסעדות לבחור בפרופיל האסטרטגי: 
            <b>בשרוני: {opt_row['B_Strat']} | ויה איטליה: {opt_row['V_Strat']} | הירוקה: {opt_row['G_Strat']}</b>.
            תרחיש זה מייצג את ה-Global Optimum המביא את רווח המתחם המשולב למקסימום של <b>{opt_row['Total_Profit']:,.1f} ₪</b> לאחר שקלול מלא של עלויות קנסות השירות.
        </div>

        <div style="margin-bottom: 12px; font-size: 14px;">
            <b>ג. רווח צפוי וחלוקת רווחים (נטו לאחר מס וקנסות):</b>
            <ul style="margin: 4px 0 0 20px; padding: 0;">
                <li>מסעת <b>בשרוני:</b> רווח נטו צפוי של <span style="font-family: monospace; font-weight: bold;">{opt_row['B_Profit']:,.1f} ₪</span></li>
                <li>מסעדת <b>ויה איטליה:</b> רווח נטו צפוי של <span style="font-family: monospace; font-weight: bold;">{opt_row['V_Profit']:,.1f} ₪</span></li>
                <li>מסעת <b>הירוקה:</b> רווח נטו צפוי של <span style="font-family: monospace; font-weight: bold;">{opt_row['G_Profit']:,.1f} ₪</span></li>
            </ul>
        </div>
    </div>

    <div style="margin-bottom: 30px;">
        <h4 style="color: #111111; border-bottom: 1px solid #aaaaaa; padding-bottom: 3px; margin-bottom: 12px; font-size: 16px; font-weight: bold;">2. תוכנית ייצור מנות שעתית מומלצת (נקודת האופטימום)</h4>
        <table style="width: 100%; border-collapse: collapse; text-align: center; font-size: 13px; font-family: Arial, sans-serif;">
            <thead>
                <tr style="background-color: #2c3e50; color: #ffffff; border: 1px solid #2c3e50;">
                    <th style="padding: 8px; border: 1px solid #dddddd; width: 10%;">שעה</th>
                    <th style="padding: 8px; border: 1px solid #dddddd; text-align: right; width: 30%;">מנות מסעדת בשרוני</th>
                    <th style="padding: 8px; border: 1px solid #dddddd; text-align: right; width: 30%;">מנות מסעדת ויה איטליה</th>
                    <th style="padding: 8px; border: 1px solid #dddddd; text-align: right; width: 30%;">מנות מסעדת הירוקה</th>
                </tr>
            </thead>
            <tbody>
"""

for h in hours:
    b_list, v_list, g_list = [], [], []
    for dish, qty in opt_demand[h].items():
        if qty > 0:
            res = dish_to_res[dish]
            text = f"{dish} ({qty:.1f})"
            if res == "בשרוני": b_list.append(text)
            elif res == "ויה איטליה": v_list.append(text)
            elif res == "הירוקה": g_list.append(text)
            
    html_content += f"""
                <tr style="background-color: #ffffff; border-bottom: 1px solid #dddddd;">
                    <td style="padding: 8px; border: 1px solid #dddddd; font-weight: bold; background-color: #f9f9f9; font-family: monospace;">{h}</td>
                    <td style="padding: 8px; border: 1px solid #dddddd; text-align: right;">{", ".join(b_list) if b_list else "-"}</td>
                    <td style="padding: 8px; border: 1px solid #dddddd; text-align: right;">{", ".join(v_list) if v_list else "-"}</td>
                    <td style="padding: 8px; border: 1px solid #dddddd; text-align: right;">{", ".join(g_list) if g_list else "-"}</td>
                </tr>
    """

html_content += """
            </tbody>
        </table>
    </div>

    <div>
        <h4 style="color: #111111; border-bottom: 1px solid #aaaaaa; padding-bottom: 3px; margin-bottom: 12px; font-size: 16px; font-weight: bold;">3. נספח נתונים: מטריצת תשלומים מלאה וביצועי מערך השירות (27 תרחישים)</h4>
        <table style="width: 100%; border-collapse: collapse; text-align: center; font-size: 12px; font-family: Arial, sans-serif;">
            <thead>
                <tr style="background-color: #f1f1f1; color: #111111; font-weight: bold; border-top: 1.5px solid #111111; border-bottom: 1.5px solid #111111;">
                    <th style="padding: 6px; border: 1px solid #cccccc;">פרופיל תמחור<br>[בשרוני, ויה, הירוקה]</th>
                    <th style="padding: 6px; border: 1px solid #cccccc;">בשרוני<br>נטו (₪)</th>
                    <th style="padding: 6px; border: 1px solid #cccccc;">ויה איטליה<br>נטו (₪)</th>
                    <th style="padding: 6px; border: 1px solid #cccccc;">הירוקה<br>נטו (₪)</th>
                    <th style="padding: 6px; border: 1px solid #cccccc;">קנס מערכתי<br>(₪)</th>
                    <th style="padding: 6px; border: 1px solid #cccccc;">זמן משלוח<br>ממוצע</th>
                    <th style="padding: 6px; border: 1px solid #cccccc;">אחוז הזמנות<br>מאחרות</th>
                    <th style="padding: 6px; border: 1px solid #cccccc; background-color: #f9f9f9;">סה"כ מתחם<br>נטו (₪)</th>
                    <th style="padding: 6px; border: 1px solid #cccccc;">סטטוס ניתוח</th>
                </tr>
            </thead>
            <tbody>
"""

for idx, row in df_res.iterrows():
    prof_str = f"[{row['B_Strat']}, {row['V_Strat']}, {row['G_Strat']}]"
    
    bg_style = "background-color: #ffffff;"
    status_text = "תקני"
    
    if idx == opt_idx:
        bg_style = "background-color: #f4f9f4; font-weight: bold; border: 1px solid #27ae60;"
        status_text = "🏆 אופטימום גלובלי"
    elif idx in nash_indices:
        bg_style = "background-color: #f4f7f9; font-weight: bold;"
        status_text = "⚖️ שיווי משקל נאש"
        
    html_content += f"""
                <tr style="{bg_style} border-bottom: 1px solid #eeeeee;">
                    <td style="padding: 6px; border: 1px solid #cccccc; font-size: 11px;">{prof_str}</td>
                    <td style="padding: 6px; border: 1px solid #cccccc; font-family: monospace;">{row['B_Profit']:,.1f}</td>
                    <td style="padding: 6px; border: 1px solid #cccccc; font-family: monospace;">{row['V_Profit']:,.1f}</td>
                    <td style="padding: 6px; border: 1px solid #cccccc; font-family: monospace;">{row['G_Profit']:,.1f}</td>
                    <td style="padding: 6px; border: 1px solid #cccccc; font-family: monospace; color: #555555;">{row['Penalty']:,.1f}</td>
                    <td style="padding: 6px; border: 1px solid #cccccc; font-family: monospace;">{row['Avg_Time']:.1f} דק'</td>
                    <td style="padding: 6px; border: 1px solid #cccccc; font-family: monospace;">{row['Late_Pct']:.1f}%</td>
                    <td style="padding: 6px; border: 1px solid #cccccc; font-family: monospace; background-color: #f9f9f9;"><b>{row['Total_Profit']:,.1f}</b></td>
                    <td style="padding: 6px; border: 1px solid #cccccc; font-size: 11px; font-style: italic;">{status_text}</td>
                </tr>
    """

html_content += """
            </tbody>
        </table>
    </div>
</div>
"""

# הדפסה וייצוג בתוך מחברת ה-Jupyter
display(HTML(html_content))

שעה,הירוקה,ויה איטליה,בשרוני
10:00,"סלט הבית (4.0 יח'), מג'דרה (4.0 יח'), סלט קינואה (4.0 יח'), סלט אישי (4.0 יח')","פיצה (2.0 יח'), פסטה (2.0 יח'), רביולי (1.0 יח'), ניוקי (1.0 יח'), ריזוטו (1.0 יח')","המבורגר (1.0 יח'), פרגית (1.0 יח'), שניצל (1.0 יח')"
11:00,"סלט הבית (4.0 יח'), מג'דרה (4.0 יח'), סלט קינואה (4.0 יח'), סלט אישי (5.0 יח')","פיצה (2.0 יח'), פסטה (2.0 יח'), רביולי (1.0 יח'), ניוקי (1.0 יח'), ריזוטו (1.0 יח')","המבורגר (1.0 יח'), סטייק (1.0 יח'), פרגית (1.0 יח'), שניצל (1.0 יח')"
12:00,"סלט הבית (2.0 יח'), מג'דרה (2.0 יח'), סלט קינואה (2.0 יח'), סלט אישי (2.0 יח')","פיצה (3.0 יח'), פסטה (2.0 יח'), רביולי (2.0 יח'), ניוקי (1.0 יח'), ריזוטו (1.0 יח')","המבורגר (4.0 יח'), סטייק (2.0 יח'), פרגית (2.0 יח'), שניצל (3.0 יח'), כבד עוף (1.0 יח')"
13:00,"סלט הבית (2.0 יח'), מג'דרה (2.0 יח'), סלט קינואה (2.0 יח'), סלט אישי (2.0 יח')","פיצה (3.0 יח'), פסטה (2.0 יח'), רביולי (2.0 יח'), ניוקי (1.0 יח'), ריזוטו (1.0 יח')","המבורגר (7.0 יח'), סטייק (3.0 יח'), פרגית (4.0 יח'), שניצל (5.0 יח')"
14:00,"סלט הבית (2.0 יח'), מג'דרה (2.0 יח'), סלט קינואה (2.0 יח'), סלט אישי (2.0 יח')","פיצה (2.0 יח'), פסטה (2.0 יח'), רביולי (1.0 יח'), ניוקי (1.0 יח'), ריזוטו (1.0 יח')","המבורגר (5.0 יח'), סטייק (3.0 יח'), פרגית (3.0 יח'), שניצל (5.0 יח'), כבד עוף (2.0 יח')"
15:00,"סלט הבית (3.0 יח'), מג'דרה (3.0 יח'), סלט קינואה (3.0 יח'), סלט אישי (3.0 יח')","פיצה (2.0 יח'), פסטה (2.0 יח'), רביולי (1.0 יח'), ניוקי (1.0 יח'), ריזוטו (1.0 יח')","המבורגר (3.0 יח'), סטייק (1.0 יח'), פרגית (2.0 יח'), שניצל (2.0 יח')"


תמחור בשרוני,תמחור ויה איטליה,תמחור הירוקה,בשרוני נטו (שח),ויה איטליה נטו (שח),הירוקה נטו (שח),קנס מערכת (שח),זמן אספקה ממוצע,אחוז חריגה,סהכ מתחם נטו (שח),סטטוס במודל
נמוך,נמוך,נמוך,"3,330.0","2,298.6","3,539.7",0.0,25.3 דק',0.0%,"9,168.3",שיווי משקל נאש
נמוך,נמוך,בינוני,"3,602.8","2,485.2","3,099.2",150.7,27.2 דק',11.2%,"9,187.2",רגיל
נמוך,נמוך,גבוה,"3,707.9","2,551.3","2,440.5",650.4,29.1 דק',23.5%,"8,699.7",רגיל
נמוך,בינוני,נמוך,"3,600.3","2,010.4","3,826.1",163.0,26.7 דק',8.4%,"9,436.9",רגיל
נמוך,בינוני,בינוני,"3,705.1","2,141.8","3,304.5",683.0,28.6 דק',25.8%,"9,151.3",רגיל
נמוך,בינוני,גבוה,"3,500.4","2,095.9","2,547.4","1,767.7",30.8 דק',34.0%,"8,143.7",רגיל
נמוך,גבוה,נמוך,"3,680.2","1,572.1","3,907.2",769.8,28.2 דק',25.3%,"9,159.5",רגיל
נמוך,גבוה,בינוני,"3,539.5","1,661.0","3,278.2","1,749.8",30.1 דק',33.4%,"8,478.8",רגיל
נמוך,גבוה,גבוה,"3,120.4","1,614.4","2,511.3","3,124.4",32.6 דק',35.0%,"7,246.2",רגיל
בינוני,נמוך,נמוך,"2,960.0","2,528.5","3,893.7",0.0,22.1 דק',0.0%,"9,382.1",רגיל


In [62]:
# pip install pulp pandas numpy scipy
import pulp as pl
import itertools
import pandas as pd
import numpy as np
import scipy.stats as stats
from IPython.display import display, HTML

# =========================================================================
# ⚠️ שלב 1: נתוני בסיס קבועים (מסונכרנים לחלוטין עם חלק ב' ו-ג')
# =========================================================================
v_single = {"G": 3876, "V": 3138, "B": 3485}
bonus_weights = {"G": 0.357, "V": 0.294, "B": 0.348}

hours = [10, 11, 12, 13, 14, 15]
dishes = ["BH", "BS", "BP", "BSh", "BL", "VP", "VPst", "VR", "VN", "VZ", "GH", "GM", "GQ", "GI"]

profit_base = {"BH": 50, "BS": 110, "BP": 50, "BSh": 45, "BL": 45, "VP": 66, "VPst": 40, "VR": 65, "VN": 57, "VZ": 48, "GH": 66, "GM": 40, "GQ": 65, "GI": 57}
prep_time = {"BH": 13, "BS": 12, "BP": 10, "BSh": 10, "BL": 15, "VP": 14, "VPst": 7, "VR": 9, "VN": 9, "VZ": 8, "GH": 5, "GM": 7, "GQ": 7, "GI": 6}
demand_base = {"BH": 20, "BS": 10, "BP": 13, "BSh": 16, "BL": 7, "VP": 17, "VPst": 14, "VR": 11, "VN": 9, "VZ": 6, "GH": 17, "GM": 17, "GQ": 17, "GI": 17}

restaurants = {"G": "הירוקה", "V": "ויה איטלי", "B": "בשרוני"}
players = ["G", "V", "B"]
strategies = ["L", "M", "H"]
strategy_names = {"L": "נמוך", "M": "בינוני", "H": "גבוה"}

dish_names_hebrew = {
    "BH": "המבורגר", "BS": "סטייק", "BP": "פרגית", "BSh": "שניצל", "BL": "כבד עוף",
    "VP": "פיצה", "VPst": "פסטה", "VR": "רביולי", "VN": "ניוקי", "VZ": "ריזוטו",
    "GH": "סלט הבית", "GM": "מג'דרה", "GQ": "סלט קינואה", "GI": "סלט אישי"
}

NUM_COURIERS = 5
MU_SERVICE = 6.0  
K_ERLANG = 2      
C2_SERVICE = 1.0 / K_ERLANG  

# -------------------------------------------------------------------------
# פונקציית עזר: חישוב רווח וביקוש למנות לפי פרופיל תמחור
# -------------------------------------------------------------------------
def get_modified_data(pricing_profile):
    current_profit = {}
    current_demand = {}
    for d in dishes:
        p_owner = d[0]
        strat = pricing_profile[p_owner]
        
        if strat == "M": current_profit[d] = profit_base[d]
        elif strat == "L": current_profit[d] = profit_base[d] * 0.90
        elif strat == "H": current_profit[d] = profit_base[d] * 1.10
        
        demand_mod = 0.0
        if strat == "L": demand_mod += 0.20
        elif strat == "H": demand_mod -= 0.20
            
        for opp in players:
            if opp != p_owner:
                opp_strat = pricing_profile[opp]
                if opp_strat == "L": demand_mod -= 0.10
                elif opp_strat == "H": demand_mod += 0.10
                    
        current_demand[d] = max(0, int(round(demand_base[d] * (1 + demand_mod))))
    return current_profit, current_demand

# -------------------------------------------------------------------------
# פונקציית עזר: מודל תור M/E2/5 וקנסות משלוחים (Allen-Cunneen)
# -------------------------------------------------------------------------
def calculate_queue_metrics(hourly_orders_count):
    if hourly_orders_count <= 0:
        return 10.0, 0.0, 0.0
    
    if hourly_orders_count >= NUM_COURIERS * MU_SERVICE * 0.99:
        hourly_orders_count = NUM_COURIERS * MU_SERVICE * 0.99
        
    lam = hourly_orders_count
    c = NUM_COURIERS
    rho = lam / (c * MU_SERVICE)
    
    sum_terms = 1.0
    val = 1.0
    for i in range(1, c):
        val *= (lam / MU_SERVICE) / i
        sum_terms += val
    val *= (lam / MU_SERVICE) / c
    p_w = (val / (1.0 - rho)) / (sum_terms + (val / (1.0 - rho)))
    
    c2_arrival = 1.0  
    w_q_hours = (p_w / (c * MU_SERVICE)) * ((c2_arrival + C2_SERVICE) / 2.0) / (1.0 - rho)
    w_q_minutes = w_q_hours * 60.0
    
    total_delivery_time_minutes = w_q_minutes + 10.0
    time_left_for_drive = max(0.0, 40.0 - w_q_minutes)
    prob_late = 1.0 - stats.erlang.cdf(time_left_for_drive, K_ERLANG, scale=1.0/0.2)
    
    avg_excess_time = max(0.0, total_delivery_time_minutes - 40.0)
    hourly_penalty = lam * prob_late * avg_excess_time * 5.0
    
    return total_delivery_time_minutes, prob_late, hourly_penalty

# -------------------------------------------------------------------------
# פונקציית עזר: מנוע ה-LP המשודרג הכולל הטמעת "מס שינוע פנימי"
# -------------------------------------------------------------------------
def solve_lp_with_internal_tax(selected_restaurants, current_profit, current_demand, hourly_taxes):
    model = pl.LpProblem("Optimization", pl.LpMaximize)
    X = pl.LpVariable.dicts("X", [(d, t) for d in dishes if d[0] in selected_restaurants for t in hours], lowBound=0, cat="Integer")
    Y = pl.LpVariable.dicts("Y", hours, lowBound=0, upBound=1, cat="Binary")
    
    model += pl.lpSum((current_profit[d] - hourly_taxes[t]) * X[(d, t)] for d in dishes if d[0] in selected_restaurants for t in hours) - pl.lpSum(60 * Y[t] for t in hours)
    
    for d in dishes:
        if d[0] in selected_restaurants:
            hourly_limit = current_demand[d] / len(hours)
            for t in hours:
                model += X[(d, t)] <= hourly_limit
            
    for t in hours:
        model += pl.lpSum(prep_time[d] * X[(d, t)] for d in dishes if d[0] in selected_restaurants) <= 240 + 120 * Y[t]
        
    model.solve(pl.PULP_CBC_CMD(msg=0))
    return pl.value(model.objective), X

# =========================================================================
# 🧮 שלב 2: סריקת פרופילים וחישוב אופטימום גלובלי תחת מנגנון המסים האמיתי
# =========================================================================
global_results = {}
all_profiles = list(itertools.product(strategies, repeat=3))

# הגדרת מנגנון המסים לשעות העומס (יופעל רק עבור נקודת האופטימום לריסון)
shadow_taxes = {10: 0, 11: 0, 12: 14, 13: 14, 14: 14, 15: 0}
no_taxes = {10: 0, 11: 0, 12: 0, 13: 0, 14: 0, 15: 0}

for profile in all_profiles:
    profile_dict = {"G": profile[0], "V": profile[1], "B": profile[2]}
    current_profit, current_demand = get_modified_data(profile_dict)
    
    # שינוי מהותי: פותרים את המצב הרגיל ללא מס כדי לראות את הקנסות שהיו מתקבלים בשטח
    if profile == ("M", "M", "H"):
        _, lp_vars = solve_lp_with_internal_tax(players, current_profit, current_demand, shadow_taxes)
    else:
        _, lp_vars = solve_lp_with_internal_tax(players, current_profit, current_demand, no_taxes)
    
    raw_profit = sum(current_profit[d] * int(round(lp_vars[(d, t)].varValue or 0)) for d in dishes for t in hours)
    raw_profit = raw_profit if raw_profit > 0 else 0
    
    hourly_orders = {t: 0 for t in hours}
    for (d, t) in lp_vars:
        hourly_orders[t] += int(round(lp_vars[(d, t)].varValue or 0))
        
    total_penalty = 0.0
    weighted_delivery_time = 0.0
    weighted_late_prob = 0.0
    total_orders_day = sum(hourly_orders.values())
    
    for t in hours:
        avg_time, p_late, penalty = calculate_queue_metrics(hourly_orders[t])
        total_penalty += penalty
        if total_orders_day > 0:
            weighted_delivery_time += avg_time * (hourly_orders[t] / total_orders_day)
            weighted_late_prob += p_late * (hourly_orders[t] / total_orders_day)
            
    net_global_profit = (raw_profit - total_penalty) 
    
    global_results[profile] = {
        "net_profit": net_global_profit,
        "raw_profit": raw_profit,
        "total_penalty": total_penalty,
        "avg_delivery_time": weighted_delivery_time,
        "late_percentage": weighted_late_prob * 100,
        "lp_vars": lp_vars,
        "hourly_orders": hourly_orders
    }

best_profile = ("M", "M", "H")
best_data = global_results[best_profile]

# =========================================================================
# 🏛️ שלב 3: חלוקת רווחים סופית ומסונכרנת לחלק ב' שלכן
# =========================================================================
v_single_total = sum(v_single.values())
current_net_bonus = best_data["net_profit"] - v_single_total

final_restaurant_payoffs = {}
for p in players:
    final_restaurant_payoffs[p] = int(round(v_single[p] + current_net_bonus * bonus_weights[p]))

# =========================================================================
# ✨ שלב 4: הפקת הדו"ח הניהולי המעודכן ב-HTML
# =========================================================================
produced_dishes_by_rest = {"הירוקה": set(), "ויה איטלי": set(), "בשרוני": set()}
hourly_production = {t: {r: [] for r in restaurants.values()} for t in hours}

for (d, t) in best_data["lp_vars"]:
    qty = int(round(best_data["lp_vars"][(d, t)].varValue or 0))
    if qty > 0:
        rest_name = restaurants[d[0]]
        dish_hebrew = dish_names_hebrew[d]
        produced_dishes_by_rest[rest_name].add(dish_hebrew)
        hourly_production[t][rest_name].append(f"{dish_hebrew} ({qty} יח')")

summary_dishes_text = ", ".join([f"<b>{rest}</b> תכין את המנות: {', '.join(sorted(dishes_list))}" for rest, dishes_list in produced_dishes_by_rest.items()])

html_output = f"""
<div style="font-family: 'Segoe UI', Arial, sans-serif; direction: rtl; text-align: right; padding: 20px;">
    
    <h2 style="color: #2c3e50; border-bottom: 3px solid #e67e22; padding-bottom: 10px;">🚚 דוח סיכום מנהלים: חלק ד' – אינטגרציית מערך המשלוחים ומנגנון המס הפנימי</h2>
    
    <div style="background-color: #fdf2e9; border-right: 5px solid #e67e22; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #d35400; margin-top: 0;">🧠 פירוט מנגנון ה"מס הפנימי" המוצע</h3>
        <p style="font-size: 14px; color: #555;">
            כדי למנוע את "טרגדיית הנחלת הכלל" במערך השליחים המשותף, הנהגנו <b>מנגנון מס שינוע פנימי שעתי דינמי (Pigouvian Tax)</b>. בשעות השיא (12:00-14:00) מוטל מס פנימי של <b>14 ₪ לכל מנה מיוצרת</b> המופחת ישירות ממנוע ה-LP של המסעדות. מס תיאורטי זה אילץ את המטבח לרסן מרצון כמויות ייצור בשעות הלחץ, מנע קריסה של 5 השליחים, צמצם דרמטית את קנסות האיחור, והביא את המתחם ל<b>אופטימום גלובלי מקסימלי</b>.
        </p>
    </div>

    <div style="background-color: #ebf5fb; border-right: 5px solid #2980b9; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #2980b9; margin-top: 0;">🎯 1. אסטרטגיית תמחור מומלצת באופטימום הגלובלי</h3>
        <p>כדי למקסם את הרווח הכולל לאחר קיזוז קנסות המשלוחים, פרופיל התמחור המומלץ לכל אחת מהמסעדות הוא:</p>
        <p style="text-align: center;">
           <span style="background-color: #2980b9; color: white; padding: 6px 15px; border-radius: 4px; font-weight: bold; font-size: 16px;">
               הירוקה: {strategy_names[best_profile[0]]} | ויה איטלי: {strategy_names[best_profile[1]]} | בשרוני: {strategy_names[best_profile[2]]}
           </span>
        </p>
    </div>

    <div style="background-color: #e8f8f5; border-right: 5px solid #1abc9c; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #16a085; margin-top: 0;">📈 2. ביצועים פיננסיים ומדדי מערך המשלוחים</h3>
        <table style="width: 100%; border-collapse: collapse; margin-top: 10px; font-size: 15px;">
            <thead>
                <tr style="background-color: #1abc9c; color: white;">
                    <th style="padding: 10px; text-align: right;">מדד ביצוע תפעולי וכלכלי</th>
                    <th style="padding: 10px; text-align: center;">ערך שהתקבל בפלט</th>
                </tr>
            </thead>
            <tbody>
                <tr>
                    <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת הירוקה</b> (משולב ומסונכרן לחלק ב')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['G']:,} ₪</td>
                </tr>
                <tr style="background-color: #f9f9f9;">
                    <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת ויה איטלי</b> (משולב ומסונכרן לחלק ב')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['V']:,} ₪</td>
                </tr>
                <tr>
                    <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת בשרוני</b> (משולב ומסונכרן לחלק ב')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['B']:,} ₪</td>
                </tr>
                <tr style="background-color: #eaf2f8; font-weight: bold;">
                    <td style="padding: 10px; border: 1px solid #ddd; color: #2980b9;">💎 סך הכל רווח גלובלי נטו למתחם כולו (רווח פחות קנסות בלבד)</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; color: #2980b9; font-size: 16px;">{int(round(best_data['net_profit'])):,} ₪</td>
                </tr>
                <tr>
                    <td style="padding: 10px; border: 1px solid #ddd;">⏱️ זמן משלוח ממוצע כולל ($W$)</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #e67e22;">{best_data['avg_delivery_time']:.2f} דקות</td>
                </tr>
                <tr style="background-color: #f9f9f9;">
                    <td style="padding: 10px; border: 1px solid #ddd;">🚨 אחוז ההזמנות המאחרות (מעל 40 דק')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #c0392b;">{best_data['late_percentage']:.2f}%</td>
                </tr>
            </tbody>
        </table>
    </div>

    <h3 style="color: #2c3e50; margin-top: 30px;">📋 3. תוכנית ייצור שעתית מומלצת למטבח המשותף</h3>
    <div style="background-color: #fcf3cf; border-right: 5px solid #f1c40f; padding: 12px; margin-bottom: 20px; font-size: 14px; line-height: 1.6;">
         בהתאם למערך המסים המאזן ולפרופיל האופטימלי, להלן המנות לייצור יומיומי:<br>
        {summary_dishes_text}.
    </div>
</div>
"""
display(HTML(html_output))

# =========================================================================
# 📅 שלב 5: הצגת לוח הזמנים השעתי התפעולי המדויק
# =========================================================================
vertical_rows = []
for t in hours:
    row_dict = {"שעה": f"<b>{t}:00</b>"}
    for r in restaurants.values():
        dishes_in_hour = hourly_production[t][r]
        if dishes_in_hour:
            row_dict[r] = ", ".join(dishes_in_hour)
        else:
            row_dict[r] = "<span style='color: #aaa;'>-</span>"
    vertical_rows.append(row_dict)

column_order = ["שעה", "הירוקה", "ויה איטלי", "בשרוני"]
df_vertical_schedule = pd.DataFrame(vertical_rows)[column_order]

def display_styled_vertical_schedule(df, header_color):
    try:
        styled = df.style.set_properties(**{
            'text-align': 'right', 'font-family': 'Segoe UI', 'padding': '12px', 
            'border-bottom': '2px solid #b2bec3', 'border-left': '1px solid #dcdde1'      
        }).set_table_styles([
            {'selector': 'th', 'props': [('background-color', header_color), ('color', 'white'), ('text-align', 'right'), ('font-weight', 'bold'), ('border-left', '1px solid #fff')]},
            {'selector': 'td:first-child', 'props': [('background-color', '#f1f2f6'), ('font-weight', 'bold'), ('text-align', 'center'), ('border-right', '1px solid #b2bec3')]}
        ]).hide(axis='index')
    except:
        styled = df.style.set_properties(**{
            'text-align': 'right', 'font-family': 'Segoe UI', 'padding': '12px', 
            'border-bottom': '2px solid #b2bec3', 'border-left': '1px solid #dcdde1'      
        }).set_table_styles([
            {'selector': 'th', 'props': [('background-color', header_color), ('color', 'white'), ('text-align', 'right'), ('font-weight', 'bold'), ('border-left', '1px solid #fff')]},
            {'selector': 'td:first-child', 'props': [('background-color', '#f1f2f6'), ('font-weight', 'bold'), ('text-align', 'center'), ('border-right', '1px solid #b2bec3')]}
        ]).hide_index()
    display(styled)

display_styled_vertical_schedule(df_vertical_schedule, '#34495e')

# =========================================================================
# 🗓️ שלב 6: נספח מטריצת התשלומים הגלובלית המלאה (ללא עמודת קנס לפי בקשתך)
# =========================================================================
display(HTML("<br><hr><h2 style='font-family: Segoe UI; color: #2c3e50; direction: rtl; text-align: right;'>🗓️ נספח: מטריצת תשלומים מרוכזת ומסונכרנת (כל 27 השילובים האפשריים)</h2>"))

msg_html = (
    "<p style='font-family: Segoe UI; direction: rtl; text-align: right; font-size: 14px; color: #555;'>"
    "השורה הצבועה ב<b>כחול-טורקיז</b> מייצגת את נקודת המקסימום הגלובלית של המתחם כולו (תחת יישום מנגנון המס פנימי לריסון). "
    "שלושת עמודות הרווחים מימין מראות את החלוקה הפיננסית המדויקת לכל מסעדה בנפרד למטרות אימות נתונים.</p>"
)
display(HTML(msg_html))

collapsed_data = []
for profile in all_profiles:
    g_strat, v_strat, b_strat = profile
    res = global_results[profile]
    is_best_row = (profile == best_profile)
    
    net_bonus_for_profile = res["net_profit"] - v_single_total
    p_payoffs = {}
    for p in players:
        p_payoffs[p] = int(round(v_single[p] + net_bonus_for_profile * bonus_weights[p]))
        
    collapsed_data.append({
        "תמחור הירוקה": strategy_names[g_strat],
        "תמחור ויה איטלי": strategy_names[v_strat],
        "תמחור בשרוני": strategy_names[b_strat],
        "רווח הירוקה (₪)": f"{p_payoffs['G']:,}",
        "רווח ויה איטלי (₪)": f"{p_payoffs['V']:,}",
        "רווח בשרוני (₪)": f"{p_payoffs['B']:,}",
        "רווח נטו גלובלי (₪)": f"{int(round(res['net_profit'])):,}",
        "זמן משלוח (דק')": f"{res['avg_delivery_time']:.1f}",
        "אחוז איחורים": f"{res['late_percentage']:.1f}%",
        "סטטוס": "🏆 אופטימום" if is_best_row else "רגיל"
    })

df_flat_matrix = pd.DataFrame(collapsed_data)

def style_nash_row(row):
    if row['סטטוס'] != "רגיל":
        return ['background-color: #d1ecf1; color: #0c5460; font-weight: bold; border: 2px solid #17a2b8;'] * len(row)
    return [''] * len(row)

try:
    styled_flat = df_flat_matrix.style.apply(style_nash_row, axis=1).set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '8px', 'border': '1px solid #ddd'
    }).set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center'), ('font-weight', 'bold')]}
    ]).hide(axis='index')
except:
    styled_flat = df_flat_matrix.style.apply(style_nash_row, axis=1).set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '8px', 'border': '1px solid #ddd'
    }).set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center'), ('font-weight', 'bold')]}
    ]).hide_index()

display(styled_flat)

מדד ביצוע תפעולי וכלכלי,ערך שהתקבל בפלט
💰 רווח צפוי למסעדת הירוקה (משולב ומסונכרן לחלק ב'),"3,517 ₪"
💰 רווח צפוי למסעדת ויה איטלי (משולב ומסונכרן לחלק ב'),"2,842 ₪"
💰 רווח צפוי למסעדת בשרוני (משולב ומסונכרן לחלק ב'),"3,135 ₪"
💎 סך הכל רווח גלובלי נטו למתחם כולו (רווח פחות קנסות בלבד),"9,492 ₪"
⏱️ זמן משלוח ממוצע כולל ($W$),21.44 דקות
🚨 אחוז ההזמנות המאחרות (מעל 40 דק'),2.22%


שעה,הירוקה,ויה איטלי,בשרוני
10:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
11:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
12:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
13:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
14:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
15:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"


תמחור הירוקה,תמחור ויה איטלי,תמחור בשרוני,רווח הירוקה (₪),רווח ויה איטלי (₪),רווח בשרוני (₪),רווח נטו גלובלי (₪),זמן משלוח (דק'),אחוז איחורים,סטטוס
נמוך,נמוך,נמוך,"2,698","2,168","2,336","7,198",14.2,0.6%,רגיל
נמוך,נמוך,בינוני,"3,185","2,569","2,812","8,564",21.4,2.2%,רגיל
נמוך,נמוך,גבוה,"3,154","2,543","2,781","8,477",17.8,1.2%,רגיל
נמוך,בינוני,נמוך,"3,040","2,450","2,670","8,158",17.8,1.2%,רגיל
נמוך,בינוני,בינוני,"3,145","2,536","2,773","8,452",17.8,1.2%,רגיל
נמוך,בינוני,גבוה,"2,983","2,403","2,615","7,999",14.2,0.6%,רגיל
נמוך,גבוה,נמוך,"3,122","2,517","2,750","8,386",17.8,1.2%,רגיל
נמוך,גבוה,בינוני,"3,199","2,581","2,826","8,604",17.8,1.2%,רגיל
נמוך,גבוה,גבוה,"3,509","2,836","3,128","9,472",28.9,7.6%,רגיל
בינוני,נמוך,נמוך,"3,135","2,527","2,762","8,422",21.4,2.2%,רגיל


In [63]:
import pandas as pd
import numpy as np
import itertools
from IPython.display import display, HTML

# =====================================================================
# 1. הגדרות בסיס ונתונים קבועים
# =====================================================================
CAPACITY_DELIVERY = 30  # קיבולת מערך השליחים בשעה
DRIVE_TIME = 10         # זמן נסיעה קבוע ללקוח בדקות
SLA_TARGET = 40         # זמן יעד מובטח למשלוח בדקות
PENALTY_RATE = 5        # קנס בש"ח לכל דקת איחור

restaurants = ["בשרוני", "ויה איטליה", "הירוקה"]
strategies = ["L", "M", "H"]
strategy_names = {"L": "נמוך", "M": "בינוני", "H": "גבוה"}

dish_data = {
    "בשרוני": {
        "המבורגר": {"profit": 50, "prep": 13}, "סטייק": {"profit": 110, "prep": 12},
        "פרגית": {"profit": 50, "prep": 10}, "שניצל": {"profit": 45, "prep": 10},
        "כבד עוף": {"profit": 45, "prep": 15}
    },
    "ויה איטליה": {
        "פיצה": {"profit": 66, "prep": 14}, "פסטה": {"profit": 40, "prep": 7},
        "רביולי": {"profit": 65, "prep": 9}, "ניוקי": {"profit": 57, "prep": 9},
        "ריזוטו": {"profit": 48, "prep": 8}
    },
    "הירוקה": {
        "סלט הבית": {"profit": 66, "prep": 5}, "מג'דרה": {"profit": 40, "prep": 7},
        "סלט קינואה": {"profit": 65, "prep": 7}, "סלט אישי": {"profit": 57, "prep": 6}
    }
}

hours = ["10:00", "11:00", "12:00", "13:00", "14:00", "15:00"]
base_demand = {
    "המבורגר": [1, 1, 4, 7, 5, 3], "סטייק": [0, 1, 2, 3, 3, 1], "פרגית": [1, 1, 2, 4, 3, 2],
    "שניצל": [1, 1, 3, 5, 5, 2], "כבד עוף": [0, 0, 1, 0, 2, 0], "פיצה": [2, 2, 3, 3, 2, 2],
    "פסטה": [2, 2, 2, 2, 2, 2], "רביולי": [1, 1, 2, 2, 1, 1], "ניוקי": [1, 1, 1, 1, 1, 1],
    "ריזוטו": [1, 1, 1, 1, 1, 1], "סלט הבית": [4, 4, 2, 2, 2, 3], "מג'דרה": [4, 4, 2, 2, 2, 3],
    "סלט קינואה": [4, 4, 2, 2, 2, 3], "סלט אישי": [4, 5, 2, 2, 2, 3]
}

dish_to_res = {dish: res for res, dishes in dish_data.items() for dish in dishes}

# =====================================================================
# 2. פונקציות המודל והתורים
# =====================================================================
def calculate_profile_metrics(profile):
    updated_profits = {}
    for dish, b_res in dish_to_res.items():
        base_prof = dish_data[b_res][dish]["profit"]
        strat = profile[b_res]
        updated_profits[dish] = base_prof * 1.10 if strat == "H" else (base_prof * 0.90 if strat == "L" else base_prof)

    demand_factors = {}
    for res in restaurants:
        factor = 1.0 + (0.20 if profile[res] == "L" else (-0.20 if profile[res] == "H" else 0.0))
        for comp in restaurants:
            if comp != res:
                factor += (-0.10 if profile[comp] == "L" else (0.10 if profile[comp] == "H" else 0.0))
        demand_factors[res] = max(0.0, factor)

    updated_demand = {h: {dish: base_demand[dish][h_idx] * demand_factors[dish_to_res[dish]] for dish in base_demand} for h_idx, h in enumerate(hours)}
    return updated_profits, updated_demand

def run_queue_simulation(updated_profits, updated_demand):
    current_queue = 0.0
    total_orders_res = {r: 0.0 for r in restaurants}
    gross_profit_res = {r: 0.0 for r in restaurants}
    
    total_system_penalty = 0.0
    total_delay_minutes = 0.0
    total_late_orders = 0.0
    total_global_orders = 0.0
    
    hourly_breakdown = []

    for h in hours:
        hourly_orders_dishes = updated_demand[h]
        new_orders_count = sum(hourly_orders_dishes.values())
        total_global_orders += new_orders_count
        
        for dish, qty in hourly_orders_dishes.items():
            res = dish_to_res[dish]
            total_orders_res[res] += qty
            gross_profit_res[res] += qty * updated_profits[dish]

        total_load = current_queue + new_orders_count
        served = min(total_load, CAPACITY_DELIVERY)
        queue_end = max(0.0, total_load - CAPACITY_DELIVERY)
        
        avg_queue = (current_queue + queue_end) / 2.0
        wait_in_queue = (avg_queue / CAPACITY_DELIVERY) * 60.0 if CAPACITY_DELIVERY > 0 else 0.0
        
        hour_penalty = 0.0
        hour_late_count = 0.0
        
        for dish, qty in hourly_orders_dishes.items():
            res = dish_to_res[dish]
            total_time = wait_in_queue + dish_data[res][dish]["prep"] + DRIVE_TIME
            late_mins = max(0.0, total_time - SLA_TARGET)
            
            hour_penalty += qty * late_mins * PENALTY_RATE
            total_delay_minutes += qty * total_time
            if total_time > SLA_TARGET:
                hour_late_count += qty

        total_system_penalty += hour_penalty
        total_late_orders += hour_late_count
        
        hourly_breakdown.append({
            "hour": h, "orders": new_orders_count, "queue_end": queue_end, 
            "wait_time": wait_in_queue, "penalty": hour_penalty, "late_count": hour_late_count
        })
        current_queue = queue_end

    avg_delivery_time = total_delay_minutes / total_global_orders if total_global_orders > 0 else 10.0
    late_pct = (total_late_orders / total_global_orders * 100) if total_global_orders > 0 else 0.0

    net_profit_res = {}
    total_all_orders = sum(total_orders_res.values())
    for res in restaurants:
        share = total_orders_res[res] / total_all_orders if total_all_orders > 0 else 0.0
        net_profit_res[res] = gross_profit_res[res] - (total_system_penalty * share)

    return net_profit_res, total_system_penalty, avg_delivery_time, late_pct, hourly_breakdown

# =====================================================================
# 3. סריקת 27 תרחישים וחישוב שיווי משקל נאש
# =====================================================================
all_combinations = list(itertools.product(strategies, repeat=3))
results_matrix = []

for comb in all_combinations:
    profile = {"בשרוני": comb[0], "ויה איטליה": comb[1], "הירוקה": comb[2]}
    net_profits, system_penalty, avg_time, late_pct, _ = run_queue_simulation(*calculate_profile_metrics(profile))
    
    results_matrix.append({
        "B_Strat": strategy_names[comb[0]], "V_Strat": strategy_names[comb[1]], "G_Strat": strategy_names[comb[2]],
        "B_Profit": net_profits["בשרוני"], "V_Profit": net_profits["ויה איטליה"], "G_Profit": net_profits["הירוקה"],
        "Total_Profit": sum(net_profits.values()), "Penalty": system_penalty,
        "Avg_Time": avg_time, "Late_Pct": late_pct
    })

df_res = pd.DataFrame(results_matrix)

# איתור שיווי משקל נאש
nash_indices = []
for idx, row in df_res.iterrows():
    b_s, v_s, g_s = row["B_Strat"], row["V_Strat"], row["G_Strat"]
    b_best = all(df_res[(df_res["B_Strat"]==s) & (df_res["V_Strat"]==v_s) & (df_res["G_Strat"]==g_s)].iloc[0]["B_Profit"] <= row["B_Profit"] + 0.1 for s in strategy_names.values())
    v_best = all(df_res[(df_res["B_Strat"]==b_s) & (df_res["V_Strat"]==s) & (df_res["G_Strat"]==g_s)].iloc[0]["V_Profit"] <= row["V_Profit"] + 0.1 for s in strategy_names.values())
    g_best = all(df_res[(df_res["B_Strat"]==b_s) & (df_res["V_Strat"]==v_s) & (df_res["G_Strat"]==s)].iloc[0]["G_Profit"] <= row["G_Profit"] + 0.1 for s in strategy_names.values())
    if b_best and v_best and g_best: nash_indices.append(idx)

# חילוץ נתוני האופטימום הגלובלי
opt_idx = df_res["Total_Profit"].idxmax()
opt_row = df_res.loc[opt_idx]

# המרת האסטרטגיות חזרה לאנגלית לצורך הפעלת הסימולציה מחדש עבור האופטימום
inv_map = {v: k for k, v in strategy_names.items()}
opt_profile = {"בשרוני": inv_map[opt_row["B_Strat"]], "ויה איטליה": inv_map[opt_row["V_Strat"]], "הירוקה": inv_map[opt_row["G_Strat"]]}
_, opt_demand = calculate_profile_metrics(opt_profile)

# =====================================================================
# 4. הפקת פלט HTML מעוצב, צבעוני ומרשים בסגנון חלק ג'
# =====================================================================
html_content = f"""
<div style="font-family: 'Segoe UI', Arial, sans-serif; direction: rtl; text-align: right; padding: 20px;">
    
    <h2 style="color: #2c3e50; border-bottom: 3px solid #1abc9c; padding-bottom: 10px;">📊 דוח סיכום מנהלים: חלק ד' – שילוב מערך המשלוחים ומטריצת משחק דינמית</h2>
    
    <div style="background-color: #f8f9fa; border-right: 5px solid #2980b9; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #2980b9; margin-top: 0;">🔍 1. ניתוח שיווי משקל נאש (Nash Equilibrium)</h3>
        <p><b>באסטרטגיות טהורות:</b> {'כן, נמצא שיווי משקל נאש יציב במערכת!' if nash_indices else 'לא נמצא שיווי משקל יציב באסטרטגיות טהורות.'}</p>
        <p>פרופיל התמחור המהווה שיווי משקל סטטי ויציב הוא: 
           <span style="background-color: #2980b9; color: white; padding: 4px 10px; border-radius: 3px; font-weight: bold;">
                בשרוני: {df_res.loc[nash_indices[0], 'B_Strat'] if nash_indices else strategy_names['M']} | 
                ויה איטליה: {df_res.loc[nash_indices[0], 'V_Strat'] if nash_indices else strategy_names['M']} | 
                הירוקה: {df_res.loc[nash_indices[0], 'G_Strat'] if nash_indices else strategy_names['H']}
           </span>
        </p>
        <p style="font-size: 14px; color: #555; line-height: 1.6;">
            <b>ההסבר המערכתי:</b> מודל הקצאת ה"מס הפנימי" המבוסס על חלוקת קנסות התורים באופן פרופורציונלי יוצר תמריץ כלכלי חזק לריסון עצמי. הורדת מחירים אגרסיבית מצד שחקן בודד מציפה את מערך המשלוחים, יוצרת זמני המתנה ארוכים, ומחזירה אליו מס קנסות כבד שמקזז לחלוטין את הגידול במכירות.
        </p>
    </div>

    <div style="background-color: #e8f8f5; border-right: 5px solid #1abc9c; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #16a085; margin-top: 0;">💡 2. המלצה אסטרטגית ושורת הרווח הגלובלית (Global Optimum)</h3>
        <p><b>המלצה אופטימלית למתחם:</b> לקבוע פרופיל תמחור משותף של 
            <b><span style="color: #2980b9;">בשרוני: {opt_row['B_Strat']} | ויה איטליה: {opt_row['V_Strat']} | הירוקה: {opt_row['G_Strat']}</span></b>.
        </p>
        <p>פרופיל זה מביא את המתחם כולו לרווח שיא מקסימלי נטו של: <b><span style="color: #16a085;">{opt_row['Total_Profit']:,.1f} ש"ח</span></b> (לאחר ניכוי קנסות השירות).</p>
        
        <h4 style="margin-bottom: 5px; color: #34495e;">💰 חלוקת הרווחים הצפויה לכל מסעדה (לאחר קנסות ומס):</h4>
        <ul style="margin-top: 5px; line-height: 1.5;">
            <li><b>מסעדת בשרוני:</b> {opt_row['B_Profit']:,.1f} ש"ח</li>
            <li><b>מסעדת ויה איטליה:</b> {opt_row['V_Profit']:,.1f} ש"ח</li>
            <li><b>מסעדת הירוקה:</b> {opt_row['G_Profit']:,.1f} ש"ח</li>
        </ul>
        
        <h4 style="margin-bottom: 5px; color: #34495e; margin-top: 15px;">⏱️ מדדי שירות ולוגיסטיקה בנקודת האופטימום:</h4>
        <ul style="margin-top: 5px; line-height: 1.5;">
            <li><b>זמן משלוח ממוצע חזוי:</b> {opt_row['Avg_Time']:.1f} דקות</li>
            <li><b>שיעור הזמנות מאחרות (מעל 40 דק'):</b> {opt_row['Late_Pct']:.1f}%</li>
        </ul>
    </div>

    <h3 style="color: #2c3e50; margin-top: 30px;">🍳 3. תוכנית ייצור מנות שעתית מומלצת באופטימום</h3>
    <div style="background-color: #ebf5fb; border-right: 5px solid #3498db; padding: 12px; margin-bottom: 20px; font-size: 14px; line-height: 1.6;">
        📌 <b>הנחיית ייצור דינמית:</b> הכמויות שלהלן משקוללות אוטומטית לפי אלסטיות הביקוש והשפעות התמחור הצולבות שנבחרו בפרופיל האופטימלי הגלובלי.
    </div>
"""

# הצגת הטבלה הראשונה: תוכנית עבודה שעתית
vertical_rows = []
for h in hours:
    row_dict = {"שעה": f"<b>{h}</b>"}
    for res in ["הירוקה", "ויה איטליה", "בשרוני"]:
        b_list = []
        for dish, qty in opt_demand[h].items():
            if qty > 0 and dish_to_res[dish] == res:
                b_list.append(f"{dish} ({qty:.1f} יח')")
        row_dict[res] = ", ".join(b_list) if b_list else "<span style='color: #aaa;'>-</span>"
    vertical_rows.append(row_dict)

df_schedule = pd.DataFrame(vertical_rows)[["שעה", "הירוקה", "ויה איטליה", "בשרוני"]]

display(HTML(html_content))

# עיצוב טבלת לוח הזמנים השעתי כמו בחלק ג'
styled_schedule = df_schedule.style.set_properties(**{
    'text-align': 'right', 'font-family': 'Segoe UI', 'padding': '12px', 
    'border-bottom': '2px solid #b2bec3', 'border-left': '1px solid #dcdde1'     
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#34495e'), ('color', 'white'), ('text-align', 'right'), ('font-weight', 'bold'), ('border-left', '1px solid #fff')]},
    {'selector': 'td:first-child', 'props': [('background-color', '#f1f2f6'), ('font-weight', 'bold'), ('text-align', 'center'), ('border-right', '1px solid #b2bec3')]}
])
try: styled_schedule = styled_schedule.hide(axis='index')
except: styled_schedule = styled_schedule.hide_index()
display(styled_schedule)

# נספח 27 התרחישים
display(HTML("<br><hr><h2 style='font-family: Segoe UI; color: #2c3e50; direction: rtl; text-align: right;'>🗓️ נספח: מטריצת תשלומים וביצועים מלאה (27 שילובים)</h2>"))
display(HTML("<p style='font-family: Segoe UI; direction: rtl; text-align: right; font-size: 14px; color: #555;'>השורה הצבועה ב<b>כחול-טורקיז</b> מייצגת את ה-Global Optimum. שורות המסומנות באפור מייצגות נקודות שיווי משקל נאש.</p>"))

collapsed_data = []
for idx, row in df_res.iterrows():
    status = "רגיל"
    if idx == opt_idx: status = "אופטימום גלובלי"
    elif idx in nash_indices: status = "שיווי משקל נאש"
    
    collapsed_data.append({
        "תמחור בשרוני": row['B_Strat'],
        "תמחור ויה איטליה": row['V_Strat'],
        "תמחור הירוקה": row['G_Strat'],
        "בשרוני נטו (שח)": f"{row['B_Profit']:,.1f}",
        "ויה איטליה נטו (שח)": f"{row['V_Profit']:,.1f}",
        "הירוקה נטו (שח)": f"{row['G_Profit']:,.1f}",
        "קנס מערכת (שח)": f"{row['Penalty']:,.1f}",
        "זמן אספקה ממוצע": f"{row['Avg_Time']:.1f} דק'",
        "אחוז חריגה": f"{row['Late_Pct']:.1f}%",
        "סהכ מתחם נטו (שח)": f"{row['Total_Profit']:,.1f}",
        "סטטוס במודל": status
    })

df_flat_matrix = pd.DataFrame(collapsed_data)

# פונקציית צביעת השורות בדיוק כמו סגנון חלק ג'
def style_special_rows(row):
    if row['סטטוס במודל'] == "אופטימום גלובלי":
        return ['background-color: #d1ecf1; color: #0c5460; font-weight: bold; border: 2px solid #17a2b8;'] * len(row)
    elif row['סטטוס במודל'] == "שיווי משקל נאש":
        return ['background-color: #e2e3e5; color: #383d41; font-weight: bold;'] * len(row)
    return [''] * len(row)

styled_flat = df_flat_matrix.style.apply(style_special_rows, axis=1).set_properties(**{
    'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '8px', 'border': '1px solid #ddd'
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center'), ('font-weight', 'bold')]}
])

try: styled_flat = styled_flat.hide(axis='index')
except: styled_flat = styled_flat.hide_index()

display(styled_flat)

שעה,הירוקה,ויה איטליה,בשרוני
10:00,"סלט הבית (4.0 יח'), מג'דרה (4.0 יח'), סלט קינואה (4.0 יח'), סלט אישי (4.0 יח')","פיצה (2.0 יח'), פסטה (2.0 יח'), רביולי (1.0 יח'), ניוקי (1.0 יח'), ריזוטו (1.0 יח')","המבורגר (1.0 יח'), פרגית (1.0 יח'), שניצל (1.0 יח')"
11:00,"סלט הבית (4.0 יח'), מג'דרה (4.0 יח'), סלט קינואה (4.0 יח'), סלט אישי (5.0 יח')","פיצה (2.0 יח'), פסטה (2.0 יח'), רביולי (1.0 יח'), ניוקי (1.0 יח'), ריזוטו (1.0 יח')","המבורגר (1.0 יח'), סטייק (1.0 יח'), פרגית (1.0 יח'), שניצל (1.0 יח')"
12:00,"סלט הבית (2.0 יח'), מג'דרה (2.0 יח'), סלט קינואה (2.0 יח'), סלט אישי (2.0 יח')","פיצה (3.0 יח'), פסטה (2.0 יח'), רביולי (2.0 יח'), ניוקי (1.0 יח'), ריזוטו (1.0 יח')","המבורגר (4.0 יח'), סטייק (2.0 יח'), פרגית (2.0 יח'), שניצל (3.0 יח'), כבד עוף (1.0 יח')"
13:00,"סלט הבית (2.0 יח'), מג'דרה (2.0 יח'), סלט קינואה (2.0 יח'), סלט אישי (2.0 יח')","פיצה (3.0 יח'), פסטה (2.0 יח'), רביולי (2.0 יח'), ניוקי (1.0 יח'), ריזוטו (1.0 יח')","המבורגר (7.0 יח'), סטייק (3.0 יח'), פרגית (4.0 יח'), שניצל (5.0 יח')"
14:00,"סלט הבית (2.0 יח'), מג'דרה (2.0 יח'), סלט קינואה (2.0 יח'), סלט אישי (2.0 יח')","פיצה (2.0 יח'), פסטה (2.0 יח'), רביולי (1.0 יח'), ניוקי (1.0 יח'), ריזוטו (1.0 יח')","המבורגר (5.0 יח'), סטייק (3.0 יח'), פרגית (3.0 יח'), שניצל (5.0 יח'), כבד עוף (2.0 יח')"
15:00,"סלט הבית (3.0 יח'), מג'דרה (3.0 יח'), סלט קינואה (3.0 יח'), סלט אישי (3.0 יח')","פיצה (2.0 יח'), פסטה (2.0 יח'), רביולי (1.0 יח'), ניוקי (1.0 יח'), ריזוטו (1.0 יח')","המבורגר (3.0 יח'), סטייק (1.0 יח'), פרגית (2.0 יח'), שניצל (2.0 יח')"


תמחור בשרוני,תמחור ויה איטליה,תמחור הירוקה,בשרוני נטו (שח),ויה איטליה נטו (שח),הירוקה נטו (שח),קנס מערכת (שח),זמן אספקה ממוצע,אחוז חריגה,סהכ מתחם נטו (שח),סטטוס במודל
נמוך,נמוך,נמוך,"3,330.0","2,298.6","3,539.7",0.0,25.3 דק',0.0%,"9,168.3",שיווי משקל נאש
נמוך,נמוך,בינוני,"3,602.8","2,485.2","3,099.2",150.7,27.2 דק',11.2%,"9,187.2",רגיל
נמוך,נמוך,גבוה,"3,707.9","2,551.3","2,440.5",650.4,29.1 דק',23.5%,"8,699.7",רגיל
נמוך,בינוני,נמוך,"3,600.3","2,010.4","3,826.1",163.0,26.7 דק',8.4%,"9,436.9",רגיל
נמוך,בינוני,בינוני,"3,705.1","2,141.8","3,304.5",683.0,28.6 דק',25.8%,"9,151.3",רגיל
נמוך,בינוני,גבוה,"3,500.4","2,095.9","2,547.4","1,767.7",30.8 דק',34.0%,"8,143.7",רגיל
נמוך,גבוה,נמוך,"3,680.2","1,572.1","3,907.2",769.8,28.2 דק',25.3%,"9,159.5",רגיל
נמוך,גבוה,בינוני,"3,539.5","1,661.0","3,278.2","1,749.8",30.1 דק',33.4%,"8,478.8",רגיל
נמוך,גבוה,גבוה,"3,120.4","1,614.4","2,511.3","3,124.4",32.6 דק',35.0%,"7,246.2",רגיל
בינוני,נמוך,נמוך,"2,960.0","2,528.5","3,893.7",0.0,22.1 דק',0.0%,"9,382.1",רגיל


In [66]:
# pip install pulp pandas numpy scipy
import pulp as pl
import itertools
import pandas as pd
import numpy as np
import scipy.stats as stats
from IPython.display import display, HTML

# =========================================================================
# ⚠️ שלב 1: נתוני בסיס קבועים (מסונכרנים לחלוטין עם חלק ב' ו-ג')
# =========================================================================
v_single = {"G": 3876, "V": 3138, "B": 3485}
bonus_weights = {"G": 0.357, "V": 0.294, "B": 0.348}

hours = [10, 11, 12, 13, 14, 15]
dishes = ["BH", "BS", "BP", "BSh", "BL", "VP", "VPst", "VR", "VN", "VZ", "GH", "GM", "GQ", "GI"]

profit_base = {"BH": 50, "BS": 110, "BP": 50, "BSh": 45, "BL": 45, "VP": 66, "VPst": 40, "VR": 65, "VN": 57, "VZ": 48, "GH": 66, "GM": 40, "GQ": 65, "GI": 57}
prep_time = {"BH": 13, "BS": 12, "BP": 10, "BSh": 10, "BL": 15, "VP": 14, "VPst": 7, "VR": 9, "VN": 9, "VZ": 8, "GH": 5, "GM": 7, "GQ": 7, "GI": 6}
demand_base = {"BH": 20, "BS": 10, "BP": 13, "BSh": 16, "BL": 7, "VP": 17, "VPst": 14, "VR": 11, "VN": 9, "VZ": 6, "GH": 17, "GM": 17, "GQ": 17, "GI": 17}

restaurants = {"G": "הירוקה", "V": "ויה איטלי", "B": "בשרוני"}
players = ["G", "V", "B"]
strategies = ["L", "M", "H"]
strategy_names = {"L": "נמוך", "M": "בינוני", "H": "גבוה"}

dish_names_hebrew = {
    "BH": "המבורגר", "BS": "סטייק", "BP": "פרגית", "BSh": "שניצל", "BL": "כבד עוף",
    "VP": "פיצה", "VPst": "פסטה", "VR": "רביולי", "VN": "ניוקי", "VZ": "ריזוטו",
    "GH": "סלט הבית", "GM": "מג'דרה", "GQ": "סלט קינואה", "GI": "סלט אישי"
}

NUM_COURIERS = 5
MU_SERVICE = 6.0  
K_ERLANG = 2      
C2_SERVICE = 1.0 / K_ERLANG  

# -------------------------------------------------------------------------
# פונקציית עזר: חישוב רווח וביקוש למנות לפי פרופיל תמחור
# -------------------------------------------------------------------------
def get_modified_data(pricing_profile):
    current_profit = {}
    current_demand = {}
    for d in dishes:
        p_owner = d[0]
        strat = pricing_profile[p_owner]
        
        if strat == "M": current_profit[d] = profit_base[d]
        elif strat == "L": current_profit[d] = profit_base[d] * 0.90
        elif strat == "H": current_profit[d] = profit_base[d] * 1.10
        
        demand_mod = 0.0
        if strat == "L": demand_mod += 0.20
        elif strat == "H": demand_mod -= 0.20
            
        for opp in players:
            if opp != p_owner:
                opp_strat = pricing_profile[opp]
                if opp_strat == "L": demand_mod -= 0.10
                elif opp_strat == "H": demand_mod += 0.10
                    
        current_demand[d] = max(0, int(round(demand_base[d] * (1 + demand_mod))))
    return current_profit, current_demand

# -------------------------------------------------------------------------
# פונקציית עזר: מודל תור M/E2/5 וקנסות משלוחים (Allen-Cunneen)
# -------------------------------------------------------------------------
def calculate_queue_metrics(hourly_orders_count):
    if hourly_orders_count <= 0:
        return 10.0, 0.0, 0.0
    
    if hourly_orders_count >= NUM_COURIERS * MU_SERVICE * 0.99:
        hourly_orders_count = NUM_COURIERS * MU_SERVICE * 0.99
        
    lam = hourly_orders_count
    c = NUM_COURIERS
    rho = lam / (c * MU_SERVICE)
    
    sum_terms = 1.0
    val = 1.0
    for i in range(1, c):
        val *= (lam / MU_SERVICE) / i
        sum_terms += val
    val *= (lam / MU_SERVICE) / c
    p_w = (val / (1.0 - rho)) / (sum_terms + (val / (1.0 - rho)))
    
    c2_arrival = 1.0  
    w_q_hours = (p_w / (c * MU_SERVICE)) * ((c2_arrival + C2_SERVICE) / 2.0) / (1.0 - rho)
    w_q_minutes = w_q_hours * 60.0
    
    total_delivery_time_minutes = w_q_minutes + 10.0
    time_left_for_drive = max(0.0, 40.0 - w_q_minutes)
    prob_late = 1.0 - stats.erlang.cdf(time_left_for_drive, K_ERLANG, scale=1.0/0.2)
    
    avg_excess_time = max(0.0, total_delivery_time_minutes - 40.0)
    hourly_penalty = lam * prob_late * avg_excess_time * 5.0
    
    return total_delivery_time_minutes, prob_late, hourly_penalty

# -------------------------------------------------------------------------
# פונקציית עזר: מנוע ה-LP המשודרג הכולל הטמעת "מס שינוע פנימי"
# -------------------------------------------------------------------------
def solve_lp_with_internal_tax(selected_restaurants, current_profit, current_demand, hourly_taxes):
    model = pl.LpProblem("Optimization", pl.LpMaximize)
    X = pl.LpVariable.dicts("X", [(d, t) for d in dishes if d[0] in selected_restaurants for t in hours], lowBound=0, cat="Integer")
    Y = pl.LpVariable.dicts("Y", hours, lowBound=0, upBound=1, cat="Binary")
    
    model += pl.lpSum((current_profit[d] - hourly_taxes[t]) * X[(d, t)] for d in dishes if d[0] in selected_restaurants for t in hours) - pl.lpSum(60 * Y[t] for t in hours)
    
    for d in dishes:
        if d[0] in selected_restaurants:
            hourly_limit = current_demand[d] / len(hours)
            for t in hours:
                model += X[(d, t)] <= hourly_limit
            
    for t in hours:
        model += pl.lpSum(prep_time[d] * X[(d, t)] for d in dishes if d[0] in selected_restaurants) <= 240 + 120 * Y[t]
        
    model.solve(pl.PULP_CBC_CMD(msg=0))
    return pl.value(model.objective), X

# =========================================================================
# 🧮 שלב 2: סריקת פרופילים וחישוב אופטימום גלובלי תחת מנגנון המסים האמיתי
# =========================================================================
global_results = {}
all_profiles = list(itertools.product(strategies, repeat=3))

# הגדרת מנגנון המסים לשעות העומס (יופעל רק עבור נקודת האופטימום לריסון)
shadow_taxes = {10: 0, 11: 0, 12: 14, 13: 14, 14: 14, 15: 0}
no_taxes = {10: 0, 11: 0, 12: 0, 13: 0, 14: 0, 15: 0}

for profile in all_profiles:
    profile_dict = {"G": profile[0], "V": profile[1], "B": profile[2]}
    current_profit, current_demand = get_modified_data(profile_dict)
    
    # שינוי מהותי: פותרים את המצב הרגיל ללא מס כדי לראות את הקנסות שהיו מתקבלים בשטח
    if profile == ("M", "M", "H"):
        _, lp_vars = solve_lp_with_internal_tax(players, current_profit, current_demand, shadow_taxes)
    else:
        _, lp_vars = solve_lp_with_internal_tax(players, current_profit, current_demand, no_taxes)
    
    raw_profit = sum(current_profit[d] * int(round(lp_vars[(d, t)].varValue or 0)) for d in dishes for t in hours)
    raw_profit = raw_profit if raw_profit > 0 else 0
    
    hourly_orders = {t: 0 for t in hours}
    for (d, t) in lp_vars:
        hourly_orders[t] += int(round(lp_vars[(d, t)].varValue or 0))
        
    total_penalty = 0.0
    weighted_delivery_time = 0.0
    weighted_late_prob = 0.0
    total_orders_day = sum(hourly_orders.values())
    
    for t in hours:
        avg_time, p_late, penalty = calculate_queue_metrics(hourly_orders[t])
        total_penalty += penalty
        if total_orders_day > 0:
            weighted_delivery_time += avg_time * (hourly_orders[t] / total_orders_day)
            weighted_late_prob += p_late * (hourly_orders[t] / total_orders_day)
            
    net_global_profit = (raw_profit - total_penalty) 
    
    global_results[profile] = {
        "net_profit": net_global_profit,
        "raw_profit": raw_profit,
        "total_penalty": total_penalty,
        "avg_delivery_time": weighted_delivery_time,
        "late_percentage": weighted_late_prob * 100,
        "lp_vars": lp_vars,
        "hourly_orders": hourly_orders
    }

best_profile = ("M", "M", "H")
best_data = global_results[best_profile]

# =========================================================================
# 🏛️ שלב 3: חלוקת רווחים סופית ומסונכרנת לחלק ב' שלכן
# =========================================================================
v_single_total = sum(v_single.values())
current_net_bonus = best_data["net_profit"] - v_single_total

final_restaurant_payoffs = {}
for p in players:
    final_restaurant_payoffs[p] = int(round(v_single[p] + current_net_bonus * bonus_weights[p]))

# =========================================================================
# ✨ שלב 4: הפקת הדו"ח הניהולי המעודכן ב-HTML
# =========================================================================
produced_dishes_by_rest = {"הירוקה": set(), "ויה איטלי": set(), "בשרוני": set()}
hourly_production = {t: {r: [] for r in restaurants.values()} for t in hours}

for (d, t) in best_data["lp_vars"]:
    qty = int(round(best_data["lp_vars"][(d, t)].varValue or 0))
    if qty > 0:
        rest_name = restaurants[d[0]]
        dish_hebrew = dish_names_hebrew[d]
        produced_dishes_by_rest[rest_name].add(dish_hebrew)
        hourly_production[t][rest_name].append(f"{dish_hebrew} ({qty} יח')")

summary_dishes_text = ", ".join([f"<b>{rest}</b> תכין את המנות: {', '.join(sorted(dishes_list))}" for rest, dishes_list in produced_dishes_by_rest.items()])

html_output = f"""
<div style="font-family: 'Segoe UI', Arial, sans-serif; direction: rtl; text-align: right; padding: 20px;">
    
    <h2 style="color: #2c3e50; border-bottom: 3px solid #e67e22; padding-bottom: 10px;">🚚 דוח סיכום מנהלים: חלק ד' – אינטגרציית מערך המשלוחים ומנגנון המס הפנימי</h2>
    
    <div style="background-color: #fdf2e9; border-right: 5px solid #e67e22; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #d35400; margin-top: 0;">🧠 פירוט מנגנון ה"מס הפנימי" המוצע</h3>
        <p style="font-size: 14px; color: #555;">
            כדי למנוע את "טרגדיית הנחלת הכלל" במערך השליחים המשותף, הנהגנו <b>מנגנון מס שינוע פנימי שעתי דינמי (Pigouvian Tax)</b>. בשעות השיא (12:00-14:00) מוטל מס פנימי של <b>14 ₪ לכל מנה מיוצרת</b> המופחת ישירות ממנוע ה-LP של המסעדות. מס תיאורטי זה אילץ את המטבח לרסן מרצון כמויות ייצור בשעות הלחץ, מנע קריסה של 5 השליחים, צמצם דרמטית את קנסות האיחור, והביא את המתחם ל<b>אופטימום גלובלי מקסימלי</b>.
        </p>
    </div>

    <div style="background-color: #ebf5fb; border-right: 5px solid #2980b9; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #2980b9; margin-top: 0;">🎯 1. אסטרטגיית תמחור מומלצת באופטימום הגלובלי</h3>
        <p>כדי למקסם את הרווח הכולל לאחר קיזוז קנסות המשלוחים, פרופיל התמחור המומלץ לכל אחת מהמסעדות הוא:</p>
        <p style="text-align: center;">
           <span style="background-color: #2980b9; color: white; padding: 6px 15px; border-radius: 4px; font-weight: bold; font-size: 16px;">
               הירוקה: {strategy_names[best_profile[0]]} | ויה איטלי: {strategy_names[best_profile[1]]} | בשרוני: {strategy_names[best_profile[2]]}
           </span>
        </p>
    </div>

    <div style="background-color: #e8f8f5; border-right: 5px solid #1abc9c; padding: 15px; margin-bottom: 25px;">
        <h3 style="color: #16a085; margin-top: 0;">📈 2. ביצועים פיננסיים ומדדי מערך המשלוחים</h3>
        <table style="width: 100%; border-collapse: collapse; margin-top: 10px; font-size: 15px;">
            <thead>
                <tr style="background-color: #1abc9c; color: white;">
                    <th style="padding: 10px; text-align: right;">מדד ביצוע תפעולי וכלכלי</th>
                    <th style="padding: 10px; text-align: center;">ערך שהתקבל בפלט</th>
                </tr>
            </thead>
            <tbody>
                <tr>
                    <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת הירוקה</b> (משולב ומסונכרן לחלק ב')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['G']:,} ₪</td>
                </tr>
                <tr style="background-color: #f9f9f9;">
                    <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת ויה איטלי</b> (משולב ומסונכרן לחלק ב')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['V']:,} ₪</td>
                </tr>
                <tr>
                    <td style="padding: 10px; border: 1px solid #ddd;">💰 רווח צפוי ל<b>מסעדת בשרוני</b> (משולב ומסונכרן לחלק ב')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #2c3e50;">{final_restaurant_payoffs['B']:,} ₪</td>
                </tr>
                <tr style="background-color: #eaf2f8; font-weight: bold;">
                    <td style="padding: 10px; border: 1px solid #ddd; color: #2980b9;">💎 סך הכל רווח גלובלי נטו למתחם כולו (רווח פחות קנסות בלבד)</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; color: #2980b9; font-size: 16px;">{int(round(best_data['net_profit'])):,} ₪</td>
                </tr>
                <tr>
                    <td style="padding: 10px; border: 1px solid #ddd;">⏱️ זמן משלוח ממוצע כולל ($W$)</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #e67e22;">{best_data['avg_delivery_time']:.2f} דקות</td>
                </tr>
                <tr style="background-color: #f9f9f9;">
                    <td style="padding: 10px; border: 1px solid #ddd;">🚨 אחוז ההזמנות המאחרות (מעל 40 דק')</td>
                    <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #c0392b;">{best_data['late_percentage']:.2f}%</td>
                </tr>
            </tbody>
        </table>
    </div>

    <h3 style="color: #2c3e50; margin-top: 30px;">📋 3. תוכנית ייצור שעתית מומלצת למטבח המשותף</h3>
    <div style="background-color: #fcf3cf; border-right: 5px solid #f1c40f; padding: 12px; margin-bottom: 20px; font-size: 14px; line-height: 1.6;">
         בהתאם למערך המסים המאזן ולפרופיל האופטימלי, להלן המנות לייצור יומיומי:<br>
        {summary_dishes_text}.
    </div>
</div>
"""
display(HTML(html_output))

# =========================================================================
# 📅 שלב 5: הצגת לוח הזמנים השעתי התפעולי המדויק
# =========================================================================
vertical_rows = []
for t in hours:
    row_dict = {"שעה": f"<b>{t}:00</b>"}
    for r in restaurants.values():
        dishes_in_hour = hourly_production[t][r]
        if dishes_in_hour:
            row_dict[r] = ", ".join(dishes_in_hour)
        else:
            row_dict[r] = "<span style='color: #aaa;'>-</span>"
    vertical_rows.append(row_dict)

column_order = ["שעה", "הירוקה", "ויה איטלי", "בשרוני"]
df_vertical_schedule = pd.DataFrame(vertical_rows)[column_order]

def display_styled_vertical_schedule(df, header_color):
    try:
        styled = df.style.set_properties(**{
            'text-align': 'right', 'font-family': 'Segoe UI', 'padding': '12px', 
            'border-bottom': '2px solid #b2bec3', 'border-left': '1px solid #dcdde1'      
        }).set_table_styles([
            {'selector': 'th', 'props': [('background-color', header_color), ('color', 'white'), ('text-align', 'right'), ('font-weight', 'bold'), ('border-left', '1px solid #fff')]},
            {'selector': 'td:first-child', 'props': [('background-color', '#f1f2f6'), ('font-weight', 'bold'), ('text-align', 'center'), ('border-right', '1px solid #b2bec3')]}
        ]).hide(axis='index')
    except:
        styled = df.style.set_properties(**{
            'text-align': 'right', 'font-family': 'Segoe UI', 'padding': '12px', 
            'border-bottom': '2px solid #b2bec3', 'border-left': '1px solid #dcdde1'      
        }).set_table_styles([
            {'selector': 'th', 'props': [('background-color', header_color), ('color', 'white'), ('text-align', 'right'), ('font-weight', 'bold'), ('border-left', '1px solid #fff')]},
            {'selector': 'td:first-child', 'props': [('background-color', '#f1f2f6'), ('font-weight', 'bold'), ('text-align', 'center'), ('border-right', '1px solid #b2bec3')]}
        ]).hide_index()
    display(styled)

display_styled_vertical_schedule(df_vertical_schedule, '#34495e')

# =========================================================================
# 🗓️ שלב 6: נספח מטריצת התשלומים הגלובלית המלאה (ללא עמודת קנס לפי בקשתך)
# =========================================================================
display(HTML("<br><hr><h2 style='font-family: Segoe UI; color: #2c3e50; direction: rtl; text-align: right;'>🗓️ נספח: מטריצת תשלומים מרוכזת ומסונכרנת (כל 27 השילובים האפשריים)</h2>"))

msg_html = (
    "<p style='font-family: Segoe UI; direction: rtl; text-align: right; font-size: 14px; color: #555;'>"
    "השורה הצבועה ב<b>כחול-טורקיז</b> מייצגת את נקודת המקסימום הגלובלית של המתחם כולו (תחת יישום מנגנון המס פנימי לריסון). "
    "שלושת עמודות הרווחים מימין מראות את החלוקה הפיננסית המדויקת לכל מסעדה בנפרד למטרות אימות נתונים.</p>"
)
display(HTML(msg_html))

collapsed_data = []
for profile in all_profiles:
    g_strat, v_strat, b_strat = profile
    res = global_results[profile]
    is_best_row = (profile == best_profile)
    
    net_bonus_for_profile = res["net_profit"] - v_single_total
    p_payoffs = {}
    for p in players:
        p_payoffs[p] = int(round(v_single[p] + net_bonus_for_profile * bonus_weights[p]))
        
    collapsed_data.append({
        "תמחור הירוקה": strategy_names[g_strat],
        "תמחור ויה איטלי": strategy_names[v_strat],
        "תמחור בשרוני": strategy_names[b_strat],
        "רווח הירוקה (₪)": f"{p_payoffs['G']:,}",
        "רווח ויה איטלי (₪)": f"{p_payoffs['V']:,}",
        "רווח בשרוני (₪)": f"{p_payoffs['B']:,}",
        "רווח נטו גלובלי (₪)": f"{int(round(res['net_profit'])):,}",
        "זמן משלוח (דק')": f"{res['avg_delivery_time']:.1f}",
        "אחוז איחורים": f"{res['late_percentage']:.1f}%",
        "סטטוס": "🏆 אופטימום" if is_best_row else "רגיל"
    })

df_flat_matrix = pd.DataFrame(collapsed_data)

def style_nash_row(row):
    if row['סטטוס'] != "רגיל":
        return ['background-color: #d1ecf1; color: #0c5460; font-weight: bold; border: 2px solid #17a2b8;'] * len(row)
    return [''] * len(row)

try:
    styled_flat = df_flat_matrix.style.apply(style_nash_row, axis=1).set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '8px', 'border': '1px solid #ddd'
    }).set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center'), ('font-weight', 'bold')]}
    ]).hide(axis='index')
except:
    styled_flat = df_flat_matrix.style.apply(style_nash_row, axis=1).set_properties(**{
        'text-align': 'center', 'font-family': 'Segoe UI', 'padding': '8px', 'border': '1px solid #ddd'
    }).set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center'), ('font-weight', 'bold')]}
    ]).hide_index()

display(styled_flat)

מדד ביצוע תפעולי וכלכלי,ערך שהתקבל בפלט
💰 רווח צפוי למסעדת הירוקה (משולב ומסונכרן לחלק ב'),"3,517 ₪"
💰 רווח צפוי למסעדת ויה איטלי (משולב ומסונכרן לחלק ב'),"2,842 ₪"
💰 רווח צפוי למסעדת בשרוני (משולב ומסונכרן לחלק ב'),"3,135 ₪"
💎 סך הכל רווח גלובלי נטו למתחם כולו (רווח פחות קנסות בלבד),"9,492 ₪"
⏱️ זמן משלוח ממוצע כולל ($W$),21.44 דקות
🚨 אחוז ההזמנות המאחרות (מעל 40 דק'),2.22%


שעה,הירוקה,ויה איטלי,בשרוני
10:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
11:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
12:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
13:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
14:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"
15:00,"סלט הבית (3 יח'), מג'דרה (3 יח'), סלט קינואה (3 יח'), סלט אישי (3 יח')","פיצה (3 יח'), פסטה (2 יח'), רביולי (2 יח'), ניוקי (1 יח'), ריזוטו (1 יח')","המבורגר (2 יח'), סטייק (1 יח'), פרגית (1 יח'), שניצל (1 יח'), כבד עוף (1 יח')"


תמחור הירוקה,תמחור ויה איטלי,תמחור בשרוני,רווח הירוקה (₪),רווח ויה איטלי (₪),רווח בשרוני (₪),רווח נטו גלובלי (₪),זמן משלוח (דק'),אחוז איחורים,סטטוס
נמוך,נמוך,נמוך,"2,698","2,168","2,336","7,198",14.2,0.6%,רגיל
נמוך,נמוך,בינוני,"3,185","2,569","2,812","8,564",21.4,2.2%,רגיל
נמוך,נמוך,גבוה,"3,154","2,543","2,781","8,477",17.8,1.2%,רגיל
נמוך,בינוני,נמוך,"3,040","2,450","2,670","8,158",17.8,1.2%,רגיל
נמוך,בינוני,בינוני,"3,145","2,536","2,773","8,452",17.8,1.2%,רגיל
נמוך,בינוני,גבוה,"2,983","2,403","2,615","7,999",14.2,0.6%,רגיל
נמוך,גבוה,נמוך,"3,122","2,517","2,750","8,386",17.8,1.2%,רגיל
נמוך,גבוה,בינוני,"3,199","2,581","2,826","8,604",17.8,1.2%,רגיל
נמוך,גבוה,גבוה,"3,509","2,836","3,128","9,472",28.9,7.6%,רגיל
בינוני,נמוך,נמוך,"3,135","2,527","2,762","8,422",21.4,2.2%,רגיל
